# Delhi NCR Land Surface Temperature: Non-linear Drivers, TreeSHAP Attribution & Spatial Governance Zoning

**Abstract.** This notebook builds a reproducible, GPU-accelerated explainable-AI pipeline for the
pre-monsoon (April–June) daytime land surface temperature (LST) of the Delhi National Capital Region
(~55,000 km², 25 districts) at four epochs (2010, 2015, 2020, 2025). Satellite predictors
(MODIS spectral indices, SRTM terrain, VIIRS/DMSP night lights, WorldPop/GHS-POP population and
10–30 m land cover converted into 1 km composition and configuration metrics) are extracted from
Google Earth Engine on an explicit UTM 43N grid. One pooled gradient-boosted tree model (XGBoost on
CUDA) is benchmarked against linear, ridge, random-forest, LightGBM and graph-neural-network baselines
under **random, spatial-block and temporal** cross-validation, with Moran's I of the residuals as a
spatial-leakage diagnostic. Exact TreeSHAP attributions (plus interaction values and a spatial block
bootstrap) reveal non-linear thresholds (e.g. NDVI cooling saturation, water-body cooling reach) and
are clustered into four **SHAP governance zones** (Ecological Cool Base, Riparian Buffer, Transition,
Heat Extreme Core) with zone-specific, quantified mitigation recommendations. All results are exported
as a web bundle for the companion React/deck.gl dashboard (3-D digital twin, dependence viewer,
scenario simulator, zoning and model-performance views).

## Pipeline overview
| Stage | Section | What happens |
|---|---|---|
| 0 | Setup (this section) | configuration, dependency check, GPU/CUDA detection, shared utilities |
| 1 | Data acquisition | ROI & districts, 1 km UTM grid, Earth Engine extraction (`computePixels`, tiled, cached) **or** a clearly-labelled synthetic twin |
| 2 | Features | land-cover fractions, landscape metrics (PD, ED, CONTAG), long feature table, CV folds |
| 3 | Models & validation | 6 models × triple cross-validation (random / spatial / temporal), Moran's I of residuals |
| 4 | SHAP & zoning | final XGBoost, GPU TreeSHAP + interactions, block bootstrap CIs, thresholds, K-means governance zones |
| 5 | Export | Parquet, GeoJSON, NPZ, native model, compact browser model and the dashboard web bundle |

## How to run on Kaggle
1. **Settings → Accelerator → GPU T4 x2**, **Internet → On**.
2. **Add-ons → Secrets** (attach both to this notebook):
   * `GEE_SERVICE_ACCOUNT_KEY` — the *full JSON key* of a Google Cloud service account that is
     registered for Earth Engine (Earth Engine API enabled on the project, service account granted
     the *Earth Engine Resource Viewer/Writer* role or registered at code.earthengine.google.com/register);
   * `GEE_PROJECT` — the Cloud project id used for Earth Engine billing/quota.

   Without a key, an **interactive** session can authenticate with `ee.Authenticate(auth_mode="notebook")`
   (you will be prompted); a committed (batch) run cannot, so it needs the secret.
3. *Run All*. Missing packages (`earthengine-api`, `pylandstats`, `libpysal`, `esda`, …) are pip-installed
   automatically on Kaggle.

## Run modes and overrides (environment variables, all optional)
| Variable | Values / default | Effect |
|---|---|---|
| `LST_RUN_MODE` | `auto` (default), `gee`, `synthetic` | `auto` tries Earth Engine and falls back to a synthetic twin with a LOUD banner; `gee` fails hard if Earth Engine is unavailable; `synthetic` never touches Earth Engine |
| `LST_FAST_DEV` | `0` / `1` | shrinks every loop (rounds, bootstrap, permutations, GNN epochs) for smoke tests |
| `LST_GRID_RES_M` | `1000` | analysis cell size (m) |
| `LST_FINE_RES_M` | 25 (gee) / 100 (synthetic) | land-cover resolution (m); must divide the cell size |
| `LST_N_BOOTSTRAP` | `100` (FAST_DEV: 8) | block-bootstrap replicates for SHAP confidence intervals |
| `LST_GEE_PROJECT` | – | Earth Engine Cloud project (else Kaggle secret `GEE_PROJECT`) |
| `LST_GEE_SERVICE_ACCOUNT_KEY` / `GOOGLE_APPLICATION_CREDENTIALS` | JSON text / path | service-account key outside Kaggle |
| `LST_OUTPUT_DIR` | `/kaggle/working/outputs` or `./outputs` | output root |
| `LST_USE_RAPIDS` | `0` | use cuDF/cuML (GPU dataframe assembly, GPU random forest, GPU K-means) when importable; off by default because these paths are not yet validated (Kaggle's GPU image ships RAPIDS, so without this switch it is ignored) |
| `LST_INSTALL_RAPIDS` | `0` | install cuDF/cuML from pypi.nvidia.com before numpy is imported (several minutes; constrained to the installed numpy/pandas, `LST_RAPIDS_VERSION` pins the release); implies `LST_USE_RAPIDS` |
| `LST_BOOT_BLOCK_KM` | auto | block size (km) of the SHAP block bootstrap; default = residual-correlation range of the spatial-CV XGBoost (≥ `INNER_BLOCK_KM`) |
| `LST_LST_MIN_OBS` | `3` | minimum number of valid 8-day MODIS composites for a cell's seasonal LST (fewer → no target) |
| `LST_ALLOW_PIP` | `0` | allow pip installs outside Kaggle |
| `LST_GNN_CPU_BUDGET_S` | `900` | wall-clock budget (s) per GNN fold when no CUDA GPU is available (CPU fallback only) |

## Expected runtime (2× T4)
* Earth Engine extraction: ~15–45 min on the first run (depends on your EE quota); re-runs reuse the
  on-disk tile cache under `outputs/cache/` and take ~1 min.
* Modelling at 1 km (~264k cell-epochs), extrapolated from a measured local run (GTX 1650 + 4-core CPU,
  91 min with 10 bootstrap replicates and a capped CPU GNN): triple CV ~30–40 min (dominated by the CPU-only
  random forest and LightGBM on Kaggle's 4 vCPUs; XGBoost and the GNN run on the T4s), Moran's I ~3 min,
  TreeSHAP ~1 min, SHAP interaction values for every row ~10 min, block bootstrap ~1 min per replicate per T4
  (≈ 50 min for the default 100 replicates on 2× T4), exports ~2 min: **≈ 1.3–2 h** in total. Deep trees make
  TreeSHAP the dominant cost (its work grows with trees × leaves × depth²); `LST_N_BOOTSTRAP` trades CI
  precision for time.
* Synthetic mode end-to-end: the same as above at 1 km; FAST_DEV at `LST_GRID_RES_M=2000` takes ~5–15 min.

In `auto` mode a missing/invalid Earth Engine login falls back to the synthetic twin, but once a
**service-account key was supplied** and Earth Engine initialised, a failure during extraction stops the run
(fix the cause and re-run: the tile cache resumes where it stopped) instead of silently producing synthetic output.

## Outputs
`outputs/parquet/` (feature table, predictions + SHAP, CV metrics, Moran's I), `outputs/geo/` (districts,
per-epoch zone polygons), `outputs/npz/` (SHAP interaction tensors), `outputs/models/xgb_final.json`,
`outputs/figures/*.png`, `outputs/web/` (dashboard bundle) and `outputs/web_bundle.zip`.

## Loading the results into the dashboard
Download `web_bundle.zip` from the Kaggle *Output* tab, unzip it into `dashboard/public/data/`
(so that `dashboard/public/data/manifest.json` exists), then run `npm install && npm run dev` inside
`dashboard/`. Alternatively host the files anywhere and set `VITE_DATA_BASE_URL` to their URL.
Synthetic-mode bundles are flagged in the manifest and shown with a permanent "SYNTHETIC DEMO DATA" banner.

## 0.1 Environment detection and dependencies
Installs are done with `subprocess` (no notebook magics, so the same code runs as a plain script).
Packages are only installed when an import is missing **and** we are on Kaggle (or `LST_ALLOW_PIP=1`);
otherwise a warning is logged and the dependent step falls back gracefully later.

In [ ]:
import gc
import importlib
import importlib.util
import json
import math
import os
import platform
import queue
import random
import re
import shutil
import subprocess
import sys
import threading
import time
import warnings
from concurrent.futures import ThreadPoolExecutor, as_completed
from contextlib import contextmanager
from dataclasses import asdict, dataclass, field, fields, is_dataclass
from datetime import date, datetime, timezone
from pathlib import Path
from typing import Any, Callable, Iterable, Optional, Sequence

_IS_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE")) or Path("/kaggle/working").exists()
_LOG_LOCK = threading.Lock()

# Legacy consoles (e.g. Windows cp1252) cannot encode symbols such as "°" or "→": replace instead of crashing.
for _stream in (sys.stdout, sys.stderr):
    if hasattr(_stream, "reconfigure"):
        try:
            _stream.reconfigure(errors="replace")
        except (ValueError, OSError):
            pass


def log(msg: str, level: str = "INFO") -> None:
    """Print a timestamped, levelled log line (thread-safe, flushed immediately).

    Levels are free-form but conventionally DEBUG / INFO / WARNING / ERROR. Output goes to stdout so
    that Kaggle's cell output and the local runner interleave messages in order.
    """
    stamp = datetime.now().strftime("%H:%M:%S")
    with _LOG_LOCK:
        print(f"[{stamp}] {level.upper():<7} {msg}", flush=True)


def _is_interactive() -> bool:
    """True inside a live notebook kernel that can prompt the user (not a Kaggle batch commit)."""
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE", "").lower() == "batch":
        return False
    return "ipykernel" in sys.modules


# pip distribution name -> import name (for the "is it already importable?" check)
_PIP_IMPORT_NAMES = {
    "earthengine-api": "ee",
    "geemap": "geemap",
    "pylandstats": "pylandstats",
    "libpysal": "libpysal",
    "esda": "esda",
    "pyarrow": "pyarrow",
    "shapely": "shapely",
    "pyproj": "pyproj",
    "lightgbm": "lightgbm",
    "shap": "shap",
    "cudf-cu12": "cudf",
    "cuml-cu12": "cuml",
}


def _import_name(spec: str) -> str:
    """Map a pip requirement spec (e.g. ``"shap>=0.44"``) to the module name it provides."""
    base = re.split(r"[<>=!~\[;\s]", spec.strip(), maxsplit=1)[0]
    return _PIP_IMPORT_NAMES.get(base.lower(), base.replace("-", "_").lower())


def _is_importable(module: str) -> bool:
    """Check importability without importing (cheap, no side effects)."""
    try:
        return importlib.util.find_spec(module) is not None
    except (ImportError, ValueError):
        return False


def _run_pip(specs: Sequence[str], extra_args: Sequence[str], timeout: int) -> bool:
    """Run one ``pip install`` subprocess; return True on exit code 0."""
    cmd = [sys.executable, "-m", "pip", "install", "-q", *extra_args, *specs]
    log(f"pip install {' '.join(specs)} {' '.join(extra_args)}".rstrip())
    try:
        proc = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
    except subprocess.TimeoutExpired:
        log(f"pip install timed out after {timeout}s for {list(specs)}", "WARNING")
        return False
    except OSError as exc:
        log(f"pip could not be started: {exc}", "WARNING")
        return False
    if proc.returncode != 0:
        log(f"pip failed (exit {proc.returncode}) for {list(specs)}:\n{proc.stderr[-1500:]}", "WARNING")
        return False
    return True


def pip_install(pkgs, extra_args: Sequence[str] = (), *, timeout: int = 900) -> bool:
    """Install missing packages with ``sys.executable -m pip install -q``.

    Only packages whose import is missing are installed, and only on Kaggle or when ``LST_ALLOW_PIP=1``
    (we never modify a user's local/global Python silently). A failed group install is retried per
    package so one broken wheel does not block the others.

    Returns True when every requested package is importable afterwards.
    """
    specs = [pkgs] if isinstance(pkgs, str) else list(pkgs)
    missing = [s for s in specs if not _is_importable(_import_name(s))]
    if not missing:
        return True
    if not (_IS_KAGGLE or os.environ.get("LST_ALLOW_PIP") == "1"):
        log(f"Missing optional packages {missing}; not installing outside Kaggle "
            f"(set LST_ALLOW_PIP=1 to allow). Dependent steps will use fallbacks.", "WARNING")
        return False
    if not _run_pip(missing, extra_args, timeout) and len(missing) > 1:
        for spec in missing:
            _run_pip([spec], extra_args, timeout)
    importlib.invalidate_caches()
    still_missing = [s for s in missing if not _is_importable(_import_name(s))]
    if still_missing:
        log(f"Still missing after pip: {still_missing}", "WARNING")
    return not still_missing


_REQUIRED_OPTIONAL_PKGS = ["earthengine-api", "pylandstats", "libpysal", "esda", "pyarrow",
                           "shapely", "pyproj", "lightgbm", "shap"]
pip_install(_REQUIRED_OPTIONAL_PKGS)
if _IS_KAGGLE and _is_interactive():
    # geemap is only used for the optional interactive map preview in section 1.
    pip_install(["geemap"])
log(f"Environment: {'Kaggle' if _IS_KAGGLE else 'local'} | Python {platform.python_version()} | "
    f"{platform.platform()}")

### Optional RAPIDS install (`LST_INSTALL_RAPIDS=1`)
Runs **before** numpy/pandas are imported (section 0.2): a RAPIDS wheel may pull in different numpy/pandas
versions, and swapping them under an interpreter that has already imported the old ones leaves a mixed,
broken environment. The install is constrained to the numpy/pandas versions already installed (pip then picks a
compatible RAPIDS release or fails cleanly); `LST_RAPIDS_VERSION` (e.g. `25.2.*`) pins the release. Kaggle's GPU
image already ships cuDF/cuML, so there the install is a no-op. Using RAPIDS at all is a separate opt-in
(`LST_USE_RAPIDS=1`, section 0.4).

In [ ]:
def _dist_version(dist: str) -> Optional[str]:
    """Installed version of a distribution without importing it (None when absent)."""
    try:
        from importlib.metadata import PackageNotFoundError, version
    except ImportError:  # pragma: no cover - Python < 3.8
        return None
    try:
        return version(dist)
    except PackageNotFoundError:
        return None


def _install_rapids() -> bool:
    """pip-install cuDF/cuML constrained to the installed numpy/pandas; returns True when a restart is needed."""
    pinned = (os.environ.get("LST_RAPIDS_VERSION") or "").strip()
    specs = [f"{name}=={pinned}" if pinned else name for name in ("cudf-cu12", "cuml-cu12")]
    before = {d: _dist_version(d) for d in ("numpy", "pandas")}
    already_imported = [d for d in before if d in sys.modules]
    constraint_path = None
    extra = ["--extra-index-url=https://pypi.nvidia.com"]
    pins = [f"{d}=={v}" for d, v in before.items() if v]
    if pins:
        import tempfile
        with tempfile.NamedTemporaryFile("w", suffix="_lst_constraints.txt", delete=False) as fh:
            fh.write("\n".join(pins) + "\n")
            constraint_path = fh.name
        extra += ["--constraint", constraint_path]
    log(f"LST_INSTALL_RAPIDS=1: installing {specs} constrained to {pins or 'nothing'} (several minutes)", "WARNING")
    try:
        pip_install(specs, extra_args=extra, timeout=2400)
    finally:
        if constraint_path:
            try:
                os.remove(constraint_path)
            except OSError:
                pass
    after = {d: _dist_version(d) for d in before}
    changed = {d: (before[d], after[d]) for d in before if before[d] != after[d]}
    if changed:
        loaded = f" ({', '.join(already_imported)} already imported in this kernel)" if already_imported else ""
        log(f"RAPIDS install changed {changed}{loaded}. RESTART THE KERNEL and run again before continuing.",
            "ERROR")
        return True
    return False


if os.environ.get("LST_INSTALL_RAPIDS", "").strip().lower() in {"1", "true", "yes", "on", "y"}:
    if _install_rapids():
        raise RuntimeError("RAPIDS installation changed numpy/pandas; restart the kernel and run the notebook "
                           "again (the install is skipped next time because cuDF/cuML are then importable).")

## 0.2 Imports
Core scientific stack first; every optional/accelerator library is imported defensively so that a
missing package only disables the feature that needs it. Matplotlib is forced to the non-interactive
`Agg` backend when no notebook kernel is present (headless scripts, CI).

In [ ]:
import numpy as np
import pandas as pd
import scipy
from scipy import ndimage, sparse, stats
import matplotlib

if not os.environ.get("MPLBACKEND") and "ipykernel" not in sys.modules:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
import sklearn

# plt.show() under Agg warns "FigureCanvasAgg is non-interactive" - harmless in headless runs.
warnings.filterwarnings("ignore", message=".*non-interactive.*")
warnings.filterwarnings("ignore", message=".*FigureCanvasAgg.*")


def _optional_import(module: str):
    """Import ``module`` or return None (any exception, e.g. CUDA driver errors, counts as missing)."""
    try:
        return importlib.import_module(module)
    except Exception as exc:  # noqa: BLE001 - optional dependency, any failure means "unavailable"
        log(f"optional import '{module}' unavailable: {type(exc).__name__}: {str(exc)[:160]}", "DEBUG")
        return None


torch = _optional_import("torch")
xgb = _optional_import("xgboost")
lgb = _optional_import("lightgbm")
shap = _optional_import("shap")
ee = _optional_import("ee")
libpysal = _optional_import("libpysal")
esda = _optional_import("esda")
pylandstats = _optional_import("pylandstats")
shapely = _optional_import("shapely")
pyproj = _optional_import("pyproj")

_OPTIONAL_MODULES = {
    "torch": torch, "xgboost": xgb, "lightgbm": lgb, "shap": shap, "earthengine-api": ee,
    "libpysal": libpysal, "esda": esda, "pylandstats": pylandstats, "shapely": shapely, "pyproj": pyproj,
}
log("versions: " + ", ".join(
    [f"numpy {np.__version__}", f"pandas {pd.__version__}", f"scipy {scipy.__version__}",
     f"sklearn {sklearn.__version__}", f"matplotlib {matplotlib.__version__}"]
    + [f"{name} {getattr(mod, '__version__', '?')}" if mod is not None else f"{name} MISSING"
       for name, mod in _OPTIONAL_MODULES.items()]))

## 0.3 Configuration (`CFG`)
A single dataclass holds every tunable. Environment variables override defaults (see the table in
the introduction). `FAST_DEV` shrinks all expensive loops. The fine land-cover resolution depends on
the data mode, which section 1 decides: `CFG.resolve_mode(mode)` fixes it and `CFG.fine_res()` returns it.

In [ ]:
def _env_str(name: str, default: Optional[str] = None) -> Optional[str]:
    value = os.environ.get(name)
    return value.strip() if value is not None and value.strip() else default


def _env_int(name: str, default: Optional[int] = None) -> Optional[int]:
    value = _env_str(name)
    if value is None:
        return default
    try:
        return int(float(value))
    except ValueError as exc:
        raise ValueError(f"Environment variable {name}={value!r} is not an integer") from exc


def _env_bool(name: str, default: bool = False) -> bool:
    value = _env_str(name)
    if value is None:
        return default
    return value.lower() in {"1", "true", "yes", "on", "y"}


def _kaggle_secret(name: str) -> Optional[str]:
    """Read a Kaggle notebook secret; None when not on Kaggle or the secret is not attached."""
    if not _IS_KAGGLE:
        return None
    try:
        from kaggle_secrets import UserSecretsClient  # available only inside Kaggle kernels
        value = UserSecretsClient().get_secret(name)
        return value.strip() if value else None
    except Exception:  # noqa: BLE001 - absent secret raises a generic exception
        return None


def _nearest_divisor(n: int, target: int) -> int:
    """Divisor of ``n`` closest to ``target`` (ties resolved towards the larger, cheaper divisor)."""
    divisors = [d for d in range(1, n + 1) if n % d == 0]
    return min(divisors, key=lambda d: (abs(d - target), -d))


_XGB_DEFAULT_PARAMS = {
    "objective": "reg:squarederror", "tree_method": "hist", "learning_rate": 0.03, "max_depth": 8,
    "min_child_weight": 5, "subsample": 0.8, "colsample_bytree": 0.8, "reg_lambda": 1.0,
    "reg_alpha": 0.0, "max_bin": 256, "eval_metric": "rmse",
}
_VALID_RUN_MODES = ("auto", "gee", "synthetic")


@dataclass
class LSTConfig:
    """Pipeline configuration (SPEC section 3). Construct with :meth:`from_env`."""

    RUN_MODE: str = "auto"
    FAST_DEV: bool = False
    EPOCHS: list = field(default_factory=lambda: [2010, 2015, 2020, 2025])
    SEASON: tuple = ("04-01", "06-30")
    CRS: str = "EPSG:32643"
    GRID_RES_M: int = 1000
    FINE_RES_M: Optional[int] = None          # resolved by resolve_mode(); read via fine_res()
    LM_WINDOW_CELLS: int = 1
    TARGET_MODE: str = "anomaly"
    SEED: int = 42
    N_FOLDS: int = 5
    SPATIAL_BLOCKS: tuple = (5, 5)
    INNER_BLOCK_KM: int = 10
    BOOT_BLOCK_KM: Optional[int] = None       # None = derived from the residual correlogram in section 4
    N_BOOTSTRAP: int = 100
    LST_MIN_OBS: int = 3                      # min valid 8-day LST composites per cell and season
    LST_MIN_COVERAGE: float = 0.5             # min valid-area fraction of a cell for its seasonal LST (GEE)
    K_ZONES: int = 4
    DEP_BINS: int = 40
    MORAN_K: int = 8
    MORAN_PERMS: int = 999
    XGB_PARAMS: dict = field(default_factory=lambda: dict(_XGB_DEFAULT_PARAMS))
    XGB_MAX_ROUNDS: int = 3000
    XGB_EARLY_STOP: int = 100
    GNN_EPOCHS: int = 300
    SHAP_BATCH: int = 16384
    SHAP_INTERACTION_MAX_ROWS: Optional[int] = 4000   # None on GPU (set by apply_hardware)
    GEE_PROJECT: Optional[str] = None
    GEE_TILE_PX: int = 128
    GEE_FINE_TILE_PX: int = 2048
    GEE_WORKERS: int = 6
    GEE_MAX_RETRIES: int = 6
    LC_SOURCES: dict = field(default_factory=lambda: {
        2010: "glc_fcs30d", 2015: "glc_fcs30d", 2020: "esa_worldcover", 2025: "dynamic_world"})
    INSTALL_RAPIDS: bool = False
    USE_RAPIDS: bool = False                  # cuDF/cuML paths are opt-in (LST_USE_RAPIDS / LST_INSTALL_RAPIDS)
    OUTPUT_DIR: Path = Path("outputs")
    CACHE_DIR: Path = field(init=False)
    FIG_DIR: Path = field(init=False)
    PARQUET_DIR: Path = field(init=False)
    GEO_DIR: Path = field(init=False)
    MODEL_DIR: Path = field(init=False)
    NPZ_DIR: Path = field(init=False)
    WEB_DIR: Path = field(init=False)
    DATA_MODE: Optional[str] = None           # "gee" | "synthetic" once resolved by section 1

    def __post_init__(self) -> None:
        if self.RUN_MODE not in _VALID_RUN_MODES:
            raise ValueError(f"RUN_MODE must be one of {_VALID_RUN_MODES}, got {self.RUN_MODE!r}")
        if self.TARGET_MODE not in ("anomaly", "absolute"):
            raise ValueError(f"TARGET_MODE must be 'anomaly' or 'absolute', got {self.TARGET_MODE!r}")
        if self.GRID_RES_M <= 0:
            raise ValueError(f"GRID_RES_M must be positive, got {self.GRID_RES_M}")
        if self.BOOT_BLOCK_KM is not None and self.BOOT_BLOCK_KM <= 0:
            raise ValueError(f"BOOT_BLOCK_KM must be positive or None, got {self.BOOT_BLOCK_KM}")
        if self.LST_MIN_OBS < 1:
            raise ValueError(f"LST_MIN_OBS must be >= 1, got {self.LST_MIN_OBS}")
        self.OUTPUT_DIR = Path(self.OUTPUT_DIR).expanduser().resolve()
        self.CACHE_DIR = self.OUTPUT_DIR / "cache"
        self.FIG_DIR = self.OUTPUT_DIR / "figures"
        self.PARQUET_DIR = self.OUTPUT_DIR / "parquet"
        self.GEO_DIR = self.OUTPUT_DIR / "geo"
        self.MODEL_DIR = self.OUTPUT_DIR / "models"
        self.NPZ_DIR = self.OUTPUT_DIR / "npz"
        self.WEB_DIR = self.OUTPUT_DIR / "web"

    @classmethod
    def from_env(cls) -> "LSTConfig":
        """Build the configuration from defaults + ``LST_*`` environment variables + Kaggle secrets."""
        fast = _env_bool("LST_FAST_DEV", False)
        default_out = Path("/kaggle/working/outputs") if _IS_KAGGLE else Path.cwd() / "outputs"
        cfg = cls(
            RUN_MODE=(_env_str("LST_RUN_MODE", "auto") or "auto").lower(),
            FAST_DEV=fast,
            GRID_RES_M=_env_int("LST_GRID_RES_M", 1000),
            TARGET_MODE=(_env_str("LST_TARGET_MODE", "anomaly") or "anomaly").lower(),
            N_BOOTSTRAP=_env_int("LST_N_BOOTSTRAP", 8 if fast else 100),
            GEE_PROJECT=_env_str("LST_GEE_PROJECT") or _kaggle_secret("GEE_PROJECT"),
            GEE_WORKERS=_env_int("LST_GEE_WORKERS", 6),
            INSTALL_RAPIDS=_env_bool("LST_INSTALL_RAPIDS", False),
            USE_RAPIDS=_env_bool("LST_USE_RAPIDS", False) or _env_bool("LST_INSTALL_RAPIDS", False),
            BOOT_BLOCK_KM=_env_int("LST_BOOT_BLOCK_KM", None),
            LST_MIN_OBS=_env_int("LST_LST_MIN_OBS", 3),
            OUTPUT_DIR=Path(_env_str("LST_OUTPUT_DIR") or default_out),
        )
        if fast:
            cfg.MORAN_PERMS = 99
            cfg.XGB_MAX_ROUNDS = 300
            cfg.XGB_EARLY_STOP = 30
            cfg.GNN_EPOCHS = 40
            cfg.XGB_PARAMS.update({"learning_rate": 0.08, "max_depth": 6})
        cfg.make_dirs()
        return cfg

    def make_dirs(self) -> None:
        """Create the output directory tree (idempotent)."""
        for path in (self.OUTPUT_DIR, self.CACHE_DIR, self.FIG_DIR, self.PARQUET_DIR, self.GEO_DIR,
                     self.MODEL_DIR, self.NPZ_DIR, self.WEB_DIR):
            path.mkdir(parents=True, exist_ok=True)

    def _fine_res_for(self, mode: str) -> int:
        """Requested fine resolution for ``mode`` snapped to a divisor of GRID_RES_M."""
        requested = _env_int("LST_FINE_RES_M", 25 if mode == "gee" else 100)
        if requested <= 0:
            raise ValueError(f"LST_FINE_RES_M must be positive, got {requested}")
        if self.GRID_RES_M % requested == 0:
            return requested
        snapped = _nearest_divisor(self.GRID_RES_M, requested)
        log(f"FINE_RES_M={requested} does not divide GRID_RES_M={self.GRID_RES_M}; using {snapped} m", "WARNING")
        return snapped

    def resolve_mode(self, mode: str) -> int:
        """Record the data mode decided by section 1 and fix FINE_RES_M accordingly; returns it."""
        if mode not in ("gee", "synthetic"):
            raise ValueError(f"data mode must be 'gee' or 'synthetic', got {mode!r}")
        self.DATA_MODE = mode
        self.FINE_RES_M = self._fine_res_for(mode)
        return self.FINE_RES_M

    def fine_res(self) -> int:
        """Fine land-cover resolution in metres (resolved value, else the RUN_MODE default)."""
        if self.FINE_RES_M is not None:
            return int(self.FINE_RES_M)
        return self._fine_res_for("synthetic" if self.RUN_MODE == "synthetic" else "gee")

    def apply_hardware(self, env: dict) -> None:
        """Adjust hardware-dependent settings once ``ENV`` is known."""
        self.SHAP_INTERACTION_MAX_ROWS = None if env.get("xgb_device") == "cuda" else 4000

    def as_dict(self) -> dict:
        """JSON-friendly snapshot (Paths as strings) for logging and the export manifest."""
        out = {}
        for f in fields(self):
            value = getattr(self, f.name)
            out[f.name] = str(value) if isinstance(value, Path) else value
        return out


CFG = LSTConfig.from_env()
log(f"CFG: RUN_MODE={CFG.RUN_MODE} FAST_DEV={CFG.FAST_DEV} GRID_RES_M={CFG.GRID_RES_M} "
    f"OUTPUT_DIR={CFG.OUTPUT_DIR}")

## 0.4 Hardware detection (`ENV`)
The GPU count comes from the **CUDA runtime** (`torch.cuda.device_count()`, else CuPy), which honours
`CUDA_VISIBLE_DEVICES`; `nvidia-smi` (NVML ignores that variable) is only the fallback when neither can be queried,
and its rows are filtered by `CUDA_VISIBLE_DEVICES`. XGBoost trains on CUDA only if GPUs exist **and** the
installed wheel was built with CUDA (`xgboost.build_info()["USE_CUDA"]`, with a tiny training probe as
fallback). RAPIDS (cuDF/cuML) is used only with `LST_USE_RAPIDS=1` (or `LST_INSTALL_RAPIDS=1`): Kaggle's GPU image
ships it preinstalled, but its paths (GPU dataframe assembly, cuML random forest and K-means) are not yet validated,
so by default the pipeline runs the pandas / scikit-learn paths everywhere. `ENV["rapids_used"]` records which
RAPIDS backends actually ran (the manifest's `hardware.rapids` is derived from it).

In [ ]:
cp = _optional_import("cupy")
cudf = _optional_import("cudf") if CFG.USE_RAPIDS else None
cuml = _optional_import("cuml") if CFG.USE_RAPIDS else None
if not CFG.USE_RAPIDS and (_is_importable("cudf") or _is_importable("cuml")):
    log("cuDF/cuML are installed but not used (set LST_USE_RAPIDS=1 to enable the RAPIDS paths)")


def _query_nvidia_smi() -> list:
    """Return [(index, name, total_MiB), ...] from nvidia-smi, or [] if unavailable."""
    exe = shutil.which("nvidia-smi")
    if exe is None:
        return []
    try:
        proc = subprocess.run([exe, "--query-gpu=index,name,memory.total", "--format=csv,noheader,nounits"],
                              capture_output=True, text=True, timeout=20)
    except (subprocess.TimeoutExpired, OSError) as exc:
        log(f"nvidia-smi failed: {exc}", "WARNING")
        return []
    gpus = []
    for line in proc.stdout.strip().splitlines():
        parts = [p.strip() for p in line.split(",")]
        if len(parts) >= 3 and parts[1]:
            try:
                index = int(parts[0])
            except ValueError:
                index = len(gpus)
            try:
                gpus.append((index, parts[1], int(float(parts[2]))))
            except ValueError:
                gpus.append((index, parts[1], None))
    return gpus


def _visible_device_filter(smi: list) -> list:
    """Restrict nvidia-smi rows to CUDA_VISIBLE_DEVICES (integer ordinals only; UUID/MIG lists are left as-is)."""
    raw = os.environ.get("CUDA_VISIBLE_DEVICES")
    if raw is None:
        return smi
    tokens = [t.strip() for t in raw.split(",") if t.strip()]
    if not tokens or raw.strip() in ("-1", "none", "NoDevFiles"):
        return []
    try:
        wanted = [int(t) for t in tokens]
    except ValueError:
        return smi
    by_index = {row[0]: row for row in smi}
    out = []
    for w in wanted:          # CUDA stops at the first invalid ordinal
        if w not in by_index:
            break
        out.append(by_index[w])
    return out


def _cupy_device_count() -> int:
    if cp is None:
        return 0
    try:
        return int(cp.cuda.runtime.getDeviceCount())
    except Exception:  # noqa: BLE001 - no driver / no device
        return 0


def _torch_device_count() -> int:
    if torch is None:
        return 0
    try:
        return int(torch.cuda.device_count()) if torch.cuda.is_available() else 0
    except Exception:  # noqa: BLE001 - broken CUDA runtime counts as "no GPU"
        return 0


def _xgb_has_cuda_build() -> bool:
    """Was the installed XGBoost built with CUDA? (build_info, else a 1-round GPU training probe)."""
    if xgb is None:
        return False
    try:
        return bool(xgb.build_info().get("USE_CUDA", False))
    except Exception:  # noqa: BLE001 - very old versions have no build_info
        pass
    try:
        probe = xgb.DMatrix(np.random.default_rng(0).random((64, 3)), label=np.arange(64, dtype=float))
        xgb.train({"device": "cuda", "tree_method": "hist", "verbosity": 0}, probe, num_boost_round=1)
        return True
    except Exception:  # noqa: BLE001
        return False


def _detect_env() -> dict:
    """Collect hardware/software facts into the SPEC ``ENV`` dict."""
    smi_all = _query_nvidia_smi()
    smi = _visible_device_filter(smi_all)
    torch_count = _torch_device_count()
    cuda_count = torch_count or _cupy_device_count()
    n_gpus = cuda_count if cuda_count > 0 else len(smi)
    if cuda_count > 0 and len(smi_all) and len(smi_all) != cuda_count:
        log(f"nvidia-smi lists {len(smi_all)} GPU(s) but the CUDA runtime exposes {cuda_count} "
            f"(CUDA_VISIBLE_DEVICES={os.environ.get('CUDA_VISIBLE_DEVICES')!r}); using {cuda_count}", "WARNING")
    gpu_names = [name for _, name, _ in smi][:n_gpus]
    gpu_mem = [mem for _, _, mem in smi][:n_gpus]
    if len(gpu_names) < n_gpus and torch_count:
        gpu_names = [torch.cuda.get_device_name(i) for i in range(torch_count)]
    has_cuda = n_gpus > 0
    xgb_cuda = has_cuda and _xgb_has_cuda_build()
    return {
        "is_kaggle": _IS_KAGGLE,
        "n_gpus": int(n_gpus),
        "gpu_names": gpu_names,
        "gpu_mem_mib": gpu_mem,
        "has_cuda": bool(has_cuda),
        "torch_device_count": int(torch_count),
        "has_cudf": cudf is not None,
        "has_cuml": cuml is not None,
        "has_cupy": cp is not None and has_cuda,
        "rapids_used": set(),   # filled by the stages that actually ran a cuDF/cuML backend
        "xgb_version": getattr(xgb, "__version__", None),
        "xgb_device": "cuda" if xgb_cuda else "cpu",
        "lgbm_device": "cpu",   # resolved by section 3 after a GPU probe
        "torch_version": getattr(torch, "__version__", None),
        "python": platform.python_version(),
        "platform": platform.platform(),
    }


ENV = _detect_env()
CFG.apply_hardware(ENV)
log(f"ENV: {ENV['n_gpus']} GPU(s) {ENV['gpu_names']} | xgb {ENV['xgb_version']} on {ENV['xgb_device']} | "
    f"cupy={ENV['has_cupy']} cudf={ENV['has_cudf']} cuml={ENV['has_cuml']}")
if ENV["has_cuda"] and ENV["xgb_device"] == "cpu":
    log("GPUs found but this XGBoost build has no CUDA support - XGBoost will run on CPU.", "WARNING")

## 0.5 Shared utilities
Timing, memory hygiene (host + CUDA + CuPy pools), retry with exponential backoff and jitter,
deterministic seeding, strict JSON writing (NaN → `null`, numpy/pandas aware), GPU memory reporting and a
multi-GPU work scheduler that pins one task per GPU at a time.

In [ ]:
TIMINGS: dict = {}
_RETRY_RNG = random.Random()   # private RNG so backoff jitter never perturbs seeded global streams


@contextmanager
def timer(name: str):
    """Context manager that logs and accumulates wall-clock seconds for ``name`` into ``TIMINGS``."""
    start = time.perf_counter()
    log(f">> {name}")
    try:
        yield
    finally:
        elapsed = time.perf_counter() - start
        TIMINGS[name] = round(TIMINGS.get(name, 0.0) + elapsed, 3)
        log(f"<< {name}: {elapsed:.1f}s")


def free_memory() -> None:
    """Release host and GPU memory: Python GC, PyTorch CUDA cache/IPC handles, CuPy memory pools."""
    gc.collect()
    if torch is not None:
        try:
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
                torch.cuda.ipc_collect()
        except Exception as exc:  # noqa: BLE001 - never fail a pipeline on cleanup
            log(f"torch cache cleanup failed: {exc}", "DEBUG")
    if cp is not None and ENV.get("has_cupy"):
        try:
            # get_default_memory_pool() frees only the CURRENT device's pool; blocks cached on the other GPUs
            # (e.g. CV folds run on cuda:1 by a worker thread) need their own device context.
            for device_id in range(int(cp.cuda.runtime.getDeviceCount())):
                with cp.cuda.Device(device_id):
                    cp.get_default_memory_pool().free_all_blocks()
            cp.get_default_pinned_memory_pool().free_all_blocks()
        except Exception as exc:  # noqa: BLE001
            log(f"cupy pool cleanup failed: {exc}", "DEBUG")


def retry(fn: Callable[[], Any], *, tries: int = 3, base_delay: float = 1.0, max_delay: float = 60.0,
          retry_on: tuple = (Exception,), on_retry: Optional[Callable[[int, BaseException, float], None]] = None):
    """Call ``fn()`` up to ``tries`` times with exponential backoff and jitter.

    Delay before attempt k+1 = min(max_delay, base_delay·2^(k-1)) scaled by a uniform jitter in [0.5, 1.0]
    (decorrelates parallel workers hitting the same quota). Only exceptions in ``retry_on`` are retried;
    ``on_retry(attempt, exc, delay)`` is called before sleeping. The last exception is re-raised.
    """
    if tries < 1:
        raise ValueError("tries must be >= 1")
    for attempt in range(1, tries + 1):
        try:
            return fn()
        except retry_on as exc:
            if attempt == tries:
                raise
            delay = min(max_delay, base_delay * 2 ** (attempt - 1)) * _RETRY_RNG.uniform(0.5, 1.0)
            if on_retry is not None:
                on_retry(attempt, exc, delay)
            time.sleep(delay)
    raise RuntimeError("unreachable")


def set_seeds(seed: int) -> int:
    """Seed Python, NumPy, PyTorch (CPU + all CUDA devices) and CuPy; export PYTHONHASHSEED.

    PYTHONHASHSEED only affects subprocesses (the running interpreter's hash seed is fixed at start-up).
    """
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    if torch is not None:
        torch.manual_seed(seed)
        try:
            if torch.cuda.is_available():
                torch.cuda.manual_seed_all(seed)
        except Exception:  # noqa: BLE001
            pass
    if cp is not None and ENV.get("has_cupy"):
        try:
            cp.random.seed(seed)
        except Exception:  # noqa: BLE001
            pass
    return seed


def _json_float(value: float, ndigits: Optional[int]):
    if not math.isfinite(value):
        return None
    return round(value, ndigits) if ndigits is not None else value


def _to_jsonable(obj: Any, ndigits: Optional[int]):
    """Recursively convert numpy/pandas/pathlib/dataclass objects into strict-JSON-compatible values."""
    if obj is None or isinstance(obj, (bool, str)):
        return obj
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    if isinstance(obj, (int, np.integer)):
        return int(obj)
    if isinstance(obj, (float, np.floating)):
        return _json_float(float(obj), ndigits)
    if isinstance(obj, dict):
        return {str(_to_jsonable(k, None)) if not isinstance(k, str) else k: _to_jsonable(v, ndigits)
                for k, v in obj.items()}
    if isinstance(obj, (list, tuple, set, frozenset)):
        return [_to_jsonable(v, ndigits) for v in obj]
    if isinstance(obj, np.ndarray):
        if obj.dtype.kind in "fc":
            arr = np.asarray(obj.real if obj.dtype.kind == "c" else obj, dtype=np.float64)
            if ndigits is not None:
                arr = np.round(arr, ndigits)
            return _to_jsonable(arr.tolist(), None)
        return _to_jsonable(obj.tolist(), ndigits)
    if isinstance(obj, pd.DataFrame):
        return _to_jsonable(obj.to_dict(orient="records"), ndigits)
    if isinstance(obj, (pd.Series, pd.Index)):
        return _to_jsonable(obj.tolist(), ndigits)
    if obj is pd.NaT:
        return None
    if isinstance(obj, (pd.Timestamp, datetime, date)):
        return obj.isoformat()
    if isinstance(obj, Path):
        return str(obj)
    if is_dataclass(obj) and not isinstance(obj, type):
        return _to_jsonable(asdict(obj), ndigits)
    if hasattr(obj, "item") and callable(obj.item):     # 0-d arrays, cupy/torch scalars
        return _to_jsonable(obj.item(), ndigits)
    raise TypeError(f"save_json: cannot serialise object of type {type(obj).__name__}")


def save_json(obj: Any, path, *, round_floats: Optional[int] = None) -> Path:
    """Write ``obj`` as strict UTF-8 JSON (NaN/Inf → null, numpy/pandas aware, compact separators).

    ``round_floats`` rounds every float to that many decimals. The file is written atomically
    (temporary file + rename) so an interrupted run never leaves truncated JSON behind.
    """
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    text = json.dumps(_to_jsonable(obj, round_floats), ensure_ascii=False, separators=(",", ":"),
                      allow_nan=False)
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(text, encoding="utf-8")
    os.replace(tmp, path)
    return path


def gpu_mem_report(log_it: bool = True) -> list:
    """Per-GPU memory usage [{gpu, name, used_mib, total_mib, torch_allocated_mib}] (empty without GPUs)."""
    report = []
    if torch is not None and ENV.get("torch_device_count", 0) > 0:
        for i in range(ENV["torch_device_count"]):
            try:
                free_b, total_b = torch.cuda.mem_get_info(i)
                report.append({"gpu": i, "name": torch.cuda.get_device_name(i),
                               "used_mib": round((total_b - free_b) / 2**20),
                               "total_mib": round(total_b / 2**20),
                               "torch_allocated_mib": round(torch.cuda.memory_allocated(i) / 2**20)})
            except Exception as exc:  # noqa: BLE001
                log(f"mem_get_info failed on GPU {i}: {exc}", "DEBUG")
    elif ENV.get("n_gpus", 0) > 0 and shutil.which("nvidia-smi"):
        try:
            proc = subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.used,memory.total",
                                   "--format=csv,noheader,nounits"], capture_output=True, text=True, timeout=20)
            for line in proc.stdout.strip().splitlines():
                idx, name, used, total = [p.strip() for p in line.split(",")]
                report.append({"gpu": int(idx), "name": name, "used_mib": int(float(used)),
                               "total_mib": int(float(total)), "torch_allocated_mib": None})
        except (subprocess.TimeoutExpired, OSError, ValueError) as exc:
            log(f"nvidia-smi memory query failed: {exc}", "DEBUG")
    if log_it:
        if report:
            log("GPU memory: " + " | ".join(f"GPU{r['gpu']} {r['used_mib']}/{r['total_mib']} MiB" for r in report))
        else:
            log("GPU memory: no CUDA devices")
    return report


def xgb_device_for(gpu_id: int) -> str:
    """XGBoost ``device`` string for a worker: ``"cuda:{gpu_id}"`` when XGBoost can use CUDA, else ``"cpu"``."""
    if ENV.get("xgb_device") != "cuda":
        return "cpu"
    gpu_id = int(gpu_id)
    if not 0 <= gpu_id < max(1, ENV.get("n_gpus", 0)):
        raise ValueError(f"gpu_id {gpu_id} out of range for {ENV.get('n_gpus')} GPU(s)")
    return f"cuda:{gpu_id}"


def _annotate_task_error(exc: BaseException, index: int, item: Any, gpu_id: int) -> None:
    context = f"gpu_pool_map task #{index} (item={str(item)[:120]}) failed on gpu_id={gpu_id}"
    log(f"{context}: {type(exc).__name__}: {str(exc)[:300]}", "ERROR")
    if hasattr(exc, "add_note"):   # Python >= 3.11: context shows up in the traceback
        exc.add_note(context)


def gpu_pool_map(fn: Callable[[Any, int], Any], items: Iterable) -> list:
    """Run ``fn(item, gpu_id)`` for every item, one concurrent task per GPU; results keep input order.

    With ≥2 CUDA GPUs usable by XGBoost, a thread pool with one worker per GPU pulls free GPU ids from a
    queue, so each GPU runs exactly one task at a time (XGBoost/PyTorch release the GIL inside CUDA work,
    so threads give real parallelism). Otherwise tasks run sequentially with ``gpu_id=0``
    (``xgb_device_for(0)`` then yields "cpu" on CPU-only machines). The first failure cancels pending tasks
    and is re-raised with its original type, annotated with the failing item and GPU.
    """
    items = list(items)
    n_workers = ENV.get("n_gpus", 0) if ENV.get("xgb_device") == "cuda" else 0
    if n_workers <= 1 or len(items) <= 1:
        results = []
        for i, item in enumerate(items):
            try:
                results.append(fn(item, 0))
            except BaseException as exc:
                _annotate_task_error(exc, i, item, 0)
                raise
        return results

    free_gpus: "queue.Queue[int]" = queue.Queue()
    for gpu_id in range(n_workers):
        free_gpus.put(gpu_id)

    def _task(index: int, item: Any):
        gpu_id = free_gpus.get()
        try:
            return fn(item, gpu_id)
        except BaseException as exc:
            _annotate_task_error(exc, index, item, gpu_id)
            raise
        finally:
            free_gpus.put(gpu_id)

    results: list = [None] * len(items)
    with ThreadPoolExecutor(max_workers=n_workers, thread_name_prefix="gpu") as pool:
        futures = {pool.submit(_task, i, item): i for i, item in enumerate(items)}
        try:
            for future in as_completed(futures):
                results[futures[future]] = future.result()
        except BaseException:
            for future in futures:
                future.cancel()
            raise
    return results


_OOM_MARKERS = ("out of memory", "cudaerrormemoryallocation", "bad_alloc", "cuda_error_out_of_memory",
                "memoryerror", "failed to allocate", "cusparse_status_alloc_failed",
                "cusparse_status_insufficient_resources", "cublas_status_alloc_failed",
                "cudnn_status_alloc_failed", "cufft_alloc_failed")


def _oom_in_chain(exc: BaseException, *, count_host_memory_error: bool) -> bool:
    torch_oom = getattr(getattr(torch, "cuda", None), "OutOfMemoryError", None) if torch is not None else None
    cupy_oom = None
    if cp is not None:
        cupy_oom = getattr(getattr(getattr(cp, "cuda", None), "memory", None), "OutOfMemoryError", None)
    seen, current = set(), exc
    while current is not None and id(current) not in seen:
        seen.add(id(current))
        if torch_oom is not None and isinstance(current, torch_oom):
            return True
        if cupy_oom is not None and isinstance(current, cupy_oom):
            return True
        if type(current) is MemoryError:
            # a bare MemoryError is the host running out of RAM
            if count_host_memory_error:
                return True
        else:
            message = f"{type(current).__name__}: {current}".lower()
            if any(marker in message for marker in _OOM_MARKERS):
                return True
        current = current.__cause__ or current.__context__
    return False


def is_oom_error(exc: BaseException) -> bool:
    """True if ``exc`` (or its cause/context chain) is a host OR GPU out-of-memory condition.

    Recognises MemoryError, torch.cuda.OutOfMemoryError, cupy OutOfMemoryError and XGBoost/CUDA/RMM/cuSPARSE/
    cuBLAS messages ("out of memory", "cudaErrorMemoryAllocation", "bad_alloc", "CUSPARSE_STATUS_ALLOC_FAILED", ...).
    """
    return _oom_in_chain(exc, count_host_memory_error=True)


def is_gpu_oom_error(exc: BaseException) -> bool:
    """True only for a GPU out-of-memory condition (a bare host ``MemoryError`` is excluded).

    Used to decide GPU -> CPU retries: moving work to the CPU after the *host* ran out of RAM needs more host
    memory, not less, so that case must propagate instead.
    """
    return _oom_in_chain(exc, count_host_memory_error=False)


set_seeds(CFG.SEED)
log("utilities ready")

## 0.6 Configuration and hardware summary

In [ ]:
def _show(obj) -> None:
    """Rich display inside notebooks, plain print in scripts."""
    try:
        from IPython.display import display
        if "ipykernel" in sys.modules:
            display(obj)
            return
    except ImportError:
        pass
    print(obj.to_string() if isinstance(obj, pd.DataFrame) else obj)


def _print_table(title: str, rows: Sequence[tuple], width: int = 110) -> None:
    """Two-column key/value table that reads well both in notebooks and in plain logs."""
    key_w = max(len(str(k)) for k, _ in rows) + 2
    print()
    print(title)
    print("-" * min(width, key_w + 60))
    for key, value in rows:
        text = str(value)
        print(f"{str(key):<{key_w}}{text if len(text) <= width - key_w else text[:width - key_w - 3] + '...'}")


_cfg_rows = [(k, json.dumps(v, default=str) if isinstance(v, (dict, list, tuple)) else v)
             for k, v in CFG.as_dict().items()]
_hw_rows = [
    ("Platform", ENV["platform"]), ("Python", ENV["python"]), ("Kaggle", ENV["is_kaggle"]),
    ("GPUs", f"{ENV['n_gpus']} x {', '.join(ENV['gpu_names']) or '-'}"),
    ("GPU memory (MiB)", ", ".join(str(m) for m in ENV["gpu_mem_mib"]) or "-"),
    ("torch CUDA devices", ENV["torch_device_count"]), ("torch", ENV["torch_version"] or "missing"),
    ("XGBoost", f"{ENV['xgb_version'] or 'missing'} (device={ENV['xgb_device']})"),
    ("CuPy / cuDF / cuML", f"{ENV['has_cupy']} / {ENV['has_cudf']} / {ENV['has_cuml']}"),
] + [(f"lib: {name}", "ok" if mod is not None else "MISSING") for name, mod in _OPTIONAL_MODULES.items()]
_print_table("Configuration", _cfg_rows)
_print_table("Hardware / software", _hw_rows)
gpu_mem_report()
del _cfg_rows, _hw_rows

# 1. Data acquisition

**Study area.** Delhi National Capital Region (NCR): Delhi NCT plus 24 districts of Haryana, Uttar
Pradesh and Rajasthan (~55,000 km²), delineated from geoBoundaries ADM2 (fallback FAO GAUL 2015 level 2,
final fallback an approximate NCR outline).

**Grid.** All variables are brought onto one explicit grid in UTM zone 43N (`EPSG:32643`) with square
cells of `CFG.GRID_RES_M` (1 km default) whose edges are multiples of the cell size. Land cover is
extracted on a nested fine grid (`CFG.fine_res()`, 25 m in Earth Engine mode) sharing the same origin, so
every coarse cell contains exactly `f × f` fine pixels (`f = GRID_RES_M / FINE_RES_M`).

| Variable(s) | Source (Earth Engine asset) | Processing |
|---|---|---|
| `lst_day_c`, `lst_obs_count` | `MODIS/061/MOD11A2` `LST_Day_1km`, `QC_Day` | QC bits 0–1 ∈ {0,1} and LST-error bits 6–7 ≤ 1 (≤ 2 K); ×0.02 − 273.15; seasonal **median**; number of valid 8-day composites; a cell needs ≥ `CFG.LST_MIN_OBS` (3) valid composites and ≥ `CFG.LST_MIN_COVERAGE` (50 %) valid area, else no target |
| `ndvi`, `ndwi`, `ndbi` | `MODIS/061/MOD09A1` | StateQA cloud bits 0–1 = 0, shadow bit 2 = 0, internal cloud bit 10 = 0; ×0.0001; NDVI (b2,b1), NDWI McFeeters (b4,b2), NDBI (b6,b2); seasonal median; 500 m → cell mean |
| `elevation`, `slope`, `aspect_sin`, `aspect_cos` | `USGS/SRTMGL1_003` | `ee.Terrain.products`; aspect sin/cos at 30 m **before** averaging (circular variable); static |
| `ntl` | `NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG` (≥ 2014), `NOAA/DMSP-OLS/NIGHTTIME_LIGHTS` F18 (2010) | VIIRS annual median of the months with cloud-free coverage (`cf_cvg > 0`) of `avg_rad`, clamp ≥ 0, log1p; DMSP F18 2010 is first intercalibrated to F18 2013 (robust quadratic fit on pseudo-invariant pixels), then mapped into VIIRS log1p space by isotonic regression fitted on DMSP F18 2013 vs the first VIIRS year (2014; VIIRS monthly composites start in January 2014) |
| `pop_density` | `WorldPop/GP/100m/pop` (≤ 2020), `JRC/GHSL/P2023A/GHS_POP` (2025) | area-exact aggregation of counts → persons/km²; GHS-POP rescaled to the WorldPop-2020 ROI total |
| land cover (fine) | GLC_FCS30D annual (2010, 2015), ESA WorldCover v100 (2020), Dynamic World V1 annual mode (2025) | remapped to 6 harmonised classes, mode-resampled to the fine grid |

**Temporal design.** Four epochs (2010, 2015, 2020, 2025), each summarised over the pre-monsoon season
(1 April – 30 June) of that year, when the surface energy balance is dominated by sensible heat and
vegetation/water contrasts are strongest. Terrain is static.

**Caveats** (also exported in the manifest): land cover comes from three different products (class
definitions and accuracies differ between epochs); DMSP saturates in urban cores, has no on-board calibration
(2010 is intercalibrated to 2013 empirically) and is only approximately harmonised with VIIRS; Terra's orbit drifts
after 2022 (earlier overpass, slightly cooler LST); GHS-POP 2025 is a projection; MODIS LST is a 1 km radiometric
skin temperature, not air temperature. If both administrative boundary layers fail, districts fall back to an
approximate outline with nearest-HQ districts and the manifest flags `study_area.boundary_approximate`.

When Earth Engine is not available (and `LST_RUN_MODE` is not `gee`), a **synthetic twin** with the same
grid, variables, dtypes and a documented *planted* non-linear LST response is generated instead, and every
downstream artefact is flagged as synthetic.

## 1.1 Constants: districts, reference geography, land-cover classes

In [ ]:
import functools
import unicodedata

NCR_DISTRICTS = [
    {"id": 0, "name": "Delhi NCT", "state": "Delhi", "hq": (77.21, 28.61),
     "aliases": ["delhi", "nct of delhi", "national capital territory of delhi"]},
    {"id": 1, "name": "Gurugram", "state": "Haryana", "hq": (77.03, 28.46), "aliases": ["gurgaon", "gurugram"]},
    {"id": 2, "name": "Faridabad", "state": "Haryana", "hq": (77.31, 28.41), "aliases": ["faridabad"]},
    {"id": 3, "name": "Palwal", "state": "Haryana", "hq": (77.33, 28.14), "aliases": ["palwal"]},
    {"id": 4, "name": "Nuh", "state": "Haryana", "hq": (77.00, 28.10), "aliases": ["mewat", "nuh"]},
    {"id": 5, "name": "Rewari", "state": "Haryana", "hq": (76.62, 28.19), "aliases": ["rewari"]},
    {"id": 6, "name": "Jhajjar", "state": "Haryana", "hq": (76.66, 28.61), "aliases": ["jhajjar"]},
    {"id": 7, "name": "Rohtak", "state": "Haryana", "hq": (76.61, 28.90), "aliases": ["rohtak"]},
    {"id": 8, "name": "Sonipat", "state": "Haryana", "hq": (77.02, 28.99), "aliases": ["sonipat", "sonepat"]},
    {"id": 9, "name": "Panipat", "state": "Haryana", "hq": (76.97, 29.39), "aliases": ["panipat"]},
    {"id": 10, "name": "Karnal", "state": "Haryana", "hq": (76.99, 29.69), "aliases": ["karnal"]},
    {"id": 11, "name": "Jind", "state": "Haryana", "hq": (76.32, 29.32), "aliases": ["jind"]},
    {"id": 12, "name": "Bhiwani", "state": "Haryana", "hq": (76.13, 28.79), "aliases": ["bhiwani"]},
    {"id": 13, "name": "Charkhi Dadri", "state": "Haryana", "hq": (76.27, 28.59),
     "aliases": ["charkhi dadri", "dadri"]},
    {"id": 14, "name": "Mahendragarh", "state": "Haryana", "hq": (76.11, 28.05),
     "aliases": ["mahendragarh", "mahendergarh", "narnaul"]},
    {"id": 15, "name": "Meerut", "state": "Uttar Pradesh", "hq": (77.71, 28.98), "aliases": ["meerut"]},
    {"id": 16, "name": "Ghaziabad", "state": "Uttar Pradesh", "hq": (77.44, 28.67), "aliases": ["ghaziabad"]},
    {"id": 17, "name": "Gautam Buddh Nagar", "state": "Uttar Pradesh", "hq": (77.49, 28.47),
     "aliases": ["gautam buddha nagar", "gautam buddh nagar", "gautambudhnagar", "noida"]},
    {"id": 18, "name": "Bulandshahr", "state": "Uttar Pradesh", "hq": (77.85, 28.40),
     "aliases": ["bulandshahr", "bulandshahar"]},
    {"id": 19, "name": "Baghpat", "state": "Uttar Pradesh", "hq": (77.22, 28.94), "aliases": ["baghpat", "bagpat"]},
    {"id": 20, "name": "Hapur", "state": "Uttar Pradesh", "hq": (77.78, 28.73), "aliases": ["hapur", "panchsheel nagar"]},
    # geoBoundaries (v6 IND ADM2) spells the district "Samli".
    {"id": 21, "name": "Shamli", "state": "Uttar Pradesh", "hq": (77.31, 29.45),
     "aliases": ["shamli", "samli", "prabudh nagar", "prabuddh nagar"]},
    {"id": 22, "name": "Muzaffarnagar", "state": "Uttar Pradesh", "hq": (77.70, 29.47), "aliases": ["muzaffarnagar"]},
    {"id": 23, "name": "Alwar", "state": "Rajasthan", "hq": (76.60, 27.55), "aliases": ["alwar"]},
    {"id": 24, "name": "Bharatpur", "state": "Rajasthan", "hq": (77.49, 27.22), "aliases": ["bharatpur"]},
]
DISTRICTS = [{"id": d["id"], "name": d["name"], "state": d["state"]} for d in NCR_DISTRICTS]

# "Dadri" is also a tehsil of Gautam Buddh Nagar (UP): only accept it on the Haryana side.
_ALIAS_LON_MAX = {"dadri": 76.9}
# Delhi's revenue districts (geoBoundaries has no ADM1 attribute, so Delhi is matched by name + bbox).
DELHI_SUBDISTRICTS = ["Central", "East", "New Delhi", "North", "North East", "North West", "Shahdara",
                      "South", "South East", "South West", "West"]
DELHI_BBOX = (76.83, 28.40, 77.35, 28.89)          # lon_min, lat_min, lon_max, lat_max
NCR_SANITY_BBOX = (75.3, 26.6, 78.6, 30.1)         # duplicate district names exist elsewhere in India

NCR_OUTLINE = [
    (76.45, 29.95), (76.85, 29.95), (77.20, 29.75), (77.55, 29.75), (78.10, 29.55), (78.20, 29.20),
    (78.45, 28.60), (78.45, 28.20), (78.00, 28.05), (77.55, 27.95), (77.55, 27.50), (77.85, 27.20),
    (77.65, 26.75), (77.20, 26.70), (76.90, 27.05), (76.25, 27.05), (76.10, 27.40), (76.20, 27.90),
    (75.95, 28.00), (75.45, 28.40), (75.55, 28.90), (75.85, 29.05), (75.95, 29.55), (76.20, 29.60),
    (76.45, 29.95),
]
YAMUNA = [(77.19, 29.95), (77.13, 29.55), (77.12, 29.20), (77.22, 28.85), (77.25, 28.62), (77.30, 28.45),
          (77.45, 28.20), (77.52, 27.90), (77.68, 27.55)]
GANGA = [(78.03, 29.75), (78.10, 29.30), (78.18, 28.95), (78.32, 28.60), (78.45, 28.30)]
ARAVALLI = [(77.17, 28.62), (77.10, 28.40), (76.85, 28.15), (76.60, 27.80), (76.40, 27.35)]

# Harmonised land-cover classes (SPEC §1)
LC_NODATA, LC_BUILT, LC_FOREST, LC_WATER, LC_CROPLAND, LC_BARREN, LC_OTHER_VEG = 0, 1, 2, 3, 4, 5, 6
LC_CLASS_NAMES = {0: "nodata", 1: "built/impervious", 2: "tree/forest", 3: "water", 4: "cropland",
                  5: "barren/bare", 6: "other vegetation"}
LC_CLASS_COLORS = {0: "#000000", 1: "#e11d48", 2: "#15803d", 3: "#0ea5e9", 4: "#facc15", 5: "#a8a29e",
                   6: "#86efac"}
LC_REMAPS = {
    "esa_worldcover": {10: 2, 20: 6, 30: 6, 40: 4, 50: 1, 60: 5, 70: 6, 80: 3, 90: 6, 95: 2, 100: 6},
    "dynamic_world": {0: 3, 1: 2, 2: 6, 3: 6, 4: 4, 5: 6, 6: 1, 7: 5, 8: 6},
    "glc_fcs30d": {**{c: 4 for c in (10, 11, 12, 20)},
                   **{c: 2 for c in (51, 52, 61, 62, 71, 72, 81, 82, 91, 92)},
                   **{c: 6 for c in (120, 121, 122, 130, 140, 150, 152, 153, *range(181, 188))},
                   190: 1, 200: 5, 201: 5, 202: 5, 210: 3, 220: 6},
}

NODATA_FLOAT = -9999.0     # sentinel written by .unmask() for continuous bands, converted to NaN locally
NODATA_LC = 0              # land-cover nodata class
NODATA_DISTRICT = -1       # DISTRICT_IDX outside the ROI
RAW_BANDS = ["lst_day_c", "lst_obs_count", "ndvi", "ndwi", "ndbi", "elevation", "slope", "aspect_sin",
             "aspect_cos", "ntl", "pop_density"]
TERRAIN_BANDS = ["elevation", "slope", "aspect_sin", "aspect_cos"]
log(f"{len(NCR_DISTRICTS)} canonical districts, {len(RAW_BANDS)} raw bands, LC sources {CFG.LC_SOURCES}")

## 1.2 Grid definition (`GridSpec`)
Coordinates are transformed with `pyproj` (`always_xy=True`: lon, lat order). If pyproj is missing, a
closed-form transverse-Mercator series (Snyder 1987, USGS PP 1395, eqs. 8-9 … 8-25) is used for UTM
north zones — accurate to well below a metre inside the zone, far below the 25 m pixel size.

In [ ]:
_WGS84_A = 6378137.0
_WGS84_F = 1 / 298.257223563
_UTM_K0 = 0.9996


def _utm_zone(crs: str) -> int:
    """Zone number for a WGS84 / UTM north CRS code such as ``EPSG:32643``."""
    match = re.fullmatch(r"EPSG:326(\d{2})", crs.strip().upper())
    if not match:
        raise ValueError(f"pyproj is unavailable and {crs} is not a WGS84 UTM-north code; install pyproj")
    return int(match.group(1))


def _utm_forward(lon, lat, zone: int):
    """Geographic (deg) -> UTM north (m), Snyder's series."""
    e2 = _WGS84_F * (2 - _WGS84_F)
    ep2 = e2 / (1 - e2)
    phi, lam = np.radians(lat), np.radians(lon)
    lam0 = np.radians(-183.0 + 6.0 * zone)
    n = _WGS84_A / np.sqrt(1 - e2 * np.sin(phi) ** 2)
    t = np.tan(phi) ** 2
    c = ep2 * np.cos(phi) ** 2
    a = np.cos(phi) * (lam - lam0)
    m = _WGS84_A * ((1 - e2 / 4 - 3 * e2**2 / 64 - 5 * e2**3 / 256) * phi
                    - (3 * e2 / 8 + 3 * e2**2 / 32 + 45 * e2**3 / 1024) * np.sin(2 * phi)
                    + (15 * e2**2 / 256 + 45 * e2**3 / 1024) * np.sin(4 * phi)
                    - (35 * e2**3 / 3072) * np.sin(6 * phi))
    x = _UTM_K0 * n * (a + (1 - t + c) * a**3 / 6 + (5 - 18 * t + t**2 + 72 * c - 58 * ep2) * a**5 / 120) + 500000.0
    y = _UTM_K0 * (m + n * np.tan(phi) * (a**2 / 2 + (5 - t + 9 * c + 4 * c**2) * a**4 / 24
                                          + (61 - 58 * t + t**2 + 600 * c - 330 * ep2) * a**6 / 720))
    return x, y


def _utm_inverse(x, y, zone: int):
    """UTM north (m) -> geographic (deg), Snyder's footpoint-latitude series."""
    e2 = _WGS84_F * (2 - _WGS84_F)
    ep2 = e2 / (1 - e2)
    e1 = (1 - np.sqrt(1 - e2)) / (1 + np.sqrt(1 - e2))
    mu = (np.asarray(y, dtype=np.float64) / _UTM_K0) / (_WGS84_A * (1 - e2 / 4 - 3 * e2**2 / 64 - 5 * e2**3 / 256))
    phi1 = (mu + (3 * e1 / 2 - 27 * e1**3 / 32) * np.sin(2 * mu) + (21 * e1**2 / 16 - 55 * e1**4 / 32) * np.sin(4 * mu)
            + (151 * e1**3 / 96) * np.sin(6 * mu) + (1097 * e1**4 / 512) * np.sin(8 * mu))
    n1 = _WGS84_A / np.sqrt(1 - e2 * np.sin(phi1) ** 2)
    t1 = np.tan(phi1) ** 2
    c1 = ep2 * np.cos(phi1) ** 2
    r1 = _WGS84_A * (1 - e2) / (1 - e2 * np.sin(phi1) ** 2) ** 1.5
    d = (np.asarray(x, dtype=np.float64) - 500000.0) / (n1 * _UTM_K0)
    lat = phi1 - (n1 * np.tan(phi1) / r1) * (d**2 / 2 - (5 + 3 * t1 + 10 * c1 - 4 * c1**2 - 9 * ep2) * d**4 / 24
                                             + (61 + 90 * t1 + 298 * c1 + 45 * t1**2 - 252 * ep2 - 3 * c1**2) * d**6 / 720)
    lon = np.radians(-183.0 + 6.0 * zone) + (d - (1 + 2 * t1 + c1) * d**3 / 6
                                              + (5 - 2 * c1 + 28 * t1 - 3 * c1**2 + 8 * ep2 + 24 * t1**2) * d**5 / 120) / np.cos(phi1)
    return np.degrees(lon), np.degrees(lat)


@functools.lru_cache(maxsize=8)
def _grid_transformer(src: str, dst: str):
    return pyproj.Transformer.from_crs(src, dst, always_xy=True)


def _grid_lonlat_to_xy(lon, lat, crs: str):
    """(lon, lat) degrees -> projected metres in ``crs`` (arrays of any shape)."""
    lon, lat = np.asarray(lon, dtype=np.float64), np.asarray(lat, dtype=np.float64)
    if pyproj is not None:
        return _grid_transformer("EPSG:4326", crs).transform(lon, lat)
    return _utm_forward(lon, lat, _utm_zone(crs))


def _grid_xy_to_lonlat(x, y, crs: str):
    """Projected metres in ``crs`` -> (lon, lat) degrees (arrays of any shape)."""
    x, y = np.asarray(x, dtype=np.float64), np.asarray(y, dtype=np.float64)
    if pyproj is not None:
        return _grid_transformer(crs, "EPSG:4326").transform(x, y)
    return _utm_inverse(x, y, _utm_zone(crs))


def _densify_ring(points, n_per_edge: int = 40) -> np.ndarray:
    """Insert points along each edge of a lon/lat ring so its projected extent is exact (edges curve in UTM)."""
    pts = np.asarray(points, dtype=np.float64)
    t = np.linspace(0.0, 1.0, n_per_edge, endpoint=False)[:, None]
    parts = [pts[i] + t * (pts[i + 1] - pts[i]) for i in range(len(pts) - 1)]
    return np.vstack(parts + [pts[-1:]])


@dataclass(frozen=True)
class GridSpec:
    """North-up raster grid: ``x0``/``y0`` are the LEFT/TOP edges (m) in ``crs``; rows run southwards."""

    crs: str
    res: int
    x0: float
    y0: float
    width: int
    height: int

    @property
    def shape(self) -> tuple:
        return (self.height, self.width)

    @property
    def cell_area_km2(self) -> float:
        return (self.res / 1000.0) ** 2

    def x_centers(self) -> np.ndarray:
        return self.x0 + (np.arange(self.width) + 0.5) * self.res

    def y_centers(self) -> np.ndarray:
        return self.y0 - (np.arange(self.height) + 0.5) * self.res

    def cell_centers_xy(self):
        """(X, Y) 2-D arrays (float64, metres) of cell centres, shape (height, width)."""
        return np.meshgrid(self.x_centers(), self.y_centers())

    def lonlat(self):
        """(LON, LAT) 2-D arrays (float64, degrees) of cell centres."""
        x, y = self.cell_centers_xy()
        return _grid_xy_to_lonlat(x, y, self.crs)

    def affine(self) -> dict:
        """``affineTransform`` dict for ``ee.data.computePixels`` (north-up, negative scaleY)."""
        return {"scaleX": float(self.res), "shearX": 0.0, "translateX": float(self.x0),
                "shearY": 0.0, "scaleY": -float(self.res), "translateY": float(self.y0)}

    def transform_list(self) -> list:
        """Six-element affine ``[scaleX, shearX, translateX, shearY, scaleY, translateY]`` (EE crsTransform)."""
        return [float(self.res), 0.0, float(self.x0), 0.0, -float(self.res), float(self.y0)]

    def fine(self, res: int) -> "GridSpec":
        """Nested grid at ``res`` metres sharing the origin; ``res`` must divide ``self.res``."""
        res = int(res)
        if res <= 0 or self.res % res:
            raise ValueError(f"fine resolution {res} m must divide the grid resolution {self.res} m")
        f = self.res // res
        return GridSpec(self.crs, res, self.x0, self.y0, self.width * f, self.height * f)

    def tile(self, row0: int, col0: int, height: int, width: int) -> "GridSpec":
        """Sub-grid starting at (row0, col0) with the given size."""
        return GridSpec(self.crs, self.res, self.x0 + col0 * self.res, self.y0 - row0 * self.res, width, height)

    def bounds_lonlat(self) -> list:
        """[min_lon, min_lat, max_lon, max_lat] of the grid rectangle."""
        ring = [(self.x0, self.y0), (self.x0 + self.width * self.res, self.y0),
                (self.x0 + self.width * self.res, self.y0 - self.height * self.res),
                (self.x0, self.y0 - self.height * self.res), (self.x0, self.y0)]
        pts = _densify_ring(ring, 20)
        lon, lat = _grid_xy_to_lonlat(pts[:, 0], pts[:, 1], self.crs)
        return [float(lon.min()), float(lat.min()), float(lon.max()), float(lat.max())]

    @classmethod
    def from_lonlat_points(cls, lonlat_points, crs: str, res: int, margin_cells: int = 1) -> "GridSpec":
        """Smallest grid covering the points, edges snapped OUTWARD to multiples of ``res`` (+ margin)."""
        pts = np.asarray(lonlat_points, dtype=np.float64)
        x, y = _grid_lonlat_to_xy(pts[:, 0], pts[:, 1], crs)
        res = int(res)
        x_left = (math.floor(np.min(x) / res) - margin_cells) * res
        x_right = (math.ceil(np.max(x) / res) + margin_cells) * res
        y_bottom = (math.floor(np.min(y) / res) - margin_cells) * res
        y_top = (math.ceil(np.max(y) / res) + margin_cells) * res
        return cls(crs, res, float(x_left), float(y_top), int((x_right - x_left) // res), int((y_top - y_bottom) // res))


def _outline_grid(res: int) -> GridSpec:
    """Grid covering the approximate NCR outline (synthetic mode / final fallback)."""
    return GridSpec.from_lonlat_points(_densify_ring(NCR_OUTLINE), CFG.CRS, res)


def _mask_fine_inplace(lc: np.ndarray, roi_mask: np.ndarray, f: int) -> None:
    """Zero fine pixels whose parent cell is outside the ROI, without materialising a fine-resolution mask."""
    h, w = roi_mask.shape
    if not lc.flags.c_contiguous or lc.shape != (h * f, w * f):
        raise ValueError(f"fine raster must be C-contiguous with shape {(h * f, w * f)}, got {lc.shape}")
    view = lc.reshape(h, f, w, f)
    view *= roi_mask[:, None, :, None].astype(lc.dtype)


_probe_grid = _outline_grid(CFG.GRID_RES_M)
log(f"NCR outline grid preview: {_probe_grid.width}x{_probe_grid.height} cells of {_probe_grid.res} m, "
    f"lon/lat bounds {np.round(_probe_grid.bounds_lonlat(), 3).tolist()}")
del _probe_grid

## 1.3 Earth Engine initialisation and data-mode decision
Credential order: (1) Kaggle secret `GEE_SERVICE_ACCOUNT_KEY`, or env `LST_GEE_SERVICE_ACCOUNT_KEY` /
`GOOGLE_APPLICATION_CREDENTIALS` → `ee.ServiceAccountCredentials`; (2) persisted user credentials;
(3) interactive `ee.Authenticate(auth_mode="notebook")` (live notebooks only). All sessions use the
**high-volume endpoint**, which is designed for many parallel `computePixels` requests.
In `auto` mode any failure falls back to the synthetic twin with a loud banner; in `gee` mode it raises.

In [ ]:
_EE_HIGHVOLUME_URL = "https://earthengine-highvolume.googleapis.com"


def _service_account_key_text() -> Optional[str]:
    """Service-account JSON text from Kaggle secrets / env var (JSON or path) / GOOGLE_APPLICATION_CREDENTIALS.

    Logs which source was used (never the key itself) and warns loudly about a value that is set but unusable,
    because silently ignoring it would push the run into interactive OAuth login instead.
    """
    candidates = [("Kaggle secret GEE_SERVICE_ACCOUNT_KEY", _kaggle_secret("GEE_SERVICE_ACCOUNT_KEY")),
                  ("env LST_GEE_SERVICE_ACCOUNT_KEY", _env_str("LST_GEE_SERVICE_ACCOUNT_KEY")),
                  ("env GOOGLE_APPLICATION_CREDENTIALS", _env_str("GOOGLE_APPLICATION_CREDENTIALS"))]
    for source, value in candidates:
        if not value:
            continue
        text = value.strip().strip("'\"").strip()  # tolerate a key pasted inside quotes
        if text.startswith("{"):
            log(f"Earth Engine service-account key found in {source}")
            return text
        path = Path(text).expanduser()
        if path.is_file():
            file_text = path.read_text(encoding="utf-8")
            try:
                if json.loads(file_text).get("type") == "service_account":
                    log(f"Earth Engine service-account key file found via {source}: {path}")
                    return file_text
                log(f"{source} points to {path}, which is not a service-account key; ignoring", "WARNING")
            except json.JSONDecodeError:
                log(f"credential file {path} ({source}) is not valid JSON; ignoring", "WARNING")
            continue
        log(f"{source} is set but is neither JSON nor an existing file. Paste the ENTIRE key file contents "
            "(it starts with '{' and contains \"type\": \"service_account\").", "WARNING")
    if _IS_KAGGLE:
        log("No Earth Engine service-account key found. On Kaggle: Add-ons -> Secrets -> add "
            "GEE_SERVICE_ACCOUNT_KEY and make sure its checkbox is ticked for THIS notebook.", "WARNING")
    return None


def _service_account_hint(message: str) -> str:
    """Map common service-account failures to the setup step that fixes them."""
    low = message.lower()
    if "not signed up" in low or "not registered" in low or "register" in low:
        return ("the Cloud project is not registered for Earth Engine: register it at "
                "https://code.earthengine.google.com/register")
    if "serviceusage" in low or "user_project_denied" in low or "caller does not have required permission" in low:
        return ("grant the service account the roles 'Service Usage Consumer' and 'Earth Engine Resource Viewer' "
                "in the Cloud project, and enable the Earth Engine API there")
    if "has not been used in project" in low or "is disabled" in low:
        return "enable the 'Google Earth Engine API' in the Cloud project (APIs & Services -> Library)"
    if "invalid_grant" in low or "invalid jwt" in low or "account not found" in low:
        return "the key was deleted/revoked or belongs to a deleted account: create a new JSON key"
    if "project" in low and ("not found" in low or "no project" in low or "required" in low):
        return "set the Kaggle secret GEE_PROJECT (or LST_GEE_PROJECT) to the Cloud project id"
    if "keyerror" in low or "client_email" in low or "jsondecodeerror" in low:
        return "the secret is not a complete service-account key: paste the whole JSON key file"
    return "check the key, the project id and the service account's roles"


def _ee_smoke_test() -> None:
    """Round-trip a trivial computation so we fail fast on bad credentials / unregistered projects."""
    if ee.Number(1).add(1).getInfo() != 2:
        raise RuntimeError("Earth Engine smoke test returned an unexpected value")


def init_gee(project: Optional[str] = None) -> tuple:
    """Initialise Earth Engine on the high-volume endpoint. Returns ``(ok, message)`` and never raises."""
    if ee is None:
        return False, "earthengine-api is not installed"
    project = project or CFG.GEE_PROJECT
    log(f"Earth Engine project: {project or '(not set - will use the key file project_id)'}")
    errors = []
    key_text = _service_account_key_text()
    if key_text:
        # A supplied key is the user's explicit choice: report ITS failure instead of silently falling through to
        # interactive OAuth, whose own errors (e.g. "incompatible OAuth2 Client configuration") hide the real cause.
        email = "?"
        try:
            info = json.loads(key_text)
            email = info["client_email"]
            credentials = ee.ServiceAccountCredentials(email, key_data=key_text)
            use_project = project or info.get("project_id")
            ee.Initialize(credentials, project=use_project, opt_url=_EE_HIGHVOLUME_URL)
            _ee_smoke_test()
            return True, f"service account {email} (project {use_project})"
        except Exception as exc:  # noqa: BLE001 - reported with a remedy below
            detail = f"{type(exc).__name__}: {str(exc)[:400]}"
            hint = _service_account_hint(detail)
            log(f"Service-account login failed for {email}: {detail}\n    Fix: {hint}", "ERROR")
            return False, f"service account {email}: {detail} | fix: {hint}"
    try:
        ee.Initialize(project=project, opt_url=_EE_HIGHVOLUME_URL)
        _ee_smoke_test()
        return True, f"persisted user credentials (project {project})"
    except Exception as exc:  # noqa: BLE001
        errors.append(f"persisted credentials: {type(exc).__name__}: {str(exc)[:300]}")
    if _is_interactive():
        log("Falling back to interactive Earth Engine login (no service-account key). If the login page says "
            "'Project has an incompatible OAuth2 Client configuration', pick (or create) a Cloud project with NO "
            "OAuth 2.0 client IDs for authentication - it may differ from GEE_PROJECT - or, better, use the "
            "GEE_SERVICE_ACCOUNT_KEY secret, which needs no browser login.", "WARNING")
        try:
            ee.Authenticate(auth_mode="notebook")
            ee.Initialize(project=project, opt_url=_EE_HIGHVOLUME_URL)
            _ee_smoke_test()
            return True, f"interactive notebook authentication (project {project})"
        except Exception as exc:  # noqa: BLE001
            errors.append(f"interactive auth: {type(exc).__name__}: {str(exc)[:300]}")
    else:
        errors.append("interactive auth skipped (not a live notebook session)")
    return False, " | ".join(errors)


def _print_synthetic_banner(reason: str) -> None:
    """Impossible-to-miss notice that every number downstream comes from the synthetic twin."""
    bar = "!" * 96
    lines = [
        "SYNTHETIC MODE - RESULTS ARE NOT REAL OBSERVATIONS",
        "Earth Engine data were NOT used. All rasters, models, SHAP values, thresholds and zones below",
        "come from a synthetic twin of Delhi NCR with a PLANTED LST response (see SYNTHETIC_TRUTH).",
        "Use them only to test the pipeline and the dashboard. Every export is flagged data_mode=synthetic.",
        f"Reason: {reason[:260]}",
        "To use real data: attach Kaggle secrets GEE_SERVICE_ACCOUNT_KEY + GEE_PROJECT (or authenticate",
        "interactively) and re-run with LST_RUN_MODE=gee.",
    ]
    print("\n" + bar + "\n" + bar)
    for line in lines:
        print(f"!!!  {line:<88}!!!")
    print(bar + "\n" + bar + "\n", flush=True)


if CFG.RUN_MODE == "synthetic":
    DATA_MODE = "synthetic"
    _MODE_REASON = "LST_RUN_MODE=synthetic"
else:
    _gee_ok, _gee_msg = init_gee()
    if _gee_ok:
        DATA_MODE = "gee"
        _MODE_REASON = f"Earth Engine initialised via {_gee_msg}"
        log(_MODE_REASON)
    elif CFG.RUN_MODE == "gee":
        raise RuntimeError(
            "LST_RUN_MODE=gee but Earth Engine could not be initialised.\n"
            f"Details: {_gee_msg}\n"
            "Fix: on Kaggle add Secrets GEE_SERVICE_ACCOUNT_KEY (full service-account JSON key; the account "
            "must be registered for Earth Engine) and GEE_PROJECT (Cloud project with the Earth Engine API "
            "enabled), enable Internet, or run interactively to authenticate; elsewhere set "
            "LST_GEE_SERVICE_ACCOUNT_KEY / GOOGLE_APPLICATION_CREDENTIALS and LST_GEE_PROJECT. "
            "Use LST_RUN_MODE=auto to fall back to synthetic data instead.")
    else:
        DATA_MODE = "synthetic"
        _MODE_REASON = f"Earth Engine unavailable ({_gee_msg})"
CFG.resolve_mode(DATA_MODE)
if DATA_MODE == "synthetic":
    _print_synthetic_banner(_MODE_REASON)
log(f"DATA_MODE={DATA_MODE} | GRID_RES_M={CFG.GRID_RES_M} | FINE_RES_M={CFG.fine_res()}")

## 1.4 Earth Engine: ROI and district matching
Candidate ADM2 polygons are pulled for the NCR bounding box (attributes + centroid + area only), matched to
the 25 canonical districts client-side by alias (case/space/diacritic-insensitive) with a bounding-box
sanity filter, and the matched polygons are tagged with the canonical `did`. Delhi's revenue districts are
merged into "Delhi NCT". The district raster is produced server-side with `paint()` on the analysis grid.

**Unmatched districts are located, not assumed merged.** For every canonical district still unmatched, the layer
is queried for the polygon containing its headquarters point: an *unmatched* polygon there (a spelling the alias
table does not know) is matched by location with a loud warning; a polygon already matched to another district
means the district is genuinely merged into that parent in this layer (acceptable, SPEC §1.1); no polygon at all is
reported as an error.

In [ ]:
_GEOBOUNDARIES_ADM2 = "WM/geoLab/geoBoundaries/600/ADM2"
_GAUL_ADM2 = "FAO/GAUL/2015/level2"


def _norm_name(text) -> str:
    """Lower-case ASCII alphanumerics only ("Gautam Buddha Nagar" -> "gautambuddhanagar")."""
    ascii_text = unicodedata.normalize("NFKD", str(text or "")).encode("ascii", "ignore").decode()
    return re.sub(r"[^a-z0-9]", "", ascii_text.lower())


def _strip_admin_words(norm: str) -> str:
    for word in ("district", "delhi", "nctof", "nct"):
        norm = norm.replace(word, "")
    return norm


_ALIAS_TO_ID = {_norm_name(a): d["id"] for d in NCR_DISTRICTS[1:] for a in d["aliases"]}
_DELHI_NAME_KEYS = {_strip_admin_words(_norm_name(s)) for s in DELHI_SUBDISTRICTS} | {""}
_DELHI_ADM1_KEYS = {_norm_name(a) for a in NCR_DISTRICTS[0]["aliases"]}


def _in_bbox(lon: float, lat: float, bbox) -> bool:
    return bbox[0] <= lon <= bbox[2] and bbox[1] <= lat <= bbox[3]


def _match_district(record: dict) -> Optional[int]:
    """Canonical district id for one candidate ADM2 record, or None if it is not an NCR district."""
    lon, lat = record.get("lon"), record.get("lat")
    if lon is None or lat is None or not _in_bbox(lon, lat, NCR_SANITY_BBOX):
        return None
    name = _norm_name(record.get("name"))
    adm1 = _norm_name(record.get("adm1")) if record.get("adm1") else None
    stripped = _strip_admin_words(name)
    for key in (name, stripped):
        did = _ALIAS_TO_ID.get(key)
        if did is None:
            continue
        if key in _ALIAS_LON_MAX and lon > _ALIAS_LON_MAX[key]:
            return None
        state_norm = _norm_name(NCR_DISTRICTS[did]["state"])
        if adm1 is not None and state_norm not in adm1 and adm1 not in state_norm:
            continue
        return did
    in_delhi_box = _in_bbox(lon, lat, DELHI_BBOX)
    if adm1 is not None:
        return 0 if (adm1 in _DELHI_ADM1_KEYS and in_delhi_box) else None
    return 0 if (stripped in _DELHI_NAME_KEYS and in_delhi_box) else None


def _gee_candidates(source: str):
    """(server FeatureCollection, client records) of ADM2 units intersecting the NCR sanity box."""
    rect = ee.Geometry.Rectangle(list(NCR_SANITY_BBOX), None, False)
    if source == "geoboundaries":
        fc = ee.FeatureCollection(_GEOBOUNDARIES_ADM2).filter(ee.Filter.eq("shapeGroup", "IND")).filterBounds(rect)
        name_prop, adm1_prop = "shapeName", None
    else:
        fc = ee.FeatureCollection(_GAUL_ADM2).filter(ee.Filter.eq("ADM0_NAME", "India")).filterBounds(rect)
        name_prop, adm1_prop = "ADM2_NAME", "ADM1_NAME"

    def _summarise(feature):
        geom = feature.geometry()
        centroid = geom.centroid(500).coordinates()
        props = {"name": feature.get(name_prop), "lon": centroid.get(0), "lat": centroid.get(1),
                 "area_km2": geom.area(500).divide(1e6)}
        if adm1_prop:
            props["adm1"] = feature.get(adm1_prop)
        return ee.Feature(None, props)

    info = retry(lambda: fc.map(_summarise).getInfo(), tries=4, base_delay=3.0, max_delay=60.0)
    records = [{"key": feat["id"], **feat["properties"]} for feat in info.get("features", [])]
    return fc, records


def _gee_hq_hits(fc, dids) -> dict:
    """{did: [system:index of every candidate polygon containing the district's HQ point]} (one server call)."""
    if not dids:
        return {}
    hits = ee.Dictionary({str(did): fc.filterBounds(ee.Geometry.Point(list(NCR_DISTRICTS[did]["hq"])))
                          .aggregate_array("system:index") for did in dids})
    info = retry(lambda: hits.getInfo(), tries=4, base_delay=3.0, max_delay=60.0)
    return {int(k): [str(v) for v in (vals or [])] for k, vals in (info or {}).items()}


def _resolve_unmatched(missing_ids, hq_hits: dict, key_to_did: dict, records_by_key: dict):
    """Classify canonical districts left unmatched by name, using the polygons that contain their HQ point.

    Returns (new_matches {candidate key: did}, messages [(level, text)]). An unmatched polygon at the HQ is matched
    by location (an alias the table does not know); a polygon already matched to another district means the
    district is merged into that parent in this layer; no polygon at all is an error.
    """
    new, messages = {}, []
    for did in missing_ids:
        name = NCR_DISTRICTS[did]["name"]
        keys = list(hq_hits.get(did, []))
        free = [k for k in keys if k not in key_to_did and k not in new and k in records_by_key]
        taken = [k for k in keys if k in key_to_did or k in new]
        if free:
            key = free[0]
            new[key] = did
            messages.append(("WARNING", f"{did:2d} {name}: matched BY LOCATION to polygon "
                                        f"'{records_by_key[key].get('name')}' containing its HQ (name not in the "
                                        f"alias table - add it to SPEC §1.1)"))
        elif taken:
            parent = key_to_did.get(taken[0], new.get(taken[0]))
            messages.append(("INFO", f"{did:2d} {name}: merged into parent district "
                                     f"{NCR_DISTRICTS[parent]['name']} in this layer (HQ inside its polygon)"))
        else:
            messages.append(("ERROR", f"{did:2d} {name}: no polygon of this layer contains its HQ - the district's "
                                      "territory may be missing from the ROI"))
    return new, messages


def _gee_match_roi(source: str):
    """Match one boundary source; returns (roi_fc, matched {did: [names]}, total area) or None if unusable."""
    fc, records = _gee_candidates(source)
    records_by_key = {rec["key"]: rec for rec in records}
    key_to_did, matched, area = {}, {}, 0.0
    for rec in records:
        did = _match_district(rec)
        if did is None:
            continue
        key_to_did[rec["key"]] = did
    missing_ids = [d["id"] for d in NCR_DISTRICTS if d["id"] not in set(key_to_did.values())]
    if missing_ids:
        try:
            located, messages = _resolve_unmatched(missing_ids, _gee_hq_hits(fc, missing_ids), key_to_did,
                                                   records_by_key)
        except Exception as exc:  # noqa: BLE001 - diagnostics must not lose an otherwise usable layer
            located, messages = {}, [("WARNING", f"HQ location check failed ({type(exc).__name__}: "
                                                 f"{str(exc)[:160]}); unmatched: "
                                                 f"{[NCR_DISTRICTS[d]['name'] for d in missing_ids]}")]
        key_to_did.update(located)
        for level, text in messages:
            log(f"[{source}] {text}", level)
    for key, did in key_to_did.items():
        matched.setdefault(did, []).append(str(records_by_key[key].get("name")))
        area += float(records_by_key[key].get("area_km2") or 0.0)
    log(f"[{source}] {len(records)} candidates, matched {len(matched)}/25 districts, area {area:,.0f} km2")
    for did in sorted(matched):
        log(f"    {did:2d} {NCR_DISTRICTS[did]['name']:<20} <- {', '.join(sorted(set(matched[did])))}")
    if 0 not in matched or len(matched) < 18:
        log(f"[{source}] too few matches (Delhi matched: {0 in matched}); trying the next boundary source", "WARNING")
        return None
    if not 45_000 <= area <= 65_000:
        log(f"[{source}] matched area {area:,.0f} km2 is outside the expected 45,000-65,000 km2", "WARNING")
    mapping = ee.Dictionary({k: v for k, v in key_to_did.items()})
    roi_fc = (fc.filter(ee.Filter.inList("system:index", list(key_to_did)))
                .map(lambda f: f.set("did", mapping.get(f.get("system:index")))))
    return roi_fc, matched, area


def _round_coords(obj, ndigits: int = 5):
    """Round nested GeoJSON coordinate lists."""
    if isinstance(obj, (list, tuple)):
        if obj and isinstance(obj[0], (int, float)):
            return [round(float(v), ndigits) for v in obj]
        return [_round_coords(v, ndigits) for v in obj]
    return obj


def _gee_boundary_geojson(roi_fc, dids) -> dict:
    """Dissolved, 200 m-simplified district polygons (EPSG:4326) with properties id/name/state."""
    features = []
    for did in sorted(dids):
        meta = NCR_DISTRICTS[did]
        geom = (roi_fc.filter(ee.Filter.eq("did", did)).geometry(100)
                .dissolve(100).simplify(200))
        features.append(ee.Feature(geom, {"id": did, "name": meta["name"], "state": meta["state"]}))
    info = retry(lambda: ee.FeatureCollection(features).getInfo(), tries=4, base_delay=3.0, max_delay=60.0)
    out = []
    for feat in info["features"]:
        props = feat["properties"]
        geometry = feat["geometry"]
        geometry = {"type": geometry["type"], "coordinates": _round_coords(geometry["coordinates"])}
        out.append({"type": "Feature", "geometry": geometry,
                    "properties": {"id": int(props["id"]), "name": props["name"], "state": props["state"]}})
    return {"type": "FeatureCollection", "features": out}


def _geojson_lonlat_points(geojson: dict) -> np.ndarray:
    """All vertex (lon, lat) pairs of a FeatureCollection of (Multi)Polygons."""
    pts = []

    def _walk(coords):
        if coords and isinstance(coords[0], (int, float)):
            pts.append(coords[:2])
        else:
            for c in coords:
                _walk(c)

    for feat in geojson["features"]:
        _walk(feat["geometry"]["coordinates"])
    return np.asarray(pts, dtype=np.float64)

## 1.5 Earth Engine: tiled `computePixels` extraction with retry, subdivision and caching
Each tile is an explicit grid request (`crsCode` + `affineTransform` + `dimensions`) returning a structured
NumPy array. Transient errors (HTTP 429/5xx, "Too many concurrent aggregations", "Earth Engine (memory) capacity
exceeded", timeouts) are retried with exponential backoff; size/memory errors split the tile into quarters (up to
depth 3) and are **permanent** once the tile cannot be split further (retrying an identical request that exceeds the
user memory limit cannot succeed). On the first failure the remaining tiles are cancelled and in-flight tiles stop
retrying. Every top-level tile is cached as `.npy` under `CFG.CACHE_DIR/tiles/<cache_key>/`, so interrupted runs
resume; the cache key contains a hash of the serialised Earth Engine expression, so any change of dates, QA rules,
scale factors, the district mapping or other server-side parameters gets a fresh cache folder.

In [ ]:
_GEE_CACHE_VERSION = "v2"
_GEE_MAX_SPLIT_DEPTH = 3
_GEE_SPLIT_RE = re.compile(r"user memory limit exceeded|computation timed out|too large|request size|"
                           r"response size|out of memory|memory limit", re.IGNORECASE)
_GEE_TIMEOUT_RE = re.compile(r"timed out|deadline", re.IGNORECASE)
_GEE_TRANSIENT_RE = re.compile(r"\b(429|500|502|503|504)\b|too many requests|too many concurrent aggregations|"
                               r"capacity exceeded|"
                               r"quota|rate limit|internal error|service unavailable|backend error|deadline|"
                               r"temporar|connection|reset by peer|timed out|remote end closed|broken pipe|ssl",
                               re.IGNORECASE)


class _GEETransientError(RuntimeError):
    """Retryable Earth Engine failure (quota, 5xx, transient timeouts)."""


class _GEESplitError(RuntimeError):
    """Request too large/slow for one call: subdivide the tile."""


class _GEEAborted(RuntimeError):
    """Another tile of the same request failed permanently: stop retrying this one."""


def _gee_expression_digest(image) -> str:
    """SHA-1 of the serialised Earth Engine expression (captures every server-side parameter).

    Falls back to a random token (cache effectively disabled for that request) if serialisation fails, so a stale
    tile can never be reused under an incomplete key.
    """
    import hashlib
    import uuid

    try:
        text = image.serialize()
    except Exception as exc:  # noqa: BLE001
        log(f"could not serialise the Earth Engine expression ({type(exc).__name__}); tile cache disabled for "
            "this request", "WARNING")
        text = uuid.uuid4().hex
    return hashlib.sha1(str(text).encode("utf-8")).hexdigest()


def _gee_cache_key(name: str, grid: GridSpec, extra: Any = None, image=None) -> str:
    """Stable cache folder name tied to the grid geometry, the request description and (when ``image`` is given)
    the hash of the serialised expression."""
    expr = _gee_expression_digest(image) if image is not None else None
    payload = json.dumps([_GEE_CACHE_VERSION, asdict(grid), extra, expr], sort_keys=True, default=str)
    digest = __import__("hashlib").sha1(payload.encode()).hexdigest()[:10]
    return f"{name}_{grid.res}m_{digest}"


def _structured_to_stack(raw, bands: Sequence[str], tile: GridSpec, dtype) -> np.ndarray:
    """computePixels NUMPY_NDARRAY result (structured, one field per band) -> (bands, h, w) array."""
    raw = np.asarray(raw)
    if raw.dtype.names:
        missing = [b for b in bands if b not in raw.dtype.names]
        if missing:
            raise RuntimeError(f"computePixels result lacks bands {missing}; got {raw.dtype.names}")
        stack = np.stack([np.asarray(raw[b]) for b in bands])
    elif raw.ndim == 3:
        stack = np.moveaxis(raw, -1, 0)
    else:
        stack = raw[None]
    if stack.shape[1:] != (tile.height, tile.width):
        raise RuntimeError(f"computePixels returned shape {stack.shape[1:]}, expected {(tile.height, tile.width)}")
    return stack.astype(dtype, copy=False)


def _compute_pixels(image, tile: GridSpec, bands: Sequence[str], dtype, allow_split: bool) -> np.ndarray:
    """One computePixels call, translating EE errors into split / transient / permanent failures."""
    request = {
        "expression": image,
        "fileFormat": "NUMPY_NDARRAY",
        "bandIds": list(bands),
        "grid": {"dimensions": {"width": int(tile.width), "height": int(tile.height)},
                 "affineTransform": tile.affine(), "crsCode": tile.crs},
    }
    try:
        raw = ee.data.computePixels(request)
    except Exception as exc:  # noqa: BLE001 - classified below
        raise _classify_gee_error(exc, allow_split) from exc
    return _structured_to_stack(raw, bands, tile, dtype)


def _classify_gee_error(exc: BaseException, allow_split: bool) -> BaseException:
    """Map an Earth Engine exception to split / transient / permanent (returns the exception to raise).

    * size/memory errors split the tile while it can be split; at the maximum depth they are permanent (an
      identical request cannot suddenly fit the per-user memory limit) - except timeouts, which stay retryable;
    * load-related errors (429/5xx, concurrent aggregations, "Earth Engine (memory) capacity exceeded", timeouts,
      connection resets) are transient and retried with backoff;
    * everything else (bad asset, auth, invalid argument) is permanent.
    """
    message = str(exc)
    if _GEE_SPLIT_RE.search(message):
        if allow_split:
            return _GEESplitError(message)
        if _GEE_TIMEOUT_RE.search(message):
            return _GEETransientError(message)
        return RuntimeError(f"Earth Engine request still too large at the maximum split depth: {message}")
    if _GEE_TRANSIENT_RE.search(message):
        return _GEETransientError(message)
    return exc


def _fetch_tile(image, tile: GridSpec, bands: Sequence[str], dtype, depth: int = 0,
                abort: Optional[threading.Event] = None) -> np.ndarray:
    """Fetch one tile with backoff; quarter it recursively on size/memory errors; stop when ``abort`` is set."""
    can_split = depth < _GEE_MAX_SPLIT_DEPTH and min(tile.width, tile.height) >= 2

    def _on_retry(attempt, exc, delay):
        if abort is not None and abort.is_set():
            raise _GEEAborted("another tile failed; not retrying") from exc
        log(f"computePixels retry {attempt} for {tile.width}x{tile.height} tile in {delay:.1f}s: {str(exc)[:160]}",
            "WARNING")

    if abort is not None and abort.is_set():
        raise _GEEAborted("another tile failed; not fetching")
    try:
        return retry(lambda: _compute_pixels(image, tile, bands, dtype, can_split), tries=CFG.GEE_MAX_RETRIES,
                     base_delay=2.0, max_delay=120.0, retry_on=(_GEETransientError,), on_retry=_on_retry)
    except _GEESplitError as exc:
        log(f"splitting {tile.width}x{tile.height} tile (depth {depth + 1}): {str(exc)[:160]}", "WARNING")
    h1, w1 = tile.height // 2, tile.width // 2
    out = np.empty((len(bands), tile.height, tile.width), dtype=dtype)
    for r0, h in ((0, h1), (h1, tile.height - h1)):
        for c0, w in ((0, w1), (w1, tile.width - w1)):
            out[:, r0:r0 + h, c0:c0 + w] = _fetch_tile(image, tile.tile(r0, c0, h, w), bands, dtype, depth + 1,
                                                       abort)
    return out


def _save_npy_atomic(path: Path, array: np.ndarray) -> None:
    tmp = path.with_name(path.name + ".tmp")
    with open(tmp, "wb") as fh:
        np.save(fh, array)
    os.replace(tmp, path)


def fetch_grid(image, grid: GridSpec, bands: Sequence[str], tile_px: int, cache_key: str,
               dtype=np.float32) -> np.ndarray:
    """Download ``bands`` of an ee.Image on ``grid`` as a (len(bands), H, W) array.

    Tiles of ``tile_px`` × ``tile_px`` are fetched concurrently (``CFG.GEE_WORKERS`` threads) with retry,
    recursive subdivision and an on-disk cache. For float dtypes the ``NODATA_FLOAT`` sentinel is turned
    into NaN; integer outputs are returned as delivered.
    """
    if ee is None:
        raise RuntimeError("earthengine-api is not available")
    bands, dtype = list(bands), np.dtype(dtype)
    abort = threading.Event()
    cache_dir = CFG.CACHE_DIR / "tiles" / cache_key
    cache_dir.mkdir(parents=True, exist_ok=True)
    tiles = [(r0, c0, min(tile_px, grid.height - r0), min(tile_px, grid.width - c0))
             for r0 in range(0, grid.height, tile_px) for c0 in range(0, grid.width, tile_px)]
    out = np.empty((len(bands), grid.height, grid.width), dtype=dtype)

    def _job(spec):
        r0, c0, h, w = spec
        path = cache_dir / f"r{r0:06d}_c{c0:06d}_{h}x{w}.npy"
        if path.exists():
            try:
                cached = np.load(path)
                if cached.shape == (len(bands), h, w):
                    return spec, cached.astype(dtype, copy=False), True
            except (OSError, ValueError) as exc:
                log(f"ignoring unreadable cache tile {path.name}: {exc}", "WARNING")
        arr = _fetch_tile(image, grid.tile(r0, c0, h, w), bands, dtype, abort=abort)
        _save_npy_atomic(path, arr)
        return spec, arr, False

    n_done, n_cached, step = 0, 0, max(1, len(tiles) // 10)
    pool = ThreadPoolExecutor(max_workers=max(1, min(CFG.GEE_WORKERS, len(tiles))), thread_name_prefix="ee")
    futures = [pool.submit(_job, t) for t in tiles]
    try:
        for future in as_completed(futures):
            (r0, c0, h, w), arr, cached = future.result()
            out[:, r0:r0 + h, c0:c0 + w] = arr
            n_done += 1
            n_cached += int(cached)
            if n_done % step == 0 or n_done == len(tiles):
                log(f"  {cache_key}: {n_done}/{len(tiles)} tiles ({n_cached} from cache)")
    except BaseException:
        # Fail fast: cancel queued tiles, tell in-flight ones to stop retrying, and do not wait for them.
        abort.set()
        pool.shutdown(wait=False, cancel_futures=True)
        raise
    pool.shutdown(wait=True)
    if dtype.kind == "f":
        out[out == NODATA_FLOAT] = np.nan
    return out

## 1.6 Earth Engine: per-epoch predictor images
Every variable is composited in its **native** projection, then aggregated to the analysis grid with an
area-weighted `reduceResolution` (mean) followed by `reproject` onto the exact grid transform. Composites
lose their native projection in Earth Engine, so it is restored with `setDefaultProjection` before
aggregation. Population counts are converted to densities (count / pixel area) **before** the area-weighted
mean, which makes the aggregate exactly "persons in cell / cell area" (a weighted `sum` reducer would
weight by output-area fraction and mis-scale counts).

In [ ]:
_NATIVE_SCALE_M = {"modis_lst": 926.625, "modis_sr": 463.313, "srtm": 30.0, "viirs": 463.83, "dmsp": 927.67,
                   "worldpop": 80.0, "ghs_pop": 100.0, "glc_fcs30d": 30.0, "esa_worldcover": 10.0,
                   "dynamic_world": 10.0}


def _season_dates(year: int):
    """[start, end) date strings for the pre-monsoon season of ``year`` (end exclusive, +1 day)."""
    start = f"{year}-{CFG.SEASON[0]}"
    end = (pd.Timestamp(f"{year}-{CFG.SEASON[1]}") + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
    return start, end


def _to_grid(image, native_scale_m: float, grid: GridSpec, reducer=None):
    """Aggregate ``image`` (with a valid default projection) onto ``grid``.

    Coarsening uses reduceResolution with ``maxPixels`` sized to the scale ratio (≥ 1024); refining or
    equal scales use a plain reprojection (nearest neighbour).
    """
    if native_scale_m < grid.res * 0.999:
        ratio = (grid.res / native_scale_m) ** 2
        max_pixels = int(min(65535, max(1024, math.ceil(ratio * 2.5))))
        image = image.reduceResolution(reducer=reducer or ee.Reducer.mean(), bestEffort=False, maxPixels=max_pixels)
    return image.reproject(crs=grid.crs, crsTransform=grid.transform_list())


def _gee_lst_bands(year: int, grid: GridSpec):
    """Seasonal median day LST (°C), count of valid 8-day composites, and the masked native median (preview).

    A native pixel needs >= CFG.LST_MIN_OBS valid composites; a grid cell's LST is kept only where >=
    CFG.LST_MIN_COVERAGE of its area has such pixels (reduceResolution's mean is mask-weighted, so without this a
    small valid sliver would stand for the whole cell).
    """
    start, end = _season_dates(year)
    col = ee.ImageCollection("MODIS/061/MOD11A2").filterDate(start, end)
    native = col.first().select("LST_Day_1km").projection()

    def _prep(img):
        qc = img.select("QC_Day")
        good = qc.bitwiseAnd(3).lte(1).And(qc.rightShift(6).bitwiseAnd(3).lte(1))
        return img.select("LST_Day_1km").multiply(0.02).subtract(273.15).updateMask(good).rename("lst_day_c")

    lst_col = col.map(_prep)
    count_native = lst_col.count().rename("lst_obs_count")
    enough = count_native.gte(int(CFG.LST_MIN_OBS))
    median = lst_col.median().rename("lst_day_c").updateMask(enough).setDefaultProjection(native)
    count = count_native.unmask(0).toFloat().setDefaultProjection(native)
    coverage = median.mask().gt(0).unmask(0).toFloat().rename("coverage").setDefaultProjection(native)
    scale = _NATIVE_SCALE_M["modis_lst"]
    lst_grid = _to_grid(median, scale, grid)
    cov_grid = _to_grid(coverage, scale, grid)
    lst_grid = lst_grid.updateMask(cov_grid.gte(float(CFG.LST_MIN_COVERAGE)))
    prov = (f"MODIS/061/MOD11A2 LST_Day_1km {start}..{CFG.SEASON[1]} QC bits0-1<=1 & bits6-7<=1, "
            f"x0.02-273.15, seasonal median (pixels with >= {CFG.LST_MIN_OBS} valid composites, cells with >= "
            f"{100 * CFG.LST_MIN_COVERAGE:.0f}% valid area), mean-aggregated to {grid.res} m")
    provenance = {"lst_day_c": prov,
                  "lst_obs_count": (f"MODIS/061/MOD11A2 LST_Day_1km {start}..{CFG.SEASON[1]} count of QC-valid "
                                    f"8-day composites, mean-aggregated to {grid.res} m")}
    return [lst_grid, _to_grid(count, scale, grid)], provenance, median


def _gee_spectral_bands(year: int, grid: GridSpec):
    """Seasonal median NDVI / NDWI (McFeeters) / NDBI from cloud-screened MOD09A1."""
    start, end = _season_dates(year)
    col = ee.ImageCollection("MODIS/061/MOD09A1").filterDate(start, end)
    native = col.first().select("sur_refl_b01").projection()

    def _prep(img):
        qa = img.select("StateQA")
        clear = (qa.bitwiseAnd(3).eq(0)
                 .And(qa.rightShift(2).bitwiseAnd(1).eq(0))
                 .And(qa.rightShift(10).bitwiseAnd(1).eq(0)))
        refl = img.select(["sur_refl_b01", "sur_refl_b02", "sur_refl_b04", "sur_refl_b06"]).multiply(0.0001)
        ndvi = refl.normalizedDifference(["sur_refl_b02", "sur_refl_b01"]).rename("ndvi")
        ndwi = refl.normalizedDifference(["sur_refl_b04", "sur_refl_b02"]).rename("ndwi")
        ndbi = refl.normalizedDifference(["sur_refl_b06", "sur_refl_b02"]).rename("ndbi")
        return ee.Image.cat([ndvi, ndwi, ndbi]).clamp(-1, 1).updateMask(clear)

    median = col.map(_prep).median().setDefaultProjection(native)
    scale = _NATIVE_SCALE_M["modis_sr"]
    images = [_to_grid(median.select(b), scale, grid) for b in ("ndvi", "ndwi", "ndbi")]
    prov = (f"MODIS/061/MOD09A1 {start}..{CFG.SEASON[1]} StateQA clear (bits0-1=0, bit2=0, bit10=0), x0.0001, "
            "seasonal median of {idx}, mean-aggregated 500 m -> grid")
    return images, {b: prov.format(idx=b.upper()) for b in ("ndvi", "ndwi", "ndbi")}


def _gee_terrain_image(grid: GridSpec):
    """Static SRTM terrain: elevation, slope and aspect sin/cos (computed at 30 m before averaging)."""
    dem = ee.Image("USGS/SRTMGL1_003").select("elevation")
    terrain = ee.Terrain.products(dem)
    aspect = terrain.select("aspect").multiply(math.pi / 180.0)
    stack = ee.Image.cat([dem.toFloat().rename("elevation"), terrain.select("slope").toFloat().rename("slope"),
                          aspect.sin().rename("aspect_sin"), aspect.cos().rename("aspect_cos")])
    scale = _NATIVE_SCALE_M["srtm"]
    bands = [_to_grid(stack.select(b), scale, grid) for b in TERRAIN_BANDS]
    return ee.Image.cat(bands).unmask(NODATA_FLOAT).toFloat()


def _gee_viirs_log(years_to_try: Sequence[int], grid: GridSpec):
    """log1p of the annual median VIIRS avg_rad (clamped ≥ 0) for the first year with data; (image, prov)."""
    for coll_id in ("NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG", "NOAA/VIIRS/DNB/MONTHLY_V1/VCMCFG"):
        for year in years_to_try:
            raw = ee.ImageCollection(coll_id).filterDate(f"{year}-01-01", f"{year + 1}-01-01")
            n_months = retry(lambda: raw.size().getInfo(), tries=4, base_delay=2.0, max_delay=30.0)
            if not n_months:
                continue
            native = raw.first().select("avg_rad").projection()
            # Months without any cloud-free observation (monsoon) report unobserved radiances: mask them first.
            col = raw.map(lambda img: img.select("avg_rad").updateMask(img.select("cf_cvg").gt(0)))
            image = col.median().max(0).add(1).log().rename("ntl").setDefaultProjection(native)
            prov = (f"{coll_id} avg_rad {year} median of the {n_months} monthly composites where cf_cvg > 0 "
                    "(months without cloud-free coverage masked), clamp>=0, log1p")
            return _to_grid(image, _NATIVE_SCALE_M["viirs"], grid), prov, year
    raise RuntimeError(f"no VIIRS monthly composites found for years {list(years_to_try)}")


def _gee_dmsp(year: int, grid: GridSpec):
    """DMSP-OLS F18 stable_lights (F18 exists for 2010-2013; other years use the nearest F18 year, which
    `_validate_epochs` has already reported as a caveat)."""
    f18_year = int(min(max(year, 2010), 2013))
    image = ee.Image(f"NOAA/DMSP-OLS/NIGHTTIME_LIGHTS/F18{f18_year}").select("stable_lights").toFloat()
    return _to_grid(image, _NATIVE_SCALE_M["dmsp"], grid), f18_year


def _density_image(counts, native_proj, factor: float = 1.0):
    """Population counts -> persons/km² in the native projection (count / pixel area × 1e6)."""
    area = ee.Image.pixelArea().reproject(native_proj)
    return counts.divide(area).multiply(1e6 * factor).setDefaultProjection(native_proj)


def _gee_worldpop(year: int):
    if not 2000 <= int(year) <= 2020:
        raise ValueError(f"WorldPop/GP/100m/pop covers 2000-2020, not {year}")
    col = (ee.ImageCollection("WorldPop/GP/100m/pop").filter(ee.Filter.eq("country", "IND"))
           .filter(ee.Filter.eq("year", year)).select("population"))
    native = col.first().projection()
    return col.mosaic().setDefaultProjection(native), native


def _gee_ghs_pop(epoch: int):
    image = ee.Image(f"JRC/GHSL/P2023A/GHS_POP/{epoch}").select("population_count")
    return image, image.projection()


def _gee_roi_total(image, roi_geom) -> float:
    """Population total inside the ROI in the image's native projection (area-weighted sum)."""
    result = image.reduceRegion(reducer=ee.Reducer.sum(), geometry=roi_geom, maxPixels=1e13, tileScale=4,
                                bestEffort=True)
    values = retry(lambda: result.getInfo(), tries=4, base_delay=3.0, max_delay=60.0)
    value = next(iter(values.values()), None) if values else None
    if value is None:
        raise RuntimeError("reduceRegion returned no population total")
    return float(value)


def _ghs_scale_factor(roi_geom, grid: GridSpec, worldpop_2020_grid: Optional[np.ndarray], roi_mask) -> tuple:
    """WorldPop-2020 / GHS-POP-2020 ROI totals, used to put GHS-POP projections on the WorldPop scale."""
    try:
        wp_img, _ = _gee_worldpop(2020)
        ghs_img, _ = _gee_ghs_pop(2020)
        factor = _gee_roi_total(wp_img, roi_geom) / _gee_roi_total(ghs_img, roi_geom)
        how = "reduceRegion(sum) over ROI, native projections"
    except Exception as exc:  # noqa: BLE001 - fall back to the already-aggregated grids
        log(f"reduceRegion population totals failed ({str(exc)[:160]}); using grid totals", "WARNING")
        ghs_img, ghs_proj = _gee_ghs_pop(2020)
        ghs_image = (_to_grid(_density_image(ghs_img, ghs_proj), _NATIVE_SCALE_M["ghs_pop"], grid)
                     .rename("pop").unmask(NODATA_FLOAT).toFloat())
        ghs_grid = fetch_grid(ghs_image, grid, ["pop"], CFG.GEE_TILE_PX,
                              _gee_cache_key("ghs2020", grid, image=ghs_image), np.float32)[0]
        if worldpop_2020_grid is None:
            raise RuntimeError("GHS-POP scaling needs the WorldPop 2020 grid (add 2020 to EPOCHS)") from exc
        factor = float(np.nansum(worldpop_2020_grid[roi_mask]) / np.nansum(ghs_grid[roi_mask]))
        how = "ratio of grid totals over ROI cells"
    if not 0.3 <= factor <= 3.0:
        log(f"GHS-POP scale factor {factor:.3f} looks implausible", "WARNING")
    return factor, how


def _gee_population_band(year: int, grid: GridSpec, ghs_factor: Optional[tuple]):
    """Population density (persons/km²) band + provenance."""
    if year <= 2020:
        counts, native = _gee_worldpop(year)
        prov = f"WorldPop/GP/100m/pop IND {year}, count/pixelArea, area-weighted mean -> persons/km2"
        return _to_grid(_density_image(counts, native), _NATIVE_SCALE_M["worldpop"], grid), prov
    epoch = int(min(2030, max(2025, 5 * round(year / 5))))
    counts, native = _gee_ghs_pop(epoch)
    factor, how = ghs_factor
    prov = (f"JRC/GHSL/P2023A/GHS_POP/{epoch} (projection) x {factor:.4f} (WorldPop2020/GHS2020 ROI totals, {how}), "
            "count/pixelArea, area-weighted mean -> persons/km2")
    return _to_grid(_density_image(counts, native, factor), _NATIVE_SCALE_M["ghs_pop"], grid), prov


def _gee_epoch_image(year: int, grid: GridSpec, ghs_factor: Optional[tuple]):
    """Multi-band per-epoch image (everything except terrain and, before 2014, night lights)."""
    images, prov = [], {}
    lst_imgs, lst_prov, lst_preview = _gee_lst_bands(year, grid)
    images += lst_imgs
    prov.update(lst_prov)
    spec_imgs, spec_prov = _gee_spectral_bands(year, grid)
    images += spec_imgs
    prov.update(spec_prov)
    pop_img, prov["pop_density"] = _gee_population_band(year, grid, ghs_factor)
    images.append(pop_img.rename("pop_density"))
    bands = ["lst_day_c", "lst_obs_count", "ndvi", "ndwi", "ndbi", "pop_density"]
    if year >= 2014:
        ntl_img, prov["ntl"], used_year = _gee_viirs_log(list(range(year, 2013, -1)), grid)
        images.append(ntl_img.rename("ntl"))
        bands.append("ntl")
        if used_year != year:
            log(f"VIIRS {year} unavailable, used {used_year}", "WARNING")
    image = ee.Image.cat(images).rename(bands).unmask(NODATA_FLOAT).toFloat()
    return image, bands, prov, lst_preview


_DMSP_SATURATED_DN = 63


def dmsp_intercalibration(src: np.ndarray, ref: np.ndarray, *, keep: float = 0.8, n_iter: int = 6,
                          min_pixels: int = 200):
    """Robust DN(src year) -> DN(reference year) intercalibration of two DMSP-OLS composites.

    DMSP-OLS has no on-board calibration: the gain of the same satellite drifts from year to year, so raw DN of
    different years are not comparable. Following the invariant-region idea (Elvidge et al. 2009; Wu et al. 2013), a
    quadratic DN_ref = a + b DN_src + c DN_src² is fitted on pixels lit and unsaturated in both years; each iteration
    keeps the ``keep`` share of pixels with the smallest absolute residuals, so pixels whose lighting really changed
    (urban growth) drop out and the fit follows the pseudo-invariant majority. A non-monotone quadratic falls back to
    a linear fit. Returns (coefficients highest power first, info) or (None, info) when too few pixels overlap.
    """
    src, ref = np.asarray(src, dtype=np.float64), np.asarray(ref, dtype=np.float64)
    ok = (np.isfinite(src) & np.isfinite(ref) & (src > 0) & (ref > 0)
          & (src < _DMSP_SATURATED_DN) & (ref < _DMSP_SATURATED_DN))
    info = {"n_pixels": int(ok.sum())}
    if ok.sum() < min_pixels:
        return None, info
    x, y = src[ok], ref[ok]
    sel = np.ones(x.size, dtype=bool)
    coef = np.polyfit(x, y, 2)
    for _ in range(n_iter):   # trimmed least squares: refit on the `keep` share of ALL pixels closest to the fit
        coef = np.polyfit(x[sel], y[sel], 2)
        resid = np.abs(y - np.polyval(coef, x))
        sel = resid <= np.quantile(resid, keep)
    grid = np.linspace(0.0, float(_DMSP_SATURATED_DN), 64)
    if np.any(np.diff(np.polyval(coef, grid)) <= 0):
        lin = np.polyfit(x[sel], y[sel], 1)
        coef = np.array([0.0, lin[0], lin[1]])
        info["fallback"] = "linear (quadratic not monotone on 0..63)"
    info.update({"n_invariant": int(sel.sum()), "coef": [float(c) for c in coef],
                 "r": float(np.corrcoef(np.polyval(coef, x[sel]), y[sel])[0, 1])})
    return coef, info


def apply_dmsp_intercalibration(dn: np.ndarray, coef) -> np.ndarray:
    """Map DN with the intercalibration polynomial; dark pixels (DN 0) stay 0, the result is clipped to 0..63."""
    dn = np.asarray(dn, dtype=np.float64)
    out = np.full(dn.shape, np.nan)
    valid = np.isfinite(dn)
    out[valid] = np.clip(np.polyval(coef, dn[valid]), 0.0, float(_DMSP_SATURATED_DN))
    out[valid & (dn <= 0)] = 0.0
    return out


def _gee_calibrated_dmsp(years: Sequence[int], grid: GridSpec, roi_mask: np.ndarray):
    """DMSP -> VIIRS-log1p calibration; returns ({year: (H,W) float32}, provenance, caveats).

    Step 1 intercalibrates each DMSP year to F18 2013 (``dmsp_intercalibration``); step 2 maps F18-2013 DN into VIIRS
    log1p space with an isotonic regression fitted on DMSP F18 2013 vs the first VIIRS year (2014: the monthly VIIRS
    composites start in January 2014, so the "overlap" is a one-year offset).
    """
    from sklearn.isotonic import IsotonicRegression

    dmsp_cal, cal_year = _gee_dmsp(2013, grid)
    viirs_cal, viirs_prov, viirs_year = _gee_viirs_log([2013, 2014, 2015], grid)
    images, bands = [dmsp_cal.rename("dmsp_cal"), viirs_cal.rename("viirs_cal")], ["dmsp_cal", "viirs_cal"]
    used = {}
    for year in years:
        img, used[year] = _gee_dmsp(year, grid)
        images.append(img.rename(f"dmsp_{year}"))
        bands.append(f"dmsp_{year}")
    stack_image = ee.Image.cat(images).rename(bands).unmask(NODATA_FLOAT).toFloat()
    stack = fetch_grid(stack_image, grid, bands, CFG.GEE_TILE_PX,
                       _gee_cache_key("ntl_calibration", grid, bands, image=stack_image), np.float32)
    x, y = stack[0][roi_mask], stack[1][roi_mask]
    ok = np.isfinite(x) & np.isfinite(y)
    if ok.sum() < 100:
        raise RuntimeError(f"only {ok.sum()} valid DMSP/VIIRS overlap pixels for isotonic calibration")
    iso = IsotonicRegression(increasing=True, out_of_bounds="clip").fit(x[ok], y[ok])
    r = np.corrcoef(iso.predict(x[ok]), y[ok])[0, 1]
    log(f"DMSP F18 {cal_year} -> VIIRS {viirs_year} isotonic calibration on {ok.sum():,} ROI cells (r={r:.3f})")
    out, prov, caveats = {}, {}, []
    for i, year in enumerate(years):
        dmsp = stack[2 + i].astype(np.float64)
        inter_txt = ""
        if used[year] != cal_year:
            coef, info = dmsp_intercalibration(dmsp[roi_mask], stack[0][roi_mask])
            if coef is None:
                log(f"DMSP F18 {used[year]} -> F18 {cal_year} intercalibration impossible ({info['n_pixels']} "
                    "overlapping lit pixels); using raw DN", "WARNING")
                inter_txt = f"NOT intercalibrated to F18{cal_year} (too few overlapping lit pixels), "
                caveats.append(f"DMSP F18 {used[year]} could not be intercalibrated to F18 {cal_year}: its night "
                               "lights carry the uncalibrated inter-year gain difference.")
            else:
                dmsp = apply_dmsp_intercalibration(dmsp, coef)
                log(f"DMSP F18 {used[year]} -> F18 {cal_year} intercalibration: DN' = "
                    f"{coef[0]:.5f} DN² + {coef[1]:.4f} DN + {coef[2]:.3f} on {info['n_invariant']:,} "
                    f"pseudo-invariant of {info['n_pixels']:,} lit pixels (r={info['r']:.3f})"
                    + (f" [{info['fallback']}]" if "fallback" in info else ""))
                inter_txt = (f"intercalibrated to F18{cal_year} (robust quadratic on {info['n_invariant']} "
                             f"pseudo-invariant pixels, r={info['r']:.3f}), ")
        mapped = np.full(dmsp.shape, np.nan, dtype=np.float32)
        valid = np.isfinite(dmsp)
        mapped[valid] = iso.predict(dmsp[valid]).astype(np.float32)
        out[year] = mapped
        prov[year] = (f"NOAA/DMSP-OLS/NIGHTTIME_LIGHTS F18{used[year]} stable_lights "
                      + (f"[nearest available F18 year to {year}] " if used[year] != year else "")
                      + f"{inter_txt}mapped to VIIRS log1p space by isotonic regression fitted on DMSP "
                      f"F18{cal_year} vs {viirs_prov} (first VIIRS year {viirs_year}; ROI, r={r:.3f})")
    return out, prov, caveats

## 1.7 Earth Engine: fine land cover
Each product is remapped to the six harmonised classes in its native grid, then brought to the fine grid:
majority (`mode`) aggregation when coarsening (10 m → 25 m), nearest neighbour when refining (30 m → 25 m).
GLC_FCS30D "annual" is an ImageCollection of ~5°×5° tiles with one band per year (`b1` = 2000 … `b23` = 2022);
the band is resolved from the tile's actual band list at run time.

In [ ]:
_GLC_ANNUAL = "projects/sat-io/open-datasets/GLC-FCS30D/annual"


def _remap(image, source: str):
    table = LC_REMAPS[source]
    return image.remap(list(table.keys()), list(table.values()), 0).rename("lc")


def _grid_ee_geom(grid: GridSpec):
    """Bounding polygon of a GridSpec in EPSG:4326 as an ee.Geometry."""
    x0, y0 = grid.x0, grid.y0
    x1, y1 = x0 + grid.width * grid.res, y0 - grid.height * grid.res
    lons, lats = _grid_xy_to_lonlat([x0, x1, x1, x0, x0], [y0, y0, y1, y1, y0], grid.crs)
    ring = [[float(lo), float(la)] for lo, la in zip(lons, lats)]
    return ee.Geometry.Polygon([ring], None, False)


def _gee_landcover_image(source: str, year: int, roi_geom, fine_grid: GridSpec):
    """Harmonised fine land cover for one epoch as a uint8 ee.Image (band 'lc'), plus provenance."""
    grid_geom = _grid_ee_geom(fine_grid)

    if source == "glc_fcs30d":
        col = ee.ImageCollection(_GLC_ANNUAL).filterBounds(grid_geom)
        band_names = []
        try:
            first_img = col.first()
            band_names = retry(lambda: first_img.bandNames().getInfo(), tries=2, base_delay=1.0, max_delay=5.0)
        except Exception as exc:
            log(f"GLC_FCS30D tiles query failed or empty ({exc!r}); falling back to alternative source", "WARNING")
            band_names = []

        if band_names:
            band_year = int(min(max(year, 2000), 2000 + len(band_names) - 1))
            band = f"b{band_year - 1999}" if f"b{band_year - 1999}" in band_names else band_names[min(band_year - 2000, len(band_names) - 1)]
            if len(band_names) != 23:
                log(f"GLC_FCS30D tiles expose {len(band_names)} bands (expected 23 for 2000-2022)", "WARNING")
            classes = _remap(col.select(band).mosaic(), source).setDefaultProjection(ee.Projection(CFG.CRS).atScale(30))
            prov = f"{_GLC_ANNUAL} band {band} (= {band_year}), remapped to harmonised classes"
            if band_year != year:
                prov += f" [nearest available year to {year}]"
        else:
            fallback_source = "dynamic_world" if year >= 2015 else "esa_worldcover"
            log(f"GLC_FCS30D unavailable for {year}; falling back to {fallback_source}", "WARNING")
            return _gee_landcover_image(fallback_source, year, roi_geom, fine_grid)
    elif source == "esa_worldcover":
        col = ee.ImageCollection("ESA/WorldCover/v100").filterBounds(grid_geom)
        image = col.select("Map").mosaic()
        classes = _remap(image, source).setDefaultProjection(ee.Projection(CFG.CRS).atScale(10))
        prov = f"ESA/WorldCover/v100 (2020) Map mosaic, remapped to harmonised classes"
    elif source == "dynamic_world":
        col = (ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1").filterBounds(grid_geom)
               .filterDate(f"{year}-01-01", f"{year + 1}-01-01").select("label"))
        mode = col.reduce(ee.Reducer.mode()).rename("label")
        classes = _remap(mode, source).setDefaultProjection(ee.Projection(CFG.CRS).atScale(10))
        prov = f"GOOGLE/DYNAMICWORLD/V1 label, annual mode {year}, remapped to harmonised classes"
    else:
        raise ValueError(f"unknown land-cover source {source!r}")
    native_scale = _NATIVE_SCALE_M[source]
    if native_scale < fine_grid.res * 0.999:
        ratio = (fine_grid.res / native_scale) ** 2
        classes = classes.reduceResolution(reducer=ee.Reducer.mode(), bestEffort=False,
                                           maxPixels=int(max(1024, math.ceil(ratio * 2.5))))
        prov += f", mode-aggregated to {fine_grid.res} m"
    else:
        prov += f", nearest-neighbour to {fine_grid.res} m"
    image = classes.reproject(crs=fine_grid.crs, crsTransform=fine_grid.transform_list()).unmask(0).toUint8()
    return image.rename("lc"), prov


def _default_lc_source(year: int) -> str:
    return CFG.LC_SOURCES.get(year) or ("glc_fcs30d" if year <= 2022 else "dynamic_world")

## 1.8 Earth Engine: orchestration
ROI → grid → district raster → static terrain → per-epoch stacks → DMSP calibration → fine land cover.

In [ ]:
_GEE_PREVIEW: dict = {}


def _gee_build_roi(res: int):
    """Try geoBoundaries, then GAUL, then the NCR outline. Returns a dict describing the ROI."""
    for source in ("geoboundaries", "gaul"):
        try:
            matched = _gee_match_roi(source)
        except Exception as exc:  # noqa: BLE001 - try the next source
            log(f"[{source}] boundary query failed: {type(exc).__name__}: {str(exc)[:200]}", "WARNING")
            continue
        if matched is None:
            continue
        roi_fc, districts, area = matched
        boundary = _gee_boundary_geojson(roi_fc, districts.keys())
        grid = GridSpec.from_lonlat_points(_geojson_lonlat_points(boundary), CFG.CRS, res)
        asset = _GEOBOUNDARIES_ADM2 if source == "geoboundaries" else _GAUL_ADM2
        return {"source": asset, "fc": roi_fc, "geom": roi_fc.geometry(), "boundary": boundary,
                "grid": grid, "area_km2": area, "local_districts": None}
    log("No administrative boundary source usable; falling back to the approximate NCR outline + "
        "nearest-HQ (Voronoi) districts", "WARNING")
    grid = _outline_grid(res)
    roi_mask, district_idx, boundary = _outline_districts(grid)
    geom = ee.Geometry.Polygon([list(p) for p in NCR_OUTLINE], None, False)
    return {"source": "approximate NCR outline (SPEC §1.2) with Voronoi districts", "fc": None, "geom": geom,
            "boundary": boundary, "grid": grid, "area_km2": float(roi_mask.sum() * grid.cell_area_km2),
            "local_districts": (roi_mask, district_idx)}


def _validate_epochs(epochs: Sequence[int]) -> list:
    """Check CFG.EPOCHS against each source's availability; raise for impossible years, return caveats otherwise.

    MODIS MOD11A2/MOD09A1 start in 2000; population needs WorldPop (2000-2020) or GHS-POP (2025/2030 epochs); DMSP
    F18 exists for 2010-2013 only, so a pre-VIIRS year outside that range uses the nearest F18 year (caveat).
    """
    caveats = []
    this_year = datetime.now().year
    for year in epochs:
        year = int(year)
        if year < 2000 or year > this_year:
            raise ValueError(f"epoch {year}: MODIS LST/reflectance (MOD11A2/MOD09A1) exist from 2000 to the present")
        if year > 2030:
            raise ValueError(f"epoch {year}: no population source (GHS-POP P2023A projections end in 2030)")
        if year < 2014 and not 2010 <= year <= 2013:
            msg = (f"epoch {year}: DMSP-OLS F18 covers 2010-2013 only; night lights use F18 "
                   f"{min(max(year, 2010), 2013)} [nearest available year], intercalibrated to 2013")
            log(msg, "WARNING")
            caveats.append(msg + ".")
        if 2020 < year < 2025:
            msg = (f"epoch {year}: population uses the GHS-POP {int(min(2030, max(2025, 5 * round(year / 5))))} "
                   "projection [nearest epoch]")
            log(msg, "WARNING")
            caveats.append(msg + ".")
    return caveats


def acquire_gee_dataset() -> dict:
    """Run the complete Earth Engine extraction; returns the section-1 globals as a dict (plus GRID, CAVEATS)."""
    caveats = _validate_epochs(CFG.EPOCHS)
    res = CFG.GRID_RES_M
    roi = _gee_build_roi(res)
    grid = roi["grid"]
    log(f"GEE grid: {grid.width}x{grid.height} cells @ {res} m, origin ({grid.x0:.0f}, {grid.y0:.0f}) {grid.crs}")
    if roi["local_districts"] is None:
        paint = ee.Image.constant(NODATA_DISTRICT).toInt16().paint(roi["fc"], "did").toInt16().rename("did")
        district_idx = fetch_grid(paint, grid, ["did"], CFG.GEE_TILE_PX,
                                  _gee_cache_key("districts", grid, roi["source"], image=paint), np.int16)[0]
        roi_mask = district_idx >= 0
    else:
        roi_mask, district_idx = roi["local_districts"]
        caveats.append("District boundaries are APPROXIMATE (nearest-HQ Voronoi partition of the SPEC §1.2 outline): "
                       "the administrative boundary layers (geoBoundaries, FAO GAUL) were unavailable, so district "
                       "names and per-district statistics describe approximate areas.")
    log(f"ROI: {int(roi_mask.sum()):,} cells = {roi_mask.sum() * grid.cell_area_km2:,.0f} km2 ({roi['source']})")

    sources: dict = {b: {} for b in RAW_BANDS}
    sources["land_cover"], sources["roi"] = {}, {}
    terrain_image = _gee_terrain_image(grid)
    terrain = fetch_grid(terrain_image, grid, TERRAIN_BANDS, CFG.GEE_TILE_PX,
                         _gee_cache_key("terrain", grid, image=terrain_image), np.float32)
    terrain_prov = f"USGS/SRTMGL1_003 via ee.Terrain.products, 30 m, aspect sin/cos before averaging, mean to {res} m"

    stacks, ghs_factor = {}, None
    for year in CFG.EPOCHS:
        if year > 2020 and ghs_factor is None:
            wp2020 = stacks.get(2020, {}).get("pop_density")
            ghs_factor = _ghs_scale_factor(roi["geom"], grid, wp2020, roi_mask)
            log(f"GHS-POP scale factor = {ghs_factor[0]:.4f} ({ghs_factor[1]})")
        with timer(f"01 GEE stack {year}"):
            image, bands, prov, lst_preview = _gee_epoch_image(year, grid, ghs_factor)
            arr = fetch_grid(image, grid, bands, CFG.GEE_TILE_PX,
                             _gee_cache_key(f"epoch{year}", grid, bands, image=image), np.float32)
        stacks[year] = {b: arr[i] for i, b in enumerate(bands)}
        for i, b in enumerate(TERRAIN_BANDS):
            stacks[year][b] = terrain[i].copy()
            sources[b][year] = terrain_prov
        for b, text in prov.items():
            sources[b][year] = text
        if year == CFG.EPOCHS[-1]:
            # masked native-resolution median clipped to the ROI (EE pyramids it natively; no -9999 fill)
            _GEE_PREVIEW["lst"] = lst_preview.clip(roi["geom"])
        del arr

    pre_viirs = [y for y in CFG.EPOCHS if y < 2014]
    if pre_viirs:
        with timer("01 GEE DMSP calibration"):
            mapped, prov, dmsp_caveats = _gee_calibrated_dmsp(pre_viirs, grid, roi_mask)
        caveats.extend(dmsp_caveats)
        for year in pre_viirs:
            stacks[year]["ntl"] = mapped[year]
            sources["ntl"][year] = prov[year]

    for year in CFG.EPOCHS:
        for b in RAW_BANDS:
            band = stacks[year][b].astype(np.float32, copy=False)
            band[~roi_mask] = np.nan
            stacks[year][b] = band
        stacks[year] = {b: stacks[year][b] for b in RAW_BANDS}
        sources["roi"][year] = roi["source"]

    fine_res = CFG.fine_res()
    fine_grid = grid.fine(fine_res)
    f = grid.res // fine_res
    lc_fine = {}
    for year in CFG.EPOCHS:
        source = _default_lc_source(year)
        with timer(f"01 GEE land cover {year} ({source})"):
            image, prov = _gee_landcover_image(source, year, roi["geom"], fine_grid)
            lc = fetch_grid(image, fine_grid, ["lc"], CFG.GEE_FINE_TILE_PX,
                            _gee_cache_key(f"lc{year}_{source}", fine_grid, image=image), np.uint8)[0]
        _mask_fine_inplace(lc, roi_mask, f)
        lc_fine[year] = lc
        sources["land_cover"][year] = prov
        shares = np.bincount(lc.ravel(), minlength=7)[1:] / max(1, int((lc > 0).sum()))
        log(f"  LC {year}: " + ", ".join(f"{LC_CLASS_NAMES[c].split('/')[0]} {100 * s:.1f}%"
                                          for c, s in zip(range(1, 7), shares)))
    if roi["fc"] is not None:
        _GEE_PREVIEW["roi"] = roi["fc"]
    return {"GRID": grid, "ROI_MASK": roi_mask, "DISTRICT_IDX": district_idx.astype(np.int16),
            "BOUNDARY_GEOJSON": roi["boundary"], "RAW_STACKS": stacks, "LC_FINE": lc_fine,
            "SYNTHETIC_TRUTH": None, "DATA_SOURCES": sources, "CAVEATS": caveats,
            "BOUNDARY_APPROXIMATE": roi["local_districts"] is not None}

## 1.9 Synthetic twin (fallback / offline development)
A physically-plausible stand-in with the same grid, globals and dtypes as the Earth Engine path:

* **ROI & districts** — the approximate NCR outline, partitioned into the 25 districts by nearest
  headquarters (a Voronoi partition in UTM metres, identical for the raster and the GeoJSON).
* **Geography** — a gently N→SE sloping plain (180–260 m) plus a Gaussian Aravalli ridge whose relief grows
  from ~300 m at the Delhi ridge to ~650 m near Alwar; slope/aspect from finite differences.
* **Urbanisation** — Gaussian kernels at 20 real urban centres combined as a probabilistic union; radii grow
  12–18 % per 5-year epoch (Gurugram, Noida, Greater Noida and Manesar fastest). Built-up area only grows.
* **Land cover** (fine grid) — rivers (Yamuna, Ganga) as narrow channels with sand bars and riparian
  vegetation, random ponds, ridge forest and rocky foothills, arid barren land in SW Haryana/Rajasthan,
  parks inside cities, villages, cropland elsewhere; texture from spatially-correlated noise that is shared
  by all epochs (temporal coherence).
* **Predictors** — NDVI/NDWI/NDBI as mixtures of class signatures (pre-monsoon: harvested wheat is bare-ish,
  the western-UP sugarcane belt stays green) + noise; log1p night lights and population from urban intensity.
* **LST** — an explicitly *planted* non-linear response (saturating NDVI cooling, exponential water-cooling
  reach, impervious power law and an impervious × low-NDVI interaction) plus spatially-correlated and white
  noise; recorded in `SYNTHETIC_TRUTH` so section 4 can check whether SHAP recovers it.

In [ ]:
_SYN_URBAN_CENTRES = [
    # name, lon, lat, amplitude, 2010 radius (km), radius growth per 5-year epoch
    ("Delhi", 77.20, 28.63, 1.00, 13.0, 0.12), ("Gurugram", 77.03, 28.46, 0.90, 6.0, 0.18),
    ("Noida", 77.36, 28.57, 0.90, 5.0, 0.18), ("Greater Noida", 77.50, 28.47, 0.70, 4.0, 0.18),
    ("Ghaziabad", 77.44, 28.67, 0.85, 5.5, 0.14), ("Faridabad", 77.31, 28.40, 0.85, 5.5, 0.13),
    ("Meerut", 77.71, 28.98, 0.80, 5.0, 0.12), ("Sonipat", 77.02, 28.99, 0.65, 3.0, 0.14),
    ("Panipat", 76.97, 29.39, 0.70, 3.5, 0.13), ("Karnal", 76.99, 29.69, 0.65, 3.5, 0.12),
    ("Rohtak", 76.61, 28.90, 0.70, 3.5, 0.12), ("Bahadurgarh", 76.92, 28.69, 0.60, 2.5, 0.15),
    ("Manesar", 76.93, 28.36, 0.60, 2.5, 0.18), ("Rewari", 76.62, 28.19, 0.60, 2.5, 0.12),
    ("Alwar", 76.60, 27.55, 0.70, 3.5, 0.12), ("Bharatpur", 77.49, 27.22, 0.65, 3.0, 0.12),
    ("Bulandshahr", 77.85, 28.40, 0.60, 2.5, 0.12), ("Muzaffarnagar", 77.70, 29.47, 0.65, 3.0, 0.12),
    ("Hapur", 77.78, 28.73, 0.60, 2.5, 0.13), ("Bhiwani", 76.13, 28.79, 0.60, 3.0, 0.12),
]
_SYN_LST_BASE_C = 43.0
_SYN_EPOCH_OFFSETS = {2010: 0.4, 2015: -0.3, 2020: -0.9, 2025: 0.7}
# The planted NDVI saturation must lie INSIDE the simulated NDVI distribution (pre-monsoon NDVI of this twin has
# median ~0.21 and p99 ~0.42): a threshold above the data support cannot be recovered by any estimator, which would
# make the planted-vs-recovered check in section 4 meaningless. 0.28 sits near the 85th percentile.
_SYN_NDVI_SAT, _SYN_WATER_SCALE, _SYN_IMP_EXP = 0.28, 0.08, 0.8
_SYN_COEF = {"frac_barren": 4.0, "frac_cropland": 1.2, "frac_impervious_pow": 2.5, "impervious_x_lowndvi": 1.5,
             "ndvi_cooling": -5.0, "water_cooling": -3.5, "ndbi": 1.5, "elevation_lapse_per_m": -0.006,
             "ntl": 0.25}
# missing_fraction: LST gaps (cells dropped from DF); spectral_missing_fraction: cloud-masked MOD09A1 gaps in
# NDVI/NDWI/NDBI (kept as NaN predictors, exercising the NaN-handling paths of every model and export).
_SYN_NOISE = {"spatial_sd": 0.7, "spatial_corr_km": 8.0, "white_sd": 0.4, "missing_fraction": 0.005,
              "spectral_missing_fraction": 0.003}
_SYN_LST_FORMULA = (
    f"lst = base(43.0) + offset[year] + 4.0*frac_barren + 1.2*frac_cropland + 2.5*frac_impervious**0.8 "
    f"+ 1.5*frac_impervious*(ndvi<0.2) - 5.0*clip(ndvi,0,{_SYN_NDVI_SAT})/{_SYN_NDVI_SAT} "
    f"- 3.5*(1-exp(-frac_water/0.08)) + 1.5*ndbi - 0.006*(elevation-220) + 0.25*ntl "
    f"+ N_spatial(sd 0.7, ~8 km) + N_white(sd 0.4)")
# class signatures (pre-monsoon) of the spectral indices, order: built, forest, water, crop, barren, other
_SYN_SIGNATURES = {"ndvi": (0.10, 0.52, -0.12, 0.22, 0.10, 0.33),
                   "ndwi": (-0.10, -0.42, 0.30, -0.22, -0.12, -0.33),
                   "ndbi": (0.12, -0.22, -0.40, 0.06, 0.16, -0.08)}


def _syn_rng(seed: int, *tags: int) -> np.random.Generator:
    """Independent, reproducible random stream per (seed, tags) - stable regardless of call order."""
    return np.random.default_rng(np.random.SeedSequence([int(seed), *[int(t) for t in tags]]))


def _correlated_noise(shape, sigma_px: float, rng: np.random.Generator) -> np.ndarray:
    """Zero-mean, unit-variance Gaussian random field with correlation length ≈ ``sigma_px`` pixels.

    Large kernels are synthesised on a coarser lattice and bilinearly upsampled (identical statistics
    at a fraction of the cost of filtering a full-resolution field).
    """
    h, w = shape
    if sigma_px < 0.5:
        return rng.standard_normal(shape, dtype=np.float32)
    k = max(1, int(sigma_px // 3))
    small = rng.standard_normal((-(-h // k) + 1, -(-w // k) + 1), dtype=np.float32)
    small = ndimage.gaussian_filter(small, sigma_px / k, mode="reflect")
    field = _upsample(small, k)[:h, :w]
    field -= field.mean()
    field /= field.std() + 1e-12
    return field


def _interp_axis(a: np.ndarray, f: int, axis: int) -> np.ndarray:
    """Linear interpolation of cell-centred samples onto an f-times finer grid along one axis (edges clamped)."""
    n = a.shape[axis]
    pos = np.clip((np.arange(n * f) + 0.5) / f - 0.5, 0.0, n - 1)
    i0 = np.floor(pos).astype(np.intp)
    i1 = np.minimum(i0 + 1, n - 1)
    shape = [1] * a.ndim
    shape[axis] = -1
    weight = (pos - i0).astype(np.float32).reshape(shape)
    lo = np.take(a, i0, axis=axis)
    return lo + (np.take(a, i1, axis=axis) - lo) * weight


def _upsample(field: np.ndarray, f: int) -> np.ndarray:
    """Bilinear upsampling of a cell-centred field by an integer factor (pixel-edge aligned, float32).

    Equivalent to ``ndimage.zoom(order=1, grid_mode=True, mode="nearest")`` but separable and ~5x faster.
    """
    field = field.astype(np.float32, copy=f == 1)
    if f == 1:
        return field
    return _interp_axis(_interp_axis(field, f, 0), f, 1)


def _polyline_distance_position(xs: np.ndarray, ys: np.ndarray, line_xy: np.ndarray, chunk_rows: int = 256):
    """Distance (m) from every grid point (xs[j], ys[i]) to a polyline and the along-line position (0..1).

    Works row-chunk by row-chunk in float32 on coordinates relative to the grid origin (precision ~cm).
    """
    ox, oy = float(xs[0]), float(ys[0])
    xr = (xs - ox).astype(np.float32)[None, :]
    a = (line_xy[:-1] - (ox, oy)).astype(np.float32)
    b = (line_xy[1:] - (ox, oy)).astype(np.float32)
    seg_len = np.hypot(*(b - a).T)
    cum = np.concatenate([[0.0], np.cumsum(seg_len)]) / max(seg_len.sum(), 1e-9)
    dist = np.empty((len(ys), len(xs)), np.float32)
    pos = np.empty_like(dist)
    for r0 in range(0, len(ys), chunk_rows):
        yr = (ys[r0:r0 + chunk_rows] - oy).astype(np.float32)[:, None]
        best = np.full((len(yr), len(xs)), np.inf, np.float32)
        best_pos = np.zeros_like(best)
        for i in range(len(a)):
            (ax, ay), (dx, dy) = a[i], b[i] - a[i]
            t = np.clip(((xr - ax) * dx + (yr - ay) * dy) / max(dx * dx + dy * dy, 1e-9), 0.0, 1.0)
            d2 = (xr - ax - t * dx) ** 2 + (yr - ay - t * dy) ** 2
            closer = d2 < best
            best = np.where(closer, d2, best)
            best_pos = np.where(closer, cum[i] + t * (cum[i + 1] - cum[i]), best_pos)
        dist[r0:r0 + chunk_rows] = np.sqrt(best)
        pos[r0:r0 + chunk_rows] = best_pos
    return dist, pos


def _lonlat_array_to_xy(points) -> np.ndarray:
    pts = np.asarray(points, dtype=np.float64)
    x, y = _grid_lonlat_to_xy(pts[:, 0], pts[:, 1], CFG.CRS)
    return np.column_stack([x, y])


def _convex_hull(points: np.ndarray) -> np.ndarray:
    """Andrew's monotone chain convex hull (closed ring) - fallback when shapely is unavailable."""
    pts = np.unique(points, axis=0)
    if len(pts) < 3:
        return np.vstack([pts, pts[:1]])

    def _turn(o, a, b):  # z-component of (a - o) x (b - o): > 0 for a counter-clockwise turn
        return (a[0] - o[0]) * (b[1] - o[1]) - (a[1] - o[1]) * (b[0] - o[0])

    def _half(seq):
        hull = []
        for p in seq:
            while len(hull) >= 2 and _turn(hull[-2], hull[-1], p) <= 0:
                hull.pop()
            hull.append(p)
        return hull

    lower, upper = _half(pts), _half(pts[::-1])
    ring = np.array(lower[:-1] + upper[:-1])
    return np.vstack([ring, ring[:1]])


def _xy_ring_to_lonlat(ring_xy: np.ndarray) -> list:
    lon, lat = _grid_xy_to_lonlat(ring_xy[:, 0], ring_xy[:, 1], CFG.CRS)
    return [[round(float(a), 5), round(float(b), 5)] for a, b in zip(lon, lat)]


def _voronoi_boundary_geojson(hq_xy: np.ndarray, outline_xy: np.ndarray, district_idx, grid: GridSpec) -> dict:
    """District polygons = Voronoi cells of the HQs clipped to the outline, simplified to 200 m (EPSG:4326)."""
    features = []
    if shapely is not None:
        from shapely.geometry import MultiPoint, Point, Polygon
        from shapely.ops import voronoi_diagram

        outline = Polygon(outline_xy).buffer(0)
        cells = voronoi_diagram(MultiPoint([tuple(p) for p in hq_xy]), envelope=outline.buffer(100_000))
        for d, hq in zip(NCR_DISTRICTS, hq_xy):
            region = next(c for c in cells.geoms if c.covers(Point(hq)))
            geom = region.intersection(outline).simplify(200, preserve_topology=True)
            polys = [geom] if geom.geom_type == "Polygon" else [g for g in getattr(geom, "geoms", []) if g.geom_type == "Polygon"]
            coords = [[_xy_ring_to_lonlat(np.asarray(p.exterior.coords))]
                      + [_xy_ring_to_lonlat(np.asarray(r.coords)) for r in p.interiors] for p in polys]
            geometry = ({"type": "Polygon", "coordinates": coords[0]} if len(coords) == 1
                        else {"type": "MultiPolygon", "coordinates": coords})
            features.append({"type": "Feature", "geometry": geometry,
                             "properties": {"id": d["id"], "name": d["name"], "state": d["state"]}})
        return {"type": "FeatureCollection", "features": features}
    log("shapely unavailable: district outlines approximated by convex hulls of their cells", "WARNING")
    x, y = grid.cell_centers_xy()
    for d in NCR_DISTRICTS:
        sel = district_idx == d["id"]
        ring = _convex_hull(np.column_stack([x[sel], y[sel]])) if sel.any() else np.repeat(hq_xy[d["id"]][None], 4, 0)
        features.append({"type": "Feature", "geometry": {"type": "Polygon", "coordinates": [_xy_ring_to_lonlat(ring)]},
                         "properties": {"id": d["id"], "name": d["name"], "state": d["state"]}})
    return {"type": "FeatureCollection", "features": features}


def _outline_districts(grid: GridSpec):
    """ROI mask, nearest-HQ district raster and matching Voronoi GeoJSON for the approximate NCR outline."""
    from matplotlib.path import Path as _MplPath

    x, y = grid.cell_centers_xy()
    outline_xy = _lonlat_array_to_xy(_densify_ring(NCR_OUTLINE, 10))
    roi_mask = _MplPath(outline_xy).contains_points(np.column_stack([x.ravel(), y.ravel()])).reshape(grid.shape)
    hq_xy = _lonlat_array_to_xy([d["hq"] for d in NCR_DISTRICTS])
    d2 = (x[..., None] - hq_xy[:, 0]) ** 2 + (y[..., None] - hq_xy[:, 1]) ** 2
    district_idx = np.argmin(d2, axis=-1).astype(np.int16)
    district_idx[~roi_mask] = NODATA_DISTRICT
    boundary = _voronoi_boundary_geojson(hq_xy, outline_xy, district_idx, grid)
    return roi_mask, district_idx, boundary


def _syn_geography(grid: GridSpec, lon: np.ndarray, lat: np.ndarray, seed: int) -> dict:
    """Static coarse fields: elevation, slope, aspect sin/cos, ridge proximity, aridity."""
    xs, ys = grid.x_centers(), grid.y_centers()
    ridge_d, ridge_t = _polyline_distance_position(xs, ys, _lonlat_array_to_xy(ARAVALLI))
    relief = 80.0 + 470.0 * ridge_t ** 1.3                       # ~300 m a.s.l. at Delhi ridge, ~650 m near Alwar
    width = 3000.0 + 7000.0 * ridge_t
    undulation = 0.75 + 0.25 * np.tanh(_correlated_noise(grid.shape, 6000.0 / grid.res, _syn_rng(seed, 1)))
    ridge = relief * undulation * np.exp(-0.5 * (ridge_d / width) ** 2)
    plain = np.clip(205.0 + 22.0 * (lat - 28.3) - 12.0 * (lon - 77.2), 180.0, 260.0)
    elevation = plain + ridge + 2.0 * _correlated_noise(grid.shape, 1.5, _syn_rng(seed, 2))
    dz_row, dz_col = np.gradient(elevation, grid.res)
    gx, gy_north = dz_col, -dz_row                               # rows run southwards
    slope = np.degrees(np.arctan(np.hypot(gx, gy_north)))
    aspect = np.arctan2(-gx, -gy_north)                          # direction the slope faces, clockwise from north
    ridge_prox = np.exp(-0.5 * (ridge_d / (1.3 * width)) ** 2) * np.clip(relief / 300.0, 0.3, 1.0)
    aridity = 1.0 / (1.0 + np.exp(-((76.5 - lon) + (28.3 - lat)) / 0.35))
    return {"elevation": elevation.astype(np.float32), "slope": slope.astype(np.float32),
            "aspect_sin": np.sin(aspect).astype(np.float32), "aspect_cos": np.cos(aspect).astype(np.float32),
            "ridge_prox": ridge_prox.astype(np.float32), "aridity": aridity.astype(np.float32)}


def _syn_urban_intensity(grid: GridSpec, year: int) -> np.ndarray:
    """Probabilistic union of Gaussian urban kernels with epoch-dependent radii (0..1)."""
    x, y = grid.cell_centers_xy()
    centres = _lonlat_array_to_xy([(c[1], c[2]) for c in _SYN_URBAN_CENTRES])
    steps = (year - 2010) / 5.0
    not_urban = np.ones(grid.shape, np.float64)
    for (name, _, _, amp, r_km, growth), (cx, cy) in zip(_SYN_URBAN_CENTRES, centres):
        radius = r_km * 1000.0 * (1.0 + growth) ** steps
        not_urban *= 1.0 - amp * np.exp(-0.5 * ((x - cx) ** 2 + (y - cy) ** 2) / radius**2)
    return (1.0 - not_urban).astype(np.float32)


def _polyline_distance_capped(xs: np.ndarray, ys: np.ndarray, line_xy: np.ndarray, cap_m: float) -> np.ndarray:
    """Distance (m) to a polyline, exact up to ``cap_m`` and clipped to ``cap_m`` beyond.

    Only the window around each segment (its bounding box grown by ``cap_m``) is evaluated, so the cost scales
    with the river length rather than with the size of the fine grid (~100x faster than a full transform).
    ``xs`` must increase and ``ys`` decrease (north-up grid).
    """
    dist = np.full((len(ys), len(xs)), cap_m, np.float32)
    neg_ys = -ys
    for (ax, ay), (bx, by) in zip(line_xy[:-1], line_xy[1:]):
        c0 = int(np.searchsorted(xs, min(ax, bx) - cap_m))
        c1 = int(np.searchsorted(xs, max(ax, bx) + cap_m, side="right"))
        r0 = int(np.searchsorted(neg_ys, -(max(ay, by) + cap_m)))
        r1 = int(np.searchsorted(neg_ys, -(min(ay, by) - cap_m), side="right"))
        if c0 >= c1 or r0 >= r1:
            continue
        xw = (xs[c0:c1] - ax).astype(np.float32)[None, :]
        yw = (ys[r0:r1] - ay).astype(np.float32)[:, None]
        dx, dy = float(bx - ax), float(by - ay)
        t = np.clip((xw * dx + yw * dy) / max(dx * dx + dy * dy, 1e-9), 0.0, 1.0)
        window = dist[r0:r1, c0:c1]
        np.minimum(window, np.sqrt((xw - t * dx) ** 2 + (yw - t * dy) ** 2), out=window)
    return dist


def _syn_lakes(grid: GridSpec, fine: GridSpec, roi_mask: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    """Random elliptical ponds / jheels (~1 per 150 km² of ROI, 150-700 m semi-axes) on the fine grid."""
    f = grid.res // fine.res
    lakes = np.zeros(fine.shape, bool)
    roi_cells = np.flatnonzero(roi_mask)
    n_lakes = max(8, int(roi_cells.size * grid.cell_area_km2 / 150.0))
    for cell in rng.choice(roi_cells, size=min(n_lakes, roi_cells.size), replace=False):
        r, c = divmod(int(cell), grid.width)
        cy, cx = (r + rng.random()) * f, (c + rng.random()) * f             # fine-pixel coordinates
        ry, rx = rng.uniform(150, 700) / fine.res, rng.uniform(150, 700) / fine.res
        r0, r1 = max(0, int(cy - ry) - 1), min(fine.height, int(cy + ry) + 2)
        c0, c1 = max(0, int(cx - rx) - 1), min(fine.width, int(cx + rx) + 2)
        yy, xx = np.mgrid[r0:r1, c0:c1]
        lakes[r0:r1, c0:c1] |= ((yy + 0.5 - cy) / max(ry, 0.5)) ** 2 + ((xx + 0.5 - cx) / max(rx, 0.5)) ** 2 <= 1.0
    return lakes


def _syn_static_fine(grid: GridSpec, fine: GridSpec, geo: dict, roi_mask: np.ndarray, seed: int) -> dict:
    """Epoch-independent fine-resolution layers (shared noise => temporally coherent land cover).

    Returns the "rural" base map (cropland / other vegetation / barren / forest, before urbanisation and water),
    the static part of the built-up score, park/lawn candidates, village seeds and the water mask. Rules are
    applied in order, later rules overriding earlier ones.
    """
    fr, f = fine.res, grid.res // fine.res
    xs, ys = fine.x_centers(), fine.y_centers()
    cap = 4000.0
    meander = 120.0 * _correlated_noise(fine.shape, 800.0 / fr, _syn_rng(seed, 10))
    d_yam = _polyline_distance_capped(xs, ys, _lonlat_array_to_xy(YAMUNA), cap)
    d_gan = _polyline_distance_capped(xs, ys, _lonlat_array_to_xy(GANGA), cap)
    river_rel = np.minimum((d_yam + meander) / 170.0, (d_gan + meander) / 320.0)   # in channel half-widths
    river_m = np.minimum(d_yam, d_gan)
    del d_yam, d_gan, meander

    tex = _correlated_noise(fine.shape, 250.0 / fr, _syn_rng(seed, 12))        # field-scale texture
    patch = _correlated_noise(fine.shape, 1500.0 / fr, _syn_rng(seed, 13))     # landscape-scale patches
    ridge = _upsample(geo["ridge_prox"], f)
    base = np.full(fine.shape, LC_CROPLAND, np.uint8)
    base[0.6 * patch + 0.4 * tex > 0.72] = LC_OTHER_VEG                                  # grass / fallow / shrub
    base[(river_m < 3000) & (0.5 * patch + 0.5 * tex > 0.6)] = LC_OTHER_VEG             # floodplain grassland
    base[_upsample(geo["aridity"], f) + 0.45 * patch + 0.2 * tex > 1.25] = LC_BARREN    # SW Haryana / Rajasthan
    base[(ridge > 0.25) & (ridge + 0.35 * patch > 0.85)] = LC_BARREN                    # rocky foothills
    base[(river_rel > 1.0) & (river_rel < 1.0 + 700.0 / 170.0) & (tex > 0.3)] = LC_BARREN   # sand bars / riverbed
    base[ridge + 0.3 * tex > 0.75] = LC_FOREST                                          # Aravalli scrub forest
    base[(river_m < 1500) & (tex > 1.0)] = LC_FOREST                                    # riparian trees
    base[tex > 2.1] = LC_FOREST                                                          # groves / farm forestry
    del ridge

    park = _correlated_noise(fine.shape, 150.0 / fr, _syn_rng(seed, 14))
    static = {
        "base": base,
        "built_noise": (0.22 * tex + 0.10 * patch).astype(np.float32),
        "village": _correlated_noise(fine.shape, 120.0 / fr, _syn_rng(seed, 15)),
        "park": park > 1.9,
        "lawn": (park > 1.5) & (park <= 1.9),
        "water": (river_rel < 1.0) | _syn_lakes(grid, fine, roi_mask, _syn_rng(seed, 11)),
    }
    del tex, patch, park, river_rel, river_m
    return static


def _syn_landcover(static: dict, urban_fine: np.ndarray, roi_mask: np.ndarray, f: int,
                   previous_built: Optional[np.ndarray], epoch_step: float) -> np.ndarray:
    """Epoch land cover (uint8 1..6, 0 outside ROI): rural base + urbanisation + urban green + water."""
    urban_core = urban_fine > 0.35
    if previous_built is not None:
        urban_core &= ~previous_built                                          # urbanisation is irreversible
    parks = urban_core & static["park"]
    lawns = urban_core & static["lawn"]
    built = (urban_fine + static["built_noise"] > 0.45) | (static["village"] > 2.35 - 0.05 * epoch_step)
    built &= ~(parks | lawns)
    if previous_built is not None:
        built |= previous_built
    lc = static["base"].copy()
    lc[built] = LC_BUILT
    lc[lawns] = LC_OTHER_VEG
    lc[parks] = LC_FOREST
    lc[static["water"]] = LC_WATER
    _mask_fine_inplace(lc, roi_mask, f)
    return lc


def _lc_block_fractions(lc: np.ndarray, f: int) -> np.ndarray:
    """(H, W, 6) fractions of classes 1..6 inside each f×f block (0 where the block is nodata)."""
    h, w = lc.shape[0] // f, lc.shape[1] // f
    block_of_row = (np.arange(lc.shape[0]) // f) * w
    block_of_col = np.arange(lc.shape[1]) // f
    key = (block_of_row[:, None] + block_of_col[None, :]) * 7 + lc          # (block id, class) -> one integer
    counts = np.bincount(key.ravel(), minlength=h * w * 7).reshape(h, w, 7)
    return (counts[..., 1:] / float(f * f)).astype(np.float32)


def _syn_epoch_stack(year: int, step: float, fracs: np.ndarray, urban: np.ndarray, geo: dict, lon, lat,
                     roi_mask: np.ndarray, static_lst_noise: np.ndarray, grid: GridSpec, seed: int) -> dict:
    """All 1 km bands for one epoch from land-cover fractions, urban intensity and geography."""
    shape = grid.shape
    rng = _syn_rng(seed, 100, year)
    corr = lambda km, tag: _correlated_noise(shape, km * 1000.0 / grid.res, _syn_rng(seed, 200 + tag, year))  # noqa: E731
    imp, forest, water, crop, barren, other = (fracs[..., i] for i in range(6))

    signatures = {k: np.tensordot(fracs, np.asarray(v, np.float32), axes=([-1], [0])) for k, v in _SYN_SIGNATURES.items()}
    sugarcane = 1.0 / (1.0 + np.exp(-((lon - 77.45) / 0.12))) * 1.0 / (1.0 + np.exp(-((lat - 28.75) / 0.12)))
    crop_green = crop * (0.12 * sugarcane + 0.04 * corr(15, 1))                 # green sugarcane belt, irrigation
    ndvi = signatures["ndvi"] + crop_green + 0.035 * corr(5, 2) + 0.015 * rng.standard_normal(shape) - 0.01 * step
    ndwi = signatures["ndwi"] - 0.6 * crop_green + 0.03 * corr(5, 3) + 0.015 * rng.standard_normal(shape)
    ndbi = signatures["ndbi"] - 0.5 * crop_green + 0.03 * corr(5, 4) + 0.015 * rng.standard_normal(shape)
    ndvi, ndwi, ndbi = (np.clip(a, -1.0, 1.0) for a in (ndvi, ndwi, ndbi))

    radiance = (0.25 + 70.0 * urban ** 1.4 + 12.0 * imp) * (1.0 + 0.1 * step) * np.exp(0.15 * corr(10, 5))
    ntl = np.clip(np.log1p(radiance) + 0.05 * rng.standard_normal(shape), 0.0, 5.2)
    rural = 380.0 * 1.05 ** step * np.exp(0.3 * corr(12, 6))
    pop = np.clip((rural + 32000.0 * urban ** 3.0 + 1500.0 * imp) * (1.0 - water), 0.0, 40000.0)

    epoch_noise = corr(_SYN_NOISE["spatial_corr_km"], 7)
    spatial = _SYN_NOISE["spatial_sd"] * (np.sqrt(0.6) * static_lst_noise + np.sqrt(0.4) * epoch_noise)
    lst = (_SYN_LST_BASE_C + _SYN_EPOCH_OFFSETS.get(year, 0.0)
           + _SYN_COEF["frac_barren"] * barren + _SYN_COEF["frac_cropland"] * crop
           + _SYN_COEF["frac_impervious_pow"] * imp ** _SYN_IMP_EXP
           + _SYN_COEF["impervious_x_lowndvi"] * imp * (ndvi < 0.2)
           + _SYN_COEF["ndvi_cooling"] * np.clip(ndvi, 0.0, _SYN_NDVI_SAT) / _SYN_NDVI_SAT
           + _SYN_COEF["water_cooling"] * (1.0 - np.exp(-water / _SYN_WATER_SCALE))
           + _SYN_COEF["ndbi"] * ndbi + _SYN_COEF["elevation_lapse_per_m"] * (geo["elevation"] - 220.0)
           + _SYN_COEF["ntl"] * ntl + spatial + _SYN_NOISE["white_sd"] * rng.standard_normal(shape))
    obs_count = np.clip(np.rint(10.2 + 1.3 * corr(20, 8) + 0.7 * rng.standard_normal(shape)), 5, 12)

    missing = roi_mask & (rng.random(shape) < _SYN_NOISE["missing_fraction"])
    lst[missing] = np.nan
    obs_count[missing] = 0.0
    # Cloud-masked reflectance gaps hit the three indices together (same MOD09A1 composite); the planted LST above
    # was computed from the true indices, as a real surface would not care whether the satellite saw it.
    spectral_gap = roi_mask & (rng.random(shape) < _SYN_NOISE["spectral_missing_fraction"])
    for band in (ndvi, ndwi, ndbi):
        band[spectral_gap] = np.nan
    stack = {"lst_day_c": lst, "lst_obs_count": obs_count, "ndvi": ndvi, "ndwi": ndwi, "ndbi": ndbi,
             "elevation": geo["elevation"], "slope": geo["slope"], "aspect_sin": geo["aspect_sin"],
             "aspect_cos": geo["aspect_cos"], "ntl": ntl, "pop_density": pop}
    out = {}
    for band in RAW_BANDS:
        arr = np.array(stack[band], dtype=np.float32, copy=True)
        arr[~roi_mask] = np.nan
        out[band] = arr
    return out


def _synthetic_truth(epochs: Sequence[int], seed: int, lst_means: dict) -> dict:
    """Machine-readable description of the planted LST response (for SHAP-recovery checks in section 4)."""
    return {
        "generator": "delhi-ncr-synthetic-v1", "seed": int(seed), "formula": _SYN_LST_FORMULA,
        "lst_base_c": _SYN_LST_BASE_C,
        "epoch_offsets": {int(y): float(_SYN_EPOCH_OFFSETS.get(y, 0.0)) for y in epochs},
        "epoch_roi_mean_lst": {int(y): float(v) for y, v in lst_means.items()},
        "coefficients": dict(_SYN_COEF),
        "ndvi_saturation": _SYN_NDVI_SAT,
        "water_cooling_scale": _SYN_WATER_SCALE, "water_scale": _SYN_WATER_SCALE,
        "impervious_exponent": _SYN_IMP_EXP,
        "interaction": {"features": ["frac_impervious", "ndvi"], "coefficient": 1.5, "ndvi_below": 0.2,
                        "description": "extra warming of impervious surface where NDVI < 0.2"},
        # Values are expressed in the SPEC 4.4 threshold definitions so section 4 can compare like with like:
        # * ndvi: piecewise-linear cooling, slope exactly 0 above the saturation -> saturation = _SYN_NDVI_SAT;
        # * frac_water: -3.5 (1 - exp(-w / s)) has slope (3.5 / s) exp(-w / s); SPEC "saturation" is where |slope|
        #   drops below 10 % of its maximum (at w = 0), i.e. w = s ln 10 (95 % of the effect is reached at 3 s).
        "planted_thresholds": {
            "ndvi": {"saturation": _SYN_NDVI_SAT, "direction": "cooling"},
            "frac_water": {"saturation": round(_SYN_WATER_SCALE * math.log(10.0), 4), "e_folding": _SYN_WATER_SCALE,
                           "effect_95pct_at": round(3 * _SYN_WATER_SCALE, 4), "direction": "cooling"},
            "frac_impervious": {"exponent": _SYN_IMP_EXP, "direction": "warming"},
            "frac_barren": {"direction": "warming"}, "frac_cropland": {"direction": "warming"},
            "ndbi": {"direction": "warming"}, "elevation": {"direction": "cooling"}, "ntl": {"direction": "warming"},
        },
        "noise": dict(_SYN_NOISE),
        "irrelevant_by_construction": ["slope", "aspect_sin", "aspect_cos", "log_pop", "frac_forest", "lm_pd",
                                       "lm_ed", "lm_contag"],
        "note": "Features not in the formula act only through correlation with planted drivers.",
    }


def generate_synthetic_dataset(grid: GridSpec, fine_res: int, epochs: Sequence[int], seed: int) -> dict:
    """Build the synthetic twin; returns the section-1 globals (same keys/dtypes as the Earth Engine path)."""
    if grid.res % fine_res:
        raise ValueError(f"fine_res {fine_res} must divide grid res {grid.res}")
    f = grid.res // fine_res
    fine = grid.fine(fine_res)
    if fine.width * fine.height > 60_000_000:
        log(f"synthetic fine grid is large ({fine.width}x{fine.height}); expect high memory use", "WARNING")
    t0 = time.perf_counter()
    lon, lat = grid.lonlat()
    roi_mask, district_idx, boundary = _outline_districts(grid)
    geo = _syn_geography(grid, lon, lat, seed)
    static = _syn_static_fine(grid, fine, geo, roi_mask, seed)
    static_lst_noise = _correlated_noise(grid.shape, _SYN_NOISE["spatial_corr_km"] * 1000.0 / grid.res, _syn_rng(seed, 3))
    log(f"synthetic static fields ready in {time.perf_counter() - t0:.1f}s (fine grid {fine.width}x{fine.height})")

    stacks, lc_fine, previous_built, lst_means = {}, {}, None, {}
    for year in epochs:
        step = (year - 2010) / 5.0
        urban = _syn_urban_intensity(grid, year)
        lc = _syn_landcover(static, _upsample(urban, f), roi_mask, f, previous_built, step)
        previous_built = lc == LC_BUILT
        fracs = _lc_block_fractions(lc, f)
        stacks[year] = _syn_epoch_stack(year, step, fracs, urban, geo, lon, lat, roi_mask, static_lst_noise, grid, seed)
        lc_fine[year] = lc
        lst_means[year] = float(np.nanmean(stacks[year]["lst_day_c"][roi_mask]))
        del urban, fracs
    del static, previous_built

    desc = f"SYNTHETIC twin (seed={seed}, generator delhi-ncr-synthetic-v1)"
    sources = {b: {y: f"{desc}: {b}" for y in epochs} for b in RAW_BANDS}
    sources["lst_day_c"] = {y: f"{desc}: planted response {_SYN_LST_FORMULA}" for y in epochs}
    sources["land_cover"] = {y: f"{desc}: rule-based land cover at {fine_res} m" for y in epochs}
    sources["roi"] = {y: "approximate NCR outline (SPEC §1.2) with nearest-HQ (Voronoi) districts" for y in epochs}
    log(f"synthetic dataset generated in {time.perf_counter() - t0:.1f}s")
    return {"ROI_MASK": roi_mask, "DISTRICT_IDX": district_idx, "BOUNDARY_GEOJSON": boundary,
            "RAW_STACKS": stacks, "LC_FINE": lc_fine, "SYNTHETIC_TRUTH": _synthetic_truth(epochs, seed, lst_means),
            "DATA_SOURCES": sources}

## 1.10 Run the acquisition
Earth Engine first when available. In `auto` mode an extraction failure falls back to the synthetic twin (with
the banner) **only when no service-account key was supplied**: with a key the user clearly wants real data, so the
error is raised instead of silently replacing hours of extraction by synthetic output (re-running resumes from the
tile cache).

In [ ]:
_ACQ = None
if DATA_MODE == "gee":
    try:
        with timer("01 GEE acquisition"):
            _ACQ = acquire_gee_dataset()
        GRID = _ACQ.pop("GRID")
    except Exception as _exc:  # noqa: BLE001 - auto mode must not crash
        if CFG.RUN_MODE == "gee" or _service_account_key_text():
            if CFG.RUN_MODE != "gee":
                log("Earth Engine extraction failed although a service-account key was supplied; stopping instead "
                    "of falling back to synthetic data (set LST_RUN_MODE=synthetic to force the synthetic twin)",
                    "ERROR")
            raise
        log(f"Earth Engine extraction failed: {type(_exc).__name__}: {_exc}", "ERROR")
        DATA_MODE = "synthetic"
        CFG.resolve_mode(DATA_MODE)
        _MODE_REASON = f"Earth Engine extraction failed: {type(_exc).__name__}: {str(_exc)[:300]}"
        _print_synthetic_banner(_MODE_REASON)
        _ACQ = None
if DATA_MODE == "synthetic":
    GRID = _outline_grid(CFG.GRID_RES_M)
    with timer("01 synthetic dataset"):
        _ACQ = generate_synthetic_dataset(GRID, CFG.fine_res(), CFG.EPOCHS, CFG.SEED)

# Why this data mode was used (exported as manifest.data_mode_reason; the dashboard banner shows it).
DATA_MODE_REASON = str(_MODE_REASON)
ROI_MASK = _ACQ["ROI_MASK"]
DISTRICT_IDX = _ACQ["DISTRICT_IDX"]
BOUNDARY_GEOJSON = _ACQ["BOUNDARY_GEOJSON"]
RAW_STACKS = _ACQ["RAW_STACKS"]
LC_FINE = _ACQ["LC_FINE"]
SYNTHETIC_TRUTH = _ACQ["SYNTHETIC_TRUTH"]
DATA_SOURCES = _ACQ["DATA_SOURCES"]
# True when district boundaries are the approximate outline + nearest-HQ partition (synthetic mode, or GEE mode
# after both boundary layers failed); exported as manifest.study_area.boundary_approximate.
BOUNDARY_APPROXIMATE = bool(_ACQ.get("BOUNDARY_APPROXIMATE", DATA_MODE == "synthetic"))
# Caveats for the manifest "notes" (kept separate so DATA_SOURCES stays strictly variable -> {year: provenance}).
DATA_CAVEATS = [
    "Land cover comes from different products per epoch (GLC_FCS30D, ESA WorldCover, Dynamic World); "
    "class definitions and accuracies differ, so part of the land-cover change is product inconsistency.",
    "DMSP-OLS (2010) saturates in urban cores and has no on-board calibration: 2010 DN are intercalibrated to "
    "F18 2013 with a robust fit on pseudo-invariant pixels and then mapped to VIIRS log radiance by an isotonic "
    "regression fitted on DMSP 2013 vs VIIRS 2014; both steps are approximate, so 2010-vs-later night-light "
    "differences should not be over-read.",
    "Terra's orbit drifts after 2022 (earlier overpass), biasing 2025 MODIS day LST slightly low.",
    "GHS-POP 2025 is a model projection rescaled to WorldPop 2020, not a census estimate.",
] + list(_ACQ.get("CAVEATS") or []) + (
    ["SYNTHETIC DATA: every variable is simulated from a planted response; see SYNTHETIC_TRUTH."]
    if DATA_MODE == "synthetic" else [])
for _caveat in DATA_CAVEATS:
    log(f"caveat: {_caveat}")
del _ACQ
log(f"GRID {GRID.width}x{GRID.height} @ {GRID.res} m | ROI {int(ROI_MASK.sum()):,} cells "
    f"({ROI_MASK.sum() * GRID.cell_area_km2:,.0f} km2) | fine {CFG.fine_res()} m | mode {DATA_MODE}")

## 1.11 Contract check
Verifies the section-1 interface (SPEC §3) so downstream sections can rely on shapes and dtypes.

In [ ]:
def _validate_acquisition() -> None:
    """Raise AssertionError with a precise message if any section-1 global violates the contract."""
    h, w = GRID.shape
    f = GRID.res // CFG.fine_res()
    assert DATA_MODE in ("gee", "synthetic"), DATA_MODE
    assert ROI_MASK.shape == (h, w) and ROI_MASK.dtype == bool, (ROI_MASK.shape, ROI_MASK.dtype)
    assert DISTRICT_IDX.shape == (h, w) and DISTRICT_IDX.dtype == np.int16, (DISTRICT_IDX.shape, DISTRICT_IDX.dtype)
    assert np.all(DISTRICT_IDX[~ROI_MASK] == NODATA_DISTRICT), "DISTRICT_IDX must be -1 outside the ROI"
    assert DISTRICT_IDX.max() < len(DISTRICTS), "district id out of range"
    assert len(DISTRICTS) == 25 and all(set(d) == {"id", "name", "state"} for d in DISTRICTS)
    assert BOUNDARY_GEOJSON["type"] == "FeatureCollection" and BOUNDARY_GEOJSON["features"]
    for feat in BOUNDARY_GEOJSON["features"]:
        assert set(feat["properties"]) == {"id", "name", "state"}, feat["properties"]
    assert sorted(RAW_STACKS) == sorted(CFG.EPOCHS), sorted(RAW_STACKS)
    for year in CFG.EPOCHS:
        assert list(RAW_STACKS[year]) == RAW_BANDS, (year, list(RAW_STACKS[year]))
        for band, arr in RAW_STACKS[year].items():
            assert arr.shape == (h, w) and arr.dtype == np.float32, (year, band, arr.shape, arr.dtype)
            assert np.all(np.isnan(arr[~ROI_MASK])), f"{band} {year} must be NaN outside the ROI"
        lc = LC_FINE[year]
        assert lc.shape == (h * f, w * f) and lc.dtype == np.uint8, (year, lc.shape, lc.dtype)
        assert int(lc.max()) <= 6, f"LC_FINE[{year}] has classes > 6"
    assert (SYNTHETIC_TRUTH is None) == (DATA_MODE == "gee")


_validate_acquisition()
log("section-1 contract check passed")

## 1.12 Optional interactive preview (geemap)
Only in live notebooks with Earth Engine data and geemap installed; skipped otherwise.

In [ ]:
if DATA_MODE == "gee" and _is_interactive() and _is_importable("geemap") and "lst" in _GEE_PREVIEW:
    try:
        import geemap
        from IPython.display import display

        _map = geemap.Map(center=[28.6, 77.2], zoom=8)
        _map.addLayer(_GEE_PREVIEW["lst"], {"min": 35, "max": 50,
                                            "palette": ["313695", "74add1", "fed976", "feb24c", "f03b20", "bd0026"]},
                      f"LST {CFG.EPOCHS[-1]} (°C)")
        if "roi" in _GEE_PREVIEW:
            _map.addLayer(_GEE_PREVIEW["roi"].style(color="ffffff", fillColor="00000000", width=1), {}, "districts")
        display(_map)
    except Exception as _exc:  # noqa: BLE001 - preview is cosmetic
        log(f"geemap preview failed: {_exc}", "WARNING")
else:
    log("geemap preview skipped (needs gee mode, a live notebook and geemap)")

## 1.13 Quality-assurance figures and per-epoch statistics

In [ ]:
def _masked(arr: np.ndarray) -> np.ma.MaskedArray:
    return np.ma.masked_invalid(np.where(ROI_MASK, arr, np.nan))


def _save_show(fig, name: str) -> None:
    fig.savefig(CFG.FIG_DIR / name, dpi=110, bbox_inches="tight")
    plt.show()
    plt.close(fig)


_mode_tag = " [SYNTHETIC]" if DATA_MODE == "synthetic" else ""
_extent_km = [0, GRID.width * GRID.res / 1000, 0, GRID.height * GRID.res / 1000]

_all_lst = np.concatenate([RAW_STACKS[y]["lst_day_c"][ROI_MASK] for y in CFG.EPOCHS])
_vmin, _vmax = np.nanpercentile(_all_lst, [2, 98])
fig, axes = plt.subplots(1, len(CFG.EPOCHS), figsize=(4.2 * len(CFG.EPOCHS), 4.6), constrained_layout=True)
for ax, year in zip(np.atleast_1d(axes), CFG.EPOCHS):
    im = ax.imshow(_masked(RAW_STACKS[year]["lst_day_c"]), cmap="inferno", vmin=_vmin, vmax=_vmax, extent=_extent_km)
    ax.set_title(f"LST {year}{_mode_tag}")
    ax.set_xlabel("km")
fig.colorbar(im, ax=axes, shrink=0.8, label="Day LST (°C), shared scale")
_save_show(fig, "01_lst_epochs.png")

_last = CFG.EPOCHS[-1]
_driver_panels = [("ndvi", "NDVI", "YlGn", None), ("ndbi", "NDBI", "RdPu", None),
                  ("ntl", "Night lights log1p(nW cm⁻² sr⁻¹)", "magma", None),
                  ("pop_density", "log10 population (persons/km²)", "viridis", np.log10),
                  ("elevation", "Elevation (m)", "terrain", None), ("lst_obs_count", "Valid LST composites", "Blues", None)]
fig, axes = plt.subplots(2, 3, figsize=(14, 9), constrained_layout=True)
for ax, (band, label, cmap, transform) in zip(axes.ravel(), _driver_panels):
    data = RAW_STACKS[_last][band]
    if transform is not None:
        data = transform(np.maximum(data, 1.0))
    im = ax.imshow(_masked(data), cmap=cmap, extent=_extent_km)
    ax.set_title(f"{label} — {_last}{_mode_tag}", fontsize=10)
    fig.colorbar(im, ax=ax, shrink=0.8)
_save_show(fig, f"01_drivers_{_last}.png")

from matplotlib.colors import BoundaryNorm, ListedColormap
_lc_cmap = ListedColormap([LC_CLASS_COLORS[c] for c in range(7)])
_lc_norm = BoundaryNorm(np.arange(-0.5, 7.5), _lc_cmap.N)
_stride = max(1, max(LC_FINE[_last].shape) // 1200)
fig, axes = plt.subplots(1, len(CFG.EPOCHS), figsize=(4.2 * len(CFG.EPOCHS), 4.8), constrained_layout=True)
for ax, year in zip(np.atleast_1d(axes), CFG.EPOCHS):
    ax.imshow(LC_FINE[year][::_stride, ::_stride], cmap=_lc_cmap, norm=_lc_norm, extent=_extent_km,
              interpolation="nearest")
    ax.set_title(f"Land cover {year}{_mode_tag}")
_handles = [plt.matplotlib.patches.Patch(color=LC_CLASS_COLORS[c], label=LC_CLASS_NAMES[c]) for c in range(1, 7)]
fig.legend(handles=_handles, loc="lower center", ncol=6, fontsize=8, bbox_to_anchor=(0.5, -0.06))
_save_show(fig, "01_landcover_epochs.png")

_rows = []
_f = GRID.res // CFG.fine_res()
for year in CFG.EPOCHS:
    stack = RAW_STACKS[year]
    lst = stack["lst_day_c"][ROI_MASK]
    lc_counts = np.bincount(LC_FINE[year].ravel(), minlength=7)[1:]
    lc_share = lc_counts / max(1, lc_counts.sum())
    _rows.append({"year": year, "cells": int(ROI_MASK.sum()), "lst_valid": int(np.isfinite(lst).sum()),
                  "lst_mean": np.nanmean(lst), "lst_std": np.nanstd(lst), "lst_p02": np.nanpercentile(lst, 2),
                  "lst_p98": np.nanpercentile(lst, 98), "ndvi_mean": np.nanmean(stack["ndvi"][ROI_MASK]),
                  "ntl_mean": np.nanmean(stack["ntl"][ROI_MASK]),
                  "pop_total_M": np.nansum(stack["pop_density"][ROI_MASK]) * GRID.cell_area_km2 / 1e6,
                  **{f"lc_{LC_CLASS_NAMES[c].split('/')[0].split(' ')[0]}_%": 100 * lc_share[c - 1] for c in range(1, 7)}})
_EPOCH_QA = pd.DataFrame(_rows).set_index("year").round(3)
print(f"Per-epoch ROI statistics{_mode_tag}")
_show(_EPOCH_QA)
_district_cells = pd.Series(np.bincount(DISTRICT_IDX[ROI_MASK], minlength=len(DISTRICTS)),
                            index=[d["name"] for d in DISTRICTS], name="cells")
log("cells per district: " + ", ".join(f"{k} {v}" for k, v in _district_cells.items()))
if (_district_cells == 0).any():
    log(f"districts without cells (absent in the boundary layer): {list(_district_cells[_district_cells == 0].index)}",
        "WARNING")
del _all_lst, _rows, _f
free_memory()

## 2. Feature engineering

One row of the modelling table is one 1 km grid cell in one epoch. The 17 predictors fall into five
physically motivated groups; their **order in `FEATURES` is the model's column order and is never changed**.

| Group | Features | Why they matter for daytime LST |
|---|---|---|
| Spectral | `ndvi`, `ndwi`, `ndbi` | Vegetation vigour drives evapotranspirative cooling (NDVI); surface wetness / open water raises latent heat flux (NDWI, McFeeters); built-up / bare signal tracks low-albedo, high-heat-capacity materials (NDBI). |
| Terrain | `elevation`, `slope`, `aspect_sin`, `aspect_cos` | Environmental lapse rate (~6.5 °C km⁻¹) and illumination geometry (the Aravalli ridge). Aspect is circular, so it enters as its sine/cosine pair (averaged at 30 m *before* aggregation, see §2 of the SPEC). |
| Socio-economic | `ntl`, `log_pop` | Anthropogenic heat and urban intensity. Night-time lights are already `log1p` radiance; population density is heavily right-skewed, so `log_pop = log1p(pop_density)`. |
| Land-cover composition | `frac_impervious`, `frac_forest`, `frac_water`, `frac_cropland`, `frac_barren` | Share of each harmonised class inside the cell, from the fine land-cover raster. "Other vegetation" (class 6) is deliberately **left out**: the six shares sum to one, so including all six would make the design matrix exactly collinear. Class 6 is the implicit reference category. |
| Landscape configuration | `lm_pd`, `lm_ed`, `lm_contag` | *How* the classes are arranged, not only how much there is: fragmentation (patch density), edge-mediated heat exchange (edge density) and aggregation (contagion). These come from FRAGSTATS (McGarigal & Marks) and are computed here with a vectorised implementation that is checked against `pylandstats`. |

### Landscape metrics (per window, valid pixels only)
A window is the block of fine land-cover pixels under `CFG.LM_WINDOW_CELLS` × `CFG.LM_WINDOW_CELLS` grid cells centred on
the cell (window 1 = the cell's own block; odd sizes ≥ 3 = overlapping moving window; pixels beyond the raster edge are nodata).
Let $A$ be the valid area (m²) of the window, $N_p$ the number of patches (8-connected components of each class 1–6, summed
over classes), $g_{ik}$ the number of 4-neighbour pixel adjacencies between classes $i$ and $k$ counted **in both directions**
(the FRAGSTATS "double-count" method, like adjacencies included) and $m$ the number of classes present.

$$\mathrm{PD} = \frac{N_p}{A}\,10^{4}\times 100 \quad [\text{patches per }100\ \text{ha}]$$

$$\mathrm{ED} = \frac{E}{A}\,10^{4}, \qquad E = r\sum_{i<k} g_{ik}\quad [\text{m ha}^{-1}]$$

where $r$ is the fine pixel size, i.e. $E$ is the length of every pixel side shared by two *different valid* classes.
The landscape boundary and sides touching nodata are not edges (FRAGSTATS "no border" / pylandstats `count_boundary=False`).

$$\mathrm{CONTAG} = \left[1 + \frac{\sum_{i=1}^{m}\sum_{k=1}^{m} p_{ik}\,\ln p_{ik}}{2\ln m}\right]\times 100,
  \qquad p_{ik} = \frac{g_{ik}}{\sum_{i,k} g_{ik}}, \qquad 0\ln 0 := 0 .$$

FRAGSTATS writes the joint probability as $P_i\,g_{ik}/\sum_k g_{ik}$ with $P_i$ the area share of class $i$; we use the
adjacency-share estimator $g_{ik}/\sum g$ exactly as `pylandstats` 3.1 does (`1 − joint_entropy / (2 ln m)`). The two
coincide except for pixels on the window/nodata border and the difference is small; matching pylandstats lets the
validation below be an exact regression test. A single-class window has no configurational information and gets
CONTAG = 100 (pylandstats returns NaN there).

### Missing-data policy
* A cell–epoch enters `DF` only if it lies in the ROI **and** has a valid LST target (no target imputation).
* Predictor NaNs are **kept** (`float32` NaN). XGBoost / LightGBM route them with learned default directions; linear
  models, random forest and the GNN impute with the **training-fold** median inside each CV fold (no leakage).
* Land-cover fractions and landscape metrics are NaN when fewer than 50 % of the window's fine pixels are valid.
* Missingness per feature is reported below so gaps from cloud masking or product coverage are visible.

In [ ]:
import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from numpy.lib.stride_tricks import sliding_window_view
from scipy import ndimage

FEATURES = [
    "ndvi", "ndwi", "ndbi", "elevation", "slope", "aspect_sin", "aspect_cos", "ntl", "log_pop",
    "frac_impervious", "frac_forest", "frac_water", "frac_cropland", "frac_barren", "lm_pd", "lm_ed", "lm_contag",
]
TARGET_COL = "target"

# Harmonised land-cover classes (SPEC §1): 0 = nodata, 1..6 = valid classes.
_N_LC_CLASSES = 6
_FRACTION_FEATURES = {  # feature name -> harmonised class id (class 6 "other vegetation" is the reference)
    "frac_impervious": 1, "frac_forest": 2, "frac_water": 3, "frac_cropland": 4, "frac_barren": 5,
}
_MIN_VALID_SHARE = 0.5          # window needs >= 50 % valid fine pixels
_LM_CHUNK_PIXELS = 1 << 23      # ~8.4 M fine pixels per chunk -> peak RAM well below 1 GB
# 8-connectivity inside each window and NO connectivity along the stacked-window axis (axis 0).
_LM_STRUCTURE = np.zeros((3, 3, 3), dtype=bool)
_LM_STRUCTURE[1] = True

### 2.1 Land-cover composition
`lc_fractions` reshapes the fine raster to `(H, f, W, f)` blocks and counts every class per block, processing row chunks
so a 25 m raster of the whole NCR (~130 M pixels) never needs more than a few hundred MB.

In [ ]:
def lc_fractions(lc_fine, f):
    """Per-cell class shares of valid fine pixels.

    Parameters
    ----------
    lc_fine : (H*f, W*f) uint8 array of harmonised classes (0 = nodata).
    f : int, fine pixels per grid cell along one axis.

    Returns
    -------
    (H, W, 6) float32 array; ``[..., c-1]`` is the share of class ``c`` among the cell's valid pixels,
    NaN where fewer than 50 % of the ``f*f`` pixels are valid.
    """
    f = int(f)
    if lc_fine.ndim != 2 or lc_fine.shape[0] % f or lc_fine.shape[1] % f:
        raise ValueError(f"lc_fine shape {lc_fine.shape} is not a multiple of the block size f={f}")
    n_rows, n_cols = lc_fine.shape[0] // f, lc_fine.shape[1] // f
    out = np.full((n_rows, n_cols, _N_LC_CLASSES), np.nan, dtype=np.float32)
    rows_per_chunk = max(1, _LM_CHUNK_PIXELS // (f * f * max(n_cols, 1)))
    for r0 in range(0, n_rows, rows_per_chunk):
        r1 = min(n_rows, r0 + rows_per_chunk)
        blocks = lc_fine[r0 * f:r1 * f].reshape(r1 - r0, f, n_cols, f)
        counts = np.stack([(blocks == c).sum(axis=(1, 3)) for c in range(1, _N_LC_CLASSES + 1)], axis=-1)
        valid = counts.sum(axis=-1, keepdims=True)
        with np.errstate(invalid="ignore", divide="ignore"):
            frac = counts / np.maximum(valid, 1)
        frac[np.broadcast_to(valid < _MIN_VALID_SHARE * f * f, frac.shape)] = np.nan
        out[r0:r1] = frac
    return out

### 2.2 Vectorised landscape metrics
All windows of a chunk are stacked into one `(n_windows, b, b)` array:
* **patches** – `ndimage.label` per class with a 3-D structuring element that is 8-connected inside a window and has
  no connectivity along the stack axis; each label's owning window is found from any of its pixels and a `bincount`
  gives patches per window;
* **adjacency** – every horizontal and vertical pixel pair with two valid classes contributes
  `window·C² + c_i·C + c_j` **and** its transpose to one `bincount`, giving the `(n, C, C)` double-count adjacency
  matrices from which ED (off-diagonal) and CONTAG (joint entropy) follow.

With `ENV["has_cupy"]` the same array code runs on the GPU (`cupyx.scipy.ndimage.label`); any GPU failure falls back to
NumPy for the remaining chunks.

In [ ]:
def _lm_backend():
    """Return (array module, label function, backend name) for landscape metrics."""
    if ENV.get("has_cupy"):
        try:
            import cupy as cp
            import cupyx.scipy.ndimage as cndi

            return cp, cndi.label, "cupy"
        except Exception as exc:  # broken CUDA install -> CPU
            log(f"cupy unavailable for landscape metrics ({exc!r}); using NumPy", "WARNING")
    return np, ndimage.label, "numpy"


def _window_stack_view(lc_fine, f, window_cells):
    """Strided view (H, W, b, b) of the fine pixels under each cell's window (no copy)."""
    if window_cells < 1 or (window_cells > 1 and window_cells % 2 == 0):
        raise ValueError(f"LM_WINDOW_CELLS must be 1 or an odd number >= 3, got {window_cells}")
    half_px = (window_cells - 1) // 2 * f
    padded = np.pad(lc_fine, half_px, mode="constant", constant_values=0) if half_px else lc_fine
    size = window_cells * f
    return sliding_window_view(padded, (size, size))[::f, ::f]


def _stack_metrics(stack, fine_res, xp=np, label_fn=ndimage.label):
    """Class counts, PD, ED and CONTAG for a stack of windows.

    Parameters
    ----------
    stack : (n, b, b) uint8 array (NumPy or CuPy) of harmonised classes, 0 = nodata.
    fine_res : fine pixel size in metres.

    Returns
    -------
    dict of NumPy arrays: ``counts`` (n, 7) pixel counts per class 0..6, ``pd``, ``ed``, ``contag`` (n,) float64,
    NaN where the window has < 50 % valid pixels.
    """
    n_win, size, _ = stack.shape
    n_cls = _N_LC_CLASSES
    stack = xp.asarray(stack)
    win_id = xp.arange(n_win, dtype=xp.int64)[:, None, None]

    counts = xp.bincount((win_id * (n_cls + 1) + stack).ravel(), minlength=n_win * (n_cls + 1))
    counts = counts.reshape(n_win, n_cls + 1)
    valid = counts[:, 1:].sum(axis=1)

    # --- patches: one 3-D labelling per class present in the chunk --------------------------------------------
    n_patches = xp.zeros(n_win, dtype=xp.int64)
    px_per_win = size * size
    for cls in range(1, n_cls + 1):
        if int(counts[:, cls].sum()) == 0:
            continue
        labels, n_labels = label_fn(stack == cls, structure=xp.asarray(_LM_STRUCTURE))
        n_labels = int(n_labels)
        if n_labels == 0:
            continue
        flat = labels.ravel()
        pix = xp.flatnonzero(flat)
        owner = xp.zeros(n_labels + 1, dtype=xp.int64)
        owner[flat[pix]] = pix // px_per_win  # every pixel of a label lies in the same window
        n_patches += xp.bincount(owner[1:], minlength=n_win)
        del labels, flat, pix, owner

    # --- 4-neighbour adjacency, double counted -------------------------------------------------------------------
    adjacency = xp.zeros(n_win * n_cls * n_cls, dtype=xp.int64)
    for a, b in ((stack[:, :, :-1], stack[:, :, 1:]), (stack[:, :-1, :], stack[:, 1:, :])):
        ok = (a > 0) & (b > 0)
        wid = xp.broadcast_to(win_id, a.shape)[ok]
        ca = a[ok].astype(xp.int64) - 1
        cb = b[ok].astype(xp.int64) - 1
        base = wid * (n_cls * n_cls)
        adjacency += xp.bincount(base + ca * n_cls + cb, minlength=n_win * n_cls * n_cls)
        adjacency += xp.bincount(base + cb * n_cls + ca, minlength=n_win * n_cls * n_cls)
        del ok, wid, ca, cb, base
    adjacency = adjacency.reshape(n_win, n_cls, n_cls)

    to_np = (lambda arr: arr) if xp is np else xp.asnumpy
    counts, valid, n_patches, adjacency = map(to_np, (counts, valid, n_patches, adjacency))
    return _metrics_from_counts(counts, valid, n_patches, adjacency, fine_res, px_per_win)


def _metrics_from_counts(counts, valid, n_patches, adjacency, fine_res, px_per_win):
    """Turn per-window counts into PD, ED and CONTAG (NumPy, float64)."""
    area_ha = valid * fine_res * fine_res / 1e4
    total_adj = adjacency.sum(axis=(1, 2)).astype(np.float64)
    like_adj = np.einsum("nii->n", adjacency).astype(np.float64)
    edge_len_m = (total_adj - like_adj) / 2.0 * fine_res  # each unlike pair is counted twice
    n_present = (counts[:, 1:] > 0).sum(axis=1)

    with np.errstate(divide="ignore", invalid="ignore"):
        pd_ = n_patches / area_ha * 100.0
        ed = edge_len_m / area_ha
        prob = adjacency / total_adj[:, None, None]
        plogp = np.where(prob > 0, prob * np.log(np.where(prob > 0, prob, 1.0)), 0.0)
        joint_entropy = -plogp.sum(axis=(1, 2))
        contag = (1.0 - joint_entropy / (2.0 * np.log(np.maximum(n_present, 2)))) * 100.0
    contag = np.where(n_present == 1, 100.0, contag)
    contag = np.where((n_present >= 2) & (total_adj == 0), np.nan, contag)

    insufficient = valid < _MIN_VALID_SHARE * px_per_win
    for arr in (pd_, ed, contag):
        arr[insufficient] = np.nan
    return {"counts": counts, "pd": pd_, "ed": ed, "contag": contag}


def _landscape_metrics(lc_fine, f, fine_res, window_cells, cell_mask):
    """PD / ED / CONTAG rasters (H, W) float32 for the cells in ``cell_mask`` (others NaN).

    Windows are processed in row-major chunks of ~``_LM_CHUNK_PIXELS`` fine pixels.
    """
    n_rows, n_cols = cell_mask.shape
    view = _window_stack_view(lc_fine, f, window_cells)
    size = view.shape[-1]
    rows, cols = np.nonzero(cell_mask)
    out = {k: np.full((n_rows, n_cols), np.nan, dtype=np.float32) for k in ("lm_pd", "lm_ed", "lm_contag")}
    xp, label_fn, backend = _lm_backend()
    per_chunk = max(1, _LM_CHUNK_PIXELS // (size * size))
    for start in range(0, rows.size, per_chunk):
        r_idx, c_idx = rows[start:start + per_chunk], cols[start:start + per_chunk]
        stack = np.ascontiguousarray(view[r_idx, c_idx])
        try:
            res = _stack_metrics(stack, fine_res, xp, label_fn)
        except Exception as exc:
            if xp is np:
                raise
            log(f"GPU landscape metrics failed ({exc!r}); continuing on CPU", "WARNING")
            xp, label_fn, backend = np, ndimage.label, "numpy"
            free_memory()
            res = _stack_metrics(stack, fine_res, xp, label_fn)
        out["lm_pd"][r_idx, c_idx] = res["pd"]
        out["lm_ed"][r_idx, c_idx] = res["ed"]
        out["lm_contag"][r_idx, c_idx] = res["contag"]
        del stack, res
    out["backend"] = backend
    return out

### 2.3 Validation against pylandstats
120 random windows are re-computed with `pylandstats.Landscape(array, res=(r, r), nodata=0, neighborhood_rule="8")`
(`patch_density()`, `edge_density(count_boundary=False)`, `contagion()`). pylandstats 3.1 uses exactly the definitions
above: PD per 100 ha of **valid** area, ED from the unique unlike 4-neighbour adjacencies (nodata excluded), CONTAG from
the joint entropy of the double-counted adjacency matrix with $m$ = classes present. Its only divergence is CONTAG for a
single-class window (NaN vs our 100), so those windows are excluded from the CONTAG comparison. Mismatches warn but
never stop the pipeline.

In [ ]:
def validate_landscape_metrics(n=120, tol=1e-6):
    """Compare the vectorised metrics with pylandstats on ``n`` random valid windows (all epochs pooled).

    Returns the ``LM_VALIDATION`` dict: ``n``, per-metric ``*_max_abs_diff`` / ``*_mean_abs_diff``, ``status``
    ("ok" | "warn" | "skipped" | "error") and a ``reason``/``note``.
    """
    try:
        import pylandstats as pls
    except ImportError:
        log("pylandstats not installed - landscape metric validation skipped", "WARNING")
        return {"n": 0, "status": "skipped", "reason": "pylandstats not installed"}

    try:
        rng = np.random.default_rng(CFG.SEED)
        years = sorted(LC_FINE)
        per_year = np.bincount(rng.integers(0, len(years), size=n), minlength=len(years))
        ours, ref = {"pd": [], "ed": [], "contag": []}, {"pd": [], "ed": [], "contag": []}
        n_single_class = 0
        for year, n_year in zip(years, per_year):
            if n_year == 0:
                continue
            lc = LC_FINE[year]
            f = lc.shape[0] // ROI_MASK.shape[0]
            fine_res = float(GRID.res) / f
            view = _window_stack_view(lc, f, CFG.LM_WINDOW_CELLS)
            size = view.shape[-1]
            cand_r, cand_c = np.nonzero(ROI_MASK)
            pick = rng.choice(cand_r.size, size=min(int(n_year) * 3, cand_r.size), replace=False)
            stack = np.ascontiguousarray(view[cand_r[pick], cand_c[pick]])
            res = _stack_metrics(stack, fine_res)
            keep = np.flatnonzero(np.isfinite(res["pd"]))[: int(n_year)]
            for i in keep:
                window = stack[i]
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore")
                    ls = pls.Landscape(window, res=(fine_res, fine_res), nodata=0, neighborhood_rule="8")
                    ref_vals = (ls.patch_density(), ls.edge_density(count_boundary=False), ls.contagion())
                ours["pd"].append(res["pd"][i])
                ref["pd"].append(ref_vals[0])
                ours["ed"].append(res["ed"][i])
                ref["ed"].append(ref_vals[1])
                if np.isfinite(ref_vals[2]):
                    ours["contag"].append(res["contag"][i])
                    ref["contag"].append(ref_vals[2])
                else:
                    n_single_class += 1
            del view, stack, res
    except Exception as exc:
        log(f"landscape metric validation failed: {exc!r}", "WARNING")
        return {"n": 0, "status": "error", "reason": repr(exc)}

    out = {"n": len(ours["pd"]), "window_cells": int(CFG.LM_WINDOW_CELLS), "n_single_class": n_single_class,
           "pylandstats_version": getattr(pls, "__version__", "unknown")}
    worst = 0.0
    for metric in ("pd", "ed", "contag"):
        a, b = np.asarray(ours[metric], float), np.asarray(ref[metric], float)
        diff = np.abs(a - b) if a.size else np.array([np.nan])
        out[f"{metric}_max_abs_diff"] = float(np.nanmax(diff)) if a.size else None
        out[f"{metric}_mean_abs_diff"] = float(np.nanmean(diff)) if a.size else None
        out[f"{metric}_n"] = int(a.size)
        if a.size:
            worst = max(worst, float(np.nanmax(diff / np.maximum(np.abs(b), 1.0))))
    out["status"] = "ok" if worst <= tol else "warn"
    out["note"] = ("CONTAG compared only on windows with >= 2 classes (pylandstats returns NaN otherwise; "
                   "this pipeline uses 100). Relative tolerance %.0e." % tol)
    if out["status"] != "ok":
        log(f"landscape metrics differ from pylandstats (max rel diff {worst:.3g}): {out}", "WARNING")
    else:
        log(f"landscape metrics match pylandstats on {out['n']} windows (max rel diff {worst:.2e})")
    return out

### 2.4 Long-format feature table
Cells are enumerated in row-major order per epoch, so `cell_id = row·W + col` is already ascending; the table is still
sorted explicitly by (`year`, `cell_id`) — the row-order contract every downstream per-row array relies on.

**Cross-validation groupings** (stored as columns so every later stage reuses identical folds):
* `spatial_block` – 5 × 5 tiles over the grid extent (`row·5 // H`, `col·5 // W`);
* `fold_spatial` – non-empty tiles sorted by size (largest first) and greedily given to the currently smallest fold,
  which balances fold sizes deterministically; a cell keeps its fold in every epoch;
* `fold_random` – a seeded random permutation of rows (the optimistic baseline);
* `block10_id` – 10 km blocks (≥ 1 cell each) for inner early-stopping splits and the block bootstrap.

When RAPIDS cuDF is present the assembly and group-bys run on the GPU and are converted back with `.to_pandas()`;
the pandas path gives identical results.

In [ ]:
_DF_DTYPES = {
    "cell_id": "int32", "row": "int32", "col": "int32", "x": "float64", "y": "float64",
    "lon": "float64", "lat": "float64", "district": "int16", "year": "int16",
    **{name: "float32" for name in FEATURES},
    "pop_density": "float32", "lst": "float32", "lst_obs_count": "float32", "epoch_mean": "float32",
    "target": "float32", "spatial_block": "int16", "fold_spatial": "int8", "fold_random": "int8",
    "block10_id": "int32",
}
_SPECTRAL_BANDS = ["ndvi", "ndwi", "ndbi", "elevation", "slope", "aspect_sin", "aspect_cos", "ntl"]


def _fine_factor(year):
    """Fine pixels per cell (f) and fine resolution (m) for an epoch's land-cover raster."""
    lc = LC_FINE[year]
    n_rows, n_cols = ROI_MASK.shape
    f = lc.shape[0] // n_rows
    if f < 1 or lc.shape != (n_rows * f, n_cols * f):
        raise ValueError(f"LC_FINE[{year}] shape {lc.shape} incompatible with grid {ROI_MASK.shape}")
    return f, float(GRID.res) / f


def _epoch_columns(year, cell_mask_lm, geo):
    """Numpy column dict for one epoch (ROI cells with a valid LST only)."""
    stack = RAW_STACKS[year]
    lst = stack["lst_day_c"]
    obs = stack["lst_obs_count"]
    valid = ROI_MASK & np.isfinite(lst)
    # A seasonal median from fewer than CFG.LST_MIN_OBS valid 8-day composites is too noisy to be a target (the
    # Earth Engine path already masks such cells server-side; this guard also covers cached or external stacks).
    few = valid & np.isfinite(obs) & (obs < int(CFG.LST_MIN_OBS))
    if few.any():
        log(f"{year}: {int(few.sum()):,} cells dropped (< {CFG.LST_MIN_OBS} valid LST composites)", "WARNING")
    rows, cols = np.nonzero(valid & ~few)
    n_cols_grid = ROI_MASK.shape[1]
    columns = {
        "cell_id": (rows * n_cols_grid + cols).astype(np.int32),
        "row": rows.astype(np.int32), "col": cols.astype(np.int32),
        "x": geo["x"][rows, cols], "y": geo["y"][rows, cols],
        "lon": geo["lon"][rows, cols], "lat": geo["lat"][rows, cols],
        "district": DISTRICT_IDX[rows, cols].astype(np.int16),
        "year": np.full(rows.size, year, dtype=np.int16),
    }
    for band in _SPECTRAL_BANDS:
        columns[band] = stack[band][rows, cols].astype(np.float32)
    pop = stack["pop_density"][rows, cols].astype(np.float32)
    columns["log_pop"] = np.log1p(np.clip(pop, 0.0, None)).astype(np.float32)

    if year in LC_FINE:
        f, fine_res = _fine_factor(year)
        with timer(f"lc_fractions_{year}"):
            fractions = lc_fractions(LC_FINE[year], f)
        for name, cls in _FRACTION_FEATURES.items():
            columns[name] = fractions[rows, cols, cls - 1]
        with timer(f"landscape_metrics_{year}"):
            lm = _landscape_metrics(LC_FINE[year], f, fine_res, int(CFG.LM_WINDOW_CELLS), cell_mask_lm)
        log(f"{year}: landscape metrics on {int(cell_mask_lm.sum()):,} windows "
            f"({CFG.LM_WINDOW_CELLS}x{CFG.LM_WINDOW_CELLS} cells, f={f}, backend={lm['backend']})")
        for name in ("lm_pd", "lm_ed", "lm_contag"):
            columns[name] = lm[name][rows, cols]
        del fractions, lm
    else:
        log(f"LC_FINE has no {year} raster: land-cover features set to NaN", "WARNING")
        for name in (*_FRACTION_FEATURES, "lm_pd", "lm_ed", "lm_contag"):
            columns[name] = np.full(rows.size, np.nan, dtype=np.float32)

    columns["pop_density"] = pop
    columns["lst"] = lst[rows, cols].astype(np.float32)
    columns["lst_obs_count"] = stack["lst_obs_count"][rows, cols].astype(np.float32)
    return columns


def _frame_backend():
    """cuDF when available (GPU group-bys), else pandas."""
    if ENV.get("has_cudf"):
        try:
            import cudf

            return cudf, "cudf"
        except Exception as exc:
            log(f"cuDF import failed ({exc!r}); using pandas", "WARNING")
    return pd, "pandas"


def _assemble_long(per_epoch):
    """Concatenate epochs, sort by (year, cell_id) and compute the group-by statistics.

    Returns (pandas DataFrame, epoch means {year: float}, row counts per spatial_block {block: int}).
    """
    lib, name = _frame_backend()
    try:
        frame = lib.concat([lib.DataFrame(cols) for cols in per_epoch], ignore_index=True)
        frame = frame.sort_values(["year", "cell_id"]).reset_index(drop=True)
        frame["_lst64"] = frame["lst"].astype("float64")
        means = frame.groupby("year")["_lst64"].mean()
        blocks = frame.groupby("spatial_block")["cell_id"].count()
        means = means.to_pandas() if name == "cudf" else means
        blocks = blocks.to_pandas() if name == "cudf" else blocks
        frame = frame.drop(columns=["_lst64"])
        frame = frame.to_pandas() if name == "cudf" else frame
        if name == "cudf":
            ENV.setdefault("rapids_used", set()).add("cudf")
    except Exception as exc:
        if lib is pd:
            raise
        log(f"cuDF assembly failed ({exc!r}); redoing with pandas", "WARNING")
        ENV["has_cudf"] = False
        return _assemble_long(per_epoch)
    log(f"long-format table assembled with {name}")
    epoch_means = {int(k): float(v) for k, v in means.items()}
    block_counts = {int(k): int(v) for k, v in blocks.items()}
    return frame, epoch_means, block_counts


def _greedy_block_folds(block_counts, n_folds):
    """Deterministic size-balanced assignment: largest block first -> currently smallest fold (ties: lowest id)."""
    order = sorted((b for b, c in block_counts.items() if c > 0), key=lambda b: (-block_counts[b], b))
    if len(order) < n_folds:
        log(f"only {len(order)} non-empty spatial blocks for {n_folds} folds - some folds will be empty", "WARNING")
    load = np.zeros(n_folds, dtype=np.int64)
    assignment = {}
    for block in order:
        fold = int(np.argmin(load))
        assignment[block] = fold
        load[fold] += block_counts[block]
    return assignment, load


def _build_feature_table():
    """Build DF, EPOCH_MEANS and the fold columns from RAW_STACKS / LC_FINE (SPEC §3)."""
    n_rows, n_cols = ROI_MASK.shape
    X, Y = GRID.cell_centers_xy()
    LON, LAT = GRID.lonlat()
    geo = {"x": np.asarray(X, np.float64), "y": np.asarray(Y, np.float64),
           "lon": np.asarray(LON, np.float64), "lat": np.asarray(LAT, np.float64)}
    years = [y for y in CFG.EPOCHS if y in RAW_STACKS] or sorted(RAW_STACKS)
    missing = sorted(set(CFG.EPOCHS) - set(RAW_STACKS))
    if missing:
        log(f"RAW_STACKS lacks epochs {missing}; continuing with {years}", "WARNING")

    # spatial CV block ids are needed before assembly (group-by on blocks)
    nby, nbx = CFG.SPATIAL_BLOCKS
    grid_rows, grid_cols = np.indices((n_rows, n_cols))
    block_grid = ((grid_rows * nby // n_rows) * nbx + grid_cols * nbx // n_cols).astype(np.int16)
    b10 = max(1, int(round(CFG.INNER_BLOCK_KM * 1000.0 / float(GRID.res))))
    block10_grid = ((grid_rows // b10) * math.ceil(n_cols / b10) + grid_cols // b10).astype(np.int32)

    per_epoch = []
    for year in years:
        cols = _epoch_columns(year, ROI_MASK, geo)
        cols["spatial_block"] = block_grid[cols["row"], cols["col"]]
        cols["block10_id"] = block10_grid[cols["row"], cols["col"]]
        per_epoch.append(cols)
        log(f"{year}: {cols['cell_id'].size:,} cells with valid LST")
    frame, epoch_means, block_counts = _assemble_long(per_epoch)
    del per_epoch

    if frame.empty:
        raise RuntimeError("feature table is empty - no ROI cell has a valid LST target")

    epoch_mean = frame["year"].map(epoch_means).to_numpy(np.float64)
    frame["epoch_mean"] = epoch_mean.astype(np.float32)
    lst64 = frame["lst"].to_numpy(np.float64)
    if CFG.TARGET_MODE == "anomaly":
        frame["target"] = (lst64 - epoch_mean).astype(np.float32)
    elif CFG.TARGET_MODE == "absolute":
        frame["target"] = lst64.astype(np.float32)
    else:
        raise ValueError(f"unknown CFG.TARGET_MODE {CFG.TARGET_MODE!r}")

    assignment, load = _greedy_block_folds(block_counts, int(CFG.N_FOLDS))
    frame["fold_spatial"] = frame["spatial_block"].map(assignment).to_numpy(np.int8)
    perm = np.random.default_rng(CFG.SEED).permutation(len(frame))
    fold_random = np.empty(len(frame), dtype=np.int8)
    fold_random[perm] = np.arange(len(frame)) % int(CFG.N_FOLDS)
    frame["fold_random"] = fold_random

    frame = frame[list(_DF_DTYPES)].astype(_DF_DTYPES)
    frame.index = pd.RangeIndex(len(frame))
    log(f"spatial folds (rows): {load.tolist()} from {len(assignment)} non-empty 5x5 blocks; "
        f"10 km blocks = {b10}x{b10} cells, {frame['block10_id'].nunique()} used")
    return frame, epoch_means


with timer("features"):
    DF, EPOCH_MEANS = _build_feature_table()
LM_VALIDATION = validate_landscape_metrics(n=120)
log(f"DF: {len(DF):,} rows x {DF.shape[1]} columns; epochs {sorted(EPOCH_MEANS)}; "
    f"epoch means (degC) {({k: round(v, 2) for k, v in EPOCH_MEANS.items()})}")

### 2.5 Feature metadata and statistics
`FEATURE_META` drives every label in the dashboard; `actionable` marks the levers planners can move: the five
land-cover fractions, NDVI (greening within the existing covers) and the three landscape-configuration metrics.
NDWI and NDBI are spectral *diagnostics*, not interventions: nobody "lowers NDBI" directly, it changes only when the
land cover changes. They are therefore `actionable = False`; in coupled scenarios they follow land-cover changes
through the fitted coupling slopes (4.10). Terrain and socio-economic context are held fixed.

In [ ]:
_ACTIONABLE_GROUPS = {"landcover", "landscape"}
_ACTIONABLE_EXTRA = {"ndvi"}          # the one directly actionable spectral index (greening)
_META_ROWS = [
    # name, label, unit, group, display, source, description
    ("ndvi", "NDVI", "", "spectral", "index", "MODIS MOD09A1 (500 m), seasonal median",
     "Normalised Difference Vegetation Index; greener, denser vegetation cools the surface by transpiration and shading."),
    ("ndwi", "NDWI", "", "spectral", "index", "MODIS MOD09A1 (500 m), seasonal median",
     "McFeeters Normalised Difference Water Index (green vs NIR); high values mark open water and wet surfaces."),
    ("ndbi", "NDBI", "", "spectral", "index", "MODIS MOD09A1 (500 m), seasonal median",
     "Normalised Difference Built-up Index (SWIR vs NIR); high values mark built-up and bare, dry surfaces."),
    ("elevation", "Elevation", "m", "terrain", "number", "SRTM GL1 (30 m), cell mean",
     "Mean terrain height above sea level; air and surface temperature fall with altitude."),
    ("slope", "Slope", "°", "terrain", "number", "SRTM GL1 (30 m), cell mean",
     "Mean terrain slope; steep ridges (Aravalli) change solar exposure and land use."),
    ("aspect_sin", "Aspect (east-west)", "", "terrain", "index", "SRTM GL1 (30 m), sin(aspect) averaged at 30 m",
     "Sine of terrain aspect: +1 east-facing, -1 west-facing slopes."),
    ("aspect_cos", "Aspect (north-south)", "", "terrain", "index", "SRTM GL1 (30 m), cos(aspect) averaged at 30 m",
     "Cosine of terrain aspect: +1 north-facing, -1 south-facing (sun-exposed) slopes."),
    ("ntl", "Night-time lights", "log1p(nW/cm²/sr)", "socioeconomic", "number",
     "VIIRS DNB monthly (2015+) / DMSP-OLS harmonised by isotonic regression (2010)",
     "Log night-time radiance; proxy for urban intensity and anthropogenic heat release."),
    ("log_pop", "Population density (log)", "log1p(persons/km²)", "socioeconomic", "number",
     "WorldPop 100 m (2010-2020) / GHS-POP rescaled (2025)",
     "Log of residents per km²; proxy for building density and human heat emission."),
    ("frac_impervious", "Impervious cover", "fraction", "landcover", "percent", "Harmonised fine land cover (class 1)",
     "Share of the cell covered by built-up / impervious surfaces."),
    ("frac_forest", "Tree cover", "fraction", "landcover", "percent", "Harmonised fine land cover (class 2)",
     "Share of the cell covered by trees / forest."),
    ("frac_water", "Water", "fraction", "landcover", "percent", "Harmonised fine land cover (class 3)",
     "Share of the cell covered by open water (rivers, canals, lakes)."),
    ("frac_cropland", "Cropland", "fraction", "landcover", "percent", "Harmonised fine land cover (class 4)",
     "Share of the cell under cultivation."),
    ("frac_barren", "Barren land", "fraction", "landcover", "percent", "Harmonised fine land cover (class 5)",
     "Share of the cell that is bare soil, sand or rock."),
    ("lm_pd", "Patch density", "patches/100 ha", "landscape", "number", "FRAGSTATS PD on fine land cover",
     "Number of land-cover patches (8-connected) per 100 ha; higher = more fragmented mosaic."),
    ("lm_ed", "Edge density", "m/ha", "landscape", "number", "FRAGSTATS ED on fine land cover",
     "Length of boundaries between different land-cover classes per hectare."),
    ("lm_contag", "Contagion", "%", "landscape", "number", "FRAGSTATS CONTAG on fine land cover",
     "Aggregation of land-cover classes (0 = maximally interspersed, 100 = single contiguous class)."),
]
FEATURE_META = {
    name: {"label": label, "unit": unit, "group": group, "description": desc, "source": source,
           "actionable": group in _ACTIONABLE_GROUPS or name in _ACTIONABLE_EXTRA, "display": display}
    for name, label, unit, group, display, source, desc in _META_ROWS
}
if list(FEATURE_META) != FEATURES:
    raise AssertionError("FEATURE_META order must match FEATURES")


def _feature_stats(frame):
    """Robust summary per feature over all DF rows (NaN-aware; NaN when a feature is entirely missing)."""
    stats = {}
    for name in FEATURES:
        v = frame[name].to_numpy(np.float64)
        v = v[np.isfinite(v)]
        if v.size == 0:
            stats[name] = {k: float("nan") for k in ("min", "max", "p01", "p99", "median", "mean", "std")}
            continue
        p01, p99, med = np.percentile(v, [1, 99, 50])
        stats[name] = {"min": float(v.min()), "max": float(v.max()), "p01": float(p01), "p99": float(p99),
                       "median": float(med), "mean": float(v.mean()), "std": float(v.std(ddof=1)) if v.size > 1 else 0.0}
    return stats


FEATURE_STATS = _feature_stats(DF)

### 2.6 Diagnostics
* per-epoch descriptive statistics (shifts in the land-surface mix and in LST),
* Spearman rank correlations (robust to the skewed socio-economic variables); pairs with |ρ| > 0.85 are flagged because
  they split SHAP credit arbitrarily between them,
* variance inflation factors $\mathrm{VIF}_j = 1/(1-R_j^2)$ from least-squares regressions of each standardised
  feature on all others (complete cases),
* missingness per feature and epoch.

In [ ]:
def _show_table(obj):
    """display() inside a notebook kernel, full-width print() in plain scripts (named apart from 00's `_show`:
    all sections share one namespace)."""
    import sys

    if "ipykernel" in sys.modules:
        try:
            from IPython.display import display

            display(obj)
            return
        except Exception:
            pass
    print(obj.to_string() if isinstance(obj, pd.DataFrame) else obj)


def _vif(frame):
    """VIF per feature via numpy least squares on standardised complete cases (NaN if undefined)."""
    data = frame[FEATURES].to_numpy(np.float64)
    data = data[np.isfinite(data).all(axis=1)]
    out = pd.Series(np.nan, index=FEATURES, name="VIF")
    if data.shape[0] < len(FEATURES) + 2:
        log(f"VIF skipped: only {data.shape[0]} complete rows", "WARNING")
        return out
    std = data.std(axis=0)
    usable = std > 0
    z = (data[:, usable] - data[:, usable].mean(axis=0)) / std[usable]
    names = [n for n, u in zip(FEATURES, usable) if u]
    for j, name in enumerate(names):
        others = np.delete(z, j, axis=1)
        design = np.column_stack([np.ones(len(z)), others])
        coef, *_ = np.linalg.lstsq(design, z[:, j], rcond=None)
        resid = z[:, j] - design @ coef
        r2 = 1.0 - resid @ resid / (z[:, j] @ z[:, j])
        out[name] = np.inf if r2 >= 1.0 - 1e-12 else 1.0 / (1.0 - r2)
    return out


def _plot_spearman(corr):
    """Annotated Spearman heatmap (diverging blue-grey-red, fixed [-1, 1])."""
    import matplotlib.pyplot as plt
    from matplotlib.colors import LinearSegmentedColormap

    cmap = LinearSegmentedColormap.from_list("div", ["#2a78d6", "#f0efec", "#e34948"])
    labels = [FEATURE_META[f]["label"] for f in corr.columns]
    fig, ax = plt.subplots(figsize=(11, 9))
    im = ax.imshow(corr.to_numpy(), cmap=cmap, vmin=-1, vmax=1)
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=60, ha="right", fontsize=8)
    ax.set_yticks(range(len(labels)))
    ax.set_yticklabels(labels, fontsize=8)
    for i in range(len(labels)):
        for j in range(len(labels)):
            val = corr.iat[i, j]
            if np.isfinite(val):
                flag = abs(val) > 0.85 and i != j
                ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=6,
                        fontweight="bold" if flag else "normal", color="#0b0b0b")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="Spearman ρ")
    ax.set_title("Spearman rank correlation of predictors (bold: |ρ| > 0.85)")
    fig.tight_layout()
    fig.savefig(Path(CFG.FIG_DIR) / "02_spearman_correlation.png", dpi=150)
    plt.show()
    plt.close(fig)


def _run_feature_diagnostics(frame):
    """Print/plot descriptive tables, correlation, VIF and missingness; returns a dict of the tables."""
    Path(CFG.FIG_DIR).mkdir(parents=True, exist_ok=True)
    by_epoch = frame.groupby("year")[["lst", "target"] + FEATURES].agg(["mean", "std"]).T
    print("Per-epoch mean / std (rows = variable, statistic):")
    _show_table(by_epoch.round(3))
    print("Rows per epoch:", frame.groupby("year").size().to_dict())

    corr = frame[FEATURES].corr(method="spearman")
    high = [(a, b, float(corr.at[a, b])) for i, a in enumerate(FEATURES) for b in FEATURES[i + 1:]
            if np.isfinite(corr.at[a, b]) and abs(corr.at[a, b]) > 0.85]
    for a, b, rho in high:
        log(f"highly correlated pair: {a} ~ {b} (Spearman rho = {rho:+.2f})", "WARNING")
    if not high:
        log("no predictor pair exceeds |Spearman rho| = 0.85")
    _plot_spearman(corr)

    vif = _vif(frame)
    missing = frame.groupby("year")[FEATURES].apply(lambda g: g.isna().mean()).T
    missing["overall"] = frame[FEATURES].isna().mean()
    summary = pd.DataFrame({"VIF": vif, "missing_overall": missing["overall"]})
    print("VIF (complete cases) and missing share per feature:")
    _show_table(summary.round(3))
    print("Missing share by epoch:")
    _show_table(missing.round(4))
    for name, value in vif.items():
        if np.isfinite(value) and value > 10:
            log(f"VIF({name}) = {value:.1f} > 10 - strong multicollinearity", "WARNING")
    return {"by_epoch": by_epoch, "spearman": corr, "high_corr_pairs": high, "vif": vif, "missing": missing}


_FEATURE_DIAGNOSTICS = _run_feature_diagnostics(DF)

### 2.7 Persist the feature table
`features_long.parquet` (pyarrow, zstd) is the reproducible hand-off of the modelling table.

In [ ]:
def _save_features(frame):
    out_dir = Path(CFG.PARQUET_DIR)
    out_dir.mkdir(parents=True, exist_ok=True)
    path = out_dir / "features_long.parquet"
    frame.to_parquet(path, engine="pyarrow", compression="zstd", index=False)
    log(f"saved {path} ({path.stat().st_size / 1e6:.1f} MB)")
    return path


_save_features(DF)
free_memory()

## 3. Benchmark models and triple cross-validation

**Why three validation schemes?** Land surface temperature is strongly spatially autocorrelated and every cell appears
once per epoch. A plain random K-fold therefore tests a model on rows whose immediate neighbours — and the *same cell in
other epochs* — were in the training set, which rewards interpolation and memorisation and is optimistic.

| Scheme | Split | Question it answers |
|---|---|---|
| `random` | rows randomly assigned to 5 folds (`fold_random`) | optimistic upper bound (interpolation) |
| `spatial` | 5 × 5 tiles over the grid extent, greedily size-balanced into 5 folds (`fold_spatial`); a cell keeps its fold in every epoch | transfer to unseen **places** (~50 km tiles, well beyond the LST correlation range) |
| `temporal` | train on 2010–2020, test on 2025 (fold 0) | forward extrapolation to an unseen **epoch** |

Early stopping for the gradient-boosted models and the GNN uses an *inner* validation set of whole 10 km blocks
(`block10_id`, ~10 % of the training rows), so the outer test fold is never touched. Held-out 10 km blocks still border
fitting blocks and the residual correlation of LST reaches ~20 km, so this split *reduces* but does not remove
adjacency leakage: the chosen stopping round is mildly optimistic (it only sets the number of trees).

In anomaly mode every epoch's target is centred on that epoch's *observed* spatial mean (one constant per epoch,
computed from all its cells, test rows included). For spatial CV this leak is negligible; for temporal CV it means
the 2025 mean is given, so the temporal score measures spatial-pattern transfer, not the climate offset.

**Residual Moran's I** (KNN, k = `CFG.MORAN_K`, row-standardised, permutation inference) is reported per model, scheme
and epoch: residual spatial structure left by a model signals missing spatially structured drivers and warns that
naive standard errors would be too small. The target's own Moran's I gives the reference level of autocorrelation.

| Model | Role | Hardware |
|---|---|---|
| `linear`, `ridge` | transparent baselines (median imputation + standardisation) | CPU |
| `random_forest` | bagged trees | scikit-learn; cuML on GPU only with `LST_USE_RAPIDS=1` (same hyper-parameters) |
| `lightgbm` | leaf-wise GBM | GPU (OpenCL) if the probe succeeds, else CPU |
| `xgboost` | **the explained model** (hist, native NaN handling) | CUDA, folds spread over both T4s |
| `gnn` | spatial GraphSAGE on a KNN graph — does explicit neighbourhood context beat per-cell trees? | `cuda:1` (or `cuda:0` / CPU) |

In [ ]:
import os
import time
import traceback
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import sparse
from scipy.spatial import cKDTree

MODEL_NAMES = ["linear", "ridge", "random_forest", "lightgbm", "xgboost", "gnn"]
SCHEMES = ["random", "spatial", "temporal"]

_X_ALL = DF[FEATURES].to_numpy(np.float32)
_Y_ALL = DF[TARGET_COL].to_numpy(np.float32)
_N_ROWS = len(DF)
_LAST_EPOCH = int(DF["year"].max())


def iter_folds(scheme):
    """Yield ``(fold_id, train_idx, test_idx)`` (int64 row positions in DF order) for a validation scheme."""
    if scheme in ("random", "spatial"):
        fold_col = DF["fold_random" if scheme == "random" else "fold_spatial"].to_numpy()
        for fold_id in range(int(CFG.N_FOLDS)):
            test_idx = np.flatnonzero(fold_col == fold_id)
            train_idx = np.flatnonzero(fold_col != fold_id)
            if test_idx.size == 0 or train_idx.size == 0:
                log(f"{scheme} fold {fold_id} is empty - skipped", "WARNING")
                continue
            yield fold_id, train_idx, test_idx
    elif scheme == "temporal":
        years = DF["year"].to_numpy()
        train_idx, test_idx = np.flatnonzero(years < _LAST_EPOCH), np.flatnonzero(years == _LAST_EPOCH)
        if train_idx.size == 0:
            log("temporal scheme needs at least two epochs - skipped", "WARNING")
            return
        yield 0, train_idx, test_idx
    else:
        raise ValueError(f"unknown scheme {scheme!r}")


def inner_split(train_idx, val_share=0.10):
    """Split training rows into (fit_idx, val_idx) holding out whole 10 km blocks (~``val_share`` of rows).

    Blocks are drawn in a seeded random order until the share is reached; with a single block the split falls back to
    a seeded random row split. Used for early stopping only.
    """
    rng = np.random.default_rng(CFG.SEED)
    groups = DF["block10_id"].to_numpy()[train_idx]
    uniq, counts = np.unique(groups, return_counts=True)
    if uniq.size < 2:
        is_val = np.zeros(train_idx.size, dtype=bool)
        is_val[rng.permutation(train_idx.size)[: max(1, int(round(val_share * train_idx.size)))]] = True
    else:
        order = rng.permutation(uniq.size)
        cum = np.cumsum(counts[order])
        n_take = int(np.searchsorted(cum, val_share * train_idx.size) + 1)
        n_take = min(n_take, uniq.size - 1)  # always keep at least one block for fitting
        is_val = np.isin(groups, uniq[order[:n_take]])
    return train_idx[~is_val], train_idx[is_val]


def _train_medians(train_idx):
    """Column medians of the training rows (0 for all-NaN columns) used for imputation."""
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        med = np.nanmedian(_X_ALL[train_idx], axis=0)
    return np.where(np.isfinite(med), med, 0.0).astype(np.float32)


def _impute(x, medians):
    return np.where(np.isnan(x), medians, x).astype(np.float32)

### 3.1 Model implementations
Each `_fit_<model>(train_idx, test_idx, gpu_id)` fits on the training rows and returns predictions for the test rows
(target units) plus an info dict (`device`, `best_iteration`). GPU paths are real (XGBoost CUDA, LightGBM OpenCL, cuML,
PyTorch). XGBoost and the GNN retry a fold on the CPU after a GPU out-of-memory error; LightGBM and cuML fall back
to the CPU on any GPU error. `best_iteration` is XGBoost's 0-based best round, LightGBM's 1-based
`best_iteration_` and the GNN's best epoch.

In [ ]:
def _fit_sklearn_linear(train_idx, test_idx, regressor):
    from sklearn.impute import SimpleImputer
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler

    model = Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler()), ("reg", regressor)])
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", UserWarning)  # all-NaN columns are dropped by the imputer
        model.fit(_X_ALL[train_idx], _Y_ALL[train_idx])
        pred = model.predict(_X_ALL[test_idx])
    return pred.astype(np.float32), {"device": "cpu", "best_iteration": None}


def _fit_linear(train_idx, test_idx, gpu_id):
    from sklearn.linear_model import LinearRegression

    return _fit_sklearn_linear(train_idx, test_idx, LinearRegression())


def _fit_ridge(train_idx, test_idx, gpu_id):
    from sklearn.linear_model import RidgeCV

    return _fit_sklearn_linear(train_idx, test_idx, RidgeCV(alphas=np.logspace(-3, 3, 13)))


def _fit_random_forest(train_idx, test_idx, gpu_id):
    """cuML RF on GPU when RAPIDS is enabled (LST_USE_RAPIDS), else scikit-learn (train-median-imputed float32).

    Both backends get the same hyper-parameters (300 trees, depth 16, 1/3 of the features per split, >= 2 rows per
    leaf); cuML additionally uses 128-bin histogram splits and one CUDA stream (n_streams=1) so that random_state
    makes it reproducible.
    """
    medians = _train_medians(train_idx)
    x_train, x_test = _impute(_X_ALL[train_idx], medians), _impute(_X_ALL[test_idx], medians)
    n_trees = 60 if CFG.FAST_DEV else 300
    if ENV.get("has_cuml"):
        try:
            import cupy as cp
            from cuml.ensemble import RandomForestRegressor as CuRF

            with cp.cuda.Device(gpu_id):
                model = CuRF(n_estimators=n_trees, max_depth=16, max_features=0.33, min_samples_leaf=2,
                             n_bins=128, n_streams=1, random_state=CFG.SEED)
                model.fit(x_train, _Y_ALL[train_idx])
                pred = np.asarray(model.predict(x_test), dtype=np.float32)
                del model
                cp.get_default_memory_pool().free_all_blocks()
            ENV.setdefault("rapids_used", set()).add("cuml_random_forest")
            return pred, {"device": f"cuda:{gpu_id}", "best_iteration": None}
        except Exception as exc:
            log(f"cuML random forest failed on GPU {gpu_id} ({exc!r}); falling back to scikit-learn", "WARNING")
            free_memory()
    from sklearn.ensemble import RandomForestRegressor

    # Same hyper-parameters as the cuML path; the depth cap also keeps memory bounded (unbounded trees on ~200k
    # rows would need several GB per forest).
    model = RandomForestRegressor(n_estimators=n_trees, max_depth=16, max_features=0.33, min_samples_leaf=2,
                                  n_jobs=-1, random_state=CFG.SEED)
    model.fit(x_train, _Y_ALL[train_idx])
    pred = model.predict(x_test).astype(np.float32)
    del model
    return pred, {"device": "cpu", "best_iteration": None}


def _lgbm_params(device, gpu_id):
    params = dict(n_estimators=300 if CFG.FAST_DEV else 3000, learning_rate=0.03, num_leaves=63, subsample=0.8,
                  subsample_freq=1, colsample_bytree=0.8, min_child_samples=20, reg_lambda=1.0,
                  random_state=CFG.SEED, verbose=-1)
    if device == "gpu":
        params.update(device="gpu", gpu_device_id=int(gpu_id), gpu_platform_id=0)
    return params


def _resolve_lgbm_device():
    """Probe LightGBM GPU support once per session (tiny fit) and cache the answer in ENV['lgbm_device']."""
    if globals().get("_LGBM_DEVICE_PROBED"):
        return ENV["lgbm_device"]
    device = "cpu"
    if ENV.get("n_gpus", 0) > 0:
        try:
            import lightgbm as lgb

            rng = np.random.default_rng(0)
            x_probe, y_probe = rng.normal(size=(256, 4)), rng.normal(size=256)
            lgb.LGBMRegressor(n_estimators=2, device="gpu", verbose=-1).fit(x_probe, y_probe)
            device = "gpu"
        except Exception as exc:
            log(f"LightGBM GPU probe failed ({type(exc).__name__}: {exc}); LightGBM will use the CPU")
    ENV["lgbm_device"] = device
    globals()["_LGBM_DEVICE_PROBED"] = True
    log(f"LightGBM device: {device}")
    return device


def _lgbm_eval_kwargs(x_val, y_val):
    """Validation-set keyword arguments for LGBMRegressor.fit: ``eval_X``/``eval_y`` on LightGBM >= 4.6
    (where ``eval_set`` is deprecated), ``eval_set`` on older releases such as Kaggle's."""
    import inspect

    import lightgbm as lgb

    if "eval_X" in inspect.signature(lgb.LGBMRegressor.fit).parameters:
        return {"eval_X": (x_val,), "eval_y": (y_val,)}
    return {"eval_set": [(x_val, y_val)]}


def _fit_lightgbm(train_idx, test_idx, gpu_id):
    import lightgbm as lgb

    fit_idx, val_idx = inner_split(train_idx)
    device = ENV.get("lgbm_device") or "cpu"
    for attempt in ("primary", "cpu_fallback"):
        try:
            model = lgb.LGBMRegressor(**_lgbm_params(device, gpu_id))
            model.fit(_X_ALL[fit_idx], _Y_ALL[fit_idx], eval_metric="l2",
                      callbacks=[lgb.early_stopping(int(CFG.XGB_EARLY_STOP), verbose=False)],
                      **_lgbm_eval_kwargs(_X_ALL[val_idx], _Y_ALL[val_idx]))
            best = int(model.best_iteration_ or model.n_estimators)  # LightGBM counts iterations from 1
            pred = model.predict(_X_ALL[test_idx], num_iteration=best).astype(np.float32)
            return pred, {"device": "cpu" if device == "cpu" else f"gpu:{gpu_id}", "best_iteration": best}
        except Exception as exc:
            if device == "cpu" or attempt == "cpu_fallback":
                raise
            log(f"LightGBM on GPU {gpu_id} failed ({exc!r}); retrying this fold on CPU", "WARNING")
            free_memory()
            device = "cpu"


def _xgb_params(device):
    params = dict(CFG.XGB_PARAMS)
    params.setdefault("objective", "reg:squarederror")
    params.setdefault("eval_metric", "rmse")
    params.setdefault("tree_method", "hist")
    params.setdefault("max_bin", 256)
    params.update(device=device, seed=int(CFG.SEED))
    return params


def _xgb_on_device(device, gpu_id, fit_idx, val_idx, test_idx):
    """One XGBoost fit/predict on ``device``; data go to the GPU as CuPy arrays when possible."""
    import xgboost as xgb

    params = _xgb_params(device)
    use_cupy = device.startswith("cuda") and ENV.get("has_cupy")
    if use_cupy:
        import cupy as cp

        ctx = cp.cuda.Device(gpu_id)
        to_dev = cp.asarray
    else:
        import contextlib

        ctx = contextlib.nullcontext()
        to_dev = np.asarray
    with ctx:
        dtrain = xgb.QuantileDMatrix(to_dev(_X_ALL[fit_idx]), label=to_dev(_Y_ALL[fit_idx]),
                                     max_bin=int(params["max_bin"]))
        dval = xgb.QuantileDMatrix(to_dev(_X_ALL[val_idx]), label=to_dev(_Y_ALL[val_idx]), ref=dtrain)
        booster = xgb.train(params, dtrain, num_boost_round=int(CFG.XGB_MAX_ROUNDS), evals=[(dval, "val")],
                            early_stopping_rounds=int(CFG.XGB_EARLY_STOP), verbose_eval=False)
        best = int(getattr(booster, "best_iteration", booster.num_boosted_rounds() - 1))
        if use_cupy:
            pred = cp.asnumpy(booster.inplace_predict(cp.asarray(_X_ALL[test_idx]), iteration_range=(0, best + 1)))
        else:
            booster.set_param({"device": "cpu"})  # host data -> host prediction (no device mismatch copies)
            pred = booster.inplace_predict(_X_ALL[test_idx], iteration_range=(0, best + 1))
        del dtrain, dval, booster
        if use_cupy:
            # free this device's CuPy pool inside its own context (free_memory() on the main thread only
            # reaches the current device)
            cp.get_default_memory_pool().free_all_blocks()
    return np.asarray(pred, dtype=np.float32), best


def _fit_xgboost(train_idx, test_idx, gpu_id):
    """xgb.train with QuantileDMatrix + inner-block early stopping; CUDA OOM -> retry on CPU."""
    fit_idx, val_idx = inner_split(train_idx)
    device = xgb_device_for(gpu_id)
    try:
        pred, best = _xgb_on_device(device, gpu_id, fit_idx, val_idx, test_idx)
    except Exception as exc:
        if device == "cpu" or not is_gpu_oom_error(exc):
            raise
        log(f"XGBoost out of memory on {device}; retrying fold on CPU", "WARNING")
        free_memory()
        device = "cpu"
        pred, best = _xgb_on_device(device, gpu_id, fit_idx, val_idx, test_idx)
    return pred, {"device": device, "best_iteration": best}

### 3.2 Spatial GraphSAGE (pure PyTorch)
Nodes are all cell–epochs; edges connect each cell to its k = 8 nearest cells **within the same epoch** (block-diagonal
adjacency across epochs, row-normalised so $A h$ is the neighbour mean). Each layer is
$h' = \mathrm{LayerNorm}(\mathrm{GELU}(W_{self} h + W_{neigh} A h)) + h$ with dropout 0.1; three layers give a 3-hop
(~3–5 km) receptive field. Training is transductive and full-batch: all node features are visible (they are known at
prediction time), the loss uses only training labels, early stopping uses the inner 10 km block validation nodes, and
test labels are never seen. On a CPU-only machine each fold gets a wall-clock budget (`_GNN_CPU_FOLD_BUDGET_S`,
measured after two epochs); the GPU path always trains for the full `CFG.GNN_EPOCHS` (with early stopping).

In [ ]:
_GNN_GRAPH = None


def _gnn_edges(k=8):
    """Directed KNN edges (src -> neighbour) within each epoch, as global DF row positions (cached)."""
    global _GNN_GRAPH
    if _GNN_GRAPH is not None and _GNN_GRAPH["n"] == _N_ROWS:
        return _GNN_GRAPH
    years = DF["year"].to_numpy()
    xy = DF[["x", "y"]].to_numpy(np.float64)
    src_all, dst_all = [], []
    for year in np.unique(years):
        rows = np.flatnonzero(years == year)
        neigh = _knn_indices(xy[rows], k)
        if neigh is None:
            continue
        src_all.append(np.repeat(rows, neigh.shape[1]))
        dst_all.append(rows[neigh.ravel()])
    src = np.concatenate(src_all) if src_all else np.zeros(0, np.int64)
    dst = np.concatenate(dst_all) if dst_all else np.zeros(0, np.int64)
    _GNN_GRAPH = {"n": _N_ROWS, "src": src.astype(np.int64), "dst": dst.astype(np.int64)}
    return _GNN_GRAPH


def _knn_indices(xy, k):
    """(n, k_eff) indices of the k nearest *other* points (exact cKDTree in float64).

    A GPU KNN is deliberately not used: cuML's euclidean metric expands |x|² + |y|² - 2x·y in float32, which on raw
    UTM coordinates (|x|² ~ 1e13 m²) loses more precision than the 1 km cell spacing, and cKDTree needs ~0.1 s per
    epoch anyway.
    """
    n = xy.shape[0]
    k_eff = min(int(k), n - 1)
    if k_eff < 1:
        return None
    _, idx = cKDTree(np.asarray(xy, dtype=np.float64)).query(xy, k=k_eff + 1)
    idx = np.asarray(idx, dtype=np.int64)
    self_first = idx[:, :1] == np.arange(n)[:, None]
    # drop the query point itself (normally column 0; with exact ties it may be elsewhere -> drop the last column)
    return np.where(self_first, idx[:, 1:], idx[:, :-1])


def _build_sage(n_in, hidden=128, n_layers=3, dropout=0.1):
    """Construct the GraphSAGE network (torch imported lazily so the notebook runs without it)."""
    import torch
    from torch import nn

    class _SageLayer(nn.Module):
        def __init__(self):
            super().__init__()
            self.w_self = nn.Linear(hidden, hidden)
            self.w_neigh = nn.Linear(hidden, hidden, bias=False)
            self.norm = nn.LayerNorm(hidden)
            self.drop = nn.Dropout(dropout)

        def forward(self, h, adj):
            agg = torch.sparse.mm(adj, h)
            return self.drop(self.norm(nn.functional.gelu(self.w_self(h) + self.w_neigh(agg)))) + h

    class SpatialSAGE(nn.Module):
        def __init__(self):
            super().__init__()
            self.inp = nn.Linear(n_in, hidden)
            self.layers = nn.ModuleList([_SageLayer() for _ in range(n_layers)])
            self.head = nn.Sequential(nn.Linear(hidden, hidden // 2), nn.GELU(), nn.Linear(hidden // 2, 1))

        def forward(self, x, adj):
            h = self.inp(x)
            for layer in self.layers:
                h = layer(h, adj)
            return self.head(h).squeeze(-1)

    return SpatialSAGE()


def _gnn_device():
    import torch

    n_dev = torch.cuda.device_count() if torch.cuda.is_available() else 0
    return "cuda:1" if n_dev >= 2 else ("cuda:0" if n_dev == 1 else "cpu")


# CPU fallback only: wall-clock budget per fold (env LST_GNN_CPU_BUDGET_S). Full-batch GraphSAGE costs ~0.2 MFLOP per
# node per forward pass, i.e. ~5 s per epoch for 265k cell-epochs on an 8-core CPU versus ~0.02 s on a T4.
_GNN_CPU_FOLD_BUDGET_S = float(os.environ.get("LST_GNN_CPU_BUDGET_S", "900"))


def _cpu_epoch_cap(max_epochs, sec_per_epoch):
    """Cap CPU training epochs to the per-fold budget (the GPU path always runs CFG.GNN_EPOCHS)."""
    affordable = max(20, int(_GNN_CPU_FOLD_BUDGET_S / max(sec_per_epoch, 1e-6)))
    if affordable < max_epochs:
        log(f"GNN on CPU: {sec_per_epoch:.2f} s/epoch -> capping this fold at {affordable} of {max_epochs} epochs "
            f"({_GNN_CPU_FOLD_BUDGET_S:.0f} s budget); use a GPU for the full schedule", "WARNING")
        return affordable
    return max_epochs


def _gnn_train(device, train_idx, test_idx):
    """Full-batch transductive training on ``device``; returns (test predictions, best epoch)."""
    import torch

    set_seeds(int(CFG.SEED))
    fit_idx, val_idx = inner_split(train_idx)
    medians = _train_medians(train_idx)
    x = _impute(_X_ALL, medians)
    mu, sd = x[train_idx].mean(axis=0), x[train_idx].std(axis=0)
    x = (x - mu) / np.where(sd > 0, sd, 1.0)
    y_mu, y_sd = float(_Y_ALL[fit_idx].mean()), float(_Y_ALL[fit_idx].std() or 1.0)

    graph = _gnn_edges(8)
    deg = np.bincount(graph["src"], minlength=_N_ROWS).astype(np.float32)
    weights = 1.0 / deg[graph["src"]]
    dev = torch.device(device)
    adj = torch.sparse_coo_tensor(torch.from_numpy(np.vstack([graph["src"], graph["dst"]])),
                                  torch.from_numpy(weights), (_N_ROWS, _N_ROWS)).coalesce().to(dev)
    if dev.type == "cpu":  # CSR SpMM is ~2x faster than COO on CPU; CUDA keeps the long-supported COO autograd path
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", UserWarning)  # "sparse CSR support is in beta"
            adj = adj.to_sparse_csr()
    x_t = torch.from_numpy(x.astype(np.float32)).to(dev)
    y_t = torch.from_numpy(((_Y_ALL - y_mu) / y_sd).astype(np.float32)).to(dev)
    fit_t, val_t = torch.from_numpy(fit_idx).to(dev), torch.from_numpy(val_idx).to(dev)

    model = _build_sage(x.shape[1]).to(dev)
    opt = torch.optim.AdamW(model.parameters(), lr=3e-3, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=10, min_lr=1e-5)
    best_loss, best_epoch, best_state, stale = np.inf, 0, None, 0
    max_epochs, t_start = int(CFG.GNN_EPOCHS), time.time()
    epoch = 0
    while epoch < max_epochs:
        model.train()
        opt.zero_grad(set_to_none=True)
        loss = torch.nn.functional.mse_loss(model(x_t, adj)[fit_t], y_t[fit_t])
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        model.eval()
        with torch.no_grad():
            val_loss = float(torch.nn.functional.mse_loss(model(x_t, adj)[val_t], y_t[val_t]))
        sched.step(val_loss)
        if val_loss < best_loss - 1e-6:
            best_loss, best_epoch, stale = val_loss, epoch, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            stale += 1
            if stale >= 30:
                break
        if epoch == 1 and dev.type == "cpu":
            max_epochs = _cpu_epoch_cap(max_epochs, (time.time() - t_start) / 2.0)
        epoch += 1
    if best_state is not None:
        model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        pred = model(x_t, adj)[torch.from_numpy(test_idx).to(dev)].float().cpu().numpy()
    del model, opt, adj, x_t, y_t, best_state
    return (pred * y_sd + y_mu).astype(np.float32), best_epoch


def _fit_gnn(train_idx, test_idx, gpu_id):
    """GraphSAGE fold; CUDA OOM -> retry on CPU; the CUDA cache is emptied after every fold."""
    import torch

    device = _gnn_device()
    try:
        pred, best = _gnn_train(device, train_idx, test_idx)
    except Exception as exc:
        if device == "cpu" or not is_gpu_oom_error(exc):
            raise
        log(f"GNN out of memory on {device}; retrying fold on CPU", "WARNING")
        torch.cuda.empty_cache()
        free_memory()
        device = "cpu"
        pred, best = _gnn_train(device, train_idx, test_idx)
    finally:
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    return pred, {"device": device, "best_iteration": best}


_MODEL_FITTERS = {"linear": _fit_linear, "ridge": _fit_ridge, "random_forest": _fit_random_forest,
                  "lightgbm": _fit_lightgbm, "xgboost": _fit_xgboost, "gnn": _fit_gnn}

### 3.3 Cross-validation driver
XGBoost (CUDA) and (with `LST_USE_RAPIDS=1`) cuML random forest distribute their folds over the GPUs with `gpu_pool_map` (one worker per T4,
so two folds run concurrently on `cuda:0`/`cuda:1`); LightGBM-GPU folds run one at a time (OpenCL thread safety). A failing fold or model is logged with its traceback and the run continues; the
affected rows stay NaN in `OOF_PRED` and `n_folds` in `CV_SUMMARY` shows how many folds succeeded.

In [ ]:
def _model_available(model):
    """Skip models whose library is missing (with a warning) instead of failing the run."""
    module = {"lightgbm": "lightgbm", "xgboost": "xgboost", "gnn": "torch"}.get(model)
    if module is None:
        return True
    try:
        __import__(module)
        return True
    except ImportError:
        log(f"{module} is not installed - model '{model}' skipped", "WARNING")
        return False


def _parallel_over_gpus(model):
    """Models whose folds are spread over the GPUs with gpu_pool_map (one fold per GPU at a time).

    LightGBM's OpenCL learner makes no thread-safety guarantee for concurrent boosters in one process (a native
    crash would kill the kernel), so its GPU folds run sequentially.
    """
    return ((model == "xgboost" and ENV.get("xgb_device") == "cuda")
            or (model == "random_forest" and bool(ENV.get("has_cuml"))))


def _regression_metrics(y_true, y_pred):
    y_true, y_pred = y_true.astype(np.float64), y_pred.astype(np.float64)
    resid = y_true - y_pred
    ss_tot = float(((y_true - y_true.mean()) ** 2).sum())
    return {"r2": 1.0 - float(resid @ resid) / ss_tot if ss_tot > 0 else np.nan,
            "rmse": float(np.sqrt(np.mean(resid ** 2))), "mae": float(np.mean(np.abs(resid)))}


def _run_fold(scheme, model, fold, gpu_id):
    """Fit one (scheme, model, fold); returns a result dict (never raises)."""
    fold_id, train_idx, test_idx = fold
    t0 = time.time()
    try:
        pred, info = _MODEL_FITTERS[model](train_idx, test_idx, gpu_id)
        if pred.shape != test_idx.shape or not np.all(np.isfinite(pred)):
            raise ValueError(f"invalid predictions (shape {pred.shape}, finite {np.isfinite(pred).mean():.3f})")
    except Exception:
        return {"fold": fold_id, "error": traceback.format_exc()}
    return {"fold": fold_id, "test_idx": test_idx, "pred": pred, "info": info, "n_train": int(train_idx.size),
            "seconds": time.time() - t0}


def run_scheme(scheme):
    """Run every model on every fold of ``scheme``.

    Returns (list of CV_FOLD_METRICS rows, {model: OOF array}, {model: [info dicts]}).
    """
    folds = list(iter_folds(scheme))
    rows, oof, infos = [], {}, {}
    for model in MODEL_NAMES:
        oof[model] = np.full(_N_ROWS, np.nan, dtype=np.float32)
        infos[model] = []
        if not folds or not _model_available(model):
            continue
        with timer(f"cv_{scheme}_{model}"):
            if _parallel_over_gpus(model):
                results = gpu_pool_map(lambda fold, gpu_id: _run_fold(scheme, model, fold, gpu_id), folds)
            else:
                results = [_run_fold(scheme, model, fold, 0) for fold in folds]
        for res in sorted(results, key=lambda r: r["fold"]):
            if "error" in res:
                log(f"{scheme}/{model} fold {res['fold']} FAILED:\n{res['error']}", "ERROR")
                continue
            oof[model][res["test_idx"]] = res["pred"]
            infos[model].append({"fold": res["fold"], **res["info"]})
            rows.append({"scheme": scheme, "model": model, "fold": int(res["fold"]),
                         **_regression_metrics(_Y_ALL[res["test_idx"]], res["pred"]),
                         "n_train": res["n_train"], "n_test": int(res["test_idx"].size),
                         "fit_seconds": float(res["seconds"]), "device": str(res["info"]["device"])})
        done = [r for r in rows if r["model"] == model]
        if done:
            log(f"{scheme:>8} | {model:<13} R2 = {np.mean([r['r2'] for r in done]):.3f}  "
                f"RMSE = {np.mean([r['rmse'] for r in done]):.3f}  ({len(done)} folds, {done[0]['device']})")
        free_memory()
    return rows, oof, infos


_resolve_lgbm_device()
_fold_rows, OOF_PRED, _FOLD_INFO = [], {}, {}
for _scheme in SCHEMES:
    with timer(f"cv_{_scheme}"):
        _rows, OOF_PRED[_scheme], _FOLD_INFO[_scheme] = run_scheme(_scheme)
    _fold_rows.extend(_rows)

_FOLD_COLUMNS = ["scheme", "model", "fold", "r2", "rmse", "mae", "n_train", "n_test", "fit_seconds", "device"]
CV_FOLD_METRICS = pd.DataFrame(_fold_rows, columns=_FOLD_COLUMNS).astype(
    {"fold": "int64", "n_train": "int64", "n_test": "int64", "r2": "float64", "rmse": "float64", "mae": "float64",
     "fit_seconds": "float64"})
del _fold_rows, _rows

### 3.4 Summary and XGBoost round budget
`std` is the across-fold sample standard deviation (0 for the single temporal fold). `XGB_BEST_ROUNDS` (number of
boosting rounds = best iteration + 1 of each spatial fold) sets the final model's round budget in section 4.

In [ ]:
def _cv_summary(fold_metrics):
    rows = []
    for scheme in SCHEMES:
        for model in MODEL_NAMES:
            sub = fold_metrics[(fold_metrics["scheme"] == scheme) & (fold_metrics["model"] == model)]
            if sub.empty:
                continue
            row = {"scheme": scheme, "model": model}
            for metric in ("r2", "rmse", "mae"):
                row[f"{metric}_mean"] = float(sub[metric].mean())
                row[f"{metric}_std"] = float(sub[metric].std(ddof=1)) if len(sub) > 1 else 0.0
            row["n_folds"] = int(len(sub))
            rows.append(row)
    cols = ["scheme", "model", "r2_mean", "r2_std", "rmse_mean", "rmse_std", "mae_mean", "mae_std", "n_folds"]
    return pd.DataFrame(rows, columns=cols)


def _best_rounds():
    for scheme in ("spatial", "random", "temporal"):
        rounds = [int(i["best_iteration"]) + 1 for i in _FOLD_INFO.get(scheme, {}).get("xgboost", [])
                  if i.get("best_iteration") is not None]
        if rounds:
            if scheme != "spatial":
                log(f"no spatial-CV XGBoost folds succeeded; using {scheme}-CV best rounds", "WARNING")
            return rounds
    fallback = max(1, int(CFG.XGB_MAX_ROUNDS) // 3)
    log(f"no XGBoost CV fold succeeded; XGB_BEST_ROUNDS falls back to [{fallback}]", "WARNING")
    return [fallback]


CV_SUMMARY = _cv_summary(CV_FOLD_METRICS)
XGB_BEST_ROUNDS = _best_rounds()
log(f"XGB_BEST_ROUNDS (spatial CV) = {XGB_BEST_ROUNDS}")
try:
    from IPython.display import display

    display(CV_SUMMARY.round(4))
except Exception:
    print(CV_SUMMARY.round(4).to_string())

### 3.5 Spatial autocorrelation of residuals (Moran's I)
$$I = \frac{n}{S_0}\,\frac{\sum_i\sum_j w_{ij} z_i z_j}{\sum_i z_i^2},\qquad E[I] = -\frac{1}{n-1}$$
with $z$ the mean-centred residuals, $w_{ij}$ row-standardised KNN weights ($S_0 = n$). Inference is by random
permutation (`CFG.MORAN_PERMS`): the pseudo p-value is the one-sided folded count $(\min(\#\{I_{sim} \ge I\}, P - \#\{\cdot\}) + 1)/(P + 1)$
and $z = (I - \bar I_{sim})/\mathrm{sd}(I_{sim})$ — exactly PySAL's `p_sim` / `z_sim`. `esda.Moran` is used when
installed; the vectorised NumPy/SciPy fallback computes the same statistics (checked below).

In [ ]:
def _knn_weights(xy, k):
    """Row-standardised KNN weights (csr, n x n) over the given coordinates."""
    n = xy.shape[0]
    neigh = _knn_indices(xy, k)
    if neigh is None:
        return sparse.csr_matrix((n, n), dtype=np.float64)
    k_eff = neigh.shape[1]
    rows = np.repeat(np.arange(n), k_eff)
    return sparse.csr_matrix((np.full(rows.size, 1.0 / k_eff), (rows, neigh.ravel())), shape=(n, n))


_EPOCH_ROWS = {int(y): np.flatnonzero(DF["year"].to_numpy() == y) for y in np.unique(DF["year"].to_numpy())}
_XY_ALL = DF[["x", "y"]].to_numpy(np.float64)
SPATIAL_WEIGHTS = {year: _knn_weights(_XY_ALL[rows], int(CFG.MORAN_K)) for year, rows in _EPOCH_ROWS.items()}

_PYSAL_W_CACHE = {}


def _pysal_weights(W):
    """libpysal W for a csr matrix (cached per matrix object; the object is kept alive by the cache)."""
    key = id(W)
    if key not in _PYSAL_W_CACHE:
        from libpysal.weights import W as PysalW

        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            _PYSAL_W_CACHE[key] = (W, PysalW.from_sparse(sparse.csr_matrix(W)))
    return _PYSAL_W_CACHE[key][1]


def _moran_numpy(y, W, permutations, seed):
    """Moran's I with permutation inference, vectorised in batches of permutations (NumPy/SciPy)."""
    n = y.size
    z = y - y.mean()
    z2 = float(z @ z)
    s0 = float(W.sum())
    if z2 == 0 or s0 == 0:
        return {"I": np.nan, "expected_I": -1.0 / (n - 1), "z": np.nan, "p": np.nan}
    scale = n / s0 / z2
    moran = scale * float(z @ (W @ z))
    out = {"I": moran, "expected_I": -1.0 / (n - 1), "z": np.nan, "p": np.nan}
    if permutations:
        rng = np.random.default_rng(seed)
        batch = max(1, min(int(permutations), 4_000_000 // max(n, 1)))
        sims = []
        for start in range(0, int(permutations), batch):
            perm = np.tile(z, (min(batch, int(permutations) - start), 1))
            rng.permuted(perm, axis=1, out=perm)
            lag = (W @ perm.T).T
            sims.append(scale * np.einsum("bi,bi->b", perm, lag))
        sims = np.concatenate(sims)
        larger = int((sims >= moran).sum())
        larger = min(larger, int(permutations) - larger)
        sd = sims.std()
        out["p"] = (larger + 1.0) / (permutations + 1.0)
        out["z"] = (moran - sims.mean()) / sd if sd > 0 else np.nan
    return out


def morans_i(values, W, permutations=None, *, backend="auto"):
    """Global Moran's I of ``values`` with row-standardised weights ``W`` (csr).

    Returns ``{"I", "expected_I", "z", "p", "backend"}`` with permutation-based z and pseudo-p
    (``esda`` ``z_sim``/``p_sim``; analytic normal z/p when ``permutations == 0``).
    ``backend``: "auto" (esda if installed), "esda" or "numpy".
    """
    y = np.asarray(values, dtype=np.float64).ravel()
    if y.size != W.shape[0]:
        raise ValueError(f"values ({y.size}) and weights ({W.shape}) differ in size")
    perms = int(CFG.MORAN_PERMS if permutations is None else permutations)
    if y.size < 3 or not np.all(np.isfinite(y)):
        return {"I": np.nan, "expected_I": np.nan, "z": np.nan, "p": np.nan, "backend": "none"}
    if backend in ("auto", "esda"):
        try:
            import esda

            state = np.random.get_state()
            np.random.seed(int(CFG.SEED))  # esda permutes with the global NumPy RNG
            try:
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore")
                    mi = esda.Moran(y, _pysal_weights(W), transformation="r", permutations=perms)
            finally:
                np.random.set_state(state)
            z, p = (mi.z_sim, mi.p_sim) if perms else (mi.z_norm, mi.p_norm)
            return {"I": float(mi.I), "expected_I": float(mi.EI), "z": float(z), "p": float(p), "backend": "esda"}
        except ImportError:
            if backend == "esda":
                raise
    res = _moran_numpy(y, W, perms, int(CFG.SEED))
    if not perms:  # analytic z under normality for row-standardised W
        s0 = float(W.sum())
        s1 = 0.5 * float((W + W.T).multiply(W + W.T).sum())
        s2 = float(((np.asarray(W.sum(axis=1)).ravel() + np.asarray(W.sum(axis=0)).ravel()) ** 2).sum())
        n = y.size
        var = (n * n * s1 - n * s2 + 3 * s0 * s0) / ((n - 1) * (n + 1) * s0 * s0) - (1.0 / (n - 1)) ** 2
        from scipy.stats import norm

        res["z"] = (res["I"] - res["expected_I"]) / np.sqrt(var)
        res["p"] = float(2 * norm.sf(abs(res["z"])))
    res["backend"] = "numpy"
    return {k: (float(v) if k != "backend" else v) for k, v in res.items()}


def _check_moran_backends():
    """Runtime consistency check: esda and the NumPy fallback must give the same I on the first epoch."""
    year = min(_EPOCH_ROWS)
    y = _Y_ALL[_EPOCH_ROWS[year]]
    ref = morans_i(y, SPATIAL_WEIGHTS[year], 0, backend="numpy")
    try:
        other = morans_i(y, SPATIAL_WEIGHTS[year], 0, backend="esda")
    except ImportError:
        log("esda/libpysal not installed - Moran's I uses the NumPy/SciPy implementation")
        return
    diff = abs(ref["I"] - other["I"]) + abs(ref["z"] - other["z"])
    level = "INFO" if diff < 1e-8 else "WARNING"
    log(f"Moran backends agree: |dI| + |dz| = {diff:.2e} (esda vs numpy, year {year})", level)


def _residual_moran(scheme, model, year):
    """Moran's I of OOF residuals for one epoch; KNN rebuilt on the predicted subset when rows are missing."""
    rows = _EPOCH_ROWS[year]
    pred = OOF_PRED[scheme][model][rows]
    ok = np.isfinite(pred)
    if ok.sum() < max(10, int(CFG.MORAN_K) + 2):
        return None
    resid = _Y_ALL[rows].astype(np.float64) - pred.astype(np.float64)
    if ok.all():
        return morans_i(resid, SPATIAL_WEIGHTS[year])
    W_sub = _knn_weights(_XY_ALL[rows[ok]], int(CFG.MORAN_K))
    return morans_i(resid[ok], W_sub)


def _moran_tables():
    target_rows, result_rows = [], []
    for year, rows in _EPOCH_ROWS.items():
        res = morans_i(_Y_ALL[rows], SPATIAL_WEIGHTS[year])
        target_rows.append({"year": year, **{k: res[k] for k in ("I", "expected_I", "z", "p")}})
    for scheme in SCHEMES:
        for model in MODEL_NAMES:
            for year in _EPOCH_ROWS:
                res = _residual_moran(scheme, model, year)
                if res is not None:
                    result_rows.append({"scheme": scheme, "model": model, "year": year,
                                        **{k: res[k] for k in ("I", "expected_I", "z", "p")}})
    cols = ["scheme", "model", "year", "I", "expected_I", "z", "p"]
    return (pd.DataFrame(result_rows, columns=cols).astype({"year": "int64"}),
            pd.DataFrame(target_rows, columns=["year", "I", "expected_I", "z", "p"]).astype({"year": "int64"}))


def _moran_scatter(max_points=2000):
    """Moran scatter data per model (spatial scheme, last epoch): standardised residual z vs spatial lag W z."""
    out = {}
    rows = _EPOCH_ROWS[_LAST_EPOCH]
    W = SPATIAL_WEIGHTS[_LAST_EPOCH]
    rng = np.random.default_rng(CFG.SEED)
    for model in MODEL_NAMES:
        pred = OOF_PRED["spatial"][model][rows]
        if not np.all(np.isfinite(pred)):
            continue
        resid = _Y_ALL[rows].astype(np.float64) - pred
        sd = resid.std()
        if sd == 0:
            continue
        z = (resid - resid.mean()) / sd
        lag = W @ z
        sel = MORAN_RESULTS[(MORAN_RESULTS["scheme"] == "spatial") & (MORAN_RESULTS["model"] == model)
                            & (MORAN_RESULTS["year"] == _LAST_EPOCH)]
        slope = float(sel["I"].iloc[0]) if len(sel) else float(z @ lag / (z @ z))
        pick = np.sort(rng.choice(z.size, size=min(max_points, z.size), replace=False))
        out[model] = {"year": _LAST_EPOCH, "z": z[pick].astype(float).tolist(),
                      "lag": np.asarray(lag)[pick].astype(float).tolist(), "slope": slope}
    return out


with timer("moran"):
    _check_moran_backends()
    MORAN_RESULTS, MORAN_TARGET = _moran_tables()
    MORAN_SCATTER = _moran_scatter()
print("Moran's I of the target by epoch:")
print(MORAN_TARGET.round(4).to_string(index=False))
print("Moran's I of spatial-CV residuals (model x year):")
print(MORAN_RESULTS[MORAN_RESULTS["scheme"] == "spatial"].pivot(index="model", columns="year", values="I")
      .reindex([m for m in MODEL_NAMES if m in set(MORAN_RESULTS["model"])]).round(3).to_string())

### 3.6 Figures
Grouped bars compare models across the three schemes (error bars = across-fold s.d.); the heatmap shows residual
Moran's I by model and epoch under spatial CV; the hexbin checks calibration of the spatial-CV XGBoost predictions.

In [ ]:
_SCHEME_COLORS = {"random": "#2a78d6", "spatial": "#eb6834", "temporal": "#1baf7a"}


def _plot_cv_bars():
    import matplotlib.pyplot as plt

    models = [m for m in MODEL_NAMES if m in set(CV_SUMMARY["model"])]
    if not models:
        log("no CV results to plot", "WARNING")
        return
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
    width = 0.8 / len(SCHEMES)
    xpos = np.arange(len(models))
    for ax, metric, label in ((axes[0], "r2", "R²"), (axes[1], "rmse", "RMSE (°C)")):
        for s_i, scheme in enumerate(SCHEMES):
            sub = CV_SUMMARY[CV_SUMMARY["scheme"] == scheme].set_index("model").reindex(models)
            ax.bar(xpos + (s_i - (len(SCHEMES) - 1) / 2) * width, sub[f"{metric}_mean"], width * 0.92,
                   yerr=sub[f"{metric}_std"], capsize=3, color=_SCHEME_COLORS[scheme], label=scheme,
                   error_kw={"elinewidth": 1, "ecolor": "#52514e"})
        ax.set_xticks(xpos)
        ax.set_xticklabels(models, rotation=20)
        ax.set_ylabel(label)
        ax.grid(axis="y", color="#e0dfdb", linewidth=0.6)
        ax.set_axisbelow(True)
        for side in ("top", "right"):
            ax.spines[side].set_visible(False)
    axes[0].legend(title="CV scheme", frameon=False)
    axes[0].set_title("Out-of-fold R² by model and scheme")
    axes[1].set_title("Out-of-fold RMSE by model and scheme")
    fig.tight_layout()
    fig.savefig(Path(CFG.FIG_DIR) / "03_cv_performance.png", dpi=150)
    plt.show()
    plt.close(fig)


def _plot_moran_heatmap():
    import matplotlib.pyplot as plt
    from matplotlib.colors import LinearSegmentedColormap

    sub = MORAN_RESULTS[MORAN_RESULTS["scheme"] == "spatial"]
    if sub.empty:
        log("no spatial-CV Moran results to plot", "WARNING")
        return
    table = sub.pivot(index="model", columns="year", values="I")
    table = table.reindex([m for m in MODEL_NAMES if m in table.index])
    vmax = max(0.05, float(np.nanmax(np.abs(table.to_numpy()))))
    cmap = LinearSegmentedColormap.from_list("div", ["#2a78d6", "#f0efec", "#e34948"])
    fig, ax = plt.subplots(figsize=(1.6 * table.shape[1] + 3, 0.55 * table.shape[0] + 1.8))
    im = ax.imshow(table.to_numpy(), cmap=cmap, vmin=-vmax, vmax=vmax, aspect="auto")
    ax.set_xticks(range(table.shape[1]))
    ax.set_xticklabels(table.columns)
    ax.set_yticks(range(table.shape[0]))
    ax.set_yticklabels(table.index)
    for i in range(table.shape[0]):
        for j in range(table.shape[1]):
            if np.isfinite(table.iat[i, j]):
                ax.text(j, i, f"{table.iat[i, j]:.2f}", ha="center", va="center", fontsize=9, color="#0b0b0b")
    fig.colorbar(im, ax=ax, label="Moran's I of residuals")
    ax.set_title("Residual spatial autocorrelation (spatial CV)")
    fig.tight_layout()
    fig.savefig(Path(CFG.FIG_DIR) / "03_moran_residuals_heatmap.png", dpi=150)
    plt.show()
    plt.close(fig)


def _plot_pred_vs_obs():
    """Hexbin of spatial-CV XGBoost OOF predictions against observations.

    The axes are in °C (anomaly + observed epoch mean) for readability, but the title reports the metrics in TARGET
    units (the anomaly in the default mode) - the quantity the CV actually scores. Adding the per-epoch means back to
    both axes would credit the inter-epoch variance to the model for free and inflate R².
    """
    import matplotlib.pyplot as plt

    pred = OOF_PRED["spatial"].get("xgboost")
    ok = np.isfinite(pred) if pred is not None else None
    if pred is None or not ok.any():
        log("no spatial XGBoost OOF predictions to plot", "WARNING")
        return
    offset = DF["epoch_mean"].to_numpy(np.float64) if CFG.TARGET_MODE == "anomaly" else 0.0
    pred_c = (pred + offset)[ok]
    obs_c = DF["lst"].to_numpy(np.float64)[ok]
    metrics = _regression_metrics(_Y_ALL[ok], pred[ok])   # target units (anomaly), as in CV_SUMMARY
    lo, hi = np.percentile(np.concatenate([obs_c, pred_c]), [0.5, 99.5])
    fig, ax = plt.subplots(figsize=(6, 5.4))
    hb = ax.hexbin(obs_c, pred_c, gridsize=60, extent=(lo, hi, lo, hi), mincnt=1, cmap="Blues", bins="log")
    ax.plot([lo, hi], [lo, hi], color="#52514e", linewidth=1, linestyle="--", label="1:1")
    unit = "anomaly" if CFG.TARGET_MODE == "anomaly" else "°C"
    suffix = "; anomaly + observed epoch mean" if CFG.TARGET_MODE == "anomaly" else ""
    ax.set_xlabel(f"Observed LST (°C{suffix})")
    ax.set_ylabel(f"Predicted LST, spatial-CV OOF (°C{suffix})")
    ax.set_title(f"XGBoost spatial CV ({unit}): R² = {metrics['r2']:.3f}, RMSE = {metrics['rmse']:.2f} °C",
                 fontsize=10)
    ax.legend(frameon=False, loc="upper left")
    fig.colorbar(hb, ax=ax, label="cells (log)")
    fig.tight_layout()
    fig.savefig(Path(CFG.FIG_DIR) / "03_pred_vs_obs_hexbin.png", dpi=150)
    plt.show()
    plt.close(fig)


Path(CFG.FIG_DIR).mkdir(parents=True, exist_ok=True)
for _plot in (_plot_cv_bars, _plot_moran_heatmap, _plot_pred_vs_obs):
    try:
        _plot()
    except Exception:
        log(f"figure {_plot.__name__} failed:\n{traceback.format_exc()}", "WARNING")

### 3.7 Persist validation tables

In [ ]:
def _save_validation_tables():
    out_dir = Path(CFG.PARQUET_DIR)
    out_dir.mkdir(parents=True, exist_ok=True)
    for name, frame in (("cv_fold_metrics", CV_FOLD_METRICS), ("cv_summary", CV_SUMMARY),
                        ("moran_results", MORAN_RESULTS)):
        path = out_dir / f"{name}.parquet"
        frame.to_parquet(path, engine="pyarrow", compression="zstd", index=False)
        log(f"saved {path} ({len(frame)} rows)")


_save_validation_tables()
_PYSAL_W_CACHE.clear()
_GNN_GRAPH = None
free_memory()

# 4. Explainability: final model, GPU TreeSHAP, interactions, bootstrap thresholds and governance zones

## 4.1 Why TreeSHAP
For a model prediction $f(x)$ the **Shapley value** of feature $j$ is its average marginal contribution over
all orderings in which features can be "switched on":

$$\phi_j(x)=\sum_{S\subseteq F\setminus\{j\}}\frac{|S|!\,(|F|-|S|-1)!}{|F|!}\big[f_x(S\cup\{j\})-f_x(S)\big],\qquad f_x(S)=E[f(X)\mid X_S=x_S].$$

It is the unique attribution satisfying *efficiency/additivity* ($f(x)=\phi_0+\sum_j\phi_j$, with the bias
$\phi_0=E[f(X)]$), *symmetry*, *dummy* and *linearity*. **TreeSHAP** (Lundberg et al. 2020) computes the exact
values for tree ensembles in $O(TLD^2)$ by pushing all subsets down each tree simultaneously, using the tree's
own cover statistics as the conditional expectation ("path-dependent" TreeSHAP). XGBoost ships a CUDA
implementation (GPUTreeShap, Mitchell et al. 2022) reached through `predict(..., pred_contribs=True)`. For the
final model here (~900 trees of depth 8) a mid-range GPU is ~20× faster than a multi-core CPU (measured: GTX 1650
≈ 2,200 rows/s vs a 4-core CPU ≈ 100 rows/s; a T4 is ~2× the GTX 1650), which is what makes explaining every
cell × epoch in every bootstrap replicate below feasible.

**SHAP interaction values** split each $\phi_j$ into a main effect $\phi_{jj}$ and pairwise terms
$\phi_{jk}=\phi_{kj}$ (each holding half of the joint interaction) such that $\sum_k\phi_{jk}=\phi_j$. We store the
mean $|\phi_{jk}|$ matrix (diagonal = main effects) globally and per zone; the strongest off-diagonal partner of a
feature colours its dependence plot.

**Additivity is checked, not assumed:** for every row, $\phi_0+\sum_j\phi_j$ must equal the raw margin
`predict(output_margin=True)` within $10^{-3}$ (target units), otherwise the section stops.

**What SHAP is not.** $\phi_j(x)$ says how far the *current value* of feature $j$ pushes this cell's prediction away
from the average cell's prediction (a level), not how much the prediction would change if the feature moved (a
slope). A cell whose NDVI is already past the cooling saturation has a large negative NDVI SHAP value but responds
no further to more greening. SHAP explains the model, not causation.

## 4.2 Why cluster in SHAP space ("supervised clustering")
Clustering raw features groups cells that *look* alike (e.g. all bright surfaces), regardless of whether
that matters for temperature. Clustering the SHAP matrix groups cells whose **predicted temperature anomaly is
attributed to the same features in similar amounts**: every coordinate is already expressed in the model's °C
attribution, so no feature scaling is required and irrelevant features contribute ~0 distance. The resulting
K-means clusters (K = 4, pooled over epochs so zones are comparable through time) are *governance zones*: areas
with a similar current state relative to the average cell. This is **not** a measure of how responsive the zones
are to interventions - that is estimated separately, per zone, with model counterfactuals (4.10, 4.14).

**Zone-labelling heuristic** (canonical ids are fixed; names follow the SHAP signatures):
1. **3 · Heat Extreme Core** = the cluster with the highest mean total SHAP $\sum_j\phi_j$ (hottest predicted anomaly);
2. **1 · Riparian Buffer** = among the remaining three, the one with the most negative mean
   $(\phi_{\text{frac\_water}}+\phi_{\text{ndwi}})$; if none of the three has a negative mean, the one with the highest
   mean water fraction;
3. **0 · Ecological Cool Base** = of the remaining two, the one with the lower mean total SHAP;
4. **2 · Transition** = the last one.

The water cluster is named before the cool/transition split, so a small, strongly water-cooled cluster becomes the
Riparian Buffer rather than the Cool Base. The run prints which cluster got which name and why (also exported as
`zones.json → labelling`).

## 4.3 Spatial block bootstrap
Uncertainty of explanations comes mostly from *which places* the model learned from. Cells are spatially
autocorrelated, so resampling rows would understate variance, and so do blocks smaller than the correlation
range. The block size is `CFG.BOOT_BLOCK_KM` or, by default, the distance at which the correlogram of the
spatial-CV XGBoost residuals (last epoch) drops below 0.1, rounded up to 5 km (at least `INNER_BLOCK_KM`, and
shrunk if fewer than 50 blocks would remain). Blocks are resampled with replacement; each row inherits its block's
multiplicity as a sample weight, and a fresh XGBoost model with `FINAL_ROUNDS` trees (no early stopping, so every
replicate has the same capacity) is trained per replicate. Replicates are spread across the GPUs (`gpu_pool_map`).
Each replicate is explained on the full data (GPU) or on ≤ 20k rows plus the whole last epoch (CPU) and
**immediately reduced** to: global mean |SHAP|, binned dependence curves (fixed quantile bins), threshold
estimates, last-epoch per-cell SHAP (float16) and the zone recommendation counterfactuals - so memory stays
bounded however many replicates are run.

**Intervals.** Replicates are refits on ~63 % of the unique blocks with duplicates as weights, so their estimates
are biased relative to the full-data model (e.g. weak features get more mean |SHAP|). Plain percentile intervals
assume no bias and then often exclude the full-data estimate. All intervals are therefore **bias-corrected
percentile intervals**: the 2.5/97.5 percentiles of the replicate distribution re-centred on the full-data estimate,
$[\hat\theta + q_{2.5} - \tilde\theta^*,\ \hat\theta + q_{97.5} - \tilde\theta^*]$ ($\tilde\theta^*$ = replicate median),
which always contain $\hat\theta$; the bootstrap bias $\tilde\theta^* - \hat\theta$ is reported next to the estimate.
With fewer than 50 replicates the 2.5/97.5 percentiles are essentially the extreme replicates, so the interval is
labelled "replicate range" instead of "95% CI" (`BOOTSTRAP_INFO["ci_label"]`, `replicate_range = true`; both are
exported to the manifest). The method code is `ci_method = "recentred-percentile"`.

## 4.4 Threshold definitions (binned mean SHAP curve, quantile bins fixed on the full data)
Bins are `CFG.DEP_BINS` equal-count quantile bins; a value that holds ≥ 1/`DEP_BINS` of all rows (e.g.
`frac_water = 0` in most cells, `frac_cropland = 1`, CONTAG = 100) gets a bin of its own and the other bins are
quantiles of the remaining values — otherwise such point masses collapse the quantiles and leave a curve of only
2–3 points exactly where the effect of a rare land cover lives.

* **zero_crossing** – x where the curve changes sign (linear interpolation). Reported only when the curve has a
  single sign change, or one crossing whose jump |Δ| is at least twice that of every other; otherwise null with
  `zero_crossing_flag = "multiple"`. Zero means "the prediction of the average cell" (the SHAP baseline), so a
  crossing is where the feature stops pushing the prediction above/below average - not an intrinsic physical
  threshold. Bootstrap replicates use the crossing nearest to the full-data one.
* **breakpoint** – best continuous two-segment (hinge) weighted least-squares fit, hinge candidates between the
  10th and 90th percentile bins; reported with the slopes before/after. It is withheld (null) when the hinge
  explains < 10 % of the residual variance of a straight line (no real kink), when the best hinge sits at the edge
  of the candidate range (a range limit, not a kink), or when either side has fewer than 5 bins or < 5 % of the rows.
* **saturation** – after the steepest part of the smoothed curve, the first x where |slope| stays below 10 % of
  its maximum for ≥ 3 bins; refined to the local level-off knee to remove the one-bin lag of the smoother.
* **direction** – sign of Spearman ρ(x, SHAP); |ρ| < 0.2 → "mixed".
* `<k>_support` = share of bootstrap replicates that produced the threshold. The interval `<k>_ci` is given only
  when the point estimate exists and support ≥ 0.5; thresholds with support < 0.8 should be read as tentative.

In [ ]:
import json
import math
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xgboost as xgb

_K = len(FEATURES)
_EXPL_X = DF[FEATURES].to_numpy(dtype=np.float32)          # (N, K), row-aligned with DF
_EXPL_Y = DF[TARGET_COL].to_numpy(dtype=np.float32)
_YEARS = DF["year"].to_numpy().astype(np.int64)
_EPOCH_LIST = sorted(int(y) for y in np.unique(_YEARS))
_LAST_YEAR = _EPOCH_LIST[-1]
# Offset that converts target units back to °C (anomaly mode adds the epoch spatial mean back).
_EPOCH_OFFSET = (
    np.array([float(EPOCH_MEANS[int(y)]) for y in _YEARS], dtype=np.float64)
    if CFG.TARGET_MODE == "anomaly" else np.zeros(len(DF), dtype=np.float64)
)
_PRIMARY_DEVICE = xgb_device_for(0)
_MIN_PRED_BATCH = 16
_CPU_INTERACTION_CAP = 4000        # rows per epoch when the interaction pass has to run on the CPU
# sklearn-style or deprecated keys that must not reach xgb.train.
_XGB_NON_TRAIN_KEYS = {
    "n_estimators", "num_boost_round", "early_stopping_rounds", "n_jobs", "gpu_id", "predictor",
    "device", "random_state", "verbose", "enable_categorical", "callbacks",
}
_PRED_KWARGS = {
    "margin": {"output_margin": True},
    "contribs": {"pred_contribs": True},
    "interactions": {"pred_interactions": True},
}


def _final_xgb_params(device, seed):
    """Training parameters: CFG.XGB_PARAMS minus sklearn-only keys, forced to `hist` on `device`."""
    params = {k: v for k, v in dict(CFG.XGB_PARAMS).items() if k not in _XGB_NON_TRAIN_KEYS}
    params["tree_method"] = "hist"            # the only method that runs on both CUDA and CPU
    params.setdefault("objective", "reg:squarederror")
    params.setdefault("max_bin", 256)
    params["device"] = device
    params["seed"] = int(seed)
    params["verbosity"] = 0
    return params


def _train_once(X, y, weight, rounds, device, seed):
    params = _final_xgb_params(device, seed)
    # QuantileDMatrix sketches the histogram bins once without materialising a second copy of X.
    dtrain = xgb.QuantileDMatrix(X, label=y, weight=weight, max_bin=int(params["max_bin"]),
                                 feature_names=list(FEATURES))
    try:
        return xgb.train(params, dtrain, num_boost_round=int(rounds), verbose_eval=False)
    finally:
        del dtrain


def _train_booster(X, y, *, rounds, device, seed, weight=None):
    """Train an XGBoost booster; a CUDA out-of-memory error falls back to the CPU (host MemoryError propagates)."""
    try:
        return _train_once(X, y, weight, rounds, device, seed)
    except Exception as exc:
        if device == "cpu" or not is_gpu_oom_error(exc):
            raise
        log(f"GPU OOM while training on {device} ({exc}); retrying on CPU", "WARNING")
        return _train_once(X, y, weight, rounds, "cpu", seed)


def _iter_predict(booster, X, *, device, batch, kind, state=None):
    """Yield (start, stop, output) over row batches of X.

    kind: "margin" | "contribs" | "interactions". On an out-of-memory error the batch is halved and retried; once
    the batch is at its minimum a *GPU* OOM moves the booster to the CPU (recorded in ``state["cpu_fallback"]`` when a
    dict is passed, so callers can shrink the remaining work) and a host OOM propagates.
    """
    booster.set_param({"device": device})
    batch = max(_MIN_PRED_BATCH, int(batch))
    start, n = 0, len(X)
    while start < n:
        stop = min(n, start + batch)
        try:
            dmat = xgb.DMatrix(X[start:stop], missing=np.nan, feature_names=list(FEATURES))
            out = booster.predict(dmat, **_PRED_KWARGS[kind])
            del dmat
        except Exception as exc:
            gpu_oom = device != "cpu" and is_gpu_oom_error(exc)
            if not (gpu_oom or is_oom_error(exc)):
                raise
            if batch > _MIN_PRED_BATCH:
                batch = max(_MIN_PRED_BATCH, batch // 2)
                log(f"OOM in {kind} prediction; batch halved to {batch}", "WARNING")
            elif gpu_oom:
                device = "cpu"
                booster.set_param({"device": "cpu"})
                if state is not None:
                    state["cpu_fallback"] = True
                log(f"GPU OOM in {kind} prediction at minimum batch; falling back to CPU", "WARNING")
            else:
                raise
            continue
        yield start, stop, out
        start = stop


def _predict_all(booster, X, *, device, batch, kind):
    """Concatenate `_iter_predict` output into one float32 array."""
    parts = [out.astype(np.float32, copy=False) for _, _, out in
             _iter_predict(booster, X, device=device, batch=batch, kind=kind)]
    return np.concatenate(parts, axis=0)

## 4.5 Final model (all rows)
One pooled XGBoost model on every cell × epoch. The number of trees is 1.1 × the median early-stopping round
of the spatial-CV folds (more data than any CV fold → slightly more trees), falling back to 800.

In [ ]:
_final_best_rounds = [int(r) for r in (XGB_BEST_ROUNDS or []) if r is not None and int(r) > 0]
FINAL_ROUNDS = int(round(1.1 * float(np.median(_final_best_rounds)))) if _final_best_rounds else 800
log(f"Final model: {FINAL_ROUNDS} rounds (spatial-CV best rounds {_final_best_rounds or 'n/a'}), "
    f"{len(DF):,} rows x {_K} features on {_PRIMARY_DEVICE}")
with timer("04_final_model"):
    FINAL_MODEL = _train_booster(_EXPL_X, _EXPL_Y, rounds=FINAL_ROUNDS, device=_PRIMARY_DEVICE, seed=CFG.SEED)
    _FINAL_MARGIN = _predict_all(FINAL_MODEL, _EXPL_X, device=_PRIMARY_DEVICE,
                                 batch=CFG.SHAP_BATCH, kind="margin").astype(np.float64)

DF["lst_pred"] = (_FINAL_MARGIN + _EPOCH_OFFSET).astype(np.float32)
_oof = OOF_PRED.get("spatial", {}).get("xgboost")
if _oof is None or len(_oof) != len(DF):
    log("Spatial-CV XGBoost OOF predictions missing; lst_pred_oof is NaN", "WARNING")
    DF["lst_pred_oof"] = np.full(len(DF), np.nan, dtype=np.float32)
else:
    DF["lst_pred_oof"] = (np.asarray(_oof, dtype=np.float64) + _EPOCH_OFFSET).astype(np.float32)
DF["resid_oof"] = (DF["lst"].astype(np.float64) - DF["lst_pred_oof"].astype(np.float64)).astype(np.float32)
_train_rmse = float(np.sqrt(np.mean((DF["lst_pred"].to_numpy(np.float64) - DF["lst"].to_numpy(np.float64)) ** 2)))
_oof_rmse = float(np.sqrt(np.nanmean(DF["resid_oof"].to_numpy(np.float64) ** 2)))
log(f"Final model in-sample RMSE {_train_rmse:.3f} °C; spatial-CV OOF RMSE {_oof_rmse:.3f} °C")
free_memory()

## 4.6 GPU TreeSHAP for every cell × epoch (additivity-checked)

In [ ]:
with timer("04_treeshap"):
    _contribs = _predict_all(FINAL_MODEL, _EXPL_X, device=_PRIMARY_DEVICE, batch=CFG.SHAP_BATCH, kind="contribs")
    SHAP_VALUES = np.ascontiguousarray(_contribs[:, :_K], dtype=np.float32)
    _bias = _contribs[:, _K].astype(np.float64)
    SHAP_BASE = float(np.median(_bias))
    if float(np.ptp(_bias)) > 1e-4:
        log(f"SHAP bias column varies by {np.ptp(_bias):.2e} across rows (expected constant)", "WARNING")
    _additivity_err = float(np.max(np.abs(SHAP_VALUES.sum(axis=1, dtype=np.float64) + _bias - _FINAL_MARGIN)))
    del _contribs
    if not _additivity_err < 1e-3:
        raise RuntimeError(f"TreeSHAP additivity violated: max |phi0 + sum(phi) - margin| = {_additivity_err:.3e}")
    log(f"SHAP_VALUES {SHAP_VALUES.shape}, base {SHAP_BASE:.4f} (target units), "
        f"additivity max error {_additivity_err:.2e}")


def _crosscheck_with_shap_package(n_rows=300):
    """Compare XGBoost's native TreeSHAP with shap.TreeExplainer on a few rows (warn-only)."""
    try:
        import shap
    except ImportError:
        log("shap package not installed; skipping TreeExplainer cross-check")
        return None
    rows = np.random.default_rng(CFG.SEED).choice(len(DF), size=min(n_rows, len(DF)), replace=False)
    try:
        FINAL_MODEL.set_param({"device": "cpu"})
        explainer = shap.TreeExplainer(FINAL_MODEL)
        ref = np.asarray(explainer.shap_values(pd.DataFrame(_EXPL_X[rows], columns=FEATURES)), dtype=np.float64)
    except Exception as exc:  # shap/xgboost version skew must not break the pipeline
        log(f"shap.TreeExplainer cross-check skipped: {type(exc).__name__}: {exc}", "WARNING")
        return None
    finally:
        FINAL_MODEL.set_param({"device": _PRIMARY_DEVICE})
    diff = float(np.max(np.abs(ref - SHAP_VALUES[rows])))
    if diff > 1e-3:
        log(f"shap.TreeExplainer differs from XGBoost TreeSHAP by {diff:.2e} (> 1e-3)", "WARNING")
    else:
        log(f"shap.TreeExplainer cross-check OK on {len(rows)} rows (max diff {diff:.2e})")
    return diff


_SHAP_CROSSCHECK_DIFF = _crosscheck_with_shap_package()
free_memory()

## 4.7 Governance zones: K-means in SHAP space
K-means (K = 4, 20 initialisations) on the SHAP matrix pooled over all epochs — cuML on the GPU when RAPIDS
is enabled (`LST_USE_RAPIDS=1`), otherwise scikit-learn. **K = 4 is fixed by the governance contract**; the
diagnostics for K = 2..8 (inertia, silhouette on a fixed 20k-row sample; every K, including 4, fitted with the
same 10 initialisations so the curves are comparable) are reported for transparency, not as evidence that 4 is
optimal - the silhouette may well peak at another K.

In [ ]:
_ZONE_META = [
    {"id": 0, "name": "Ecological Cool Base", "color": "#34d399"},
    {"id": 1, "name": "Riparian Buffer", "color": "#22d3ee"},
    {"id": 2, "name": "Transition", "color": "#a78bfa"},
    {"id": 3, "name": "Heat Extreme Core", "color": "#fb7185"},
]
_N_ZONES = 4
_DIAG_N_INIT = 10
if int(CFG.K_ZONES) != _N_ZONES:
    log(f"CFG.K_ZONES={CFG.K_ZONES} but the zone contract has 4 canonical zones; using K=4", "WARNING")


def _to_numpy(arr):
    """Host numpy array from numpy / cupy / cudf outputs."""
    if hasattr(arr, "to_numpy"):
        arr = arr.to_numpy()
    if hasattr(arr, "get"):
        arr = arr.get()
    return np.asarray(arr)


def _kmeans_fit(X, k, n_init, seed):
    """Return (labels int32, inertia) from cuML KMeans when enabled, else scikit-learn."""
    if ENV.get("has_cuml"):
        try:
            from cuml.cluster import KMeans as _CuKMeans
            model = _CuKMeans(n_clusters=k, n_init=n_init, random_state=seed, max_iter=300)
            model.fit(X)
            labels, inertia = _to_numpy(model.labels_).astype(np.int32), float(model.inertia_)
            ENV.setdefault("rapids_used", set()).add("cuml_kmeans")
            return labels, inertia
        except Exception as exc:
            log(f"cuML KMeans failed ({type(exc).__name__}: {exc}); using scikit-learn", "WARNING")
    from sklearn.cluster import KMeans
    model = KMeans(n_clusters=k, n_init=n_init, random_state=seed).fit(X)
    return model.labels_.astype(np.int32), float(model.inertia_)


def _relabel_zones(labels, shap_values, features, x_values):
    """Map raw cluster labels to canonical zone ids with the documented heuristic (4.2).

    Returns (zones int8, lut, info) where lut[raw_label] = canonical id and info documents, per cluster, the
    statistics the names were derived from.
    """
    raw_ids = np.unique(labels)
    if len(raw_ids) != _N_ZONES:
        raise ValueError(f"expected {_N_ZONES} non-empty clusters, got {len(raw_ids)}")
    jw, jn = features.index("frac_water"), features.index("ndwi")
    total = shap_values.sum(axis=1, dtype=np.float64)
    water = shap_values[:, jw].astype(np.float64) + shap_values[:, jn].astype(np.float64)
    frac_water = x_values[:, jw].astype(np.float64)
    stats = []
    for c in range(_N_ZONES):
        sel = labels == c
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)
            fw = float(np.nanmean(frac_water[sel])) if np.isfinite(frac_water[sel]).any() else float("nan")
        stats.append({"raw_cluster": c, "n": int(sel.sum()), "mean_total_shap": float(total[sel].mean()),
                      "mean_water_shap": float(water[sel].mean()), "mean_frac_water": fw})
    hot = int(np.argmax([s["mean_total_shap"] for s in stats]))
    rest = [c for c in range(_N_ZONES) if c != hot]
    negative = [c for c in rest if stats[c]["mean_water_shap"] < 0]
    if negative:
        riparian = min(negative, key=lambda c: stats[c]["mean_water_shap"])
        riparian_reason = "most negative mean SHAP(frac_water) + SHAP(ndwi) among the non-hottest clusters"
    else:
        riparian = max(rest, key=lambda c: (stats[c]["mean_frac_water"]
                                             if np.isfinite(stats[c]["mean_frac_water"]) else -np.inf))
        riparian_reason = ("no non-hottest cluster has a negative mean water SHAP: fallback to the highest mean "
                           "water fraction")
    rest2 = [c for c in rest if c != riparian]
    cool = min(rest2, key=lambda c: stats[c]["mean_total_shap"])
    transition = [c for c in rest2 if c != cool][0]
    lut = np.empty(_N_ZONES, dtype=np.int8)
    lut[cool], lut[riparian], lut[transition], lut[hot] = 0, 1, 2, 3
    reasons = {hot: "highest mean total SHAP", riparian: riparian_reason,
               cool: "lower mean total SHAP of the last two clusters", transition: "remaining cluster"}
    clusters = sorted(({**stats[c], "zone": int(lut[c]), "name": _ZONE_META[int(lut[c])]["name"],
                        "reason": reasons[c]} for c in range(_N_ZONES)), key=lambda d: d["zone"])
    info = {"rule": ("3 Heat Extreme Core = highest mean total SHAP; 1 Riparian Buffer = most negative mean "
                     "SHAP(frac_water)+SHAP(ndwi) among the other three (fallback: highest mean frac_water); "
                     "0 Ecological Cool Base = lower mean total SHAP of the last two; 2 Transition = the last one"),
            "clusters": clusters}
    return lut[labels].astype(np.int8), lut, info


def _silhouette(X, labels, sample_idx):
    from sklearn.metrics import silhouette_score
    return float(silhouette_score(X[sample_idx], labels[sample_idx]))


with timer("04_zoning"):
    _sil_idx = np.sort(np.random.default_rng(CFG.SEED).choice(len(DF), size=min(20000, len(DF)), replace=False))
    _raw_labels, _zone_inertia = _kmeans_fit(SHAP_VALUES, _N_ZONES, n_init=20, seed=CFG.SEED)
    ZONE_SILHOUETTE = _silhouette(SHAP_VALUES, _raw_labels, _sil_idx)
    ZONE_DIAGNOSTICS = {"k": [], "inertia": [], "silhouette": [], "n_init": _DIAG_N_INIT}
    for _k in range(2, 9):
        _lab_k, _in_k = _kmeans_fit(SHAP_VALUES, _k, n_init=_DIAG_N_INIT, seed=CFG.SEED)
        _sil_k = _silhouette(SHAP_VALUES, _lab_k, _sil_idx)
        ZONE_DIAGNOSTICS["k"].append(_k)
        ZONE_DIAGNOSTICS["inertia"].append(_in_k)
        ZONE_DIAGNOSTICS["silhouette"].append(_sil_k)
        log(f"  K={_k}: inertia {_in_k:,.1f}, silhouette {_sil_k:.3f}")
    _best_k = int(ZONE_DIAGNOSTICS["k"][int(np.argmax(ZONE_DIAGNOSTICS["silhouette"]))])
    log(f"Silhouette peaks at K={_best_k}; the zones use K=4 (governance contract), "
        f"zone fit with 20 initialisations: silhouette {ZONE_SILHOUETTE:.3f}")
    ZONES, _zone_lut, ZONE_LABELLING = _relabel_zones(_raw_labels, SHAP_VALUES, FEATURES, _EXPL_X)
    DF["zone"] = ZONES
    del _raw_labels, _lab_k
log(f"Zones (K=4, silhouette {ZONE_SILHOUETTE:.3f}): " + ", ".join(
    f"{m['name']}={int((ZONES == m['id']).sum()):,}" for m in _ZONE_META))
print("Zone labelling (" + ZONE_LABELLING["rule"] + "):")
for _c in ZONE_LABELLING["clusters"]:
    print(f"  {_c['zone']} {_c['name']:<22s} <- cluster {_c['raw_cluster']}: n={_c['n']:,}, "
          f"mean total SHAP {_c['mean_total_shap']:+.3f}, mean water+NDWI SHAP {_c['mean_water_shap']:+.3f}, "
          f"mean frac_water {_c['mean_frac_water']:.3f} [{_c['reason']}]")
free_memory()

## 4.8 SHAP interaction values (streamed per epoch)
`pred_interactions=True` returns an (n, K+1, K+1) tensor per batch. Each epoch is streamed batch by batch into
float16 chunks written to `npz/shap_interactions_<year>.npz`, while |φ_jk| sums are accumulated globally and per
zone in the same pass (zones are already known), so RAM stays bounded. On CPU the pass is limited to
`CFG.SHAP_INTERACTION_MAX_ROWS` random rows per epoch; if a GPU pass has to fall back to the CPU part-way (GPU
out of memory at the minimum batch), the rest of that epoch is subsampled to the same CPU cap and the event is
recorded (`cpu_fallback_epochs`), so a fallback can never turn into hours of CPU work.

In [ ]:
def _interaction_rows_for_epoch(year, rng):
    rows = np.flatnonzero(_YEARS == year)
    cap = CFG.SHAP_INTERACTION_MAX_ROWS
    if cap is not None and len(rows) > int(cap):
        rows = np.sort(rng.choice(rows, size=int(cap), replace=False))
    return rows


def _interaction_pass():
    """Stream interaction values for every epoch; return (global mean |phi|, per-zone mean |phi|, info)."""
    npz_dir = Path(CFG.NPZ_DIR)
    npz_dir.mkdir(parents=True, exist_ok=True)
    rng = np.random.default_rng(CFG.SEED + 7)
    batch = max(64, CFG.SHAP_BATCH // 16) if _PRIMARY_DEVICE != "cpu" else 256
    sum_all = np.zeros((_K, _K), dtype=np.float64)
    sum_zone = np.zeros((_N_ZONES, _K, _K), dtype=np.float64)
    n_zone = np.zeros(_N_ZONES, dtype=np.int64)
    acc = {"n_total": 0, "worst": 0.0}
    cpu_fallback_epochs = []

    def _consume(row_idx, inter, chunks):
        chunks.append((row_idx, inter.astype(np.float16)))
        # Interaction rows must sum back to the SHAP values (consistency of the decomposition).
        recon = inter[:, :_K, :].sum(axis=2, dtype=np.float64)
        acc["worst"] = max(acc["worst"], float(np.max(np.abs(recon - SHAP_VALUES[row_idx]))))
        mag = np.abs(inter[:, :_K, :_K]).astype(np.float64)
        sum_all[:] += mag.sum(axis=0)
        zb = ZONES[row_idx]
        for z in range(_N_ZONES):
            sel = zb == z
            if sel.any():
                sum_zone[z] += mag[sel].sum(axis=0)
                n_zone[z] += int(sel.sum())

    for year in _EPOCH_LIST:
        rows = _interaction_rows_for_epoch(year, rng)
        chunks = []
        state = {"cpu_fallback": False}
        gen = _iter_predict(FINAL_MODEL, _EXPL_X[rows], device=_PRIMARY_DEVICE, batch=batch,
                            kind="interactions", state=state)
        for start, stop, inter in gen:
            _consume(rows[start:stop], inter, chunks)
            del inter
            remaining = rows[stop:]
            if state["cpu_fallback"] and remaining.size > _CPU_INTERACTION_CAP:
                gen.close()
                keep = np.sort(rng.choice(remaining, size=_CPU_INTERACTION_CAP, replace=False))
                log(f"  interactions {year}: GPU fallback to CPU - subsampling the remaining {remaining.size:,} "
                    f"rows to {keep.size:,}", "WARNING")
                for s2, e2, inter2 in _iter_predict(FINAL_MODEL, _EXPL_X[keep], device="cpu", batch=256,
                                                    kind="interactions"):
                    _consume(keep[s2:e2], inter2, chunks)
                cpu_fallback_epochs.append(int(year))
                break
        FINAL_MODEL.set_param({"device": _PRIMARY_DEVICE})
        rows_used = np.concatenate([c[0] for c in chunks])
        store = np.concatenate([c[1] for c in chunks], axis=0)
        del chunks
        path = npz_dir / f"shap_interactions_{year}.npz"
        np.savez_compressed(path, interactions=store,
                            cell_id=DF["cell_id"].to_numpy()[rows_used].astype(np.int32),
                            features=np.array(list(FEATURES) + ["bias"]))
        acc["n_total"] += len(rows_used)
        log(f"  interactions {year}: {len(rows_used):,} rows -> {path.name} ({path.stat().st_size / 1e6:.1f} MB)")
        del store
        free_memory()
    if acc["worst"] > 1e-2:
        log(f"Interaction rows deviate from SHAP values by up to {acc['worst']:.3e}", "WARNING")
    glob = sum_all / max(acc["n_total"], 1)
    by_zone = {z: (sum_zone[z] / n_zone[z] if n_zone[z] else np.zeros((_K, _K))) for z in range(_N_ZONES)}
    return glob, by_zone, {"rows": acc["n_total"], "max_consistency_error": acc["worst"],
                           "cpu_fallback_epochs": cpu_fallback_epochs}


with timer("04_interactions"):
    INTERACTION_GLOBAL, INTERACTION_BY_ZONE, _INTERACTION_INFO = _interaction_pass()
    FINAL_MODEL.set_param({"device": _PRIMARY_DEVICE})
_offdiag = INTERACTION_GLOBAL.copy()
np.fill_diagonal(_offdiag, -np.inf)
_i, _j = np.unravel_index(np.argmax(_offdiag), _offdiag.shape)
log(f"Interactions on {_INTERACTION_INFO['rows']:,} rows; strongest pair {FEATURES[_i]} x {FEATURES[_j]} "
    f"(mean |phi| {INTERACTION_GLOBAL[_i, _j]:.4f})")
free_memory()

## 4.9 Dependence curves and threshold estimators
Pure functions shared by the final-model analysis and every bootstrap replicate. The final-model point
estimates are computed here, before the bootstrap, because the recommendation planning (4.10) needs them.

In [ ]:
def _quantile_edges(x, n_bins):
    """Quantile bin edges fixed on the full data, robust to point masses ("atoms"); ≥ 2 edges, no empty bin.

    Plain quantiles collapse when one value holds a large share of the data: `frac_water` is exactly 0 in ~95 %
    of cells, so 40 quantile edges reduce to 3 distinct values and the dependence curve (and every threshold)
    loses all resolution precisely where the water effect lives. Here every value holding ≥ 1/n_bins of the
    data (0 or 1 for land-cover fractions, 100 for single-class CONTAG, k/f² fractions at coarse grids) gets a bin
    of its own, and the remaining bins are equal-count quantiles of the other values. Continuous features have
    no atoms and get the ordinary quantile bins. Bins are [e_i, e_{i+1}) with the last one closed (`_bin_index`).
    """
    xf = np.asarray(x, dtype=np.float64)
    xf = xf[np.isfinite(xf)]
    if xf.size == 0:
        return np.array([0.0, 0.0])
    n_bins = max(1, int(n_bins))
    values, counts = np.unique(xf, return_counts=True)
    atoms = values[counts * n_bins >= xf.size]
    rest = xf[~np.isin(xf, atoms)] if atoms.size else xf
    parts = [values[[0, -1]], atoms, np.nextafter(atoms, np.inf)]      # [a, a⁺) isolates atom a
    if rest.size:
        n_rest = max(1, n_bins - atoms.size)
        parts.append(np.quantile(rest, np.linspace(0.0, 1.0, n_rest + 1)))
    edges = np.unique(np.concatenate(parts))
    if edges.size < 2:
        return np.array([edges[0], edges[0]])
    # Merge empty bins (e.g. the gap between an isolated atom and the first quantile of the rest) into their
    # lower neighbour so every bin has data and bootstrap curves stay aligned with the final-model curve.
    while edges.size > 2:
        occupied = np.bincount(_bin_index(xf, edges), minlength=edges.size - 1)
        empty = np.flatnonzero(occupied == 0)
        if empty.size == 0:
            break
        i = int(empty[0])
        edges = np.delete(edges, i if i > 0 else 1)
    # An atom inside the range splits a quantile bin in two and an atom at the maximum leaves a sliver bin just
    # below it, so the count can exceed n_bins: merge the smallest non-atom bin into its smaller non-atom
    # neighbour until at most n_bins remain (atom bins are never merged, so they stay pure).
    atom_set = set(atoms.tolist())
    while edges.size - 1 > n_bins:
        occupied = np.bincount(_bin_index(xf, edges), minlength=edges.size - 1)
        is_atom = np.array([float(edges[b]) in atom_set and edges[b + 1] == np.nextafter(edges[b], np.inf)
                            for b in range(edges.size - 1)])
        merged = False
        for b in np.argsort(occupied, kind="stable"):
            if is_atom[b]:
                continue
            nbrs = [n for n in (b - 1, b + 1) if 0 <= n < edges.size - 1 and not is_atom[n]]
            if not nbrs:
                continue
            n = min(nbrs, key=lambda k: occupied[k])
            edges = np.delete(edges, max(b, n))          # the edge shared by bins b and n
            merged = True
            break
        if not merged:
            break
    return edges


def _bin_index(x, edges):
    """Bin id of each value (0..nb-1, the last bin is closed), -1 for NaN."""
    x = np.asarray(x, dtype=np.float64)
    nb = len(edges) - 1
    idx = np.clip(np.searchsorted(edges, x, side="right") - 1, 0, nb - 1)
    idx[~np.isfinite(x)] = -1
    return idx


def binned_curve(x, s, edges):
    """Binned mean of SHAP values s against feature x.

    Returns (centers, mean, counts): centers are the mean x inside each bin (edge midpoint if empty),
    mean is NaN for empty bins.
    """
    x = np.asarray(x, dtype=np.float64)
    s = np.asarray(s, dtype=np.float64)
    nb = len(edges) - 1
    idx = _bin_index(x, edges)
    ok = (idx >= 0) & np.isfinite(s)
    counts = np.bincount(idx[ok], minlength=nb)
    sum_s = np.bincount(idx[ok], weights=s[ok], minlength=nb)
    sum_x = np.bincount(idx[ok], weights=x[ok], minlength=nb)
    with np.errstate(invalid="ignore", divide="ignore"):
        mean = sum_s / counts
        centers = sum_x / counts
    mids = 0.5 * (np.asarray(edges[:-1], dtype=np.float64) + np.asarray(edges[1:], dtype=np.float64))
    centers = np.where(counts > 0, centers, mids)
    return centers, np.where(counts > 0, mean, np.nan), counts.astype(np.int64)


def _finite_curve(centers, mean, counts=None):
    c = np.asarray(centers, dtype=np.float64)
    m = np.asarray(mean, dtype=np.float64)
    ok = np.isfinite(c) & np.isfinite(m)
    if counts is not None:
        w = np.asarray(counts, dtype=np.float64)
        ok &= w > 0
        return c[ok], m[ok], w[ok]
    return c[ok], m[ok], np.ones(int(ok.sum()))


def _crossings(centers, mean):
    """All sign changes of the curve: (x of each crossing by linear interpolation, |jump| across it)."""
    c, m, _ = _finite_curve(centers, mean)
    if c.size < 2:
        return np.zeros(0), np.zeros(0)
    m0, m1 = m[:-1], m[1:]
    cross = np.flatnonzero(((m0 < 0) & (m1 >= 0)) | ((m0 > 0) & (m1 <= 0)))
    if cross.size == 0:
        return np.zeros(0), np.zeros(0)
    x = c[cross] - m[cross] * (c[cross + 1] - c[cross]) / (m[cross + 1] - m[cross])
    return x.astype(np.float64), np.abs(m1[cross] - m0[cross]).astype(np.float64)


def zero_crossing(centers, mean, ref=None, dominance=2.0):
    """x where the curve changes sign.

    Without ``ref``: the single crossing, or the crossing whose |jump| is ≥ ``dominance`` × every other one; None
    when there is no crossing or several comparable ones. With ``ref`` (bootstrap replicates): the crossing nearest
    to the full-data crossing ``ref``, so replicates never mix different crossings.
    """
    x, jump = _crossings(centers, mean)
    if x.size == 0:
        return None
    if ref is not None and np.isfinite(ref):
        return float(x[int(np.argmin(np.abs(x - float(ref))))])
    if x.size == 1:
        return float(x[0])
    order = np.argsort(jump)[::-1]
    if jump[order[0]] >= dominance * jump[order[1]]:
        return float(x[order[0]])
    return None


def zero_crossing_is_multiple(centers, mean, dominance=2.0):
    """True when the curve has several sign changes and none of them dominates (zero_crossing -> None)."""
    x, jump = _crossings(centers, mean)
    if x.size < 2:
        return False
    top2 = np.sort(jump)[::-1][:2]
    return bool(top2[0] < dominance * top2[1])


def _hinge_fit(c, m, w, candidates):
    """Weighted continuous two-segment fit y = b0 + b1 x + b2 max(0, x - h) for each hinge h.

    Solved in closed form (batched 3x3 normal equations). Returns (sse per candidate, coefficients).
    """
    h = np.asarray(candidates, dtype=np.float64)[:, None]
    A = np.stack([np.ones((h.shape[0], c.size)), np.broadcast_to(c, (h.shape[0], c.size)),
                  np.maximum(0.0, c[None, :] - h)], axis=2)                  # (C, n, 3)
    Aw = A * w[None, :, None]
    ata = np.einsum("cni,cnj->cij", Aw, A) + 1e-12 * np.eye(3)[None]
    aty = np.einsum("cni,n->ci", Aw, m)
    beta = np.linalg.solve(ata, aty[..., None])[..., 0]                    # (C, 3)
    resid = m[None, :] - np.einsum("cni,ci->cn", A, beta)
    return (w[None, :] * resid ** 2).sum(axis=1), beta


_HINGE_MIN_SIDE_BINS = 5
_HINGE_MIN_SIDE_WEIGHT = 0.05


def hinge_breakpoint(centers, mean, counts):
    """Best continuous hinge (count-weighted least squares) with the hinge between the 10th-90th pct bins.

    Returns (breakpoint, slope_before, slope_after) or (None, None, None) when there are < 5 bins, the hinge removes
    < 10 % of the straight-line residual sum of squares (no real change of slope), the best hinge lies at the edge of
    the candidate range (a range limit, not a kink), or either side of it has < 5 bins or < 5 % of the row weight.
    """
    c, m, w = _finite_curve(centers, mean, counts)
    n = c.size
    if n < 5:
        return None, None, None
    lo, hi = max(1, int(np.floor(0.1 * (n - 1)))), min(n - 2, int(np.ceil(0.9 * (n - 1))))
    if hi <= lo:
        return None, None, None
    candidates = np.unique(np.concatenate([c[lo:hi + 1], np.linspace(c[lo], c[hi], 121)]))
    sse, beta = _hinge_fit(c, m, w, candidates)
    lin = np.polyfit(c, m, 1, w=np.sqrt(w))
    sse_lin = float(np.sum(w * (m - np.polyval(lin, c)) ** 2))
    best = int(np.argmin(sse))
    if sse_lin <= 1e-12 or sse[best] > 0.9 * sse_lin:
        return None, None, None
    h = float(candidates[best])
    eps = 1e-9 * max(1.0, float(np.max(np.abs(c))))
    if h <= c[lo] + eps or h >= c[hi] - eps:
        return None, None, None
    left, right = c <= h, c > h
    total_w = float(w.sum())
    if (left.sum() < _HINGE_MIN_SIDE_BINS or right.sum() < _HINGE_MIN_SIDE_BINS
            or w[left].sum() < _HINGE_MIN_SIDE_WEIGHT * total_w or w[right].sum() < _HINGE_MIN_SIDE_WEIGHT * total_w):
        return None, None, None
    b1, b2 = float(beta[best, 1]), float(beta[best, 2])
    return h, b1, b1 + b2


def _smooth3(m):
    """Centered 3-bin running mean (edges use the available neighbours)."""
    num = np.convolve(m, np.ones(3), mode="same")
    den = np.convolve(np.ones_like(m), np.ones(3), mode="same")
    return num / den


def saturation_point(centers, mean):
    """First x after the steepest section where |smoothed slope| < 10% of its max for ≥ 3 consecutive bins.

    The detected flat-onset bin is refined to the level-off knee by a local hinge fit on the raw curve over
    the steep→flat transition (removes the ~1-bin lag introduced by the smoother).
    """
    c, m, _ = _finite_curve(centers, mean)
    if c.size < 5 or np.any(np.diff(c) <= 0):
        return None
    slope = np.abs(np.gradient(_smooth3(m), c))
    smax = float(np.max(slope))
    if not np.isfinite(smax) or smax <= 0:
        return None
    steep = int(np.argmax(slope))
    flat = slope < 0.1 * smax
    onset = next((j for j in range(steep + 1, c.size - 2) if flat[j:j + 3].all()), None)
    if onset is None:
        return None
    lo, hi = max(steep, onset - 4), min(c.size - 1, onset + 2)
    seg_c, seg_m = c[lo:hi + 1], m[lo:hi + 1]
    if seg_c.size < 4:
        return float(c[onset])
    candidates = np.linspace(c[lo], c[onset], 81)
    sse, _ = _hinge_fit(seg_c, seg_m, np.ones(seg_c.size), candidates)
    return float(candidates[int(np.argmin(sse))])


def effect_direction(x, s):
    """'warming' / 'cooling' from the sign of Spearman rho(x, SHAP); |rho| < 0.2 -> 'mixed'."""
    x = np.asarray(x, dtype=np.float64)
    s = np.asarray(s, dtype=np.float64)
    ok = np.isfinite(x) & np.isfinite(s)
    if ok.sum() < 10 or np.ptp(x[ok]) == 0 or np.ptp(s[ok]) == 0:
        return "mixed"
    try:
        from scipy.stats import spearmanr
        rho = float(spearmanr(x[ok], s[ok]).correlation)
    except ImportError:
        rx = pd.Series(x[ok]).rank().to_numpy()
        rs = pd.Series(s[ok]).rank().to_numpy()
        rho = float(np.corrcoef(rx, rs)[0, 1])
    if not np.isfinite(rho) or abs(rho) < 0.2:
        return "mixed"
    return "warming" if rho > 0 else "cooling"


def _curve_thresholds(centers, mean, counts, zc_ref=None):
    """(zero_crossing, breakpoint, saturation) as floats with NaN for 'not found' (replicates pass zc_ref)."""
    zc = zero_crossing(centers, mean, ref=zc_ref)
    bp, _, _ = hinge_breakpoint(centers, mean, counts)
    sat = saturation_point(centers, mean)
    return [np.nan if v is None else v for v in (zc, bp, sat)]


# Fixed quantile bins on the full data (shared by the final model and all replicates).
_DEP_EDGES = [_quantile_edges(_EXPL_X[:, j], CFG.DEP_BINS) for j in range(_K)]
_DEP_BIN_IDX = [_bin_index(_EXPL_X[:, j], _DEP_EDGES[j]) for j in range(_K)]
_DEP_CURVES = [binned_curve(_EXPL_X[:, j], SHAP_VALUES[:, j], _DEP_EDGES[j]) for j in range(_K)]
_MAX_BINS = max(len(e) - 1 for e in _DEP_EDGES)


def _point_thresholds(j):
    centers, mean, counts = _DEP_CURVES[j]
    bp, slope_before, slope_after = hinge_breakpoint(centers, mean, counts)
    return {"zero_crossing": zero_crossing(centers, mean),
            "zero_crossing_multiple": zero_crossing_is_multiple(centers, mean),
            "breakpoint": bp, "slope_before": slope_before, "slope_after": slope_after,
            "saturation": saturation_point(centers, mean),
            "direction": effect_direction(_EXPL_X[:, j], SHAP_VALUES[:, j])}


_POINT_THRESHOLDS = [_point_thresholds(j) for j in range(_K)]
_ZC_REF = [np.nan if t["zero_crossing"] is None else float(t["zero_crossing"]) for t in _POINT_THRESHOLDS]

## 4.10 Scenario coupling and zone recommendation planning (model counterfactuals)
**Coupling.** The dashboard's scenario simulator can move a land-cover fraction and (in coupled mode) shift NDVI /
NDWI / NDBI consistently. The coupling is the partial slope of each index on the five fractions (OLS with
intercept, all cells × epochs), i.e. d index / d fraction holding the other fractions fixed; "other vegetation"
(class 6, grass/shrub/wetland) is the omitted reference class.

**Recommendations** are *plausible interventions* evaluated with the final model itself, not read off the pooled
marginal SHAP curve (which ignores interactions and is not zone-specific):
* **Levers**: the five land-cover fractions, NDVI (greening within the existing covers) and the three landscape
  metrics. NDWI and NDBI are spectral diagnostics (`actionable = false`); they move only through land-cover change.
  Allowed directions: impervious and barren only *decrease*, tree cover and water only *increase*, cropland, and the
  landscape metrics either way; NDVI only increases and is not offered in water-dominated zones (median water
  fraction > 0.15), where low NDVI is caused by the water itself.
* **Counterfactual**: every cell of the zone (last epoch) is shifted by the same step (target − current). A fraction
  change is compositional, exactly as in the dashboard: an increase takes land from the other four fractions and the
  implicit other-vegetation share in proportion to their current values (capped by the land available), a decrease
  gives the freed land to them in the same proportions; NDVI/NDWI/NDBI then follow through the coupling slopes
  (clamped to [−1, 1]). NDVI and landscape metrics are shifted directly (clamped to their physical ranges).
* **Feasibility**: the target may not go beyond the zone's own 90th percentile (increase) / 10th percentile
  (decrease) of that feature, and the step is capped at 0.20 for fractions and 0.15 for NDVI.
* **Target choice**: candidate targets are the feature's breakpoint and saturation (when inside the feasible range)
  and a grid up to the feasibility bound; the recommended target is the *smallest* step that achieves ≥ 90 % of
  the best cooling available within the bounds (so a saturating response stops at its saturation, and a
  flat-then-steep response goes past its breakpoint).
* **Effect**: expected Δ = mean over the zone's applicable cells of the change in the final model's prediction
  (a model-based ceteris-paribus/ICE estimate, **associated with** the change, not a causal effect). Actions with
  Δ > −0.05 °C are dropped; the best four (most negative Δ) are kept per zone. The interval comes from applying the
  identical counterfactual to every bootstrap replicate (bias-corrected percentile interval, 4.3).

In [ ]:
_COUPLING_FRACTIONS = ["frac_impervious", "frac_forest", "frac_water", "frac_cropland", "frac_barren"]
_COUPLED_INDICES = ["ndvi", "ndwi", "ndbi"]


def _scenario_coupling():
    cols = [FEATURES.index(f) for f in _COUPLING_FRACTIONS]
    coupling = {f: {} for f in _COUPLING_FRACTIONS}
    r2, intercept, n_used = {}, {}, {}
    for index in _COUPLED_INDICES:
        y = _EXPL_X[:, FEATURES.index(index)].astype(np.float64)
        A = _EXPL_X[:, cols].astype(np.float64)
        ok = np.isfinite(y) & np.isfinite(A).all(axis=1)
        design = np.column_stack([np.ones(int(ok.sum())), A[ok]])
        beta, *_ = np.linalg.lstsq(design, y[ok], rcond=None)
        resid = y[ok] - design @ beta
        sst = float(np.sum((y[ok] - y[ok].mean()) ** 2))
        r2[index] = 1.0 - float(np.sum(resid ** 2)) / sst if sst > 0 else None
        intercept[index] = float(beta[0])
        n_used[index] = int(ok.sum())
        for k, f in enumerate(_COUPLING_FRACTIONS):
            coupling[f][index] = float(beta[k + 1])
    return {
        "description": ("Partial OLS slopes (d index / d fraction) of NDVI, NDWI and NDBI on the five land-cover "
                        "fractions (with intercept; other vegetation is the omitted reference class), fitted on "
                        "all cells and epochs. Used by the coupled scenario mode: when a fraction changes, the land "
                        "is exchanged with the other fractions and the implicit other-vegetation share in "
                        "proportion to their current values, and each index shifts by the sum of slope x actual "
                        "fraction change."),
        "fraction_features": list(_COUPLING_FRACTIONS),
        "coupling": coupling, "r2": r2, "intercept": intercept, "n": n_used,
    }


SCENARIO_COUPLING = _scenario_coupling()
print(pd.DataFrame(SCENARIO_COUPLING["coupling"]).T.round(3).to_string())
print("R2:", {k: (round(v, 3) if v is not None else None) for k, v in SCENARIO_COUPLING["r2"].items()})


def rebalance_fractions(F, k, delta):
    """Compositional change of fraction column ``k`` by ``delta`` for every row of ``F`` (n, 5).

    Mirrors the dashboard's `rebalanceFractions` (SPEC 4.8) with the implicit other-vegetation share
    ``1 - sum(F)`` as a sixth donor/recipient: an increase is capped by the land the others can give up and taken
    from them in proportion to their current values; a decrease (capped at the current value) gives the freed land to
    them in the same proportions (equally when they are all 0). Returns (F_after, other_after, applied).
    """
    F = np.asarray(F, dtype=np.float64)
    x = F[:, k]
    other = np.clip(1.0 - F.sum(axis=1), 0.0, None)
    donors = np.column_stack([np.delete(F, k, axis=1), other])              # (n, 5): 4 fractions + other veg
    dsum = donors.sum(axis=1)
    if delta > 0:
        applied = np.clip(np.minimum(np.minimum(delta, 1.0 - x), dsum), 0.0, None)
    else:
        applied = np.maximum(delta, -x)
    with np.errstate(invalid="ignore", divide="ignore"):
        share = np.where(dsum[:, None] > 0, donors / dsum[:, None], 1.0 / donors.shape[1])
    donors_after = np.clip(donors - applied[:, None] * share, 0.0, 1.0)
    F_after = F.copy()
    F_after[:, k] = np.clip(x + applied, 0.0, 1.0)
    others = [i for i in range(F.shape[1]) if i != k]
    F_after[:, others] = donors_after[:, :-1]
    return F_after, donors_after[:, -1], applied


_FRAC_COLS = [FEATURES.index(f) for f in _COUPLING_FRACTIONS]
_FEATURE_BOUNDS = {"ndvi": (-1.0, 1.0), "ndwi": (-1.0, 1.0), "ndbi": (-1.0, 1.0), "lm_contag": (0.0, 100.0),
                   "lm_pd": (0.0, np.inf), "lm_ed": (0.0, np.inf),
                   **{f: (0.0, 1.0) for f in _COUPLING_FRACTIONS}}


def counterfactual(X, feature, step, coupled=True):
    """Shift ``feature`` by ``step`` in every row of X (n, K) -> (X_cf float32, applicable bool mask, donor info).

    Fractions: compositional (``rebalance_fractions``) and, when ``coupled``, NDVI/NDWI/NDBI shift by the coupling
    slopes × the actual fraction changes. Other levers: direct shift clamped to their physical range. Rows where the
    feature (or, for fractions, any fraction) is missing are left unchanged and flagged not applicable.
    """
    X = np.asarray(X, dtype=np.float32)
    j = FEATURES.index(feature)
    Xcf = X.astype(np.float64, copy=True)
    ok = np.isfinite(Xcf[:, j])
    donors = {}
    if feature in _COUPLING_FRACTIONS:
        ok &= np.isfinite(Xcf[:, _FRAC_COLS]).all(axis=1)
        F = Xcf[np.ix_(ok, _FRAC_COLS)]
        other_before = np.clip(1.0 - F.sum(axis=1), 0.0, None)
        F_after, other_after, _ = rebalance_fractions(F, _COUPLING_FRACTIONS.index(feature), float(step))
        Xcf[np.ix_(ok, _FRAC_COLS)] = F_after
        d_frac = F_after - F
        donors = {f: float(d_frac[:, i].mean()) for i, f in enumerate(_COUPLING_FRACTIONS)
                  if f != feature and d_frac.size}
        donors["other_vegetation"] = float((other_after - other_before).mean()) if d_frac.size else 0.0
        if coupled:
            for index in _COUPLED_INDICES:
                ji = FEATURES.index(index)
                slopes = np.array([SCENARIO_COUPLING["coupling"][f][index] for f in _COUPLING_FRACTIONS])
                shift = d_frac @ slopes
                old = Xcf[ok, ji]
                Xcf[ok, ji] = np.where(np.isfinite(old), np.clip(old + shift, -1.0, 1.0), old)
    else:
        lo, hi = _FEATURE_BOUNDS.get(feature, (-np.inf, np.inf))
        Xcf[ok, j] = np.clip(Xcf[ok, j] + float(step), lo, hi)
    return Xcf.astype(np.float32), ok, donors


_REC_DIRECTIONS = {"frac_impervious": ("decrease",), "frac_barren": ("decrease",), "frac_forest": ("increase",),
                   "frac_water": ("increase",), "frac_cropland": ("increase", "decrease"), "ndvi": ("increase",),
                   "lm_pd": ("increase", "decrease"), "lm_ed": ("increase", "decrease"),
                   "lm_contag": ("increase", "decrease")}
_REC_MAX_STEP = {**{f: 0.20 for f in _COUPLING_FRACTIONS}, "ndvi": 0.15}
_REC_MIN_COOLING_C = 0.05          # drop actions whose expected Δ is > -0.05 °C
_REC_EFFICIENCY = 0.90             # smallest step achieving >= 90 % of the best cooling within the bounds
_REC_WATER_ZONE = 0.15             # zone median water fraction above which NDVI greening is not offered
_REC_MAX_PER_ZONE = 4
_REC_LEVERS = [f for f in FEATURES if FEATURE_META.get(f, {}).get("actionable")]
_unknown_levers = [f for f in _REC_LEVERS if f not in _REC_DIRECTIONS]
if _unknown_levers:
    log(f"actionable features without an intervention rule are not recommended: {_unknown_levers}", "WARNING")


def _zone_rows(zone):
    rows = np.flatnonzero((ZONES == zone) & (_YEARS == _LAST_YEAR))
    return rows if rows.size >= 10 else np.flatnonzero(ZONES == zone)


def _margin(booster, X, device):
    return _predict_all(booster, X, device=device, batch=CFG.SHAP_BATCH, kind="margin").astype(np.float64)


def _plan_zone(zone):
    """Candidate actions for one zone evaluated with the final model; returns plans sorted by Δ (best first)."""
    rows = _zone_rows(zone)
    X = _EXPL_X[rows]
    base = _FINAL_MARGIN[rows]
    jw = FEATURES.index("frac_water")
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        water_median = float(np.nanmedian(X[:, jw])) if np.isfinite(X[:, jw]).any() else 0.0
    plans = []
    for feature in _REC_LEVERS:
        if feature not in _REC_DIRECTIONS:
            continue
        if feature == "ndvi" and water_median > _REC_WATER_ZONE:
            continue
        j = FEATURES.index(feature)
        vals = X[:, j][np.isfinite(X[:, j])].astype(np.float64)
        if vals.size < 10:
            continue
        current = float(np.median(vals))
        p10, p90 = (float(v) for v in np.percentile(vals, [10, 90]))
        stats = FEATURE_STATS[feature]
        min_step = max(1e-6, 0.01 * (float(stats["p99"]) - float(stats["p01"])))
        lo_phys, hi_phys = _FEATURE_BOUNDS.get(feature, (-np.inf, np.inf))
        cap = _REC_MAX_STEP.get(feature, np.inf)
        th = _POINT_THRESHOLDS[j]
        best_plan = None
        for direction in _REC_DIRECTIONS[feature]:
            if direction == "increase":
                bound = min(p90, current + cap, hi_phys)
                if bound - current < min_step:
                    continue
                inside = lambda t: current + min_step <= t <= bound  # noqa: E731
            else:
                bound = max(p10, current - cap, lo_phys)
                if current - bound < min_step:
                    continue
                inside = lambda t: bound <= t <= current - min_step  # noqa: E731
            cands = [(current + (bound - current) * q, "partial") for q in (0.25, 0.5, 0.75)] + [(bound, "bound")]
            cands += [(float(th[k]), k) for k in ("breakpoint", "saturation")
                      if th[k] is not None and inside(float(th[k]))]
            results = []
            for target, kind in cands:
                Xcf, ok, donors = counterfactual(X, feature, target - current)
                if not ok.any():
                    continue
                pred = _margin(FINAL_MODEL, Xcf[ok], _PRIMARY_DEVICE)
                results.append({"target": float(target), "kind": kind, "delta": float(np.mean(pred - base[ok])),
                                "n_cells": int(ok.sum()), "donors": donors})
            if not results:
                continue
            best = min(r["delta"] for r in results)
            if not best < -_REC_MIN_COOLING_C:
                continue
            good = [r for r in results if r["delta"] <= _REC_EFFICIENCY * best]
            # smallest step first; on ties prefer a named threshold over a grid point
            chosen = min(good, key=lambda r: (abs(r["target"] - current), r["kind"] not in ("saturation", "breakpoint")))
            plan = {"zone": int(zone), "feature": feature, "j": j, "action": direction, "current": current,
                    "target": chosen["target"], "step": chosen["target"] - current, "kind": chosen["kind"],
                    "expected_delta_c": chosen["delta"], "best_delta_c": best, "n_cells": chosen["n_cells"],
                    "donors": chosen["donors"], "p10": p10, "p90": p90, "bound": bound, "rows": rows}
            if best_plan is None or plan["expected_delta_c"] < best_plan["expected_delta_c"]:
                best_plan = plan
        if best_plan is not None:
            plans.append(best_plan)
    plans.sort(key=lambda p: p["expected_delta_c"])
    return plans[:_REC_MAX_PER_ZONE]


with timer("04_recommendation_planning"):
    _REC_PLAN = [p for m in _ZONE_META for p in _plan_zone(m["id"])]
    FINAL_MODEL.set_param({"device": _PRIMARY_DEVICE})
# Pre-built counterfactual matrices (identical for the final model and every bootstrap replicate).
_REC_BASE_ROWS = np.unique(np.concatenate([p["rows"] for p in _REC_PLAN])) if _REC_PLAN else np.zeros(0, np.int64)
_REC_XCF, _REC_SLICES = [], []
for _p in _REC_PLAN:
    _xcf, _ok, _ = counterfactual(_EXPL_X[_p["rows"]], _p["feature"], _p["step"])
    _p["applicable"] = _ok
    _p["base_pos"] = np.searchsorted(_REC_BASE_ROWS, _p["rows"][_ok])
    _start = sum(len(x) for x in _REC_XCF)
    _REC_XCF.append(_xcf[_ok])
    _REC_SLICES.append(slice(_start, _start + int(_ok.sum())))
_REC_XCF = np.concatenate(_REC_XCF, axis=0) if _REC_XCF else np.zeros((0, _K), np.float32)
del _p


def _plan_deltas(booster, device):
    """Mean prediction change of every planned action under ``booster`` (same counterfactuals for all models)."""
    if not _REC_PLAN:
        return np.zeros(0)
    base = _margin(booster, _EXPL_X[_REC_BASE_ROWS], device)
    cf = _margin(booster, _REC_XCF, device)
    return np.array([float(np.mean(cf[sl] - base[p["base_pos"]])) for p, sl in zip(_REC_PLAN, _REC_SLICES)])


_REC_FINAL_DELTAS = _plan_deltas(FINAL_MODEL, _PRIMARY_DEVICE)
FINAL_MODEL.set_param({"device": _PRIMARY_DEVICE})
for _p, _d in zip(_REC_PLAN, _REC_FINAL_DELTAS):
    if abs(_d - _p["expected_delta_c"]) > 1e-4:
        log(f"recommendation {_p['zone']}/{_p['feature']}: replayed delta {_d:.5f} != planned "
            f"{_p['expected_delta_c']:.5f}", "WARNING")
    _p["expected_delta_c"] = float(_d)
log(f"Recommendation plan: {len(_REC_PLAN)} action(s) over {_N_ZONES} zones, "
    f"{len(_REC_XCF):,} counterfactual rows per model")

## 4.11 Spatial block bootstrap (multi-GPU)
The block size comes from the residual correlogram (4.3) unless `CFG.BOOT_BLOCK_KM` is set.

In [ ]:
_BOOT_ON_GPU = _PRIMARY_DEVICE != "cpu"
_boot_rng = np.random.default_rng(CFG.SEED + 1000)
_rows_last = np.flatnonzero(_YEARS == _LAST_YEAR)
if _BOOT_ON_GPU:
    _rows_sample = np.arange(len(DF))
else:  # CPU: explain a fixed ≤20k-row sample plus the complete last epoch
    _rows_sample = np.sort(_boot_rng.choice(len(DF), size=min(20000, len(DF)), replace=False))
_BOOT_EVAL_IDX = np.union1d(_rows_sample, _rows_last)
_BOOT_POS_S = np.searchsorted(_BOOT_EVAL_IDX, _rows_sample)
_BOOT_POS_L = np.searchsorted(_BOOT_EVAL_IDX, _rows_last)
_BOOT_BIN_IDX_S = [idx[_rows_sample] for idx in _DEP_BIN_IDX]
_BOOT_COUNTS_S = [np.bincount(idx[idx >= 0], minlength=len(_DEP_EDGES[j]) - 1)
                  for j, idx in enumerate(_BOOT_BIN_IDX_S)]
_BOOT_MIN_BLOCKS = 50


def _residual_correlogram(max_points=3000, bin_km=5.0, max_km=100.0):
    """Pair correlogram of the standardised last-epoch spatial-CV residuals (target anomaly if OOF is missing).

    Returns (bin upper edges in km, correlation per bin, variable name).
    """
    rows = _rows_last
    values = DF["resid_oof"].to_numpy(np.float64)[rows]
    name = "spatial-CV XGBoost residual"
    if np.isfinite(values).sum() < 100:
        values, name = _EXPL_Y[rows].astype(np.float64), "target"
    ok = np.isfinite(values)
    rows, values = rows[ok], values[ok]
    rng = np.random.default_rng(CFG.SEED + 55)
    if rows.size > max_points:
        pick = np.sort(rng.choice(rows.size, size=max_points, replace=False))
        rows, values = rows[pick], values[pick]
    z = (values - values.mean()) / (values.std() or 1.0)
    xy = DF[["x", "y"]].to_numpy(np.float64)[rows] / 1000.0
    edges = np.arange(0.0, max_km + bin_km, bin_km)
    sums = np.zeros(edges.size - 1)
    counts = np.zeros(edges.size - 1)
    for start in range(0, len(z), 500):
        d = np.sqrt(((xy[start:start + 500, None, :] - xy[None, :, :]) ** 2).sum(axis=2))
        prod = z[start:start + 500, None] * z[None, :]
        valid = (d > 0) & (d < max_km)
        b = np.clip((d[valid] // bin_km).astype(np.int64), 0, edges.size - 2)
        sums += np.bincount(b, weights=prod[valid], minlength=edges.size - 1)
        counts += np.bincount(b, minlength=edges.size - 1)
    with np.errstate(invalid="ignore", divide="ignore"):
        corr = sums / counts
    return edges[1:], corr, name


def _block_ids(block_km):
    b = max(1, int(round(float(block_km) * 1000.0 / float(GRID.res))))
    n_cols = int(math.ceil(GRID.width / b))
    rows, cols = DF["row"].to_numpy(np.int64), DF["col"].to_numpy(np.int64)
    return (rows // b) * n_cols + cols // b


def _choose_bootstrap_blocks():
    """(block codes per row, block ids, block size km, info) following 4.3."""
    info = {}
    if CFG.BOOT_BLOCK_KM:
        block_km, source = float(CFG.BOOT_BLOCK_KM), "CFG.BOOT_BLOCK_KM"
    else:
        upper, corr, name = _residual_correlogram()
        below = np.flatnonzero(np.isfinite(corr) & (corr < 0.1))
        range_km = float(upper[below[0]] - 2.5) if below.size else float(upper[-1])
        info = {"residual_corr_range_km": range_km, "correlogram_variable": name,
                "correlogram": {f"{u - 5:.0f}-{u:.0f}": (None if not np.isfinite(c) else round(float(c), 3))
                                for u, c in zip(upper[:12], corr[:12])}}
        block_km = max(float(CFG.INNER_BLOCK_KM), 5.0 * math.ceil(range_km / 5.0))
        source = f"{name} correlogram (< 0.1 beyond ~{range_km:.0f} km)"
        log(f"Residual correlogram ({name}): " + ", ".join(f"{k} km {v}" for k, v in info["correlogram"].items()))
    codes, ids = pd.factorize(_block_ids(block_km), sort=True)
    while len(ids) < _BOOT_MIN_BLOCKS and block_km - 5.0 >= float(CFG.INNER_BLOCK_KM) and not CFG.BOOT_BLOCK_KM:
        block_km -= 5.0
        codes, ids = pd.factorize(_block_ids(block_km), sort=True)
        source += f"; shrunk to keep >= {_BOOT_MIN_BLOCKS} blocks"
    if len(ids) < _BOOT_MIN_BLOCKS:
        log(f"Only {len(ids)} bootstrap blocks of {block_km:.0f} km (< {_BOOT_MIN_BLOCKS}); intervals will be "
            "coarse", "WARNING")
    return codes, len(ids), block_km, {**info, "block_size_source": source}


_block_codes, _N_BLOCKS, _BOOT_BLOCK_KM, _BOOT_BLOCK_INFO = _choose_bootstrap_blocks()
log(f"Bootstrap blocks: {_N_BLOCKS} blocks of {_BOOT_BLOCK_KM:.0f} km ({_BOOT_BLOCK_INFO['block_size_source']})")


def _replicate_curves(shap_sample):
    """Binned mean curves (K, _MAX_BINS) and thresholds (K, 3) for one replicate's SHAP sample."""
    curves = np.full((_K, _MAX_BINS), np.nan)
    thresholds = np.full((_K, 3), np.nan)
    for j in range(_K):
        idx = _BOOT_BIN_IDX_S[j]
        ok = idx >= 0
        nb = len(_DEP_EDGES[j]) - 1
        counts = _BOOT_COUNTS_S[j]
        sums = np.bincount(idx[ok], weights=shap_sample[ok, j].astype(np.float64), minlength=nb)
        with np.errstate(invalid="ignore", divide="ignore"):
            mean = np.where(counts > 0, sums / np.maximum(counts, 1), np.nan)
        curves[j, :nb] = mean
        thresholds[j] = _curve_thresholds(_DEP_CURVES[j][0], mean, counts, zc_ref=_ZC_REF[j])
    return curves, thresholds


def _bootstrap_replicate(b, gpu_id):
    """Train one block-bootstrap replicate on `gpu_id` and reduce its SHAP values immediately."""
    t0 = time.time()
    device = xgb_device_for(gpu_id) if _BOOT_ON_GPU else "cpu"
    try:
        rng = np.random.default_rng(CFG.SEED + b)
        multiplicity = np.bincount(rng.integers(0, _N_BLOCKS, _N_BLOCKS), minlength=_N_BLOCKS)[_block_codes]
        rows = np.flatnonzero(multiplicity)
        booster = _train_booster(_EXPL_X[rows], _EXPL_Y[rows], weight=multiplicity[rows].astype(np.float32),
                                 rounds=FINAL_ROUNDS, device=device, seed=CFG.SEED + b)
        contribs = _predict_all(booster, _EXPL_X[_BOOT_EVAL_IDX], device=device,
                                batch=CFG.SHAP_BATCH, kind="contribs")
        rec_deltas = _plan_deltas(booster, device)
        del booster
        shap_eval = contribs[:, :_K]
        shap_sample = shap_eval[_BOOT_POS_S]
        curves, thresholds = _replicate_curves(shap_sample)
        result = {
            "ok": True, "b": b, "device": device,
            "mean_abs": np.abs(shap_sample).mean(axis=0, dtype=np.float64),
            "curves": curves, "thresholds": thresholds, "rec_deltas": rec_deltas,
            "last": shap_eval[_BOOT_POS_L].astype(np.float16),
            "seconds": time.time() - t0,
        }
        del contribs, shap_eval, shap_sample
        log(f"  bootstrap {b + 1}/{CFG.N_BOOTSTRAP} on {device}: {len(rows):,} rows, {result['seconds']:.1f}s")
        return result
    except Exception as exc:
        log(f"  bootstrap replicate {b} failed on {device}: {type(exc).__name__}: {exc}", "WARNING")
        return {"ok": False, "b": b, "device": device, "error": f"{type(exc).__name__}: {exc}"}


def _recentred_interval(theta_hat, boot, lower=None):
    """Bias-corrected percentile interval (4.3), elementwise over the leading replicate axis of ``boot``.

    Returns (lo, hi, bias) with lo/hi = theta_hat + q2.5/q97.5 - median and bias = median - theta_hat; NaN where
    fewer than 2 finite replicates exist. ``lower`` clips the interval (e.g. 0 for mean |SHAP|).
    """
    boot = np.asarray(boot, dtype=np.float64)
    theta_hat = np.asarray(theta_hat, dtype=np.float64)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        n_ok = np.isfinite(boot).sum(axis=0)
        q_lo, q_med, q_hi = np.nanpercentile(boot, [2.5, 50.0, 97.5], axis=0)
    lo, hi = theta_hat + (q_lo - q_med), theta_hat + (q_hi - q_med)
    bad = n_ok < 2
    lo, hi = np.where(bad, np.nan, lo), np.where(bad, np.nan, hi)
    if lower is not None:
        lo = np.maximum(lo, lower)
    return lo, hi, np.where(bad, np.nan, q_med - theta_hat)


with timer("04_bootstrap"):
    log(f"Block bootstrap: {CFG.N_BOOTSTRAP} replicates x {FINAL_ROUNDS} rounds, {_N_BLOCKS} blocks of "
        f"{_BOOT_BLOCK_KM:.0f} km, explaining {len(_BOOT_EVAL_IDX):,} rows per replicate "
        f"({'GPU, full data' if _BOOT_ON_GPU else 'CPU, sampled'}; {max(1, ENV.get('n_gpus', 0))} worker(s))")
    _boot_results = gpu_pool_map(_bootstrap_replicate, list(range(int(CFG.N_BOOTSTRAP))))
    _boot_ok = [r for r in _boot_results if r is not None and r.get("ok")]
    _boot_failed = [r for r in _boot_results if r is None or not r.get("ok")]
    if not _boot_ok:
        raise RuntimeError(f"All {CFG.N_BOOTSTRAP} bootstrap replicates failed: "
                           f"{[r.get('error') for r in _boot_failed if r]}")
    _BOOT_MEAN_ABS = np.stack([r["mean_abs"] for r in _boot_ok])            # (B, K)
    _BOOT_CURVES = np.stack([r["curves"] for r in _boot_ok])                # (B, K, bins)
    _BOOT_THRESHOLDS = np.stack([r["thresholds"] for r in _boot_ok])        # (B, K, 3)
    _BOOT_REC_DELTAS = np.stack([r["rec_deltas"] for r in _boot_ok])        # (B, n_plan)
    _boot_last = np.stack([r["last"] for r in _boot_ok])                    # (B, n_last, K) float16
    del _boot_results
    free_memory()

    _ci_cols = {"cell_id": DF["cell_id"].to_numpy()[_rows_last].astype(np.int32)}
    for _jj, _f in enumerate(FEATURES):
        _lo, _hi, _ = _recentred_interval(SHAP_VALUES[_rows_last, _jj], _boot_last[:, :, _jj].astype(np.float32))
        _ci_cols[f"{_f}_lo"] = _lo.astype(np.float32)
        _ci_cols[f"{_f}_hi"] = _hi.astype(np.float32)
    SHAP_CI_LATEST = pd.DataFrame(_ci_cols)
    del _boot_last, _ci_cols
    _n_boot = len(_boot_ok)
    BOOTSTRAP_INFO = {
        "n": _n_boot, "n_requested": int(CFG.N_BOOTSTRAP), "mode": "block",
        "block_size_km": float(_BOOT_BLOCK_KM), "n_blocks": int(_N_BLOCKS), "rounds": int(FINAL_ROUNDS),
        **_BOOT_BLOCK_INFO,
        "ci_level": 0.95,
        "ci_method": "recentred-percentile",
        "ci_method_description": ("bias-corrected percentile interval: the 2.5/97.5 replicate percentiles shifted by "
                                  "(full-data estimate - replicate median); always contains the estimate"),
        "ci_label": "95% CI" if _n_boot >= 50 else "replicate range",
        "replicate_range": bool(_n_boot < 50),
        "eval_rows": int(len(_BOOT_EVAL_IDX)), "full_data": bool(_BOOT_ON_GPU),
        "rows_sampled": not bool(_BOOT_ON_GPU),
        "devices": sorted({r["device"] for r in _boot_ok}),
        "mean_seconds": float(np.mean([r["seconds"] for r in _boot_ok])),
        "failures": [{"b": r.get("b"), "error": r.get("error")} for r in _boot_failed if r],
    }
    del _boot_ok, _boot_failed
log(f"Bootstrap done: {BOOTSTRAP_INFO['n']}/{BOOTSTRAP_INFO['n_requested']} replicates OK "
    f"({BOOTSTRAP_INFO['mean_seconds']:.1f}s each, devices {BOOTSTRAP_INFO['devices']}); intervals labelled "
    f"'{BOOTSTRAP_INFO['ci_label']}'" + ("" if _BOOT_ON_GPU else "; CPU: interval spreads come from a 20k-row sample"))
if BOOTSTRAP_INFO["n"] < 50:
    log(f"Only {BOOTSTRAP_INFO['n']} bootstrap replicates: the 2.5/97.5 percentiles are close to the extreme "
        "replicates, so intervals are reported as a 'replicate range', not a 95% CI", "WARNING")
free_memory()

## 4.12 SHAP importance (global with bootstrap interval, by epoch, by zone)

In [ ]:
def _importance_block(mask):
    sv = SHAP_VALUES[mask].astype(np.float64)
    return {"mean_abs": np.abs(sv).mean(axis=0).tolist(), "mean": sv.mean(axis=0).tolist()}


_imp_full = np.abs(SHAP_VALUES).mean(axis=0, dtype=np.float64)
# Bias reference: the final model on the same rows the replicates were explained on (all rows on GPU).
_imp_ref = np.abs(SHAP_VALUES[_rows_sample]).mean(axis=0, dtype=np.float64)
_imp_lo, _imp_hi, _ = _recentred_interval(_imp_full, _BOOT_MEAN_ABS, lower=0.0)
with warnings.catch_warnings():
    warnings.simplefilter("ignore", RuntimeWarning)
    _imp_bias = np.nanmedian(_BOOT_MEAN_ABS, axis=0) - _imp_ref
SHAP_IMPORTANCE = {
    "global": {"mean_abs": _imp_full.tolist(), "ci_lo": _imp_lo.tolist(), "ci_hi": _imp_hi.tolist(),
               "bias": _imp_bias.tolist()},
    "by_epoch": {y: _importance_block(_YEARS == y) for y in _EPOCH_LIST},
    "by_zone": {z: _importance_block(ZONES == z) for z in range(_N_ZONES) if (ZONES == z).any()},
}
_IMPORTANCE_ORDER = list(np.argsort(SHAP_IMPORTANCE["global"]["mean_abs"])[::-1])
for _r, _jj in enumerate(_IMPORTANCE_ORDER[:10], start=1):
    print(f"{_r:2d}. {FEATURES[_jj]:<16s} mean|SHAP| {SHAP_IMPORTANCE['global']['mean_abs'][_jj]:.3f} °C "
          f"({BOOTSTRAP_INFO['ci_label']} {SHAP_IMPORTANCE['global']['ci_lo'][_jj]:.3f}-"
          f"{SHAP_IMPORTANCE['global']['ci_hi'][_jj]:.3f}; bootstrap bias {SHAP_IMPORTANCE['global']['bias'][_jj]:+.3f})")

## 4.13 Dependence curves, thresholds with bootstrap intervals, and planted-vs-recovered validation

In [ ]:
def _boot_ci(values, point):
    """(interval, support) for one threshold: support = share of replicates with a finite value; the interval
    (bias-corrected percentile, re-centred on ``point``) only when the point exists and support >= 0.5."""
    v = np.asarray(values, dtype=np.float64)
    support = float(np.isfinite(v).mean()) if v.size else 0.0
    if point is None or support < 0.5 or np.isfinite(v).sum() < 2:
        return None, (None if point is None else support)
    lo, hi, _ = _recentred_interval(np.array([float(point)]), v[:, None])
    return [float(lo[0]), float(hi[0])], support


def _none_if_nan(v):
    return None if v is None or not np.isfinite(v) else float(v)


def _scatter_sample(j, partner, rng, max_points=1500):
    """≤ max_points rows with finite x, stratified equally across epochs."""
    per_epoch = max_points // len(_EPOCH_LIST)
    picks = []
    for year in _EPOCH_LIST:
        rows = np.flatnonzero((_YEARS == year) & np.isfinite(_EXPL_X[:, j]))
        if rows.size:
            picks.append(rng.choice(rows, size=min(per_epoch, rows.size), replace=False))
    rows = np.sort(np.concatenate(picks)) if picks else np.array([], dtype=np.int64)
    return {
        "x": _EXPL_X[rows, j].astype(np.float64).tolist(),
        "shap": SHAP_VALUES[rows, j].astype(np.float64).tolist(),
        "color": _EXPL_X[rows, partner].astype(np.float64).tolist(),
        "color_feature": FEATURES[partner],
        "zone": ZONES[rows].astype(int).tolist(),
        "year": _YEARS[rows].astype(int).tolist(),
    }


def _build_threshold(j):
    centers, mean, counts = _DEP_CURVES[j]
    pt = _POINT_THRESHOLDS[j]
    finite_mean = mean[np.isfinite(mean)]
    out = {}
    for col, key in enumerate(("zero_crossing", "breakpoint", "saturation")):
        ci, support = _boot_ci(_BOOT_THRESHOLDS[:, j, col], pt[key])
        out[key] = pt[key]
        out[f"{key}_ci"] = ci
        out[f"{key}_support"] = support
    return {
        "zero_crossing": out["zero_crossing"], "zero_crossing_ci": out["zero_crossing_ci"],
        "zero_crossing_support": out["zero_crossing_support"],
        # several comparable sign changes: no single crossing is reported (SPEC 4.4)
        "zero_crossing_flag": "multiple" if pt["zero_crossing_multiple"] and pt["zero_crossing"] is None else None,
        "breakpoint": out["breakpoint"], "breakpoint_ci": out["breakpoint_ci"],
        "breakpoint_support": out["breakpoint_support"],
        "slope_before": pt["slope_before"], "slope_after": pt["slope_after"],
        "saturation": out["saturation"], "saturation_ci": out["saturation_ci"],
        "saturation_support": out["saturation_support"],
        "effect_range": float(np.ptp(finite_mean)) if finite_mean.size else None,
        "direction": pt["direction"],
    }


with timer("04_dependence"):
    _partner_matrix = INTERACTION_GLOBAL.copy()
    np.fill_diagonal(_partner_matrix, -np.inf)
    _dep_rng = np.random.default_rng(CFG.SEED + 11)
    THRESHOLDS, DEPENDENCE = {}, {}
    for _jj, _f in enumerate(FEATURES):
        _centers, _mean, _counts = _DEP_CURVES[_jj]
        _nb = len(_centers)
        _lo, _hi, _ = _recentred_interval(_mean, _BOOT_CURVES[:, _jj, :_nb])
        THRESHOLDS[_f] = _build_threshold(_jj)
        DEPENDENCE[_f] = {
            "bin_centers": _centers.tolist(), "mean": _mean.tolist(),
            "ci_lo": _lo.tolist(), "ci_hi": _hi.tolist(), "count": _counts.astype(int).tolist(),
            "scatter": _scatter_sample(_jj, int(np.argmax(_partner_matrix[_jj])), _dep_rng),
            "threshold": THRESHOLDS[_f],
        }
_bad_thr = [f for f, t in THRESHOLDS.items() for k in ("zero_crossing", "breakpoint", "saturation")
            if t[k] is None and t[f"{k}_ci"] is not None]
if _bad_thr:
    raise AssertionError(f"threshold interval without a point estimate: {_bad_thr}")


def _fmt_opt(v, ci=None, support=None):
    if v is None:
        return "—"
    text = f"{v:.3f}" + (f" [{ci[0]:.3f}, {ci[1]:.3f}]" if ci else "")
    return text + (f" (support {support:.0%})" if support is not None and support < 0.8 else "")


_thr_table = pd.DataFrame([
    {"feature": f, "direction": t["direction"], "effect_range_C": round(t["effect_range"] or 0.0, 3),
     "zero_crossing": ("multiple" if t["zero_crossing_flag"] == "multiple"
                       else _fmt_opt(t["zero_crossing"], t["zero_crossing_ci"], t["zero_crossing_support"])),
     "breakpoint": _fmt_opt(t["breakpoint"], t["breakpoint_ci"], t["breakpoint_support"]),
     "saturation": _fmt_opt(t["saturation"], t["saturation_ci"], t["saturation_support"])}
    for f, t in THRESHOLDS.items()
])
print(f"Thresholds ({BOOTSTRAP_INFO['ci_label']} in brackets):")
print(_thr_table.to_string(index=False))

_THRESHOLD_KINDS = {"saturation": "saturation", "breakpoint": "breakpoint", "knee": "breakpoint",
                    "threshold": "breakpoint", "zero": "zero_crossing", "crossing": "zero_crossing"}


def _planted_rows(feature, kind, planted):
    """Compare one planted property of `feature` with what the SHAP pipeline recovered."""
    th = THRESHOLDS[feature]
    target = next((v for k, v in _THRESHOLD_KINDS.items() if k in kind), None)
    if kind == "direction":
        return {"feature": feature, "planted": kind, "value": planted, "recovered": th["direction"],
                "abs_error": None, "match": th["direction"] == planted}
    if target is not None and isinstance(planted, (int, float, np.integer, np.floating)):
        rec = th[target]
        return {"feature": feature, "planted": kind, "value": float(planted),
                "recovered": f"{target} {_fmt_opt(rec, th[f'{target}_ci'])}",
                "abs_error": None if rec is None else round(abs(rec - float(planted)), 3), "match": None}
    return {"feature": feature, "planted": kind, "value": planted,
            "recovered": f"direction {th['direction']}, effect range {th['effect_range'] or 0:.2f} °C",
            "abs_error": None, "match": None}


def _planted_vs_recovered(truth):
    """Planted-vs-recovered table from SYNTHETIC_TRUTH.

    Uses the structured `planted_thresholds` {feature: {kind: value}} when present, otherwise flat
    '<feature>_<kind>' keys (e.g. 'ndvi_saturation').
    """
    rows = []
    structured = truth.get("planted_thresholds")
    if isinstance(structured, dict):
        for feature, props in structured.items():
            if feature in THRESHOLDS and isinstance(props, dict):
                rows.extend(_planted_rows(feature, str(kind), value) for kind, value in props.items())
    else:
        for key, planted in truth.items():
            feat = max((f for f in FEATURES if key.startswith(f + "_")), key=len, default=None)
            if feat is not None:
                rows.append(_planted_rows(feat, key[len(feat) + 1:], planted))
    return pd.DataFrame(rows)


def _irrelevant_feature_ranks(truth):
    """Importance rank of features that act on LST only through correlation (should rank low)."""
    names = [f for f in truth.get("irrelevant_by_construction", []) if f in FEATURES]
    ranks = {FEATURES[j]: r for r, j in enumerate(_IMPORTANCE_ORDER, start=1)}
    return pd.DataFrame([{"feature": f, "importance_rank": ranks[f],
                          "mean_abs_shap_C": round(SHAP_IMPORTANCE["global"]["mean_abs"][FEATURES.index(f)], 4)}
                         for f in names])


if SYNTHETIC_TRUTH:
    _pvr = _planted_vs_recovered(SYNTHETIC_TRUTH)
    print("\nPlanted (synthetic generator) vs recovered (SHAP pipeline):")
    print(_pvr.to_string(index=False) if len(_pvr) else "  no feature-level entries in SYNTHETIC_TRUTH")
    _irr = _irrelevant_feature_ranks(SYNTHETIC_TRUTH)
    if len(_irr):
        print(f"\nFeatures irrelevant by construction (importance rank of {_K}; should be low):")
        print(_irr.to_string(index=False))
free_memory()

## 4.14 Zone profiles, transitions and recommendations
The planned actions (4.10) get their bootstrap intervals here. `transitions` count, for each pair of
**consecutive** epochs, the cells present in both epochs by (zone in the earlier epoch, zone in the later epoch).

In [ ]:
_RES_KM2 = (float(CFG.GRID_RES_M) / 1000.0) ** 2
_MIN_DRIVER_C = 0.01   # |mean SHAP| below this is not reported as a zone driver
_DISTRICT_NAMES = {int(d["id"]): d["name"] for d in DISTRICTS}
_DONOR_LABELS = {"other_vegetation": "other vegetation"}


def _as_exported(value, scale=1):
    """`value` rounded to the 4 significant digits written to zones.json (05), times `scale`, as an exact Decimal.

    The rationale text is formatted from this with round-half-up, like the dashboard's Intl formatting of the
    exported number, so the sentence and the table never disagree on a tie (e.g. 0.0275 -> 2.8%, not 2.7%).
    """
    from decimal import Decimal
    return Decimal(repr(float(f"{float(value):.4g}"))) * scale


def _fmt_dp(value, dp, scale=1):
    from decimal import ROUND_HALF_UP, Decimal
    q = _as_exported(value, scale).quantize(Decimal(1).scaleb(-dp), rounding=ROUND_HALF_UP)
    return f"{abs(q) if q == 0 else q:f}"  # never "-0.0"


def _fmt_sig(value, sig=3):
    from decimal import ROUND_HALF_UP, Decimal
    d = _as_exported(value)
    if d == 0:
        return "0"
    q = d.quantize(Decimal(1).scaleb(d.adjusted() - (sig - 1)), rounding=ROUND_HALF_UP)
    text = f"{q:,f}"
    return text.rstrip("0").rstrip(".") if "." in text else text


def _fmt_feature_value(feature, value):
    display = FEATURE_META.get(feature, {}).get("display", "number")
    if display == "percent":
        pct = float(_as_exported(value, 100))
        return f"{_fmt_dp(value, 1, 100)}%" if 0 < abs(pct) < 9.95 else f"{_fmt_dp(value, 0, 100)}%"
    if display == "index":
        return _fmt_dp(value, 2)
    return _fmt_sig(value, 3)


def _pp(frac):
    """Percentage points with one decimal below 10 pp (so a 2.4 pp step is not shown as '2')."""
    v = float(_as_exported(frac, 100))
    return _fmt_dp(frac, 1, 100) if abs(v) < 9.95 else _fmt_dp(frac, 0, 100)


def _donor_text(plan):
    """' The land is taken from ... mostly cropland (-12 pp) and other vegetation (-5 pp); ...' (fraction actions)."""
    moves = sorted(((k, v) for k, v in plan["donors"].items() if abs(v) >= 0.005),
                   key=lambda kv: -abs(kv[1]))[:2]
    verb = "taken from" if plan["step"] > 0 else "given to"
    lead = f" The land is {verb} the other land covers (including other vegetation) in proportion to their shares"
    if moves:
        names = [f"{_DONOR_LABELS.get(k, FEATURE_META.get(k, {}).get('label', k)).lower()} "
                 f"({'+' if v > 0 else '-'}{_pp(abs(v))} pp)" for k, v in moves]
        lead += f", mostly {' and '.join(names)}"
    return lead + "; NDVI, NDWI and NDBI follow through the land-cover coupling."


def _rationale(plan, ci):
    feature = plan["feature"]
    label = FEATURE_META.get(feature, {}).get("label", feature)
    fv = lambda v: _fmt_feature_value(feature, v)  # noqa: E731
    verb = "raising" if plan["step"] > 0 else "lowering"
    if feature in _COUPLING_FRACTIONS:
        change = (f"{verb} {label.lower()} by {_pp(abs(plan['step']))} percentage points in every cell "
                  f"(zone median {fv(plan['current'])} → {fv(plan['target'])})")
    elif feature == "ndvi":
        change = (f"{verb} NDVI by {_fmt_dp(abs(plan['step']), 2)} in every cell (zone median {fv(plan['current'])} → "
                  f"{fv(plan['target'])}), i.e. greening within the existing land covers")
    else:
        change = (f"{verb} {label.lower()} by {_fmt_sig(abs(plan['step']), 3)} in every cell (zone median "
                  f"{fv(plan['current'])} → {fv(plan['target'])})")
    donors = _donor_text(plan) if feature in _COUPLING_FRACTIONS else ""
    label_ci = BOOTSTRAP_INFO["ci_label"]
    ci_text = f"{label_ci} {ci[0]:+.2f} to {ci[1]:+.2f} °C" if ci else "no bootstrap interval"
    kind = plan["kind"]
    if kind == "saturation":
        why = (f" The target is where the {label} effect saturates ({fv(plan['target'])} on the pooled dependence "
               "curve); it already achieves ≥ 90% of the cooling available within the bounds.")
    elif kind == "breakpoint":
        why = (f" The target is where the {label} response changes slope ({fv(plan['target'])}); it already "
               "achieves ≥ 90% of the cooling available within the bounds.")
    elif kind == "bound":
        why = " The target is the feasibility bound (the zone's own 90th/10th percentile, or the maximum step)."
    else:
        why = (" The target is the smallest step that achieves ≥ 90% of the cooling available within the "
               "feasibility bounds.")
    return (f"In the final model, {change} is associated with a mean LST change of {plan['expected_delta_c']:+.2f} °C "
            f"over the zone's {plan['n_cells']:,} cells ({ci_text}).{donors} This is a model-based "
            f"ceteris-paribus estimate, not a causal effect.{why}")


def _build_recommendations():
    by_zone = {m["id"]: [] for m in _ZONE_META}
    for i, plan in enumerate(_REC_PLAN):
        delta = float(_REC_FINAL_DELTAS[i])
        plan["expected_delta_c"] = delta
        if not delta < -_REC_MIN_COOLING_C:
            continue
        boot = _BOOT_REC_DELTAS[:, i] if _BOOT_REC_DELTAS.size else np.zeros(0)
        ci, bias = None, None
        if np.isfinite(boot).sum() >= 2:
            lo, hi, b = _recentred_interval(np.array([delta]), boot[:, None])
            ci, bias = [float(lo[0]), float(hi[0])], float(b[0])
        by_zone[plan["zone"]].append({
            "feature": plan["feature"], "action": plan["action"],
            "current": plan["current"], "target": plan["target"], "expected_delta_c": delta, "ci": ci,
            # priority from the exported (2 dp) value so the bundle is self-consistent (SPEC 4.6)
            "priority": "high" if round(delta, 2) <= -1.0 else ("medium" if round(delta, 2) <= -0.3 else "low"),
            "rationale": _rationale(plan, ci),
            "kind": plan["kind"], "step": plan["step"], "n_cells": plan["n_cells"], "coupled": True,
            "bias": bias,
        })
    return {z: sorted(v, key=lambda r: r["expected_delta_c"])[:_REC_MAX_PER_ZONE] for z, v in by_zone.items()}


def _top_pairs(matrix, n=3):
    iu = np.triu_indices(_K, k=1)
    vals = matrix[iu]
    order = np.argsort(vals)[::-1][:n]
    return [{"a": FEATURES[iu[0][o]], "b": FEATURES[iu[1][o]], "value": float(vals[o])} for o in order]


def _top_districts(mask, n=5):
    d = DF["district"].to_numpy()[mask].astype(np.int64)
    total = max(int(mask.sum()), 1)
    ids, counts = np.unique(d[d >= 0], return_counts=True)
    order = np.argsort(counts)[::-1][:n]
    return [{"district": int(ids[o]), "name": _DISTRICT_NAMES.get(int(ids[o]), str(ids[o])),
             "share": float(counts[o] / total)} for o in order]


def _zone_description(meta, profile):
    last = str(_LAST_YEAR)
    share = profile["n_cells"][last] / max(int((_YEARS == _LAST_YEAR).sum()), 1)
    label = lambda f: FEATURE_META.get(f, {}).get("label", f)  # noqa: E731
    warm = ", ".join(f"{label(t['feature'])} ({t['shap']:+.2f} °C)" for t in profile["top_warming"]) or "none"
    cool = ", ".join(f"{label(t['feature'])} ({t['shap']:+.2f} °C)" for t in profile["top_cooling"]) or "none"
    dist = ", ".join(f"{t['name']} ({100 * t['share']:.0f}%)" for t in profile["top_districts"][:3]) or "n/a"
    lst = profile["lst_obs_mean"][last]
    lst_txt = f"mean observed LST {lst:.1f} °C" if lst is not None else "no cells observed"
    return (f"{meta['name']}: {profile['area_km2'][last]:,.0f} km² in {last} ({100 * share:.0f}% of the study "
            f"area), {lst_txt}. Main warming drivers: {warm}. Main cooling drivers: {cool}. "
            f"Largest shares in {dist}.")


def _zone_profile(meta):
    z = meta["id"]
    mask = ZONES == z
    n_cells, area, lst_obs, lst_pred = {}, {}, {}, {}
    for year in _EPOCH_LIST:
        m = mask & (_YEARS == year)
        n = int(m.sum())
        n_cells[str(year)] = n
        area[str(year)] = n * _RES_KM2
        lst_obs[str(year)] = float(DF["lst"].to_numpy()[m].mean()) if n else None
        lst_pred[str(year)] = float(DF["lst_pred"].to_numpy()[m].mean()) if n else None
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        feature_means = np.nanmean(_EXPL_X[mask].astype(np.float64), axis=0)
    shap_means = SHAP_VALUES[mask].mean(axis=0, dtype=np.float64)
    order = np.argsort(shap_means)
    profile = {
        "id": z, "name": meta["name"], "color": meta["color"], "description": "",
        "n_cells": n_cells, "area_km2": area, "lst_obs_mean": lst_obs, "lst_pred_mean": lst_pred,
        "feature_means": {f: float(feature_means[j]) for j, f in enumerate(FEATURES)},
        "shap_means": {f: float(shap_means[j]) for j, f in enumerate(FEATURES)},
        "top_warming": [{"feature": FEATURES[j], "shap": float(shap_means[j])}
                        for j in order[::-1][:3] if shap_means[j] >= _MIN_DRIVER_C],
        "top_cooling": [{"feature": FEATURES[j], "shap": float(shap_means[j])}
                        for j in order[:3] if shap_means[j] <= -_MIN_DRIVER_C],
        "top_interactions": _top_pairs(INTERACTION_BY_ZONE[z]),
        "top_districts": _top_districts(mask),
        "recommendations": RECOMMENDATIONS[z],
    }
    profile["description"] = _zone_description(meta, profile)
    return profile


def _zone_transitions():
    """{"<y0>-><y1>": 4x4 counts} for consecutive epoch pairs; rows = zone in y0, cols = zone in y1."""
    out = {}
    frame = pd.DataFrame({"cell_id": DF["cell_id"].to_numpy(), "year": _YEARS, "zone": ZONES.astype(np.int64)})
    for y0, y1 in zip(_EPOCH_LIST[:-1], _EPOCH_LIST[1:]):
        pair = frame[frame.year == y0].merge(frame[frame.year == y1], on="cell_id", suffixes=("_a", "_b"))
        counts = np.bincount(pair["zone_a"].to_numpy() * _N_ZONES + pair["zone_b"].to_numpy(),
                             minlength=_N_ZONES * _N_ZONES).reshape(_N_ZONES, _N_ZONES)
        out[f"{y0}->{y1}"] = counts.astype(int).tolist()
    return out


with timer("04_zone_profiles"):
    RECOMMENDATIONS = _build_recommendations()
    ZONE_PROFILES = [_zone_profile(m) for m in _ZONE_META]
    ZONE_TRANSITIONS = _zone_transitions()
for _p in ZONE_PROFILES:
    print(f"\n[{_p['id']}] {_p['description']}")
    for _rec in _p["recommendations"]:
        print(f"    - {_rec['priority'].upper():6s} {_rec['rationale']}")
    if not _p["recommendations"]:
        print("    - no feasible action with an expected cooling > 0.05 °C")

## 4.15 Figures

In [ ]:
_FIG_DIR = Path(CFG.FIG_DIR)
_FIG_DIR.mkdir(parents=True, exist_ok=True)


def _save_fig(fig, name):
    fig.savefig(_FIG_DIR / name, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)


def _label(f):
    return FEATURE_META.get(f, {}).get("label", f)


def _fig_beeswarm(n_top=12, n_rows=4000):
    """Beeswarm-style summary: SHAP per row, coloured by the within-sample rank of the feature value."""
    rng = np.random.default_rng(CFG.SEED)
    rows = rng.choice(len(DF), size=min(n_rows, len(DF)), replace=False)
    top = _IMPORTANCE_ORDER[:n_top]
    cmap = plt.get_cmap("Blues")
    fig, ax = plt.subplots(figsize=(9, 0.45 * len(top) + 1.5))
    for pos, j in enumerate(reversed(top)):
        s = SHAP_VALUES[rows, j]
        x = _EXPL_X[rows, j]
        rank = pd.Series(x).rank(pct=True).to_numpy()
        colors = np.where(np.isfinite(rank)[:, None], cmap(0.25 + 0.75 * np.nan_to_num(rank)),
                          np.array([[0.6, 0.6, 0.6, 1.0]]))
        # Jitter proportional to local density gives the beeswarm silhouette without an O(n^2) layout.
        hist, edges = np.histogram(s, bins=60)
        dens = hist[np.clip(np.searchsorted(edges, s) - 1, 0, 59)] / max(hist.max(), 1)
        ax.scatter(s, pos + rng.uniform(-0.4, 0.4, s.size) * dens, s=3, c=colors, linewidths=0)
    ax.axvline(0, color="#475569", lw=0.8)
    ax.set_yticks(range(len(top)))
    ax.set_yticklabels([_label(FEATURES[j]) for j in reversed(top)])
    ax.set_xlabel("SHAP value (°C contribution to LST, relative to the average cell)")
    ax.set_title("SHAP summary (colour: feature value rank, light = low, dark = high)")
    _save_fig(fig, "shap_summary_beeswarm.png")


def _fig_importance():
    imp = SHAP_IMPORTANCE["global"]
    order = _IMPORTANCE_ORDER[::-1]
    vals = np.array(imp["mean_abs"])[order]
    lo = vals - np.array(imp["ci_lo"])[order]
    hi = np.array(imp["ci_hi"])[order] - vals
    fig, ax = plt.subplots(figsize=(8, 0.35 * _K + 1.5))
    ax.barh(range(_K), vals, color="#22d3ee", height=0.6,
            xerr=np.vstack([np.clip(np.nan_to_num(lo), 0, None), np.clip(np.nan_to_num(hi), 0, None)]),
            ecolor="#334155", capsize=2)
    ax.set_yticks(range(_K))
    ax.set_yticklabels([_label(FEATURES[j]) for j in order])
    ax.set_xlabel(f"mean |SHAP| (°C), whiskers = {BOOTSTRAP_INFO['ci_label']} (block bootstrap, bias-corrected)")
    ax.set_title(f"Global feature importance ({BOOTSTRAP_INFO['n']} bootstrap replicates)")
    _save_fig(fig, "shap_importance_ci.png")


def _fig_dependence_grid():
    top = _IMPORTANCE_ORDER[:6]
    fig, axes = plt.subplots(2, 3, figsize=(15, 8.5))
    styles = {"zero_crossing": (":", "zero crossing"), "breakpoint": ("--", "breakpoint"),
              "saturation": ("-.", "saturation")}
    for ax, j in zip(axes.ravel(), top):
        f = FEATURES[j]
        dep = DEPENDENCE[f]
        sc = dep["scatter"]
        ax.scatter(sc["x"], sc["shap"], s=4, color="#94a3b8", alpha=0.5, linewidths=0, label="cells (sample)")
        ax.fill_between(dep["bin_centers"], dep["ci_lo"], dep["ci_hi"], color="#a78bfa", alpha=0.3,
                        label=f"{BOOTSTRAP_INFO['ci_label']} (bootstrap)")
        ax.plot(dep["bin_centers"], dep["mean"], color="#6d28d9", lw=2, label="binned mean")
        for key, (ls, name) in styles.items():
            if THRESHOLDS[f][key] is not None:
                ax.axvline(THRESHOLDS[f][key], color="#0f172a", ls=ls, lw=1, label=name)
        ax.axhline(0, color="#475569", lw=0.6)
        ax.set_title(f"{_label(f)} ({THRESHOLDS[f]['direction']})")
        ax.set_xlabel(_label(f))
        ax.set_ylabel("SHAP (°C vs average cell)")
    for ax in axes.ravel()[len(top):]:
        ax.set_visible(False)
    handles, labels = axes.ravel()[0].get_legend_handles_labels()
    for ax in axes.ravel()[1:len(top)]:
        for h, lab in zip(*ax.get_legend_handles_labels()):
            if lab not in labels:
                handles.append(h)
                labels.append(lab)
    fig.legend(handles, labels, loc="lower center", ncol=6, frameon=False)
    fig.tight_layout(rect=(0, 0.05, 1, 1))
    _save_fig(fig, "shap_dependence_top6.png")


def _fig_zone_map():
    from matplotlib.colors import ListedColormap
    from matplotlib.patches import Patch
    last = _YEARS == _LAST_YEAR
    raster = np.full((GRID.height, GRID.width), np.nan)
    raster[DF["row"].to_numpy()[last], DF["col"].to_numpy()[last]] = ZONES[last]
    cmap = ListedColormap([m["color"] for m in _ZONE_META])
    fig, ax = plt.subplots(figsize=(8, 8 * GRID.height / max(GRID.width, 1) + 0.5))
    ax.imshow(raster, cmap=cmap, vmin=-0.5, vmax=_N_ZONES - 0.5, interpolation="nearest")
    ax.set_axis_off()
    ax.legend(handles=[Patch(color=m["color"], label=f"{m['id']} {m['name']}") for m in _ZONE_META],
              loc="lower left", frameon=True, fontsize=8)
    ax.set_title(f"SHAP governance zones, {_LAST_YEAR}")
    _save_fig(fig, f"zones_map_{_LAST_YEAR}.png")


def _fig_zone_signature():
    mat = np.array([[p["shap_means"][f] for f in FEATURES] for p in ZONE_PROFILES])
    vmax = float(np.max(np.abs(mat))) or 1.0
    fig, ax = plt.subplots(figsize=(0.6 * _K + 2, 3.2))
    im = ax.imshow(mat, cmap="RdBu_r", vmin=-vmax, vmax=vmax, aspect="auto")
    for (r, c), v in np.ndenumerate(mat):
        ax.text(c, r, f"{v:.2f}", ha="center", va="center", fontsize=6,
                color="white" if abs(v) > 0.6 * vmax else "#0f172a")
    ax.set_xticks(range(_K))
    ax.set_xticklabels([_label(f) for f in FEATURES], rotation=60, ha="right", fontsize=8)
    ax.set_yticks(range(len(ZONE_PROFILES)))
    ax.set_yticklabels([p["name"] for p in ZONE_PROFILES])
    fig.colorbar(im, ax=ax, label="mean SHAP (°C)")
    ax.set_title("Zone SHAP signatures")
    _save_fig(fig, "zones_shap_signature.png")


def _fig_k_diagnostics():
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.6))
    ks = ZONE_DIAGNOSTICS["k"]
    ax1.plot(ks, ZONE_DIAGNOSTICS["inertia"], marker="o", color="#22d3ee", lw=2)
    ax1.set_title("K-means inertia (SHAP space)")
    ax2.plot(ks, ZONE_DIAGNOSTICS["silhouette"], marker="o", color="#a78bfa", lw=2)
    ax2.set_title("Silhouette (20k-row sample)")
    for ax in (ax1, ax2):
        ax.axvline(_N_ZONES, color="#fb7185", ls="--", lw=1, label="K = 4 (governance contract)")
        ax.set_xlabel(f"K ({ZONE_DIAGNOSTICS['n_init']} initialisations each)")
    ax2.legend(frameon=False, fontsize=8)
    _save_fig(fig, "zones_k_diagnostics.png")


with timer("04_figures"):
    for _fig_fn in (_fig_beeswarm, _fig_importance, _fig_dependence_grid, _fig_zone_map,
                    _fig_zone_signature, _fig_k_diagnostics):
        try:
            _fig_fn()
        except Exception as _exc:  # a broken figure must never lose the computed results
            log(f"Figure {_fig_fn.__name__} failed: {type(_exc).__name__}: {_exc}", "WARNING")
            plt.close("all")
free_memory()
log("Section 04 complete: FINAL_MODEL, SHAP_VALUES, INTERACTION_*, THRESHOLDS, ZONES, RECOMMENDATIONS ready")

# 5. Exports: Parquet, GeoJSON, native model, compact web model and the dashboard bundle

```
outputs/
├── parquet/  features_long, predictions_shap, cv_fold_metrics, cv_summary, moran_results, shap_ci_<last>
├── npz/      shap_interactions_<year>.npz  (written in section 4)
├── geo/      ncr_districts.geojson, zones_<year>.geojson (1 km cell squares, lon/lat)
├── models/   xgb_final.json (native XGBoost)
├── figures/  *.png
├── web/      manifest.json, epoch_<year>.json.gz, model_web.json, shap_global.json, dependence.json.gz,
│             zones.json, metrics.json, interactions.json, scenario_coupling.json, districts.geojson
└── web_bundle.zip
```

**Compression.** The two large file kinds (`epoch_<year>` and `dependence`) are written gzip-compressed (level 9,
fixed mtime so re-runs are byte-identical) as `*.json.gz`; `manifest.files` names them. Readers detect gzip by its
magic bytes (0x1f 0x8b), not by the file name, so a server that already sent `Content-Encoding: gzip` also works.
This shrinks a 1 km bundle from ~71 MB to ~19 MB.

**Loading into the dashboard:** copy the *contents* of `outputs/web/` (or unzip `web_bundle.zip`) into
`dashboard/public/data/`, then `npm run dev` (or `npm run build`) inside `dashboard/`. On Kaggle download
`web_bundle.zip` from the output panel. Alternatively host the files anywhere and set `VITE_DATA_BASE_URL`.

**Web model.** The dashboard runs scenario inference in the browser with a *surrogate* XGBoost model
(depth ≤ 6, ≤ 400 trees) serialised to a compact array format (`lst-xgb-compact-v1`). The export evaluates
that format with an independent numpy traverser and requires agreement with XGBoost to < 10⁻³ (including
missing values), and embeds 50 check rows so the JavaScript evaluator can prove the same.

All web JSON is compact UTF-8 with NaN/Inf → `null`; rounding: lon/lat 5 dp, °C 2 dp, SHAP 3 dp,
feature values 4 significant digits. Every file is re-read and validated against the contract at the end.

In [ ]:
import datetime as _dt
import gzip
import json
import math
import os
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import xgboost as xgb

_OUT = Path(CFG.OUTPUT_DIR)
_PARQUET_DIR, _GEO_DIR, _MODEL_DIR, _WEB_DIR = (Path(CFG.PARQUET_DIR), Path(CFG.GEO_DIR),
                                                 Path(CFG.MODEL_DIR), Path(CFG.WEB_DIR))
for _d in (_PARQUET_DIR, _GEO_DIR, _MODEL_DIR, _WEB_DIR):
    _d.mkdir(parents=True, exist_ok=True)
_EXP_EPOCHS = sorted(int(y) for y in DF["year"].unique())
_EXP_LAST = _EXP_EPOCHS[-1]
_WEB_MAX_BYTES = 6 * 1024 * 1024
_TARGET_UNIT_OFFSET = {y: (float(EPOCH_MEANS[y]) if CFG.TARGET_MODE == "anomaly" else 0.0) for y in _EXP_EPOCHS}


def _round_dp(arr, dp):
    return np.round(np.asarray(arr, dtype=np.float64), dp)


def _round_sig(arr, sig=4):
    """Round to `sig` significant digits, producing the nearest double of the decimal (clean JSON)."""
    a = np.asarray(arr, dtype=np.float64)
    out = np.full(a.shape, np.nan)
    ok = np.isfinite(a) & (a != 0)
    out[np.isfinite(a) & (a == 0)] = 0.0
    if ok.any():
        k = (sig - 1 - np.floor(np.log10(np.abs(a[ok])))).astype(np.int64)
        v = a[ok]
        pos = k >= 0
        res = np.empty_like(v)
        scale_pos = 10.0 ** k[pos]
        res[pos] = np.round(v[pos] * scale_pos) / scale_pos          # integer / 10^k: correctly rounded
        scale_neg = 10.0 ** (-k[~pos])
        res[~pos] = np.round(v[~pos] / scale_neg) * scale_neg        # integer * 10^m: exact
        out[ok] = res
    return out


def _sig(v, sig=4):
    """Scalar version of _round_sig (None for missing/non-finite)."""
    if v is None:
        return None
    r = float(_round_sig(np.array([v], dtype=np.float64), sig)[0])
    return r if math.isfinite(r) else None


def _json_list(arr):
    """numpy array -> list with None for NaN/Inf."""
    a = np.asarray(arr)
    if a.dtype.kind in "iub":
        return a.tolist()
    a = a.astype(np.float64)
    out = a.tolist()
    for i in np.flatnonzero(~np.isfinite(a)):
        out[i] = None
    return out


def _clean(obj):
    """Recursively convert numpy/pandas types to JSON-safe Python (NaN/Inf -> None, keys -> str)."""
    if isinstance(obj, dict):
        return {str(k): _clean(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_clean(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return _clean(_json_list(obj) if obj.ndim == 1 else obj.tolist())
    if isinstance(obj, (bool, np.bool_)):
        return bool(obj)
    if isinstance(obj, (int, np.integer)):
        return int(obj)
    if isinstance(obj, (float, np.floating)):
        v = float(obj)
        return v if math.isfinite(v) else None
    if isinstance(obj, Path):
        return str(obj)
    if obj is None or isinstance(obj, str):
        return obj
    if isinstance(obj, (pd.Timestamp, _dt.datetime, _dt.date)):
        return obj.isoformat()
    raise TypeError(f"Cannot serialise {type(obj).__name__} to JSON")


def _write_json(obj, path):
    """Compact UTF-8 JSON without NaN/Infinity (gzip level 9 when the name ends in .gz); returns bytes on disk."""
    data = json.dumps(_clean(obj), separators=(",", ":"), ensure_ascii=False, allow_nan=False).encode("utf-8")
    path = Path(path)
    if path.suffix == ".gz":
        data = gzip.compress(data, compresslevel=9, mtime=0)
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_bytes(data)
    os.replace(tmp, path)
    return len(data)


def _read_bytes_maybe_gzip(path):
    """File bytes, gunzipped when they start with the gzip magic number (independent of the file name)."""
    raw = Path(path).read_bytes()
    return gzip.decompress(raw) if raw[:2] == b"\x1f\x8b" else raw


def _round_nested(obj, fn):
    """Apply a scalar rounding function to every float inside nested lists/dicts."""
    if isinstance(obj, dict):
        return {k: _round_nested(v, fn) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_round_nested(v, fn) for v in obj]
    if isinstance(obj, (float, np.floating)):
        v = float(obj)
        return fn(v) if math.isfinite(v) else None
    return obj


def _dp(n):
    return lambda v: round(v, n)

## 5.1 Parquet, native model

In [ ]:
_BASE_DF_COLS = ["cell_id", "row", "col", "x", "y", "lon", "lat", "district", "year", *FEATURES, "pop_density",
                 "lst", "lst_obs_count", "epoch_mean", "target", "spatial_block", "fold_spatial", "fold_random",
                 "block10_id"]


def _to_parquet(df, name):
    path = _PARQUET_DIR / name
    df.to_parquet(path, engine="pyarrow", compression="zstd", index=False)
    log(f"  {path.name}: {len(df):,} rows, {path.stat().st_size / 1e6:.2f} MB")
    return path


def _export_parquet():
    _to_parquet(DF[[c for c in _BASE_DF_COLS if c in DF.columns]], "features_long.parquet")
    pred = DF[["cell_id", "row", "col", "lon", "lat", "district", "year", "lst", "lst_pred", "lst_pred_oof",
               "resid_oof", "zone"]].copy()
    for j, f in enumerate(FEATURES):
        pred[f"shap_{f}"] = SHAP_VALUES[:, j]
    _to_parquet(pred, "predictions_shap.parquet")
    _to_parquet(CV_FOLD_METRICS, "cv_fold_metrics.parquet")
    _to_parquet(CV_SUMMARY, "cv_summary.parquet")
    _to_parquet(MORAN_RESULTS, "moran_results.parquet")
    _to_parquet(SHAP_CI_LATEST, f"shap_ci_{_EXP_LAST}.parquet")


with timer("05_parquet_model"):
    _export_parquet()
    FINAL_MODEL.save_model(str(_MODEL_DIR / "xgb_final.json"))
    log(f"  models/xgb_final.json: {(_MODEL_DIR / 'xgb_final.json').stat().st_size / 1e6:.2f} MB")

## 5.2 GeoJSON: districts and per-epoch zone cells
Cell squares are built from the grid corners in UTM 43N and reprojected once (a (H+1)×(W+1) corner lattice)
to lon/lat, so neighbouring polygons share vertices exactly. Rings are counter-clockwise (RFC 7946).

In [ ]:
def _corner_lonlat():
    """(H+1, W+1) lon/lat arrays of the grid-cell corners."""
    try:
        from pyproj import Transformer
    except ImportError:
        pip_install(["pyproj"])
        from pyproj import Transformer
    xs = GRID.x0 + np.arange(GRID.width + 1) * GRID.res
    ys = GRID.y0 - np.arange(GRID.height + 1) * GRID.res
    XX, YY = np.meshgrid(xs, ys)
    lon, lat = Transformer.from_crs(GRID.crs, "EPSG:4326", always_xy=True).transform(XX, YY)
    return np.round(lon, 5), np.round(lat, 5)


def _write_zone_geojson(year, lon_c, lat_c):
    sub = DF[DF["year"] == year]
    r, c = sub["row"].to_numpy(), sub["col"].to_numpy()
    lst = _json_list(_round_dp(sub["lst"], 2))
    pred = _json_list(_round_dp(sub["lst_pred"], 2))
    path = _GEO_DIR / f"zones_{year}.geojson"
    with open(path, "w", encoding="utf-8") as fh:
        fh.write('{"type":"FeatureCollection","features":[')
        for i in range(len(sub)):
            # bottom-left -> bottom-right -> top-right -> top-left -> close (counter-clockwise; row 0 is north)
            corners = [(r[i] + 1, c[i]), (r[i] + 1, c[i] + 1), (r[i], c[i] + 1), (r[i], c[i]), (r[i] + 1, c[i])]
            ring = [[float(lon_c[a, b]), float(lat_c[a, b])] for a, b in corners]
            feature = {"type": "Feature",
                       "geometry": {"type": "Polygon", "coordinates": [ring]},
                       "properties": {"cell_id": int(sub["cell_id"].iat[i]), "zone": int(sub["zone"].iat[i]),
                                      "lst": lst[i], "lst_pred": pred[i]}}
            fh.write(("," if i else "") + json.dumps(feature, separators=(",", ":"), allow_nan=False))
        fh.write("]}")
    return path


def _districts_geojson():
    """BOUNDARY_GEOJSON with 5-dp coordinates and exactly the contract properties."""
    feats = []
    for feat in BOUNDARY_GEOJSON["features"]:
        props = feat.get("properties", {})
        feats.append({"type": "Feature",
                      "geometry": _round_nested(feat["geometry"], _dp(5)),
                      "properties": {"id": int(props["id"]), "name": str(props["name"]),
                                     "state": str(props["state"])}})
    return {"type": "FeatureCollection", "features": feats}


with timer("05_geojson"):
    _LON_C, _LAT_C = _corner_lonlat()
    _DISTRICTS_FC = _districts_geojson()
    _write_json(_DISTRICTS_FC, _GEO_DIR / "ncr_districts.geojson")
    for _y in _EXP_EPOCHS:
        _zp = _write_zone_geojson(_y, _LON_C, _LAT_C)
        log(f"  {_zp.name}: {_zp.stat().st_size / 1e6:.1f} MB")
    _rows_c, _cols_c = DF["row"].to_numpy(), DF["col"].to_numpy()
    _cell_lons = np.concatenate([_LON_C[_rows_c, _cols_c], _LON_C[_rows_c + 1, _cols_c + 1]])
    _cell_lats = np.concatenate([_LAT_C[_rows_c, _cols_c], _LAT_C[_rows_c + 1, _cols_c + 1]])
    _BBOX = [float(np.min(_cell_lons)), float(np.min(_cell_lats)), float(np.max(_cell_lons)),
             float(np.max(_cell_lats))]
    del _cell_lons, _cell_lats, _rows_c, _cols_c
free_memory()

## 5.3 Compact web model (surrogate) with an independent evaluator

In [ ]:
def _parse_base_score(raw):
    """learner_model_param.base_score is '4.1E1' (XGBoost 2.x) or '[4.1E1]' (3.x vector form)."""
    text = str(raw).strip().strip("[]").split(",")[0].strip()
    return float(text)


def _booster_to_compact(booster, fidelity, check):
    """Convert a gbtree booster's JSON dump to the lst-xgb-compact-v1 structure (SPEC 4.3)."""
    raw = json.loads(bytes(booster.save_raw(raw_format="json")).decode("utf-8"))
    learner = raw["learner"]
    gb = learner["gradient_booster"]
    if gb.get("name") != "gbtree":
        raise ValueError(f"compact export supports gbtree only, got {gb.get('name')}")
    trees = []
    for tree in gb["model"]["trees"]:
        left = [int(v) for v in tree["left_children"]]
        right = [int(v) for v in tree["right_children"]]
        leaf = [lc == -1 for lc in left]
        cond = tree["split_conditions"]
        trees.append({
            "f": [-1 if is_leaf else int(fi) for fi, is_leaf in zip(tree["split_indices"], leaf)],
            "t": [0 if is_leaf else float(np.float32(t)) for t, is_leaf in zip(cond, leaf)],
            "l": left,
            "r": right,
            "d": [0 if is_leaf else int(bool(dl)) for dl, is_leaf in zip(tree["default_left"], leaf)],
            "v": [float(f"{float(np.float32(v)):.9g}") if is_leaf else 0 for v, is_leaf in zip(cond, leaf)],
        })
    return {
        "format": "lst-xgb-compact-v1",
        "objective": str(learner.get("objective", {}).get("name", "reg:squarederror")),
        "target_mode": CFG.TARGET_MODE,
        "features": list(FEATURES),
        "base_score": _parse_base_score(learner["learner_model_param"]["base_score"]),
        "n_trees": len(trees),
        "trees": trees,
        "fidelity": fidelity,
        "check": check,
    }


def _compile_compact(model):
    return [tuple(np.asarray(t[k], dtype=dt) for k, dt in
                  (("f", np.int64), ("t", np.float64), ("l", np.int64), ("r", np.int64), ("d", np.int64),
                   ("v", np.float64))) for t in model["trees"]]


def _predict_compact(model, X):
    """Pure-numpy evaluator of lst-xgb-compact-v1 (NaN/None = missing). Returns target units."""
    X = np.array(X, dtype=np.float64)  # None -> NaN
    out = np.full(X.shape[0], float(model["base_score"]), dtype=np.float64)
    for f, t, left, right, dflt, v in _compile_compact(model):
        node = np.zeros(X.shape[0], dtype=np.int64)
        while True:
            active = np.flatnonzero(left[node] != -1)
            if active.size == 0:
                break
            nd = node[active]
            xv = X[active, f[nd]]
            go_left = np.where(np.isnan(xv), dflt[nd] == 1, xv < t[nd])
            node[active] = np.where(go_left, left[nd], right[nd])
        out += v[node]
    return out


def _train_surrogate(rounds):
    """Depth-6 XGBoost on all rows (CUDA with CPU fallback on OOM)."""
    params = {"objective": "reg:squarederror", "max_depth": 6, "learning_rate": 0.08, "subsample": 0.8,
              "colsample_bytree": 0.8, "min_child_weight": 5, "tree_method": "hist", "max_bin": 256,
              "seed": int(CFG.SEED), "verbosity": 0}
    X = DF[FEATURES].to_numpy(np.float32)
    y = DF[TARGET_COL].to_numpy(np.float32)
    for device in dict.fromkeys([xgb_device_for(0), "cpu"]):
        try:
            dtrain = xgb.QuantileDMatrix(X, label=y, max_bin=256, feature_names=list(FEATURES))
            booster = xgb.train({**params, "device": device}, dtrain, num_boost_round=int(rounds))
            del dtrain
            booster.set_param({"device": "cpu"})
            return booster
        except Exception as exc:
            if device == "cpu" or not is_gpu_oom_error(exc):
                raise
            log(f"GPU OOM training the surrogate on {device}; retrying on CPU", "WARNING")
    raise RuntimeError("unreachable: surrogate training loop exhausted")


def _r2(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true, np.float64), np.asarray(y_pred, np.float64)
    ok = np.isfinite(y_true) & np.isfinite(y_pred)
    sst = np.sum((y_true[ok] - y_true[ok].mean()) ** 2)
    return float(1.0 - np.sum((y_true[ok] - y_pred[ok]) ** 2) / sst) if sst > 0 else None


def _with_injected_nans(X, rng, frac=0.1):
    Xn = X.astype(np.float32).copy()
    Xn[rng.random(Xn.shape) < frac] = np.nan
    return Xn


def _build_web_model(rounds=400):
    rng = np.random.default_rng(CFG.SEED + 404)
    X_all = DF[FEATURES].to_numpy(np.float32)
    y_all = DF[TARGET_COL].to_numpy(np.float64)
    final_margin = DF["lst_pred"].to_numpy(np.float64) - DF["year"].map(_TARGET_UNIT_OFFSET).to_numpy(np.float64)
    test_rows = rng.choice(len(DF), size=max(2000, min(5000, len(DF))), replace=len(DF) < 2000)
    X_test = _with_injected_nans(X_all[test_rows], rng)
    check_X = _with_injected_nans(X_all[rng.choice(len(DF), size=50, replace=len(DF) < 50)], rng, frac=0.15)
    full = _train_surrogate(rounds)
    n_trees = full.num_boosted_rounds()
    while True:
        booster = full[:n_trees] if n_trees < full.num_boosted_rounds() else full
        dm = lambda A: xgb.DMatrix(A, missing=np.nan, feature_names=list(FEATURES))  # noqa: E731
        surrogate_all = booster.predict(dm(X_all), output_margin=True).astype(np.float64)
        fidelity = {
            "r2_vs_final": _r2(final_margin, surrogate_all),
            "rmse_vs_final": float(np.sqrt(np.mean((final_margin - surrogate_all) ** 2))),
            "r2_vs_obs": _r2(y_all, surrogate_all),
            "max_depth": 6, "rounds": int(n_trees),
        }
        check = {"rows": [_json_list(r) for r in check_X],
                 "expected": booster.predict(dm(check_X), output_margin=True).astype(np.float64).tolist()}
        model = json.loads(json.dumps(_clean(_booster_to_compact(booster, fidelity, check)),
                                      separators=(",", ":"), allow_nan=False))
        size = len(json.dumps(model, separators=(",", ":")).encode("utf-8"))
        if size <= _WEB_MAX_BYTES or n_trees <= 20:
            break
        n_trees = max(20, int(n_trees * _WEB_MAX_BYTES / size * 0.95))
        log(f"  model_web.json would be {size / 1e6:.1f} MB; truncating surrogate to {n_trees} trees")
    # Independent verification of the serialised format against XGBoost, missing values included.
    diff = float(np.max(np.abs(_predict_compact(model, X_test)
                               - booster.predict(dm(X_test), output_margin=True).astype(np.float64))))
    if not diff < 1e-3:
        raise RuntimeError(f"compact model evaluator disagrees with XGBoost: max |diff| = {diff:.3e}")
    check_diff = float(np.max(np.abs(_predict_compact(model, np.array(check["rows"], dtype=np.float64))
                                     - np.array(check["expected"]))))
    if not check_diff < 1e-3:
        raise RuntimeError(f"compact model check rows disagree: max |diff| = {check_diff:.3e}")
    log(f"  surrogate: {n_trees} trees, {size / 1e6:.2f} MB; evaluator vs XGBoost max diff {diff:.1e} "
        f"on {len(X_test):,} rows with NaNs; R2 vs final {fidelity['r2_vs_final']:.4f}, "
        f"RMSE vs final {fidelity['rmse_vs_final']:.3f}, R2 vs obs {fidelity['r2_vs_obs']:.4f}")
    del full, booster
    return model


with timer("05_web_model"):
    _MODEL_WEB = _build_web_model()
free_memory()

## 5.4 Web bundle (SPEC 4.1 – 4.8)

In [ ]:
# Data caveats (SPEC 2); fallback used only when section 1 provided neither DATA_CAVEATS nor
# DATA_SOURCES["notes"]["caveats"].
_SOURCE_CAVEATS = [
    "Land cover combines different products per epoch (GLC_FCS30D 2010/2015, ESA WorldCover 2020, Dynamic World "
    "2025); part of the apparent land-cover change is cross-product inconsistency.",
    "2010 night-time lights are DMSP-OLS mapped to the VIIRS scale by isotonic regression; DMSP saturates in "
    "urban cores, so 2010 NTL is compressed at the high end.",
    "Terra's orbit drifted after 2022 (earlier overpass time), which lowers 2025 daytime LST relative to earlier "
    "epochs; the anomaly target removes the epoch-wide offset but not spatially varying effects.",
    "GHS-POP 2025 is a projection rescaled to WorldPop 2020 totals, not a census-based estimate.",
]
_MODEL_CAVEATS = [
    "SHAP values explain the model, not causal effects. Zone recommendations are model-based counterfactuals "
    "(the final model re-evaluated on the zone's cells after a feasible land-cover / NDVI / landscape change); the "
    "cooling they report is associated with the change in the model, not a causal effect.",
    "Zones group cells whose predicted anomaly is attributed to the same features in similar amounts (current state "
    "relative to the average cell); zone names follow a fixed labelling rule (see zones.json labelling).",
    "In-browser scenarios use a depth-6 surrogate of the final model (see model_web.json fidelity).",
    "Temporal cross-validation scores the spatial anomaly pattern of the held-out last epoch (its epoch-wide mean "
    "is removed from the target by design), so it measures pattern transfer, not prediction of the climate offset.",
]


def _manifest_sources():
    """DATA_SOURCES as {variable: {year|key: provenance}}; section-1 caveats under 'notes' become notes."""
    sources, notes = {}, []
    for var, vals in DATA_SOURCES.items():
        if var == "notes" and isinstance(vals, dict):
            notes.extend(str(c) for c in vals.get("caveats", []))
        elif isinstance(vals, dict):
            sources[str(var)] = {str(k): str(v) for k, v in vals.items()}
        else:
            sources[str(var)] = {"all": str(vals)}
    return sources, notes


EPOCH_FILE = "epoch_{year}.json.gz"
DEPENDENCE_FILE = "dependence.json.gz"


def _manifest():
    sources, source_notes = _manifest_sources()
    # Section 1 publishes its caveats as DATA_CAVEATS (mode-aware: includes the SYNTHETIC notice); the static list
    # above is only a fallback when section 1 was replaced by an older version without that global.
    section1_caveats = [str(c) for c in (globals().get("DATA_CAVEATS") or [])]
    notes = (source_notes or section1_caveats or list(_SOURCE_CAVEATS)) + list(_MODEL_CAVEATS)
    if DATA_MODE == "synthetic" and not any("SYNTHETIC" in n for n in notes):
        notes.insert(0, "SYNTHETIC DEMO DATA: generated by the notebook's synthetic generator, not satellite "
                        "observations. Use for software demonstration only.")
    if CFG.TARGET_MODE == "anomaly":
        notes.append("Target is the LST anomaly from each epoch's spatial mean; predictions are converted back "
                     "to °C by adding epoch_mean.")
    if BOOTSTRAP_INFO.get("replicate_range"):
        notes.append(f"Only {BOOTSTRAP_INFO['n']} bootstrap replicates: intervals are a replicate range "
                     "(bias-corrected 2.5-97.5 percentiles of the replicates), not calibrated 95% confidence intervals.")
    features = []
    for f in FEATURES:
        meta = FEATURE_META[f]
        features.append({"name": f, "label": meta["label"], "unit": meta["unit"], "group": meta["group"],
                         "description": meta["description"], "source": meta["source"],
                         "actionable": bool(meta["actionable"]), "display": meta["display"],
                         "stats": {k: _sig(FEATURE_STATS[f][k]) for k in
                                   ("min", "max", "p01", "p99", "median", "mean", "std")}})
    center_lon, center_lat = 0.5 * (_BBOX[0] + _BBOX[2]), 0.5 * (_BBOX[1] + _BBOX[3])
    roi_prov = DATA_SOURCES.get("roi")
    roi_sources = sorted({str(v) for v in roi_prov.values()}) if isinstance(roi_prov, dict) else []
    return {
        "schema_version": "1.0",
        "generated_at": _dt.datetime.now(_dt.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "data_mode": DATA_MODE,
        "data_mode_reason": str(globals().get("DATA_MODE_REASON") or "") or None,
        "study_area": {"name": "Delhi NCR", "area_km2": round(float(ROI_MASK.sum()) * (GRID.res / 1000.0) ** 2, 1),
                       "crs": CFG.CRS, "grid_res_m": int(CFG.GRID_RES_M),
                       "bbox": [round(v, 5) for v in _BBOX], "center": [round(center_lon, 5), round(center_lat, 5)],
                       "boundary_approximate": bool(globals().get("BOUNDARY_APPROXIMATE", DATA_MODE == "synthetic")),
                       "boundary_source": "; ".join(roi_sources) or None},
        "epochs": list(_EXP_EPOCHS),
        "season": list(CFG.SEASON),
        "target_mode": CFG.TARGET_MODE,
        "epoch_means": {str(y): round(float(EPOCH_MEANS[y]), 2) for y in _EXP_EPOCHS},
        "shap_base": round(float(SHAP_BASE), 4),
        "features": features,
        "zones": [{"id": p["id"], "name": p["name"], "color": p["color"]} for p in ZONE_PROFILES],
        "districts": [{"id": int(d["id"]), "name": d["name"], "state": d["state"]} for d in DISTRICTS],
        "files": {"epochs": {str(y): EPOCH_FILE.format(year=y) for y in _EXP_EPOCHS}, "model": "model_web.json",
                  "shap_global": "shap_global.json", "dependence": DEPENDENCE_FILE, "zones": "zones.json",
                  "metrics": "metrics.json", "districts": "districts.geojson", "interactions": "interactions.json",
                  "coupling": "scenario_coupling.json"},
        "data_sources": sources,
        "synthetic_truth": SYNTHETIC_TRUTH,
        "bootstrap": {k: BOOTSTRAP_INFO[k] for k in ("n", "n_requested", "mode", "block_size_km", "n_blocks",
                                                     "block_size_source", "ci_level", "ci_method",
                                                     "ci_method_description", "ci_label", "replicate_range",
                                                     "full_data") if k in BOOTSTRAP_INFO},
        "notes": notes,
    }


def _epoch_payload(year):
    """Columnar epoch file; verifies the waterfall invariant on the rounded values before returning."""
    idx = np.flatnonzero(DF["year"].to_numpy() == year)
    sub = DF.iloc[idx]
    epoch_mean = round(float(EPOCH_MEANS[year]), 2)
    base_value_c = round(float(SHAP_BASE) + _TARGET_UNIT_OFFSET[year], 2)
    lst_pred = _round_dp(sub["lst_pred"], 2)
    shap = {f: _round_dp(SHAP_VALUES[idx, j], 3) for j, f in enumerate(FEATURES)}
    recon = base_value_c + np.sum(np.vstack(list(shap.values())), axis=0)
    err = float(np.max(np.abs(recon - lst_pred))) if len(idx) else 0.0
    if not err <= 0.05:
        raise RuntimeError(f"epoch {year}: waterfall invariant violated after rounding (max err {err:.3f} °C)")
    return {
        "year": int(year), "n": int(len(idx)), "cell_size_m": int(CFG.GRID_RES_M),
        "epoch_mean": epoch_mean, "base_value_c": base_value_c,
        "cell_id": sub["cell_id"].to_numpy().astype(np.int64),
        "lon": _round_dp(sub["lon"], 5), "lat": _round_dp(sub["lat"], 5),
        "district": sub["district"].to_numpy().astype(np.int64),
        "lst_obs": _round_dp(sub["lst"], 2), "lst_pred": lst_pred,
        "lst_pred_oof": _round_dp(sub["lst_pred_oof"], 2), "resid_oof": _round_dp(sub["resid_oof"], 2),
        "zone": sub["zone"].to_numpy().astype(np.int64),
        "features": {f: _round_sig(sub[f]) for f in FEATURES},
        "shap": shap,
    }, err


def _shap_global_payload():
    r3 = lambda v: _round_dp(v, 3)  # noqa: E731
    imp = SHAP_IMPORTANCE
    return {
        "features": list(FEATURES), "n_bootstrap": int(BOOTSTRAP_INFO["n"]), "bootstrap_mode": BOOTSTRAP_INFO["mode"],
        "global": {k: r3(imp["global"][k]) for k in ("mean_abs", "ci_lo", "ci_hi", "bias") if k in imp["global"]},
        "ci_label": BOOTSTRAP_INFO["ci_label"], "ci_method": BOOTSTRAP_INFO.get("ci_method"),
        "by_epoch": {str(y): {k: r3(v[k]) for k in ("mean_abs", "mean")} for y, v in imp["by_epoch"].items()},
        "by_zone": {str(z): {k: r3(v[k]) for k in ("mean_abs", "mean")} for z, v in imp["by_zone"].items()},
    }


def _threshold_payload(th):
    s4 = lambda v: _sig(v, 4)  # noqa: E731
    ci = lambda c: None if c is None else [s4(c[0]), s4(c[1])]  # noqa: E731
    sup = lambda v: None if v is None else round(float(v), 3)  # noqa: E731
    out = {}
    for k in ("zero_crossing", "breakpoint", "saturation"):
        point = s4(th[k])
        out[k] = point
        # an interval is only meaningful around an existing point estimate (SPEC 4.4)
        out[f"{k}_ci"] = None if point is None else ci(th[f"{k}_ci"])
        out[f"{k}_support"] = None if point is None else sup(th.get(f"{k}_support"))
    return {
        "zero_crossing": out["zero_crossing"], "zero_crossing_ci": out["zero_crossing_ci"],
        "zero_crossing_support": out["zero_crossing_support"],
        "zero_crossing_flag": "multiple" if th.get("zero_crossing_flag") == "multiple" else None,
        "breakpoint": out["breakpoint"], "breakpoint_ci": out["breakpoint_ci"],
        "breakpoint_support": out["breakpoint_support"],
        "slope_before": s4(th["slope_before"]), "slope_after": s4(th["slope_after"]),
        "saturation": out["saturation"], "saturation_ci": out["saturation_ci"],
        "saturation_support": out["saturation_support"],
        "effect_range": None if th["effect_range"] is None else round(float(th["effect_range"]), 3),
        "direction": th["direction"],
    }


def _dependence_payload():
    out = {}
    for f in FEATURES:
        d = DEPENDENCE[f]
        sc = d["scatter"]
        out[f] = {
            "bin_centers": _round_sig(d["bin_centers"]), "mean": _round_dp(d["mean"], 3),
            "ci_lo": _round_dp(d["ci_lo"], 3), "ci_hi": _round_dp(d["ci_hi"], 3),
            "count": np.asarray(d["count"], dtype=np.int64),
            "scatter": {"x": _round_sig(sc["x"]), "shap": _round_dp(sc["shap"], 3), "color": _round_sig(sc["color"]),
                        "color_feature": sc["color_feature"], "zone": np.asarray(sc["zone"], dtype=np.int64),
                        "year": np.asarray(sc["year"], dtype=np.int64)},
            "threshold": _threshold_payload(d["threshold"]),
        }
    return {"features": out}


def _zones_payload():
    zones = []
    for p in ZONE_PROFILES:
        z = dict(p)
        for key in ("area_km2", "lst_obs_mean", "lst_pred_mean"):
            z[key] = _round_nested(p[key], _dp(2))
        z["feature_means"] = {f: _sig(v) for f, v in p["feature_means"].items()}
        z["shap_means"] = _round_nested(p["shap_means"], _dp(3))
        z["top_warming"] = _round_nested(p["top_warming"], _dp(3))
        z["top_cooling"] = _round_nested(p["top_cooling"], _dp(3))
        z["top_interactions"] = _round_nested(p["top_interactions"], _dp(4))
        z["top_districts"] = _round_nested(p["top_districts"], _dp(4))
        z["recommendations"] = [
            {**r, "current": _sig(r["current"]), "target": _sig(r["target"]), "step": _sig(r.get("step")),
             "expected_delta_c": round(float(r["expected_delta_c"]), 2),
             "ci": None if r["ci"] is None else [round(float(r["ci"][0]), 2), round(float(r["ci"][1]), 2)],
             "bias": None if r.get("bias") is None else round(float(r["bias"]), 3)}
            for r in p["recommendations"]]
        zones.append(z)
    labelling = globals().get("ZONE_LABELLING")
    if labelling:
        labelling = {"rule": labelling["rule"],
                     "clusters": [{k: (round(v, 4) if isinstance(v, float) else v) for k, v in c.items()}
                                  for c in labelling["clusters"]]}
    return {"k": len(ZONE_PROFILES), "silhouette": round(float(ZONE_SILHOUETTE), 4),
            "diagnostics": {"k": [int(k) for k in ZONE_DIAGNOSTICS["k"]],
                            "inertia": _round_sig(ZONE_DIAGNOSTICS["inertia"], 6),
                            "silhouette": _round_dp(ZONE_DIAGNOSTICS["silhouette"], 4),
                            "n_init": int(ZONE_DIAGNOSTICS.get("n_init", 10))},
            "labelling": labelling,
            "zones": zones, "transitions": ZONE_TRANSITIONS}


def _records(df, dp=4):
    return _round_nested(_clean(df.to_dict(orient="records")), _dp(dp))


def _metrics_payload():
    scatter = {m: {"year": int(v["year"]), "z": _round_dp(v["z"], 3), "lag": _round_dp(v["lag"], 3),
                   "slope": round(float(v["slope"]), 4)} for m, v in MORAN_SCATTER.items()}
    return {
        "schemes": list(SCHEMES), "models": list(MODEL_NAMES),
        "summary": _records(CV_SUMMARY), "folds": _records(CV_FOLD_METRICS),
        "moran": _records(MORAN_RESULTS), "moran_target": _records(MORAN_TARGET),
        "moran_scatter": scatter,
        "hardware": {"gpus": list(ENV.get("gpu_names") or []), "xgb_device": ENV.get("xgb_device"),
                     "lgbm_device": ENV.get("lgbm_device"),
                     "rapids": bool(ENV.get("rapids_used")),
                     "rapids_used": sorted(ENV.get("rapids_used") or []),
                     "n_gpus": int(ENV.get("n_gpus") or 0), "xgb_version": ENV.get("xgb_version"),
                     "python": ENV.get("python"), "platform": ENV.get("platform")},
        "timings": {str(k): round(float(v), 2) for k, v in TIMINGS.items()},
        "bootstrap": {k: v for k, v in BOOTSTRAP_INFO.items() if k != "failures"},
    }


def _interactions_payload():
    r4 = lambda m: _round_dp(m, 4).tolist()  # noqa: E731
    return {"features": list(FEATURES), "global": r4(INTERACTION_GLOBAL),
            "by_zone": {str(z): r4(m) for z, m in INTERACTION_BY_ZONE.items()}}


def _coupling_payload():
    c = SCENARIO_COUPLING
    return {"description": c["description"], "fraction_features": list(c["fraction_features"]),
            "coupling": _round_nested(c["coupling"], lambda v: _sig(v, 4)),
            "r2": _round_nested(c["r2"], _dp(4)), "intercept": _round_nested(c["intercept"], lambda v: _sig(v, 4)),
            "n": c["n"]}


def _clear_web_dir(keep):
    """Remove bundle files from a previous export that this export does not write (e.g. plain epoch_*.json next
    to the new .json.gz), so the web folder never carries stale data."""
    for p in _WEB_DIR.iterdir():
        if p.is_file() and p.name not in keep and p.name.endswith((".json", ".json.gz", ".geojson", ".tmp")):
            p.unlink()
            log(f"  removed stale bundle file {p.name}")


def _write_web_bundle():
    sizes = {}
    EXPORT_MANIFEST = _manifest()
    files = EXPORT_MANIFEST["files"]
    _clear_web_dir({"manifest.json", *files["epochs"].values(), *(v for k, v in files.items() if k != "epochs")})
    sizes["manifest.json"] = _write_json(EXPORT_MANIFEST, _WEB_DIR / "manifest.json")
    for year in _EXP_EPOCHS:
        payload, err = _epoch_payload(year)
        name = files["epochs"][str(year)]
        sizes[name] = _write_json(payload, _WEB_DIR / name)
        log(f"  {name}: n={payload['n']:,}, waterfall max err {err:.3f} °C")
        del payload
    sizes["model_web.json"] = _write_json(_MODEL_WEB, _WEB_DIR / "model_web.json")
    sizes["shap_global.json"] = _write_json(_shap_global_payload(), _WEB_DIR / "shap_global.json")
    sizes[files["dependence"]] = _write_json(_dependence_payload(), _WEB_DIR / files["dependence"])
    sizes["zones.json"] = _write_json(_zones_payload(), _WEB_DIR / "zones.json")
    sizes["interactions.json"] = _write_json(_interactions_payload(), _WEB_DIR / "interactions.json")
    sizes["scenario_coupling.json"] = _write_json(_coupling_payload(), _WEB_DIR / "scenario_coupling.json")
    sizes["districts.geojson"] = _write_json(_DISTRICTS_FC, _WEB_DIR / "districts.geojson")
    sizes["metrics.json"] = _write_json(_metrics_payload(), _WEB_DIR / "metrics.json")  # last: includes timings
    return EXPORT_MANIFEST, sizes


with timer("05_web_bundle"):
    EXPORT_MANIFEST, _WEB_SIZES = _write_web_bundle()
for _name, _size in _WEB_SIZES.items():
    log(f"  web/{_name}: {_size / 1e6:.2f} MB")
free_memory()

## 5.5 Contract validation of the written bundle
Every file is re-read from disk (rejecting NaN/Infinity tokens) and checked against SPEC §4: keys, array
lengths, feature order, zone ids, the SHAP waterfall invariant for every cell, and the compact model's check rows.
A stricter standalone checker (numpy only; e.g. for a bundle downloaded from Kaggle) lives in the repository:
`python notebook/validate_bundle.py outputs/web` (also checks canonical zones/districts, rounding, tree depth,
float32 thresholds, recommendation priorities, transition totals and the bundle size).

In [ ]:
def _reject_constant(token):
    raise ValueError(f"non-JSON constant {token!r}")


def _load_strict(path):
    """Parse UTF-8 JSON (gzip detected by magic bytes), rejecting NaN / Infinity tokens."""
    return json.loads(_read_bytes_maybe_gzip(path).decode("utf-8"), parse_constant=_reject_constant)


def _require(cond, errors, msg):
    if not cond:
        errors.append(msg)
    return bool(cond)


def _missing_keys(obj, keys):
    return [k for k in keys if not isinstance(obj, dict) or k not in obj]


_ZONE_IDS = {0, 1, 2, 3}
_THRESHOLD_KEYS = ["zero_crossing", "zero_crossing_ci", "breakpoint", "breakpoint_ci", "slope_before",
                   "slope_after", "saturation", "saturation_ci", "effect_range", "direction"]
_REC_LEVER_NAMES = {"frac_impervious", "frac_forest", "frac_water", "frac_cropland", "frac_barren", "ndvi",
                    "lm_pd", "lm_ed", "lm_contag"}


def _check_manifest(man, errors):
    keys = ["schema_version", "generated_at", "data_mode", "study_area", "epochs", "season", "target_mode",
            "epoch_means", "features", "zones", "districts", "files", "data_sources", "synthetic_truth", "notes"]
    miss = _missing_keys(man, keys)
    if not _require(not miss, errors, f"manifest: missing keys {miss}"):
        return None
    _require(man["data_mode"] in ("gee", "synthetic"), errors, "manifest: bad data_mode")
    _require(not _missing_keys(man["study_area"], ["name", "area_km2", "crs", "grid_res_m", "bbox", "center"]),
             errors, "manifest.study_area: missing keys")
    _require(len(man["study_area"].get("bbox", [])) == 4, errors, "manifest.study_area.bbox must have 4 values")
    names = [f.get("name") for f in man["features"]]
    for f in man["features"]:
        _require(not _missing_keys(f, ["name", "label", "unit", "group", "description", "source", "actionable",
                                       "display", "stats"]), errors, f"manifest feature {f.get('name')}: keys")
        _require(f.get("display") in ("index", "percent", "number"), errors, f"feature {f.get('name')}: display")
    _require([z["id"] for z in man["zones"]] == [0, 1, 2, 3], errors, "manifest.zones must be ids 0..3")
    _require(set(man["files"]["epochs"]) == {str(y) for y in man["epochs"]}, errors, "manifest.files.epochs")
    _require(set(man["epoch_means"]) == {str(y) for y in man["epochs"]}, errors, "manifest.epoch_means")
    return names


def _check_epoch(ep, year, names, target_mode, epoch_mean, errors):
    tag = f"epoch_{year}"
    keys = ["year", "n", "cell_size_m", "epoch_mean", "base_value_c", "cell_id", "lon", "lat", "district",
            "lst_obs", "lst_pred", "lst_pred_oof", "resid_oof", "zone", "features", "shap"]
    miss = _missing_keys(ep, keys)
    if not _require(not miss, errors, f"{tag}: missing keys {miss}"):
        return
    n = ep["n"]
    _require(ep["year"] == year, errors, f"{tag}: year mismatch")
    for k in ("cell_id", "lon", "lat", "district", "lst_obs", "lst_pred", "lst_pred_oof", "resid_oof", "zone"):
        _require(len(ep[k]) == n, errors, f"{tag}.{k}: length {len(ep[k])} != n {n}")
    for grp in ("features", "shap"):
        _require(list(ep[grp]) == names, errors, f"{tag}.{grp}: keys differ from manifest feature order")
        for f, arr in ep[grp].items():
            _require(len(arr) == n, errors, f"{tag}.{grp}.{f}: length != n")
    _require(set(ep["zone"]) <= _ZONE_IDS, errors, f"{tag}: zone ids outside 0..3")
    _require(None not in ep["lst_pred"], errors, f"{tag}: lst_pred contains null")
    offset = epoch_mean if target_mode == "anomaly" else 0.0
    _require(abs(ep["epoch_mean"] - epoch_mean) < 0.011, errors, f"{tag}: epoch_mean differs from manifest")
    if n and list(ep["shap"]) == names and None not in ep["lst_pred"]:
        shap = np.array([ep["shap"][f] for f in names], dtype=np.float64)
        _require(np.isfinite(shap).all(), errors, f"{tag}: shap contains null")
        err = np.max(np.abs(ep["base_value_c"] + np.nansum(shap, axis=0) - np.array(ep["lst_pred"])))
        _require(err <= 0.05, errors, f"{tag}: waterfall invariant violated (max err {err:.3f})")
        _require(abs(ep["base_value_c"] - (float(SHAP_BASE) + offset)) < 0.011, errors,
                 f"{tag}: base_value_c != SHAP_BASE + offset")


def _check_model(model, names, errors):
    keys = ["format", "objective", "target_mode", "features", "base_score", "n_trees", "trees", "fidelity", "check"]
    miss = _missing_keys(model, keys)
    if not _require(not miss, errors, f"model_web: missing keys {miss}"):
        return
    _require(model["format"] == "lst-xgb-compact-v1", errors, "model_web: bad format")
    _require(model["features"] == names, errors, "model_web: feature order differs from manifest")
    _require(model["n_trees"] == len(model["trees"]), errors, "model_web: n_trees != len(trees)")
    _require(model["fidelity"].get("max_depth", 99) <= 6 and model["fidelity"].get("rounds", 999) <= 400, errors,
             "model_web: surrogate exceeds depth 6 / 400 rounds")
    for i, t in enumerate(model["trees"]):
        lens = {k: len(t.get(k, [])) for k in ("f", "t", "l", "r", "d", "v")}
        if not _require(len(set(lens.values())) == 1, errors, f"model_web tree {i}: array lengths {lens}"):
            return
        m = lens["f"]
        kids = [c for c in t["l"] + t["r"] if c != -1]
        _require(all(0 < c < m for c in kids), errors, f"model_web tree {i}: child index out of range")
        _require(all(0 <= f < len(names) for f, lc in zip(t["f"], t["l"]) if lc != -1), errors,
                 f"model_web tree {i}: split feature out of range")
    rows = model["check"]["rows"]
    _require(len(rows) == 50 and all(len(r) == len(names) for r in rows), errors,
             "model_web.check: need 50 rows of K values")
    _require(len(model["check"]["expected"]) == len(rows), errors, "model_web.check: expected length")
    if rows and not errors:
        pred = _predict_compact(model, np.array([[np.nan if v is None else v for v in r] for r in rows]))
        diff = float(np.max(np.abs(pred - np.array(model["check"]["expected"]))))
        _require(diff < 1e-3, errors, f"model_web.check: evaluator differs from expected by {diff:.2e}")


def _check_shap_global(sg, names, epochs, errors):
    miss = _missing_keys(sg, ["features", "n_bootstrap", "bootstrap_mode", "global", "by_epoch", "by_zone"])
    if not _require(not miss, errors, f"shap_global: missing keys {miss}"):
        return
    k = len(names)
    _require(sg["features"] == names, errors, "shap_global: feature order")
    _require(all(len(sg["global"].get(key, [])) == k for key in ("mean_abs", "ci_lo", "ci_hi")), errors,
             "shap_global.global: arrays must have K values")
    _require(set(sg["by_epoch"]) == {str(y) for y in epochs}, errors, "shap_global.by_epoch keys")
    _require(set(sg["by_zone"]) <= {str(z) for z in _ZONE_IDS}, errors, "shap_global.by_zone keys")
    for grp in ("by_epoch", "by_zone"):
        for key, v in sg[grp].items():
            _require(len(v.get("mean_abs", [])) == k and len(v.get("mean", [])) == k, errors,
                     f"shap_global.{grp}.{key}: arrays must have K values")


def _check_dependence(dep, names, errors):
    feats = dep.get("features", {}) if isinstance(dep, dict) else {}
    _require(list(feats) == names, errors, "dependence: features differ from manifest order")
    for f, d in feats.items():
        miss = _missing_keys(d, ["bin_centers", "mean", "ci_lo", "ci_hi", "count", "scatter", "threshold"])
        if not _require(not miss, errors, f"dependence.{f}: missing {miss}"):
            continue
        nb = len(d["bin_centers"])
        _require(all(len(d[k]) == nb for k in ("mean", "ci_lo", "ci_hi", "count")), errors,
                 f"dependence.{f}: bin array lengths differ")
        sc = d["scatter"]
        ns = len(sc.get("x", []))
        _require(ns <= 1500 and all(len(sc.get(k, [])) == ns for k in ("shap", "color", "zone", "year")), errors,
                 f"dependence.{f}.scatter: lengths")
        _require(sc.get("color_feature") in names, errors, f"dependence.{f}.scatter.color_feature")
        _require(not _missing_keys(d["threshold"], _THRESHOLD_KEYS), errors, f"dependence.{f}.threshold keys")
        _require(d["threshold"].get("direction") in ("cooling", "warming", "mixed"), errors,
                 f"dependence.{f}.threshold.direction")
        for k in ("zero_crossing", "breakpoint", "saturation"):
            _require(d["threshold"].get(k) is not None or d["threshold"].get(f"{k}_ci") is None, errors,
                     f"dependence.{f}.threshold.{k}_ci given without a point estimate")
        _require(d["threshold"].get("zero_crossing_flag") in (None, "multiple"), errors,
                 f"dependence.{f}.threshold.zero_crossing_flag must be 'multiple' or null")


def _check_zones(zj, names, epochs, errors):
    miss = _missing_keys(zj, ["k", "silhouette", "diagnostics", "zones", "transitions"])
    if not _require(not miss, errors, f"zones: missing keys {miss}"):
        return
    _require(zj["k"] == 4 and [z.get("id") for z in zj["zones"]] == [0, 1, 2, 3], errors, "zones: need ids 0..3")
    dg = zj["diagnostics"]
    _require(len(dg["k"]) == len(dg["inertia"]) == len(dg["silhouette"]), errors, "zones.diagnostics lengths")
    zkeys = ["id", "name", "color", "description", "n_cells", "area_km2", "lst_obs_mean", "lst_pred_mean",
             "feature_means", "shap_means", "top_warming", "top_cooling", "top_interactions", "top_districts",
             "recommendations"]
    for z in zj["zones"]:
        zmiss = _missing_keys(z, zkeys)
        if not _require(not zmiss, errors, f"zones[{z.get('id')}]: missing {zmiss}"):
            continue
        _require(set(z["n_cells"]) == {str(y) for y in epochs}, errors, f"zones[{z['id']}].n_cells epochs")
        _require(list(z["feature_means"]) == names and list(z["shap_means"]) == names, errors,
                 f"zones[{z['id']}]: feature order")
        for r in z["recommendations"]:
            _require(r.get("priority") in ("high", "medium", "low") and r.get("expected_delta_c", 0) < 0, errors,
                     f"zones[{z['id']}]: invalid recommendation {r.get('feature')}")
            _require(r.get("feature") in _REC_LEVER_NAMES, errors,
                     f"zones[{z['id']}]: {r.get('feature')} is not an intervention lever (SPEC 4.6)")
            _require("associated with" in str(r.get("rationale", "")), errors,
                     f"zones[{z['id']}] recommendation {r.get('feature')}: rationale must say 'associated with'")
            if r.get("ci") is not None:
                _require(r["ci"][0] - 0.011 <= r["expected_delta_c"] <= r["ci"][1] + 0.011, errors,
                         f"zones[{z['id']}] recommendation {r.get('feature')}: estimate outside its interval")
    for key, mat in zj["transitions"].items():
        _require(len(mat) == 4 and all(len(row) == 4 for row in mat), errors, f"zones.transitions[{key}]: 4x4")


def _check_small_files(web_dir, names, errors):
    metrics = _load_strict(web_dir / "metrics.json")
    miss = _missing_keys(metrics, ["schemes", "models", "summary", "folds", "moran", "moran_target",
                                   "moran_scatter", "hardware", "timings"])
    _require(not miss, errors, f"metrics: missing keys {miss}")
    for rec in metrics.get("summary", []):
        _require(not _missing_keys(rec, ["scheme", "model", "r2_mean", "r2_std", "rmse_mean", "rmse_std",
                                         "mae_mean", "mae_std", "n_folds"]), errors, "metrics.summary record keys")
    inter = _load_strict(web_dir / "interactions.json")
    k = len(names)
    _require(inter.get("features") == names, errors, "interactions: feature order")
    mats = [inter.get("global", [])] + list(inter.get("by_zone", {}).values())
    _require(all(len(m) == k and all(len(r) == k for r in m) for m in mats), errors, "interactions: K x K")
    cp = _load_strict(web_dir / "scenario_coupling.json")
    fr = ["frac_impervious", "frac_forest", "frac_water", "frac_cropland", "frac_barren"]
    _require(cp.get("fraction_features") == fr, errors, "scenario_coupling: fraction_features")
    _require(all(set(cp.get("coupling", {}).get(f, {})) >= {"ndvi", "ndwi", "ndbi"} for f in fr), errors,
             "scenario_coupling: coupling must give ndvi/ndwi/ndbi slopes per fraction")
    geo = _load_strict(web_dir / "districts.geojson")
    _require(geo.get("type") == "FeatureCollection", errors, "districts.geojson: not a FeatureCollection")
    _require(all(not _missing_keys(ft.get("properties", {}), ["id", "name", "state"])
                 for ft in geo.get("features", [])), errors, "districts.geojson: properties id/name/state")


def validate_web_bundle(web_dir):
    """Re-read the web bundle and check it against the SPEC §4 contract. Raises ValueError listing all problems."""
    web_dir = Path(web_dir)
    errors = []
    man = _load_strict(web_dir / "manifest.json")
    names = _check_manifest(man, errors)
    if names is None:
        raise ValueError("Web bundle invalid:\n  - " + "\n  - ".join(errors))
    for fname in list(man["files"]["epochs"].values()) + [v for k, v in man["files"].items() if k != "epochs"]:
        _require((web_dir / fname).is_file(), errors, f"missing file {fname}")
    if errors:
        raise ValueError("Web bundle invalid:\n  - " + "\n  - ".join(errors))
    for y in man["epochs"]:
        ep = _load_strict(web_dir / man["files"]["epochs"][str(y)])
        _check_epoch(ep, int(y), names, man["target_mode"], man["epoch_means"][str(y)], errors)
        del ep
    _check_model(_load_strict(web_dir / man["files"]["model"]), names, errors)
    _check_shap_global(_load_strict(web_dir / man["files"]["shap_global"]), names, man["epochs"], errors)
    _check_dependence(_load_strict(web_dir / man["files"]["dependence"]), names, errors)
    _check_zones(_load_strict(web_dir / man["files"]["zones"]), names, man["epochs"], errors)
    _check_small_files(web_dir, names, errors)
    if errors:
        raise ValueError(f"Web bundle invalid ({len(errors)} problems):\n  - " + "\n  - ".join(errors))
    log(f"Web bundle valid: {len(man['epochs'])} epochs, {len(names)} features, "
        f"{sum(p.stat().st_size for p in web_dir.iterdir() if p.is_file()) / 1e6:.1f} MB on disk")
    return True


with timer("05_validate"):
    validate_web_bundle(_WEB_DIR)

## 5.6 Bundle zip, output inventory and run summary

In [ ]:
def _zip_web(web_dir, zip_path):
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
        for p in sorted(Path(web_dir).iterdir()):
            if p.is_file():
                # already-gzipped files are stored as they are (deflating them again gains nothing)
                zf.write(p, arcname=p.name,
                         compress_type=zipfile.ZIP_STORED if p.name.endswith(".gz") else zipfile.ZIP_DEFLATED)
    return zip_path


def _print_tree(root, max_files=25):
    """Print the outputs tree with sizes; directories with many files (e.g. caches) are summarised."""
    root = Path(root)
    print(f"{root}/")
    for d in sorted([root] + [p for p in root.rglob("*") if p.is_dir()]):
        files = sorted(p for p in d.iterdir() if p.is_file())
        depth = len(d.relative_to(root).parts)
        if d != root:
            print(f"{'  ' * depth}{d.name}/")
        if len(files) > max_files:
            total = sum(p.stat().st_size for p in files)
            print(f"{'  ' * (depth + 1)}[{len(files)} files, {total / 1e6:.1f} MB]")
            continue
        for p in files:
            print(f"{'  ' * (depth + 1)}{p.name:<40s} {p.stat().st_size / 1e6:9.2f} MB")


_ZIP_PATH = _zip_web(_WEB_DIR, _OUT / "web_bundle.zip")
log(f"Web bundle zipped: {_ZIP_PATH} ({_ZIP_PATH.stat().st_size / 1e6:.1f} MB)")
_print_tree(_OUT)

_xgb_spatial = CV_SUMMARY[(CV_SUMMARY["scheme"] == "spatial") & (CV_SUMMARY["model"] == "xgboost")]
print("\n" + "=" * 78)
print(f"RUN SUMMARY  (data mode: {DATA_MODE.upper()}{' - SYNTHETIC DEMO DATA' if DATA_MODE == 'synthetic' else ''})")
print("=" * 78)
print(f"Cells x epochs: {len(DF):,}  |  epochs {_EXP_EPOCHS}  |  features {len(FEATURES)}  |  target {CFG.TARGET_MODE}")
if len(_xgb_spatial):
    _row = _xgb_spatial.iloc[0]
    print(f"XGBoost spatial-CV: R2 {_row['r2_mean']:.3f} ± {_row['r2_std']:.3f}, RMSE {_row['rmse_mean']:.3f} °C")
print(f"Final model: {FINAL_ROUNDS} rounds; SHAP base {SHAP_BASE:.3f}; bootstrap {BOOTSTRAP_INFO['n']} replicates")
print(f"Web surrogate: {_MODEL_WEB['n_trees']} trees, R2 vs final {_MODEL_WEB['fidelity']['r2_vs_final']:.4f}")
print(f"Zones (silhouette {ZONE_SILHOUETTE:.3f}), {_EXP_LAST}:")
for _zp in ZONE_PROFILES:
    print(f"  {_zp['id']} {_zp['name']:<22s} {_zp['area_km2'][str(_EXP_LAST)]:>9,.0f} km²  "
          f"LST {(_zp['lst_obs_mean'][str(_EXP_LAST)] or float('nan')):.2f} °C  "
          f"{len(_zp['recommendations'])} recommendation(s)")
print(f"\nNEXT STEP: copy {_WEB_DIR}/* to dashboard/public/data/  (or unzip {_ZIP_PATH.name} there), "
      "then run `npm run dev` in dashboard/.")
free_memory()